# CCMP gate decomposition: which gates move a route's weight?

The path interpretations show route weights changing between the CCMP gate on and off even when the gate
on every drawn hop is 1.00 (e.g. the protein-RNA / optimal-transport example on SIR-4 Biology: 9.28 → 12.44
with g = 0.9999 and 1.0015). A route's weight is the mean over its hops of ∂score/∂(edge weight) at the layer
the hop is attributed to, so the gate on the drawn hop enters only as a factor of 1; the change must come from
gates elsewhere: on the same senders at later layers (the frontier-mean responsibility falls from ≈0.48 at layer
0 to ≈0.17 at layer 5, so a sender at 0.42 is amplified ≈1.7× there), or on competing senders.

This notebook measures that directly on the same trained weights. For every interpreted (query, gold) it follows
the top routes found under the full gate and re-evaluates the **same routes** (directly along their edges) and the
gold's graph / fused rank under six inference-time gate masks:

| condition | gate applied to |
|---|---|
| gate_on | everything (the model as run) |
| gate_off | nothing (the "no CCMP" arm) |
| gate_layers_attributed | only the layers the route is attributed to (0 .. L−1) |
| gate_layers_later | only the later layers (L .. 5) |
| gate_route_senders_only | only the route's own sender nodes |
| gate_all_but_route_senders | every node except the route's senders |

Engine: `models.py` gains `_gate_layers` / `_gate_nodes` masks on the CCMP gate (interpretation only);
`fusion_trainer.py` gains `gate_decomposition()`; `interpret_paths.py` gains `+interp.mode=gate_decomp`. All
three are in the repo and in the embedded overlay. Outputs per dataset on Drive: `outputs/scan/<dataset>/
gate_decomp_<dataset>.json`, `fig_gate_decomp_<dataset>.{pdf,png,md}`. Needs `hops_frame_ccmp.json` from the
showcase run (the routes to follow) and the CCMP checkpoint. Generated by `prep/build_gate_decomp_notebook.py`
on 2026-09-09 12:43.


## 1. Datasets, environment, checkpoint spec, resume plan

In [ ]:
# 1. Datasets to run, environment, Drive, paths, the checkpoint spec. Edit DATASETS to run a subset.
DATASETS = ["sir4_biology", "sir4_cs"]     # CCMP gate decomposition on these (any dataset with a CCMP arm works)
GD_ARM   = "frame_ccmp"                    # the arm whose gate is decomposed
GD_MAX_GOLDS = 120                          # golds per dataset (cross-field first, deepest under cosine first)
GD_EXAMPLES = {"sir4_biology": ["10.1007/s11263-023-01831-9"]}   # golds always included (the OT example)
import os, sys, re, json, glob, shutil, time, zipfile, subprocess, threading, random
# Colab runs Python 3.13 and the engine's editable install refuses it, so subprocesses must be told
# where gfmrag is and which interpreter to use. Must precede every sh() snapshot of os.environ.
os.environ["PATH"] = os.path.dirname(sys.executable) + os.pathsep + os.environ.get("PATH", "")
os.environ["PYTHONPATH"] = "/content/gfm-rag" + os.pathsep + os.environ.get("PYTHONPATH", "")
from google.colab import drive
drive.mount('/content/drive')
DRIVE      = "/content/drive/MyDrive/cargo-gfmrag"
CARGO_ROOT = "/content/cargo"
DATA_ROOT  = f"{CARGO_ROOT}/kg-construction/data"
S4         = f"{CARGO_ROOT}/sir4-retrieval"
KGDIR      = f"{CARGO_ROOT}/kg-construction"
RUNS       = "/content/runs"
OP_MODEL   = "/content/qwen3"
OP_SLUG    = "_content-qwen3"
TRAIN = TEST = "mir_test_v16sc"        # config DEFAULT names only; every command overrides them on the CLI
os.environ["CARGO_ROOT"] = CARGO_ROOT
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    os.environ.setdefault("HF_TOKEN", "")
os.makedirs(RUNS, exist_ok=True)

def sh(cmd, cwd, extra=None, log=None, check=True):
    """Run a command with LIVE output (raw chunks, so tqdm bars show)."""
    t0 = time.time()
    p = subprocess.Popen(cmd, cwd=cwd, env=dict(env, **(extra or {})), shell=isinstance(cmd, str),
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    fh = open(log, "w") if log else None
    while True:
        c = os.read(p.stdout.fileno(), 8192)
        if not c: break
        s = c.decode("utf-8", "replace")
        sys.stdout.write(s); sys.stdout.flush()
        if fh: fh.write(s); fh.flush()
    p.wait()
    if fh: fh.close()
    print(f"\n[{time.time()-t0:.0f}s, exit {p.returncode}]")
    if check:
        assert p.returncode == 0, f"FAILED (exit {p.returncode}): {cmd if isinstance(cmd, str) else ' '.join(map(str, cmd))}"
    return p.returncode
env = dict(os.environ, CARGO_ROOT=CARGO_ROOT, PYTHONUNBUFFERED="1")

# --- per-dataset spec: source notebook to replay, test graphs, scorer files, checkpoints (Drive-relative) ---
def _sir4(field):
    pair = "physics+biology" if field in ("physics", "biology") else "cs+matsci"
    warm = "biology" if field in ("physics", "biology") else "cs"
    d = f"sir4_{field}"
    return dict(nb=f"colab_qualitative_{field}", frame=f"{d}_test_v16sc", openie=f"{d}_test",
                sem={"field": (f"outputs/{d}/semantic", d),                                  # the field's own scorer
                     "frame": (f"outputs/sir4_zeroshot/semantic_sir4_{warm}", f"sir4_{warm}")},  # the CCMP run's warm start
                arms=[("frame_ccmp",     f"outputs/sir4_zeroshot/scigraphir_{pair}_qwenmlp_ccmp_e10_b2", "frame",  "frame", True),
                      ("frame_ccmp_off", f"outputs/sir4_zeroshot/scigraphir_{pair}_qwenmlp_ccmp_e10_b2", "frame",  "frame", False),
                      ("frame_nocc",     [f"outputs/routing/{d}/{d}_route_control_e10_b2_s1024",
                                          f"outputs/{d}/{d}_fusion_qwenmlp_epoch10_b2"],                     "frame",  "field", None),
                      ("openie",         f"outputs/sir4_openie/{d}_openie_qwenmlp_graph_e10_b2",          "openie", "field", None)])
SPEC = {
    "mir": dict(nb="colab_mir_all", frame="mir_test_v16sc", openie="mir_test",
                extra={"hyb": "mir_test_hyb"},             # merged graph (prep/build_hybrid_graph.py; mir_hyb_bundle.zip on Drive)
                sem={"field": ("outputs/mir/semantic", "mir")},
                arms=[("frame_ccmp",     "outputs/mir/mir_qwenmlp_ccmp_e10_b2",         "frame",  "field", True),
                      ("frame_ccmp_off", "outputs/mir/mir_qwenmlp_ccmp_e10_b2",         "frame",  "field", False),
                      ("frame_nocc",     "outputs/mir/mir_qwenmlp_graph_e10_b2",        "frame",  "field", None),
                      ("openie",         "outputs/mir/mir_openie_qwenmlp_graph_e10_b2", "openie", "field", None),
                      ("hyb_nocc",       "outputs/routing_hyb/mir/mir_hyb_route_control_e10_b2_s1024", "hyb", "field", None),
                      ("hyb_ccmp",       "outputs/routing_hyb/mir/mir_hyb_route_ccmp_e10_b2_s1024",    "hyb", "field", True),
                      ("hyb_ccmp_off",   "outputs/routing_hyb/mir/mir_hyb_route_ccmp_e10_b2_s1024",    "hyb", "field", False)]),
    "tomato": dict(nb="colab_routing_tomato", frame="tomato_test_v16sc", openie="tomato_test",   # OpenIE arm: colab_tomato_openie_ablation.ipynb
                   extra={"hyb": "tomato_test_hyb"},        # merged graph (prep/build_hybrid_graph.py; tomato_hyb_bundle.zip on Drive)
                   sem={"field": ("outputs/tomato_ablations_v1/tomato/semantic", "tomato")},
                   arms=[("frame_ccmp",     "outputs/tomato_ablations_v1/tomato/tomato_fusion_qwenmlp_ccmp_epoch10_b2", "frame", "field", True),
                         ("frame_ccmp_off", "outputs/tomato_ablations_v1/tomato/tomato_fusion_qwenmlp_ccmp_epoch10_b2", "frame", "field", False),
                         ("frame_nocc",     ["outputs/tomato_ablations_v1/tomato/tomato_fusion_qwenmlp_epoch10_b2",
                                             "outputs/routing/tomato/tomato_route_control_e10_b2_s1024"],              "frame", "field", None),
                         ("openie",         "outputs/tomato_openie/tomato_openie_qwenmlp_graph_e10_b2",                  "openie", "field", None),
                         ("hyb_nocc",       "outputs/routing_hyb/tomato/tomato_hyb_route_control_e10_b2_s1024",          "hyb",    "field", None),
                         ("hyb_ccmp",       "outputs/routing_hyb/tomato/tomato_hyb_route_ccmp_e10_b2_s1024",             "hyb",    "field", True),
                         ("hyb_ccmp_off",   "outputs/routing_hyb/tomato/tomato_hyb_route_ccmp_e10_b2_s1024",             "hyb",    "field", False)]),
}
for _f in ("cs", "biology", "physics", "matsci"): SPEC[f"sir4_{_f}"] = _sir4(_f)

for d in DATASETS: assert d in SPEC, f"{d}: DATASETS must be from {sorted(SPEC)}"

# --- the CCMP arm only ---------------------------------------------------------------------------
for _d in SPEC:
    SPEC[_d].pop("extra", None)
    SPEC[_d]["arms"] = [a for a in SPEC[_d]["arms"] if a[0] == GD_ARM]
    assert SPEC[_d]["arms"], f"{_d}: no '{GD_ARM}' arm in the spec"

# --- resume plan --------------------------------------------------------------------------------------
# FORCE = "" (nothing), "fig" (redraw), "decomp" (redo the decomposition), or "all".
FORCE = ""
PLAN = {}
for d in DATASETS:
    o = f"{DRIVE}/outputs/scan/{d}"
    rel = SPEC[d]["arms"][0][1]; rels = rel if isinstance(rel, list) else [rel]
    PLAN[d] = {"ckpt": any(os.path.exists(f"{DRIVE}/{r}/model_best.pth") for r in rels),
               "hops": os.path.exists(f"{o}/hops_{GD_ARM}.json"),           # the routes to follow come from the showcase run
               "decomp": os.path.exists(f"{o}/gate_decomp_{d}.json") and FORCE not in ("decomp", "all"),
               "fig": os.path.exists(f"{o}/fig_gate_decomp_{d}.png") and FORCE == ""}
NEED_ENGINE = any(PLAN[d]["ckpt"] and PLAN[d]["hops"] and not PLAN[d]["decomp"] for d in DATASETS)
import numpy as np
print("datasets:", DATASETS, "| arm:", GD_ARM, "| FORCE:", FORCE or "(resume)")
for d in DATASETS:
    P = PLAN[d]
    print(f"  {d:14} checkpoint {'ok' if P['ckpt'] else 'MISSING'} | hops_{GD_ARM}.json {'ok' if P['hops'] else 'MISSING (run the showcase notebook first)'} "
          f"| decomposition {'cached' if P['decomp'] else 'to do'} | figure {'cached' if P['fig'] else 'to do'}")
print(f"engine + Qwen3 + component tables needed: {NEED_ENGINE}" + ("" if NEED_ENGINE else "  -> cells 2b-3g are no-ops; run cell 1, 2, 4, 5"))


## 2. Unpack every bundle + current scripts

In [ ]:
# 2. Unpack every bundle side by side into one local root (caches kept) and install the CURRENT
# repo scripts. Bundles: <dataset>_bundle.zip on Drive, each with the frame graph and (except TOMATO
# before its OpenIE run) the OpenIE graph of its test split.
# Re-running this cell in the same runtime (after fixing something downstream) re-uses what is already
# unpacked; set FORCE_UNPACK = True to wipe /content/cargo and start over.
FORCE_UNPACK = False
MARK = f"{CARGO_ROOT}/.unpacked.json"
_have = set(json.load(open(MARK))) if (os.path.exists(MARK) and not FORCE_UNPACK) else set()
_todo = [d for d in DATASETS if d not in _have]
if _todo:
    KEEP, PARK = f"{CARGO_ROOT}/outputs/caches", "/content/_caches_keep"
    if _have:                       # incremental: keep what is unpacked, add the missing bundles
        for d in _todo:
            z = f"{DRIVE}/{d}_bundle.zip"
            assert os.path.exists(z) and zipfile.is_zipfile(z), f"missing or corrupt {z}"
            t0 = time.time(); zipfile.ZipFile(z).extractall(CARGO_ROOT); print(f"unpacked {os.path.basename(z)} ({time.time()-t0:.0f}s)")
    else:
        if os.path.isdir(KEEP):
            shutil.rmtree(PARK, ignore_errors=True); shutil.move(KEEP, PARK)
        if os.path.exists(CARGO_ROOT):
            shutil.rmtree(CARGO_ROOT)
        os.makedirs(CARGO_ROOT, exist_ok=True)
        for d in DATASETS:
            z = f"{DRIVE}/{d}_bundle.zip"
            assert os.path.exists(z) and zipfile.is_zipfile(z), f"missing or corrupt {z}"
            t0 = time.time(); zipfile.ZipFile(z).extractall(CARGO_ROOT); print(f"unpacked {os.path.basename(z)} ({time.time()-t0:.0f}s)")
        if os.path.isdir(PARK):
            os.makedirs(os.path.dirname(KEEP), exist_ok=True); shutil.move(PARK, KEEP); print("restored caches")
    json.dump(sorted(_have | set(DATASETS)), open(MARK, "w"))
else:
    print("bundles already unpacked:", sorted(_have), "(FORCE_UNPACK = True to redo)")
OVERLAY = json.loads(r"""{"cargo_paths.py": "\"\"\"\ncargo_paths.py -- one place that decides WHICH corpus a pipeline script is\nworking on, and where that corpus's caches live.\n\nTHE PROBLEM THIS SOLVES. Five scripts in the construction pipeline had the\nTOMATO corpus baked in as the literal string \"tomato\", and their caches were\nkeyed by SPLIT ALONE:\n\n    kg-construction-v16/cache/frames_doc.jsonl          <- which corpus?\n    kg-construction/construct_v2/cache/probes_test.jsonl\n    outputs/caches/op_emb/test_doc.npy\n\nPoint those scripts at a second corpus and a leftover TOMATO `test_doc.npy`\nloads silently: the shape check passes whenever the two corpora happen to have a\nsimilar document count, and the run produces plausible, meaningless numbers. No\nexception, no warning. That is the failure this module exists to prevent.\n\nTHE CONTRACT. With the default dataset \"tomato\" every path returned here is\nbyte-identical to the hardcoded string it replaced, so existing caches still\nresolve and published TOMATO numbers stay reproducible. Any other dataset name\nadditionally scopes every cache into a subdirectory named after it, so two\ncorpora can never share a cache file.\n\n    DATASET=tomato   ->  .../cache/frames_doc.jsonl          (unchanged)\n    DATASET=sir4_cs  ->  .../cache/sir4_cs/frames_doc_test.jsonl\n\nScripts call `set_dataset()` once after parsing args, then use the helpers.\n\n    import sys, os\n    sys.path.insert(0, os.path.expanduser(\"~/Desktop/CARGO\"))\n    from cargo_paths import set_dataset, corpus_dir, frames_path\n\n    set_dataset(args.dataset)\n    raw = corpus_dir(args.split)              # .../data/sir4_cs_test/raw\n\"\"\"\nfrom __future__ import annotations\n\nimport os\n\n# Repo root. CARGO_ROOT lets the same scripts run somewhere that is not this\n# laptop -- Colab unzips the bundle to /content/cargo, where ~/Desktop/CARGO\n# does not exist. Unset, the default is byte-identical to the hardcoded string\n# it replaced, so nothing about a local run changes.\nCARGO = os.environ.get(\"CARGO_ROOT\") or os.path.expanduser(\"~/Desktop/CARGO\")\nKG = f\"{CARGO}/kg-construction\"\nV16 = f\"{CARGO}/kg-construction-v16\"\n\n#: Corpus currently being processed. \"tomato\" reproduces every legacy path.\nDATASET = os.environ.get(\"CARGO_DATASET\", \"tomato\")\n\n\ndef set_dataset(name: str | None) -> str:\n    \"\"\"Set the active corpus. `None` or \"\" leaves the current value alone.\"\"\"\n    global DATASET\n    if name:\n        DATASET = name\n    return DATASET\n\n\ndef is_legacy() -> bool:\n    \"\"\"True when paths must stay byte-identical to the pre-refactor strings.\"\"\"\n    return DATASET == \"tomato\"\n\n\ndef _scoped(root: str) -> str:\n    \"\"\"Cache root, with a per-dataset subdirectory for anything but TOMATO.\"\"\"\n    d = root if is_legacy() else f\"{root}/{DATASET}\"\n    os.makedirs(d, exist_ok=True)\n    return d\n\n\n# --------------------------------------------------------------------------\n# corpora and graphs\n# --------------------------------------------------------------------------\ndef corpus_name(split: str) -> str:\n    \"\"\"Directory name of a raw corpus, e.g. `tomato_test` / `sir4_cs_test`.\"\"\"\n    return f\"{DATASET}_{split}\"\n\n\ndef corpus_dir(split: str) -> str:\n    \"\"\"Absolute path to a raw corpus directory (holds raw/documents.json).\"\"\"\n    return f\"{KG}/data/{corpus_name(split)}\"\n\n\ndef graph_name(split: str, suffix: str = \"v16sc\") -> str:\n    \"\"\"Directory name of a built graph, e.g. `sir4_cs_train_v16sc`.\"\"\"\n    return f\"{DATASET}_{split}_{suffix}\"\n\n\ndef graph_dir(split: str, suffix: str = \"v16sc\") -> str:\n    return f\"{KG}/data/{graph_name(split, suffix)}\"\n\n\n# --------------------------------------------------------------------------\n# caches\n# --------------------------------------------------------------------------\ndef frames_path(side: str, split: str) -> str:\n    \"\"\"Extracted-frame cache. `side` is \"doc\" or \"query\".\n\n    TOMATO's test frames are stored WITHOUT a split suffix and its train frames\n    WITH one -- an asymmetry from when only a test split existed. That is\n    preserved exactly for TOMATO and dropped for every other dataset, where the\n    split is always in the name.\n    \"\"\"\n    root = _scoped(f\"{V16}/cache\")\n    if is_legacy() and split == \"test\":\n        return f\"{root}/frames_{side}.jsonl\"\n    return f\"{root}/frames_{side}_{split}.jsonl\"\n\n\ndef graph_cache_dir() -> str:\n    \"\"\"Scratch for graph-construction caches, e.g. `ds_{split}_emb_{type}.npy`.\n\n    Those were keyed by split alone too. A SIR-4 build silently loaded TOMATO's\n    concept embeddings from `ds_test_emb_method.npy`, which is part of why a\n    contaminated build produced TOMATO-shaped output without complaining.\n    \"\"\"\n    return _scoped(f\"{V16}/cache\")\n\n\ndef probes_path(split: str) -> str:\n    \"\"\"LLM probe cache used by the operator's S and M terms.\"\"\"\n    return f\"{_scoped(f'{KG}/construct_v2/cache')}/probes_{split}.jsonl\"\n\n\ndef emb_dir() -> str:\n    \"\"\"BGE embedding cache (`{split}_doc.npy`, `{split}_query_{hash}.npy`, ...).\n\n    The most dangerous of the three: filenames carry only the split, and a\n    document matrix from the wrong corpus fails no assertion that the callers\n    make.\n    \"\"\"\n    return _scoped(f\"{CARGO}/outputs/caches/op_emb\")\n\n\ndef add_dataset_arg(ap) -> None:\n    \"\"\"Attach the standard `--dataset` flag to an argparse parser.\"\"\"\n    ap.add_argument(\n        \"--dataset\", default=os.environ.get(\"CARGO_DATASET\", \"tomato\"),\n        help=\"corpus to operate on (default tomato; e.g. sir4_cs). \"\n             \"Anything but 'tomato' also scopes every cache under this name.\")\n\n\ndef banner() -> str:\n    return (f\"[cargo_paths] dataset={DATASET} \"\n            f\"{'(legacy TOMATO paths)' if is_legacy() else '(scoped caches)'}\")\n", "sir4-retrieval/eval/baselines_sir4.py": "\"\"\"\nbaselines_sir4.py -- the retrieval-baseline table: BM25, and any HF dense encoder.\n\nWHY NOT bge_sir4.py. That script is not just \"the BGE baseline\": score_sir4.py\ndefines the `dissimilar` slice as \"plain BGE ranks this query's best gold below\n100\", so its output file IS the slice definition. Adding encoders and pooling\nmodes to it risks a baseline run silently redefining the slice every other arm\nis measured against. This is a separate file and bge_sir4.py is left alone.\n\nPOOLING IS NOT A DETAIL. SentenceTransformer wraps a bare HF checkpoint with MEAN\npooling when the repo carries no ST config. SPECTER2 and SciNCL are both trained\nwith CLS pooling, so loading them the default way silently evaluates a different\nmodel than the paper's and understates both. `--pooling` is therefore explicit,\nand `auto` picks the documented pooling per family rather than whatever ST\nguesses.\n\nOUTPUT is byte-compatible with bge_sir4.py, so score_sir4.py consumes it\nunchanged:\n    [{id, stratum, supporting_documents, predictions:{document:[[doc_id, score]]}}]\n\nUsage\n-----\n    python3 eval/baselines_sir4.py --dataset tomato --split test --model bm25\n    python3 eval/baselines_sir4.py --dataset tomato --split test \\\n        --model allenai/specter2_base --pooling cls --tag specter2\n\"\"\"\nfrom __future__ import annotations\n\nimport argparse\nimport hashlib\nimport json\nimport os\nimport re\nimport sys\n\nimport numpy as np\n\nHERE = os.path.dirname(os.path.abspath(__file__))\nROOT = os.path.dirname(HERE)\nCARGO = os.path.dirname(ROOT)\nsys.path.insert(0, CARGO)\n\nfrom cargo_paths import add_dataset_arg, banner, corpus_dir, set_dataset  # noqa: E402\n\n# Documented pooling per family. Anything not listed falls back to mean, which is\n# what a generic sentence-transformers checkpoint expects.\nCLS_POOLED = (\"specter\", \"scincl\", \"scibert\")\n\n_WORD_RE = re.compile(r\"[a-zA-Z]+\")\n_STOP = set(\"\"\"a an and are as at be by for from has have in is it of on or that the to\nwas were will with this these those we our using used use based via study paper\"\"\".split())\n\n\ndef _tokens(text: str) -> list[str]:\n    return [t for t in (m.group(0).lower() for m in _WORD_RE.finditer(text))\n            if len(t) > 2 and t not in _STOP]\n\n\ndef run_bm25(queries, corpus, doc_ids, topk):\n    \"\"\"BM25Okapi over the same corpus text the dense arms encode.\n\n    Ranks the FULL corpus per query, so its recall@100 is comparable with the\n    dense arms rather than being capped by a candidate set.\n    \"\"\"\n    from rank_bm25 import BM25Okapi\n    print(f\"[bm25] tokenising {len(doc_ids):,} documents ...\")\n    bm = BM25Okapi([_tokens(corpus[d]) for d in doc_ids])\n    out = []\n    for i, q in enumerate(queries):\n        if i % 500 == 0:\n            print(f\"  [bm25] {i:,}/{len(queries):,}\", flush=True)\n        s = np.asarray(bm.get_scores(_tokens(q[\"question\"])), dtype=np.float32)\n        k = min(topk, len(doc_ids))\n        top = np.argpartition(-s, k - 1)[:k]\n        top = top[np.argsort(-s[top])]\n        out.append((q, [(doc_ids[j], float(s[j])) for j in top]))\n    return out\n\n\ndef run_dense(queries, corpus, doc_ids, topk, model_name, pooling, instruct, batch, cpu,\n              trust_remote_code=False, dtype=None):\n    import torch\n    from sentence_transformers import SentenceTransformer, models\n\n    dev = \"cpu\" if cpu else (\"cuda\" if torch.cuda.is_available()\n                             else \"mps\" if torch.backends.mps.is_available() else \"cpu\")\n    if pooling == \"auto\":\n        low = model_name.lower()\n        pooling = \"cls\" if any(k in low for k in CLS_POOLED) else \"st\"\n    # An 8B encoder in float32 is ~32 GB of weights before a single activation, so\n    # the big ones have to load in half precision or they do not load at all.\n    margs = {}\n    if trust_remote_code:\n        margs[\"trust_remote_code\"] = True\n    if dtype:\n        margs[\"torch_dtype\"] = getattr(torch, dtype)\n    if pooling == \"st\":\n        # The checkpoint ships its own pooling/normalisation config; trust it.\n        model = SentenceTransformer(model_name, device=dev,\n                                    trust_remote_code=trust_remote_code,\n                                    model_kwargs=margs or None)\n    else:\n        # Build the tower explicitly so the pooling is the documented one rather\n        # than sentence-transformers' mean-pooling fallback.\n        w = models.Transformer(model_name, max_seq_length=512,\n                               model_args=margs or None,\n                               tokenizer_args={\"trust_remote_code\": True}\n                               if trust_remote_code else None)\n        p = models.Pooling(w.get_word_embedding_dimension(),\n                           pooling_mode_cls_token=(pooling == \"cls\"),\n                           pooling_mode_mean_tokens=(pooling == \"mean\"))\n        model = SentenceTransformer(modules=[w, p], device=dev)\n    model.max_seq_length = 512\n    print(f\"[dense] {model_name} on {dev} | pooling={pooling} | instruct={bool(instruct)}\")\n\n    def enc(texts, tag):\n        print(f\"  [enc] {tag}: {len(texts):,} texts\")\n        v = model.encode(texts, batch_size=batch, convert_to_numpy=True,\n                         normalize_embeddings=True, show_progress_bar=True)\n        return v.astype(np.float32)\n\n    D = enc([corpus[d] for d in doc_ids], \"documents\")\n    Q = enc([(instruct + q[\"question\"]) if instruct else q[\"question\"] for q in queries],\n            \"queries\")\n    out, k = [], min(topk, len(doc_ids))\n    for i0 in range(0, len(queries), 256):          # chunked: the full matrix is large\n        S = Q[i0:i0 + 256] @ D.T\n        for r in range(S.shape[0]):\n            s = S[r]\n            top = np.argpartition(-s, k - 1)[:k]\n            top = top[np.argsort(-s[top])]\n            out.append((queries[i0 + r], [(doc_ids[j], float(s[j])) for j in top]))\n        del S\n    return out\n\n\ndef main() -> int:\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\"--split\", default=\"test\", choices=[\"train\", \"test\"])\n    ap.add_argument(\"--model\", required=True,\n                    help=\"'bm25', or any HF/sentence-transformers model id or local path\")\n    ap.add_argument(\"--pooling\", default=\"auto\", choices=[\"auto\", \"st\", \"cls\", \"mean\"],\n                    help=\"auto = ST config for generic encoders, CLS for SPECTER/SciNCL. \"\n                         \"The default ST fallback is MEAN, which is the wrong model for \"\n                         \"both of those and understates them.\")\n    ap.add_argument(\"--instruct\", default=\"\",\n                    help=\"prefix prepended to each QUERY only. Instruction-tuned \"\n                         \"retrievers (Qwen3, ReasonIR) expect one; BM25 and SPECTER do not.\")\n    ap.add_argument(\"--tag\", default=None, help=\"filename tag; defaults to a slug of --model\")\n    ap.add_argument(\"--topk\", type=int, default=300,\n                    help=\"ranked documents per query. Must exceed 100: the dissimilar \"\n                         \"rule is 'best gold below rank 100'\")\n    ap.add_argument(\"--batch\", type=int, default=32)\n    ap.add_argument(\"--cpu\", action=\"store_true\")\n    ap.add_argument(\"--trust-remote-code\", action=\"store_true\",\n                    help=\"required by ReasonIR-8B and other custom-architecture encoders\")\n    ap.add_argument(\"--dtype\", default=None, choices=[\"float16\", \"bfloat16\", \"float32\"],\n                    help=\"weight dtype. An 8B encoder needs float16/bfloat16 to fit at all.\")\n    ap.add_argument(\"--out\", default=None)\n    add_dataset_arg(ap)\n    a = ap.parse_args()\n    set_dataset(a.dataset)\n    print(banner())\n    if a.topk < 100:\n        print(\"--topk below 100 makes the dissimilar slice undefined\", file=sys.stderr)\n        return 2\n\n    corpus = json.load(open(f\"{corpus_dir(a.split)}/raw/documents.json\"))\n    queries = json.load(open(f\"{corpus_dir(a.split)}/raw/{a.split}.json\"))\n    doc_ids = list(corpus)\n    tag = a.tag or re.sub(r\"[^a-z0-9]+\", \"-\", a.model.lower()).strip(\"-\")\n    print(f\"corpus {len(doc_ids):,} docs | {len(queries):,} queries | topk={a.topk} | tag={tag}\")\n\n    if a.model.lower() == \"bm25\":\n        ranked = run_bm25(queries, corpus, doc_ids, a.topk)\n    else:\n        ranked = run_dense(queries, corpus, doc_ids, a.topk, a.model, a.pooling,\n                           a.instruct, a.batch, a.cpu, a.trust_remote_code, a.dtype)\n\n    out = [{\"id\": q[\"id\"],\n            \"stratum\": q.get(\"stratum\"),\n            \"supporting_documents\": q.get(\"supporting_documents\") or [],\n            \"predictions\": {\"document\": [[d, s] for d, s in top]}}\n           for q, top in ranked]\n    dest = a.out or f\"{ROOT}/data/predictions_{tag}_{a.dataset}_{a.split}.json\"\n    os.makedirs(os.path.dirname(dest), exist_ok=True)\n    json.dump(out, open(dest, \"w\"))\n    print(f\"wrote {dest}  ({len(out):,} queries)\")\n    return 0\n\n\nif __name__ == \"__main__\":\n    raise SystemExit(main())\n", "sir4-retrieval/eval/bge_sir4.py": "\"\"\"\nbge_sir4.py -- step 14. Plain BGE-large dense baseline on a SIR-4 split.\n\nWHY NOT kg-construction/eval/bge.py. That script is TOMATO-only in three ways\nthat cannot be flagged away: it reads a TOMATO-shaped cache\n(`outputs/caches/corpus.json`), it expects the old query schema (`query_id`,\n`b_text`, `gold_key`), and it writes ONE gold per record (`[q[\"gold_key\"]]`).\nSIR-4 averages 4.45 golds. Patching it would put TOMATO's reproduction at risk\nfor no gain, so this is a separate script and bge.py is left alone.\n\nWHY THIS MATTERS BEYOND \"a baseline\". `score_sir4.py` defines the `dissimilar`\nslice as \"plain BGE ranks this query's BEST gold below 100\". Without this file\nthere is no similar/dissimilar split at all, which is the axis the whole\ncross-domain argument rests on.\n\nThe encoder is imported from cargo_operator.py rather than re-specified, so the\nbaseline and the operator's dense term are the SAME vectors, same instruction,\nsame cache. A baseline built with a slightly different encoder would make every\ndelta unattributable.\n\nUsage\n-----\n    python3 eval/bge_sir4.py --dataset sir4_cs --split test\n    python3 eval/bge_sir4.py --dataset sir4_cs --split test --topk 300\n\"\"\"\nfrom __future__ import annotations\n\nimport argparse\nimport hashlib\nimport importlib.util\nimport json\nimport os\nimport sys\n\nimport numpy as np\n\nHERE = os.path.dirname(os.path.abspath(__file__))\nROOT = os.path.dirname(HERE)\nCARGO = os.path.dirname(ROOT)\nKG = f\"{CARGO}/kg-construction\"\nsys.path.insert(0, CARGO)\nsys.path.insert(0, KG)\nfrom cargo_paths import add_dataset_arg, banner, corpus_dir, set_dataset  # noqa: E402\n\n\nDEFAULT_MODEL = \"BAAI/bge-large-en-v1.5\"\n\n# Imported rather than restated. This used to be a second copy of the string,\n# which meant swapping the instruction in cargo_operator.py left the dense\n# baseline on the old one and the two silently disagreed.\nQWEN_QI = None          # resolved from cargo_operator in main(); see op.QWEN_QI\n\n\ndef model_slug(name: str) -> str:\n    \"\"\"'' for the default encoder, else a filename-safe tag.\n\n    THIS IS A CORRECTNESS FIX, NOT COSMETICS. cached_encode keys purely on the\n    tag it is handed, and the tag used to be `{split}_doc` with no mention of the\n    model. Running --model Qwen/... therefore loaded BGE's cached .npy and\n    reported it as Qwen3: a silent contamination of exactly the kind that has\n    already cost this project a day. Empty for the default keeps the existing\n    cache hits (and the shared-encoder guarantee with cargo_operator) intact.\n    \"\"\"\n    if name == DEFAULT_MODEL:\n        return \"\"\n    import re\n    return \"_\" + re.sub(r\"[^a-z0-9]+\", \"-\", name.lower()).strip(\"-\")\n\n\ndef load_operator_module():\n    \"\"\"Import cargo_operator.py for cached_encode + BGE_QI, so the baseline and\n    the operator's dense term are literally the same encoder and cache.\"\"\"\n    spec = importlib.util.spec_from_file_location(\"op\", f\"{KG}/eval/cargo_operator.py\")\n    op = importlib.util.module_from_spec(spec)\n    spec.loader.exec_module(op)\n    return op\n\n\ndef main() -> int:\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\"--split\", default=\"test\", choices=[\"train\", \"test\"])\n    ap.add_argument(\"--topk\", type=int, default=300,\n                    help=\"ranked documents per query. Must exceed 100: the \"\n                         \"dissimilar rule is 'best gold below rank 100'\")\n    ap.add_argument(\"--model\", default=\"BAAI/bge-large-en-v1.5\")\n    ap.add_argument(\"--cpu\", action=\"store_true\")\n    ap.add_argument(\"--out\", default=None)\n    add_dataset_arg(ap)\n    a = ap.parse_args()\n    set_dataset(a.dataset)\n    print(banner())\n    if a.topk < 100:\n        print(\"--topk below 100 makes the dissimilar slice undefined\", file=sys.stderr)\n        return 2\n\n    op = load_operator_module()\n    from sentence_transformers import SentenceTransformer\n\n    corpus = json.load(open(f\"{corpus_dir(a.split)}/raw/documents.json\"))\n    queries = json.load(open(f\"{corpus_dir(a.split)}/raw/{a.split}.json\"))\n    doc_ids = list(corpus)\n    qkey = hashlib.md5(\"|\".join(q[\"id\"] for q in queries).encode()).hexdigest()[:8]\n    print(f\"corpus {len(doc_ids):,} docs | {len(queries):,} queries | topk={a.topk}\")\n\n    import torch\n    dev = \"cpu\" if a.cpu else (\"cuda\" if torch.cuda.is_available()\n                               else \"mps\" if torch.backends.mps.is_available() else \"cpu\")\n    model = SentenceTransformer(a.model, device=dev)\n    model.max_seq_length = 512\n\n    # Same tags cargo_operator.py uses, so whatever it already encoded is reused\n    # rather than recomputed with a different seed of the same model.\n    slug = model_slug(a.model)\n    qi = op.query_instruction(a.model)      # one definition, shared with the operator\n    print(f\"encoder {a.model}  cache tag suffix {slug!r}  instruct {qi[:40]!r}...\")\n    D = op.cached_encode(model, [corpus[d] for d in doc_ids],\n                         f\"{a.split}_doc{slug}\").astype(np.float32)\n    Q = op.cached_encode(model, [q[\"question\"] for q in queries],\n                         f\"{a.split}_query_{qkey}{slug}{op.qi_tag(qi)}\",\n                         instruct=qi).astype(np.float32)\n\n    out, k = [], min(a.topk, len(doc_ids))\n    CH = 256                                    # chunked: the full score matrix is\n    for i0 in range(0, len(queries), CH):       # 5,445 x 20,203 on train\n        S = Q[i0:i0 + CH] @ D.T\n        for r in range(S.shape[0]):\n            q = queries[i0 + r]\n            s = S[r]\n            top = np.argpartition(-s, k - 1)[:k]\n            top = top[np.argsort(-s[top])]\n            out.append({\n                \"id\": q[\"id\"],\n                \"stratum\": q.get(\"stratum\"),\n                \"supporting_documents\": q.get(\"supporting_documents\") or [],\n                \"predictions\": {\"document\": [[doc_ids[i], float(s[i])] for i in top]},\n            })\n        del S\n\n    # The slug is in the DEFAULT filename too. Without it a Qwen3 run would\n    # overwrite predictions_bge_*.json, and score_sir4.py reads that exact file\n    # to define the `dissimilar` slice -- so one Qwen3 run would silently\n    # redefine the slice for every arm scored afterwards.\n    dest = a.out or f\"{ROOT}/data/predictions_bge{slug}_{a.dataset}_{a.split}.json\"\n    os.makedirs(os.path.dirname(dest), exist_ok=True)\n    json.dump(out, open(dest, \"w\"))\n    print(f\"wrote {dest}  ({len(out):,} records)\")\n\n    # Report the slice this file exists to define, so a broken run is obvious here\n    # rather than three steps later inside the scorer.\n    below = 0\n    for rec, q in zip(out, queries):\n        golds = set(q.get(\"supporting_documents\") or [])\n        rank = next((i for i, d in enumerate(rec[\"predictions\"][\"document\"], 1)\n                     if d[0] in golds), 10 ** 9)\n        below += rank > 100\n    print(f\"dissimilar slice: {below:,} of {len(out):,} queries \"\n          f\"({below/max(len(out),1):.1%}) have their best gold below rank 100\")\n    return 0\n\n\nif __name__ == \"__main__\":\n    raise SystemExit(main())\n", "sir4-retrieval/eval/score_sir4.py": "\"\"\"\nscore_sir4.py -- retrieval metrics for SIR-4, which is MULTI-GOLD.\n\nWHY THIS EXISTS. Every eval cell inherited from the TOMATO notebooks does:\n\n    gold = r[\"supporting_documents\"]\n    gd   = gold[0] if isinstance(gold, list) else gold     # <-- one gold\n    rank = ranked.index(gd) + 1\n\nTOMATO carries exactly one gold per row, so that is correct there. SIR-4 carries\na mean of 4.45 golds per query (range 1 to 10) on a single row, so the inherited\ncode silently scores against the first of four-and-a-half. It does not crash and\nthe number it prints looks reasonable, which is what makes it dangerous.\n\nMETRICS (as specified for the CS run):\n\n  MRR            reciprocal rank of the HIGHEST-ranked labelled positive.\n                 \"Did the retriever surface anything useful, and how fast.\"\n\n  MGRR           mean reciprocal rank over ALL labelled positives.\n                 \"How highly did it rank the complete flat gold list.\"\n\n  Recall@k       fraction of ALL labelled positives inside the top k.\n                 \"How much of the decomposition did it find.\"\n\n  CompleteSet@k  1 if at least one complete set from sets.json is a subset of\n                 the top k, else 0. \"Did it find a WHOLE working explanation.\"\n\nCompleteSet@k is the SIR-4-specific metric. It is only definable because SIR-4\nrecords every validated decomposition, not just one; no other benchmark in the\nMOOSE lineage carries the alternative sets. It is also strictly harder than\nRecall@k: a run can retrieve 6 of 7 golds and still score 0, because six pieces\nof two different explanations do not make one explanation.\n\nSLICES. `same` / `cross` come from the query row. `similar` / `dissimilar` need a\ndense baseline: a query is `dissimilar` when plain BGE ranks its BEST gold below\n100. Best-gold, not first-gold, so the slice definition is consistent with MRR.\nWithout --bge those two slices are skipped rather than guessed.\n\nUsage\n-----\n    python3 eval/score_sir4.py --pred preds.json --queries .../test.json \\\\\n                               --sets .../sets.json --bge predictions_bge.json\n\"\"\"\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport math\nfrom collections import defaultdict\n\n# 15 is here for ResearchBench: it retains 20% of a 75-candidate pool, so Recall@15\n# is the cutoff its published numbers are reported at. Costs nothing to compute and\n# nothing downstream reads KS positionally.\n# 25 is here for the semantic-scorer comparison, whose reporting set spans\n# Recall@3 to Recall@100. Adding a cutoff is safe: nothing downstream reads KS\n# positionally, and every consumer selects metrics by name or via --cols.\nKS = (1, 2, 3, 5, 10, 15, 20, 25, 50, 100)\nBIG = 10 ** 9\n\n\ndef ranked_docs(rec: dict) -> list[str]:\n    \"\"\"Ranked document ids from a prediction record, tolerating both shapes.\"\"\"\n    p = rec.get(\"predictions\", rec)\n    docs = p.get(\"document\", p) if isinstance(p, dict) else p\n    out = []\n    for d in docs:\n        out.append(d[0] if isinstance(d, (list, tuple)) else d)\n    return out\n\n\ndef best_gold_rank(ranked: list[str], golds: set[str]) -> int:\n    for i, d in enumerate(ranked, 1):\n        if d in golds:\n            return i\n    return BIG\n\n\ndef score_one(ranked: list[str], golds: set[str], sets_: list[list[str]]) -> dict:\n    r = best_gold_rank(ranked, golds)\n    # Standard query-level MRR uses the first relevant result.  The previous version\n    # stored the mean reciprocal rank over every gold under `mrr`; that is a useful\n    # multi-gold diagnostic, but it is not conventional MRR.  Preserve it explicitly\n    # as MGRR so old values remain reproducible without overloading a standard name.\n    #\n    # A gold outside the truncated prediction list contributes 0 rather than 1/rank.\n    # With top-300 of 20,322 the difference is below the fourth decimal, and pretending\n    # to know a rank the file does not contain would be worse.\n    #\n    # `mrr_best` is retained as a deprecated alias for compatibility with any files\n    # or notebooks that already requested it explicitly.\n    pos = {d: i + 1 for i, d in enumerate(ranked)}\n    ranks = [pos.get(g, BIG) for g in golds]\n    standard_mrr = 1.0 / r if r < BIG else 0.0\n    mean_gold_rr = ((sum(1.0 / x for x in ranks if x < BIG) / len(golds))\n                    if golds else 0.0)\n    # Average precision over the flat gold set: mean over golds of precision at that gold's\n    # rank, a gold outside the list contributing 0. Aggregated over queries this is mAP,\n    # the metric MIR (ACL 2025) reports, so its published rows line up with ours.\n    hits, ap = 0, 0.0\n    for i, d in enumerate(ranked, 1):\n        if d in golds:\n            hits += 1\n            ap += hits / i\n    out = {\n        \"mrr\": standard_mrr,\n        \"mgrr\": mean_gold_rr,\n        \"mrr_best\": standard_mrr,\n        \"map\": ap / len(golds) if golds else 0.0,\n    }\n    for k in KS:\n        topk = set(ranked[:k])\n        # Recall over ALL positives, not just the first one.\n        out[f\"recall@{k}\"] = len(topk & golds) / len(golds) if golds else 0.0\n        out[f\"hits@{k}\"] = float(r <= k)\n        # A whole validated set inside the top k. Sets larger than k can never\n        # qualify, which is correct: you cannot fit 7 papers into a top-5.\n        out[f\"completeset@{k}\"] = float(any(set(s) <= topk for s in sets_)) if sets_ else 0.0\n\n    # Binary-relevance nDCG@5 over the flat set of labelled inspiration papers.\n    # CompleteSet@5 below remains the dependency-aware companion metric.\n    rel = [float(d in golds) for d in ranked[:5]]\n    dcg = sum(v / math.log2(i + 2) for i, v in enumerate(rel))\n    ideal_n = min(len(golds), 5)\n    idcg = sum(1.0 / math.log2(i + 2) for i in range(ideal_n))\n    out[\"ndcg@5\"] = dcg / idcg if idcg else 0.0\n    return out\n\n\ndef main() -> int:\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\"--pred\", required=True, help=\"predictions json (list of records)\")\n    ap.add_argument(\"--queries\", required=True, help=\"the split's {split}.json\")\n    ap.add_argument(\"--sets\", default=None, help=\"sets.json from the SIR-4 export (for CompleteSet@k)\")\n    ap.add_argument(\"--bge\", default=None, help=\"dense baseline predictions, for the similar/dissimilar split\")\n    ap.add_argument(\"--name\", default=\"run\")\n    ap.add_argument(\"--json-out\", default=None)\n    # PER-QUERY OUTPUT EXISTS FOR PAIRED TESTS. The aggregate below reduces every\n    # slice to a mean, and two means cannot be compared with a paired test no\n    # matter how the arms were run. When two arms rank the SAME queries over the\n    # SAME candidate graphs -- which is the whole design of the ResearchBench\n    # transfer experiment -- an unpaired z-test throws away the pairing and is\n    # strictly weaker. Dump the per-query rows and bootstrap the difference.\n    ap.add_argument(\"--per-query-out\", default=None,\n                    help=\"write {query_id: {metric: value, 'stratum': ...}} before aggregation\")\n    # PER-GOLD STRATA EXIST BECAUSE RESEARCHBENCH LABELS GOLDS, NOT QUERIES.\n    # SIR-4 carries one `stratum` per query, so the slice table below works. On\n    # ResearchBench a query has a LIST of per-gold labels (`gold_strata`,\n    # positionally aligned with supporting_documents): one query can contribute a\n    # same-field gold and a cross-field gold at once. Collapsing that to a query\n    # label would have to pick a rule -- any-cross? all-cross? majority? -- and\n    # every rule discards information the benchmark actually provides. Instead\n    # answer the question the labels support directly: of the labelled cross-field\n    # golds, what fraction were retrieved by rank k. That is a different\n    # denominator from the slice table and is reported separately.\n    ap.add_argument(\"--gold-strata\", action=\"store_true\",\n                    help=\"also report recall by PER-GOLD stratum (needs gold_strata on each query)\")\n    # The default column set is the one the SIR-4 results notebook reads by name.\n    # ResearchBench wants a different set (Recall@3 and @15 are ITS operating points,\n    # at 4% and 20% of a 75-candidate pool), and quietly widening the default would\n    # change every SIR-4 table already produced. Let the caller ask instead.\n    ap.add_argument(\"--cols\", default=None,\n                    help=\"comma-separated metric keys to aggregate; default is the SIR-4 set\")\n    a = ap.parse_args()\n\n    queries = {q[\"id\"]: q for q in json.load(open(a.queries))}\n    preds = json.load(open(a.pred))\n    if isinstance(preds, dict):\n        preds = list(preds.values())\n\n    sets_by_q: dict[str, list[list[str]]] = {}\n    if a.sets:\n        raw = json.load(open(a.sets))\n        for qid, v in raw.items():\n            sets_by_q[qid] = [list(s) for s in (v.get(\"sets\") if isinstance(v, dict) else v)]\n\n    bge_rank: dict[str, int] = {}\n    if a.bge:\n        for rec in json.load(open(a.bge)):\n            q = queries.get(rec[\"id\"])\n            if q:\n                bge_rank[rec[\"id\"]] = best_gold_rank(ranked_docs(rec), set(q[\"supporting_documents\"]))\n\n    slices: dict[str, list[dict]] = defaultdict(list)\n    per_query: dict[str, dict] = {}\n    gold_rows: list[tuple[str, int]] = []          # (stratum, rank) one per gold\n    seen, missing = set(), 0\n    for rec in preds:\n        qid = rec.get(\"id\")\n        q = queries.get(qid)\n        if q is None or qid in seen:\n            missing += q is None\n            continue\n        seen.add(qid)\n        golds = set(q[\"supporting_documents\"])\n        m = score_one(ranked_docs(rec), golds, sets_by_q.get(qid, []))\n        # SIR-4 has one query-level `stratum`.  ResearchBench has per-gold\n        # labels plus the conservative derived `domain_slice` written by the\n        # corrected pool builder.  Map the latter only for the query-level\n        # table; the per-gold table below keeps the original aligned labels.\n        query_stratum = q.get(\"stratum\")\n        if not query_stratum:\n            query_stratum = {\n                \"same_only\": \"same\",\n                \"cross_involved\": \"cross\",\n            }.get(q.get(\"domain_slice\"))\n        names = [\"all\"] + ([query_stratum] if query_stratum else [])\n        if qid in bge_rank:\n            sim = \"dissimilar\" if bge_rank[qid] > 100 else \"similar\"\n            names.append(sim)\n            if query_stratum:\n                names.append(f\"{query_stratum}+{sim}\")\n        for n in names:\n            slices[n].append(m)\n        if a.gold_strata:\n            # One row per (query, gold). `ranked.index` is O(n) but this runs once\n            # per gold over a list already in memory, and clarity beats a rank map\n            # that would have to be kept in sync with ranked_docs().\n            order = ranked_docs(rec)\n            pos = {d: i + 1 for i, d in enumerate(order)}\n            strata = q.get(\"gold_strata\") or []\n            for i, g in enumerate(q[\"supporting_documents\"]):\n                gold_rows.append((strata[i] if i < len(strata) else \"unlabelled\",\n                                  pos.get(g, BIG)))\n        if a.per_query_out is not None:\n            # `slices` is carried alongside so a paired test can be restricted to a\n            # slice (cross-field only, one discipline) without re-deriving which\n            # queries belong to it. `discipline` is ResearchBench-only and absent\n            # on SIR-4; recorded when present rather than required.\n            per_query[qid] = {**m, \"slices\": names,\n                              \"n_golds\": len(golds),\n                              **({\"domain_slice\": q[\"domain_slice\"]}\n                                 if \"domain_slice\" in q else {}),\n                              **({\"discipline\": q[\"discipline\"]} if \"discipline\" in q else {})}\n\n    order = [\"all\", \"same\", \"cross\", \"similar\", \"dissimilar\", \"same+dissimilar\", \"cross+dissimilar\"]\n    cols = [\"mrr\", \"ndcg@5\", \"recall@3\", \"recall@5\", \"completeset@5\"]\n    if a.cols:\n        cols = [c.strip() for c in a.cols.split(\",\") if c.strip()]\n        probe = score_one([], set(), [])          # the metric keys score_one emits\n        unknown = [c for c in cols if c not in probe]\n        if unknown:\n            return print(f\"unknown metric(s) {unknown}; available: \"\n                         f\"{sorted(probe)}\") or 2\n    print(f\"\\n### {a.name}   ({len(seen)} queries scored\"\n          + (f\", {missing} predictions had no matching query\" if missing else \"\") + \")\")\n    if not a.sets:\n        print(\"    CompleteSet@k is 0 everywhere: --sets not given\")\n    if not a.bge:\n        print(\"    similar/dissimilar slices skipped: --bge not given\")\n    print(\"| slice | n | \" + \" | \".join(cols) + \" |\")\n    print(\"|---|--:|\" + \"--:|\" * len(cols))\n    result = {}\n    for nm in order:\n        rows = slices.get(nm)\n        if not rows:\n            continue\n        agg = {c: sum(r[c] for r in rows) / len(rows) for c in cols}\n        result[nm] = {\"n\": len(rows), **agg}\n        print(f\"| {nm} | {len(rows)} | \" + \" | \".join(f\"{agg[c]:.4f}\" for c in cols) + \" |\")\n\n    if a.gold_strata:\n        if not gold_rows:\n            print(\"\\n--gold-strata: no gold_strata on any query; nothing to report\")\n        else:\n            by = defaultdict(list)\n            for st, r in gold_rows:\n                by[st].append(r)\n                by[\"all\"].append(r)\n            gk = [1, 5, 10, 20, 100]\n            print(f\"\\n### per-GOLD recall by stratum   \"\n                  f\"(denominator is golds, not queries; {len(gold_rows):,} golds)\")\n            print(\"| stratum | golds | \" + \" | \".join(f\"R@{k}\" for k in gk) + \" | median rank |\")\n            print(\"|---|--:|\" + \"--:|\" * (len(gk) + 1))\n            gs_out = {}\n            for st in (\"all\", \"same\", \"cross\", \"ambiguous\", \"unmatched\",\n                       \"unknown\", \"unlabelled\"):\n                rs = by.get(st)\n                if not rs:\n                    continue\n                row = {f\"recall@{k}\": sum(r <= k for r in rs) / len(rs) for k in gk}\n                med = sorted(rs)[len(rs) // 2]\n                gs_out[st] = {\"n_golds\": len(rs), **row,\n                              \"median_rank\": None if med >= BIG else med}\n                print(f\"| {st} | {len(rs)} | \"\n                      + \" | \".join(f\"{row[f'recall@{k}']:.4f}\" for k in gk)\n                      + f\" | {'>corpus' if med >= BIG else med} |\")\n            result[\"_per_gold\"] = gs_out\n            print(\"NOTE: this table's rows are golds, so it does NOT sum to the \"\n                  \"query-level table above.\")\n\n    if a.json_out:\n        json.dump(result, open(a.json_out, \"w\"), indent=1)\n        print(f\"\\nwrote {a.json_out}\")\n    if a.per_query_out:\n        json.dump(per_query, open(a.per_query_out, \"w\"), indent=1)\n        print(f\"wrote {a.per_query_out}  ({len(per_query)} queries)\")\n    return 0\n\n\nif __name__ == \"__main__\":\n    raise SystemExit(main())\n", "sir4-retrieval/eval/semantic_scorer.py": "\"\"\"\nsemantic_scorer.py -- controlled comparison of semantic scorer architectures on\nSIR-4. No graph, no reasoner, no fusion.\n\n    arm \"current\"  s = w0 z(r_dir) + w1 z(S/p^beta) + w2 z(M/p^beta)\n                   S = sum_j H_qdj, M = max_j H_qdj      (the handcrafted summaries)\n\n    arm \"attention\"  s = sum_v a_v x_v,  a = softmax_v(MLP([x_v, t_v]))\n                     over the direct view and every valid hypothetical answer, so\n                     the weights sum to 1. The pooled score is a weighted average,\n                     which keeps it on the same scale as the direct view no matter\n                     how many hypothetical answers a query has.\n\n    arm \"gated\"      s = g_dir x_dir + sum_j g_qdj x_qdj,  g = sigmoid(MLP([x, t]))\n                     Independent gates that do not sum to 1, so several strong\n                     views accumulate. Kept selectable via --arms; note the summed\n                     views carry several times the spread of the direct view, so\n                     this arm has to learn the channel balance the other two get\n                     from normalisation.\n\n    arm \"dualsetmlp\" h_j   = [x_j, MLP(x_j)]\n                       u_all = sum_j h_j\n                       u_sel = sum_j softmax_j(tau x_j) h_j\n                       s     = linear([x_dir,u_all,u_sel]) + MLP([x_dir,u_all,u_sel])\n\n                     This is the automatic raw-view scorer: additive pooling\n                     preserves evidence accumulation, a learned positive\n                     temperature provides monotonic selective pooling, and the\n                     final residual MLP is not constrained to a weighted average.\n\nAll trained arms use the SAME selected loss, frozen encoder outputs, fit/dev\nsplit, and development metric. In a comparison run, the only intended change is\nthe scorer architecture.\n\nTHE OBJECTIVE. `--loss operator` (DEFAULT) is cargo_operator.py's exactly, so the\n`current` arm reproduces the scorer as it is fitted everywhere else in the project\nand the only thing varying between arms is the architecture.\n\n`--loss fixed` is the multi-gold correction. The operator objective's denominator\nruns over the whole corpus INCLUDING the query's other golds; TOMATO has one gold\nper query so that is harmless there, but SIR-4 matsci has 3.95, so every step\npushes gold 1 up by pushing golds 2..4 down. Both are available and every output\nfilename carries which one produced it, so the two can be compared rather than\nargued about.\n\nWHY THE HYPOTHETICAL VIEWS SHARE ONE SCALE. Standardising each view separately\nsets every view's spread to exactly 1, which is precisely the quantity that says\nwhether a hypothetical answer discriminates between papers at all. A view that\ngives every paper the same score would arrive at the gate looking as confident as\none that separates them. So the views are CENTRED individually and divided by a\nsingle shared scale, which preserves their relative spreads.\n\nRun (after the operator cell has produced the embedding caches):\n    python3 eval/semantic_scorer.py --dataset sir4_matsci --model /content/qwen3\n\"\"\"\nfrom __future__ import annotations\n\nimport argparse\nimport hashlib\nimport importlib.util\nimport json\nimport math\nimport os\nimport sys\nimport time\n\nimport numpy as np\n# Torch at module level, unlike the rest of this file's function-local imports:\n# DeepSetsScorer must inherit nn.Module at class-definition time so that\n# train()/eval() actually gate its dropout. Every entry point here needs torch\n# within seconds anyway.\nimport torch as _torch\nimport torch.nn as _nn\n\n_ROOT = os.environ.get(\"CARGO_ROOT\") or os.path.expanduser(\"~/Desktop/CARGO\")\nsys.path.insert(0, _ROOT)\nfrom cargo_paths import (add_dataset_arg, banner, corpus_dir, emb_dir,  # noqa: E402\n                         probes_path, set_dataset)\n\nEPS = 1e-6            # popularity floor and normalisation floor, as specified\nKS_REPORT = (1, 3, 5, 10, 25, 100)\n\n\ndef _op_module():\n    \"\"\"Load cargo_operator as a module so the embedding cache keys are IDENTICAL.\n\n    The encoder, the query instruction, the instruction fingerprint and the\n    filename layout all live there. Re-implementing any of them here would\n    produce a cache MISS at best, and at worst a hit on a file built under a\n    different instruction -- which is the same silent-wrong-baseline failure the\n    rest of this pipeline is armoured against.\n    \"\"\"\n    p = f\"{_ROOT}/kg-construction/eval/cargo_operator.py\"\n    spec = importlib.util.spec_from_file_location(\"cargo_operator\", p)\n    m = importlib.util.module_from_spec(spec)\n    spec.loader.exec_module(m)\n    return m\n\n\n# --------------------------------------------------------------------------\n# stage 1: raw semantic measurements, cached on disk\n# --------------------------------------------------------------------------\ndef build_inputs(op, model_name, split, cache_root, force=False):\n    \"\"\"Materialise dense, H, mask and total_S for one split.\n\n    H is [Q, Jmax, D] float16 in a MEMMAP, never a resident tensor: at CS scale it\n    is 5,445 x 8 x 20,203 = 1.8 GB, which is fine on disk and fine to slice a\n    minibatch out of, and not fine to hold on a GPU alongside activations.\n    \"\"\"\n    doc_ids, corpus, queries, probes = op.load_split(split)\n    D, Q = len(doc_ids), len(queries)\n    qids = [q[\"id\"] for q in queries]\n    qkey = hashlib.md5(\"|\".join(qids).encode()).hexdigest()[:8]\n    dockey = hashlib.md5(\"|\".join(doc_ids).encode()).hexdigest()[:8]\n    slug = op.model_slug(model_name)\n    qi = op.query_instruction(model_name)\n\n    # THE CACHE KEY CARRIES EVERY INPUT THAT CHANGES THE NUMBERS. Dataset and\n    # split alone are what the rest of this repo used to key on, and a matrix\n    # from the wrong corpus passes a row-count check. Document ORDER is in here\n    # too: the arrays are column-indexed by it, so a reordered corpus would\n    # misalign every score with nothing to show for it.\n    meta = {\"dataset\": os.path.basename(corpus_dir(split)).rsplit(\"_\", 1)[0],\n            \"split\": split, \"encoder\": model_name, \"query_instruct\": qi,\n            \"qkey\": qkey, \"dockey\": dockey, \"Q\": Q, \"D\": D}\n    os.makedirs(cache_root, exist_ok=True)\n    tag = f\"{split}_{qkey}_{dockey}{slug}\"\n    mpath = f\"{cache_root}/semantic_inputs_{tag}.json\"\n    hpath = f\"{cache_root}/semantic_H_{tag}.f16\"\n\n    if os.path.exists(mpath) and not force:\n        got = json.load(open(mpath))\n        stale = {k: (got.get(k), v) for k, v in meta.items() if got.get(k) != v}\n        assert not stale, f\"stale semantic cache {mpath}: {stale}\"\n        assert os.path.exists(hpath), f\"manifest without matrix: {hpath}\"\n        H = np.memmap(hpath, np.float16, \"r\", shape=(Q, got[\"Jmax\"], D))\n        z = np.load(f\"{cache_root}/semantic_side_{tag}.npz\", allow_pickle=True)\n        print(f\"[sem] loaded cached inputs {tag}: H{H.shape} f16\")\n        return dict(H=H, dense=z[\"dense\"], mask=z[\"mask\"], total_S=z[\"total_S\"],\n                    doc_ids=doc_ids, queries=queries, meta=got)\n\n    # Encode only on a genuine cache miss.  Importing sentence-transformers can\n    # initialise a multi-gigabyte model stack, so a cache-only experiment must\n    # not require it (and must not fail merely because that optional stack is\n    # unavailable on the evaluation machine).\n    dpath = f\"{emb_dir()}/{split}_doc{slug}.npy\"\n    qpath = f\"{emb_dir()}/{split}_query_{qkey}{slug}{op.qi_tag(qi)}.npy\"\n    ppath = f\"{emb_dir()}/{split}_probe_{qkey}{slug}.npy\"\n    flat, own = [], []\n    for i, q in enumerate(queries):\n        for pr in probes.get(q[\"id\"], []):\n            flat.append(pr)\n            own.append(i)\n    if not all(os.path.exists(p) for p in (dpath, qpath, ppath)):\n        from sentence_transformers import SentenceTransformer\n        print(f\"[sem] embeddings absent for {split}; encoding with {model_name}\")\n        enc = SentenceTransformer(model_name)\n        enc.max_seq_length = 512\n        op.cached_encode(enc, [corpus[d] for d in doc_ids], f\"{split}_doc{slug}\")\n        op.cached_encode(enc, [q[\"question\"] for q in queries],\n                         f\"{split}_query_{qkey}{slug}{op.qi_tag(qi)}\", instruct=qi)\n        op.cached_encode(enc, flat, f\"{split}_probe_{qkey}{slug}\")\n    de = np.load(dpath).astype(np.float32)\n    qe = np.load(qpath).astype(np.float32)\n    pe = np.load(ppath).astype(np.float32)\n    assert de.shape[0] == D and qe.shape[0] == Q and pe.shape[0] == len(flat), (\n        f\"embedding rows {de.shape[0]}/{qe.shape[0]}/{pe.shape[0]} != {D}/{Q}/{len(flat)}\")\n\n    own = np.asarray(own)\n    counts = np.bincount(own, minlength=Q) if len(own) else np.zeros(Q, int)\n    Jmax = int(counts.max()) if len(own) else 1\n    meta[\"Jmax\"] = Jmax\n    print(f\"[sem] {split}: Q={Q} D={D} Jmax={Jmax} \"\n          f\"(views/query min {counts.min()} mean {counts.mean():.1f})\")\n    assert counts.min() > 0, \"a query has no hypothetical answers; regenerate probes\"\n\n    # dense in row chunks -- one Q x D float32 is 440 MB at CS scale\n    dense = np.empty((Q, D), np.float16)\n    for i in range(0, Q, 512):\n        dense[i:i + 512] = (qe[i:i + 512] @ de.T).astype(np.float16)\n\n    H = np.memmap(hpath, np.float16, \"w+\", shape=(Q, Jmax, D))\n    mask = np.zeros((Q, Jmax), bool)\n    total_S = np.zeros(D, np.float64)\n    t0 = time.time()\n    for i in range(Q):\n        idx = np.where(own == i)[0]\n        h = np.clip(pe[idx] @ de.T, 0, None).astype(np.float32)      # [j_i, D]\n        H[i, :len(idx)] = h.astype(np.float16)\n        mask[i, :len(idx)] = True\n        total_S += h.sum(0)                       # popularity over the WHOLE split\n        if i % 500 == 0:\n            print(f\"  [sem] {split} {i}/{Q}  ({time.time() - t0:.0f}s)\", flush=True)\n    H.flush()\n\n    np.savez_compressed(f\"{cache_root}/semantic_side_{tag}.npz\",\n                        dense=dense, mask=mask, total_S=total_S.astype(np.float32))\n    json.dump(meta, open(mpath, \"w\"), indent=1)\n    print(f\"[sem] wrote {hpath} ({os.path.getsize(hpath)/1e6:.0f} MB)\")\n    return dict(H=np.memmap(hpath, np.float16, \"r\", shape=(Q, Jmax, D)),\n                dense=dense, mask=mask, total_S=total_S.astype(np.float32),\n                doc_ids=doc_ids, queries=queries, meta=meta)\n\n\ndef gold_matrix(queries, doc_ids):\n    \"\"\"Padded gold indices + validity mask, and a per-query python list.\"\"\"\n    pos = {d: i for i, d in enumerate(doc_ids)}\n    lists = []\n    for q in queries:\n        gs = [pos[g] for g in (q.get(\"supporting_documents\") or []) if g in pos]\n        lists.append(gs)\n    G = max(1, max(len(g) for g in lists))\n    idx = np.zeros((len(lists), G), np.int64)\n    val = np.zeros((len(lists), G), np.float32)\n    for i, gs in enumerate(lists):\n        idx[i, :len(gs)] = gs\n        val[i, :len(gs)] = 1.0\n    return idx, val, lists\n\n\n# --------------------------------------------------------------------------\n# stage 2: the two architectures\n# --------------------------------------------------------------------------\nclass Pop:\n    \"\"\"Per-document popularity for the anti-hub division, in one of three modes.\n\n    loo        (total_S - S) per query: the historical estimate. TRANSDUCTIVE --\n               a test paper's popularity is computed from the OTHER test queries'\n               hypothetical answers, so it cannot run on one query alone.\n    bank       mean ReLU cosine against the TRAIN-split answer bank. Query-\n               independent: one [D] vector per corpus, computable for unseen\n               papers from their embedding plus a frozen train artifact.\n    predicted  a small MLP distilled from the bank targets. Fully local: needs\n               only the paper's own embedding at inference.\n\n    The object rides in the argument slot that used to carry total_S, so every\n    scorer stays a pure function of (H, mask, dense, pop).\n    \"\"\"\n\n    def __init__(self, mode, total_S=None, vec=None, predictor=None,\n                 doc_emb=None, target=None):\n        self.mode, self.total_S, self.vec = mode, total_S, vec\n        # `joint` keeps the predictor LIVE inside the scoring path, so the\n        # retrieval gradient reaches it. `predicted` freezes its output to a\n        # vector instead. Same network, different training regime.\n        self.predictor, self.doc_emb, self.target = predictor, doc_emb, target\n\n    def parameters(self):\n        \"\"\"Predictor parameters, so the trainer can optimise them jointly.\"\"\"\n        return list(self.predictor.parameters()) if self.mode == \"joint\" else []\n\n    def state(self):\n        \"\"\"Predictor weights, or None. Part of the checkpoint under joint training.\n\n        The scorer and the predictor are ONE model: restoring the scorer to its\n        best epoch while the predictor sits at whatever the last epoch left it\n        would evaluate a pair that never existed during training.\n        \"\"\"\n        if self.mode != \"joint\":\n            return None\n        return {k: v.detach().cpu().clone() for k, v in self.predictor.state_dict().items()}\n\n    def load_state(self, st):\n        if st is None or self.mode != \"joint\":\n            return\n        dev = next(self.predictor.parameters()).device\n        self.predictor.load_state_dict({k: v.to(dev) for k, v in st.items()})\n\n    def aux_loss(self):\n        \"\"\"Anchor p_hat to the bank targets in log space.\n\n        Without it the retrieval loss is free to repurpose the predictor as extra\n        scorer capacity -- it would stop meaning \"how generally matchable is this\n        paper\" and start meaning \"whatever lowers the ranking loss\", which is a\n        different model wearing the same name. The log keeps a handful of very\n        popular papers from dominating.\n        \"\"\"\n        if self.mode != \"joint\" or self.target is None:\n            return 0.0\n        p = self.predictor(self.doc_emb)\n        return ((_torch.log(EPS + p) - _torch.log(EPS + self.target)) ** 2).mean()\n\n    @staticmethod\n    def from_data(data, device, variant=\"\"):\n        \"\"\"`variant` selects an alternative popularity stored on the same split.\n\n        The mlp arm carries its own LEARNED popularity as part of the proposal, so\n        one run needs two sources live at once: the baseline's leave-one-out and\n        the predictor's. They are stored side by side rather than in two runs.\n        \"\"\"\n        vk = f\"pop_vec{variant}\"\n        mk = f\"pop_mode{variant}\"\n        if data.get(mk, \"loo\") == \"loo\":\n            return Pop(\"loo\", total_S=_torch.as_tensor(data[\"total_S\"], device=device))\n        return Pop(data[mk], vec=_torch.as_tensor(data[vk], device=device).float())\n\n    def per_query(self, S):\n        \"\"\"[B, D] popularity, given the query's own answer-sum S [B, D].\"\"\"\n        if self.mode == \"loo\":\n            return (self.total_S.unsqueeze(0) - S).clamp_min(EPS)\n        if self.mode == \"joint\":\n            # Recomputed every step and DIFFERENTIABLE: this is the whole point.\n            return self.predictor(self.doc_emb).clamp_min(EPS).unsqueeze(0).expand_as(S)\n        # Query-independent: the same vector for every query. Includes the\n        # query's own answers when the query came from the train split, which is\n        # the definition of the bank -- one contribution among thousands.\n        return self.vec.unsqueeze(0).clamp_min(EPS).expand_as(S)\n\n\ndef _views(H, mask, dense, pop, beta):\n    \"\"\"Popularity adjustment + the two normalisations. Returns x_dir, x_hyp, Ht.\n\n    H     [B, J, D] float32   raw ReLU'd hypothetical-answer matches\n    mask  [B, J]    float32   1 for a real answer, 0 for padding\n    dense [B, D]    float32   direct question-paper cosine\n    \"\"\"\n    import torch\n    m3 = mask.unsqueeze(-1)                                   # [B, J, 1]\n    S = (H * m3).sum(1)                                       # [B, D]\n    p = pop.per_query(S)\n    Ht = H / p.unsqueeze(1).pow(beta)\n    Ht = Ht * m3                                              # padding contributes 0\n\n    D = H.shape[-1]\n    nj = mask.sum(1).clamp_min(1.0)                           # [B] real answers\n    mu = Ht.sum(-1, keepdim=True) / D                         # [B, J, 1]\n    c = (Ht - mu) * m3                                        # centre each answer\n    # ONE shared scale across this query's valid answers, so a view that barely\n    # separates papers stays narrow instead of being inflated to spread 1.\n    sig = torch.sqrt((c * c).sum((1, 2)) / (nj * D) + EPS)    # [B]\n    x_hyp = c / (sig.view(-1, 1, 1) + EPS)\n    x_dir = (dense - dense.mean(1, keepdim=True)) / (dense.std(1, keepdim=True) + EPS)\n    return x_dir, x_hyp * m3, Ht\n\n\ndef _set_views(H, mask, dense, pop, beta):\n    \"\"\"Return direct and hypothetical-answer inputs for learned set pooling.\n\n    Unlike ``_views``, this uses one mean and one scale for the complete\n    hypothetical-answer set of a query.  Consequently an answer that barely\n    separates papers stays weak, and relative offsets between answers are not\n    erased before the set network sees them.\n    \"\"\"\n    import torch\n    m3 = mask.unsqueeze(-1)\n    S = (H * m3).sum(1)\n    p = pop.per_query(S)\n    adjusted = H / p.unsqueeze(1).pow(beta)\n    adjusted = adjusted * m3\n\n    D = H.shape[-1]\n    n = (mask.sum(1) * D).clamp_min(1.0)\n    mu = adjusted.sum((1, 2)) / n\n    centred = (adjusted - mu.view(-1, 1, 1)) * m3\n    scale = torch.sqrt((centred * centred).sum((1, 2)) / n + EPS)\n    x_hyp = centred / (scale.view(-1, 1, 1) + EPS)\n    x_dir = (dense - dense.mean(1, keepdim=True)) / (dense.std(1, keepdim=True) + EPS)\n    return x_dir, x_hyp * m3\n\n\nclass MatchabilityPredictor(_nn.Module):\n    \"\"\"Query-Independent Document Matchability Estimator.\n\n        p_hat(d) = softplus( g_eta(E(d)) ),   g_eta: dim -> hidden -> 1\n\n    Distils the train-answer bank into a function of the paper embedding alone,\n    so popularity at inference needs nothing but the paper itself: no other test\n    queries, no bank matmul, no lookup table. Fitted on log targets so a few\n    extremely popular papers cannot dominate the loss.\n    \"\"\"\n\n    def __init__(self, dim, hidden=64):\n        super().__init__()\n        self.net = _nn.Sequential(_nn.Linear(dim, hidden), _nn.GELU(),\n                                  _nn.Linear(hidden, 1))\n\n    def forward(self, E):\n        return _nn.functional.softplus(self.net(E)).squeeze(-1)\n\n\ndef bank_popularity(bank_emb, doc_emb, chunk=4096):\n    \"\"\"p_bank[d] = mean over bank answers of ReLU(cos(h, d)). Pure numpy, chunked.\"\"\"\n    out = np.zeros(doc_emb.shape[0], np.float64)\n    for i in range(0, bank_emb.shape[0], chunk):\n        out += np.clip(bank_emb[i:i + chunk] @ doc_emb.T, 0, None).sum(0)\n    return (out / max(bank_emb.shape[0], 1)).astype(np.float32)\n\n\ndef fit_matchability(doc_emb, targets, device, seed=0, hidden=64, lr=1e-3,\n                     weight_decay=1e-2, epochs=200, patience=10):\n    \"\"\"Fit p_hat to the bank targets with log-MSE, early-stopped on held-out docs.\"\"\"\n    N = doc_emb.shape[0]\n    rng = np.random.default_rng(seed)\n    perm = rng.permutation(N)\n    n_val = max(1, N // 10)\n    va, fi = perm[:n_val], perm[n_val:]\n    X = _torch.as_tensor(doc_emb, device=device)\n    y = _torch.log(EPS + _torch.as_tensor(targets, device=device))\n    m = MatchabilityPredictor(doc_emb.shape[1], hidden).to(device)\n    groups = [{\"params\": [p for p in m.parameters() if p.ndim >= 2],\n               \"weight_decay\": weight_decay},\n              {\"params\": [p for p in m.parameters() if p.ndim < 2],\n               \"weight_decay\": 0.0}]\n    opt = _torch.optim.AdamW(groups, lr=lr)\n    best, best_sd, stale = float(\"inf\"), None, 0\n    for ep in range(epochs):\n        m.train()\n        for bi in range(0, len(fi), 1024):\n            sel = _torch.as_tensor(fi[bi:bi + 1024], device=device)\n            loss = ((_torch.log(EPS + m(X[sel])) - y[sel]) ** 2).mean()\n            opt.zero_grad(); loss.backward(); opt.step()\n        m.eval()\n        with _torch.no_grad():\n            v = float(((_torch.log(EPS + m(X[va])) - y[va]) ** 2).mean())\n        if v < best - 1e-6:\n            best, best_sd, stale = v, {k: t.clone() for k, t in m.state_dict().items()}, 0\n        else:\n            stale += 1\n            if stale >= patience:\n                break\n    m.load_state_dict(best_sd)\n    m.eval()\n    with _torch.no_grad():\n        pred = m(X).cpu().numpy()\n        corr = float(np.corrcoef(np.log(EPS + pred), np.log(EPS + targets))[0, 1])\n    print(f\"[matchability] fitted on {len(fi)} docs, held-out {n_val}: \"\n          f\"log-MSE {best:.4f}, log-corr {corr:.3f} (ep {ep + 1})\")\n    return m, {\"heldout_logmse\": round(best, 6), \"log_corr\": round(corr, 4),\n               \"epochs_run\": ep + 1, \"hidden\": hidden}\n\n\nclass CurrentScorer:\n    \"\"\"w0 z(dense) + w1 z(S/p^b) + w2 z(M/p^b) -- the handcrafted sum and max.\"\"\"\n\n    name = \"current\"\n\n    def __init__(self, device):\n        import torch\n        import torch.nn as nn\n        self.w = nn.Parameter(torch.tensor([1.0, 1.0, 0.2], device=device))\n        self.logbeta = nn.Parameter(torch.zeros((), device=device))\n\n    def parameters(self):\n        return [self.w, self.logbeta]\n\n    def state(self):\n        return {\"w\": self.w.detach().cpu().tolist(),\n                \"beta\": float(self.logbeta.detach().exp().cpu())}\n\n    def load(self, st):\n        import torch\n        with torch.no_grad():\n            self.w.copy_(torch.tensor(st[\"w\"], device=self.w.device))\n            self.logbeta.copy_(torch.tensor(math.log(st[\"beta\"]), device=self.w.device))\n\n    def __call__(self, H, mask, dense, pop):\n        import torch\n        beta = self.logbeta.exp()\n        m3 = mask.unsqueeze(-1)\n        S_raw = (H * m3).sum(1)\n        degb = pop.per_query(S_raw).pow(beta)\n        # max over REAL answers only: padding is 0 and every H is >= 0, so a\n        # padded slot would silently act as a floor of zero on an all-zero row.\n        M_raw = H.masked_fill(m3 == 0, -1.0).max(1).values.clamp_min(0.0)\n\n        def z(x):\n            return (x - x.mean(1, keepdim=True)) / (x.std(1, keepdim=True) + EPS)\n\n        return (self.w[0] * z(dense) + self.w[1] * z(S_raw / degb)\n                + self.w[2] * z(M_raw / degb))\n\n\nclass DenseScorer:\n    \"\"\"The dense baseline: rank by the query-document cosine alone.\n\n    No hypothetical answers, no popularity term, no parameters, no training. It is\n    here so the table shows what the hypothetical-answer machinery buys over plain\n    query-document similarity in the SAME encoder space -- without it, a reader\n    cannot tell whether `current` and `attention` are close to each other because\n    both are good or because neither adds anything over the encoder.\n    \"\"\"\n\n    name = \"dense\"\n\n    def __init__(self, device):\n        pass\n\n    def parameters(self):\n        return []\n\n    def state(self):\n        return {\"note\": \"no parameters; ranks by cos(E(q), E(d))\"}\n\n    def load(self, st):\n        pass\n\n    def __call__(self, H, mask, dense, pop):\n        return (dense - dense.mean(1, keepdim=True)) / (dense.std(1, keepdim=True) + EPS)\n\nclass DeepSetsScorer(_nn.Module):\n    \"\"\"Compact Deep Sets, ~62 parameters. No sum, no max, no handcrafted summaries.\n\n        z_qdj = phi([x_hyp_qdj, x_dir_qd])        phi: 2 -> 4 -> 4\n        u_qd  = masked mean_j z_qdj               over VALID (and kept) answers\n        s     = rho([x_dir_qd, u_qd])             rho: 5 -> 4 -> 1\n\n    Pairing each answer with the direct score inside phi lets the network encode\n    interactions (\"a strong answer AND a weak direct match\") before pooling, and\n    the masked MEAN keeps u on one scale regardless of how many answers a query\n    has -- the 610-parameter version summed, which tied its output scale to J.\n\n    REGULARISATION IS THE POINT of this version; the large one overfit (best\n    train loss of all arms, worse test).\n      * 15% whole-answer dropout, training only: an entire hypothetical answer is\n        dropped for a query, with the SAME mask for every paper of that query, so\n        the model cannot rely on any single answer existing. The masked mean\n        renormalises over the kept answers, so no 1/(1-p) scaling is needed.\n      * weight decay handled by train_arm (matrices only), dev early stopping,\n        and three seeds handled by the --seeds loop in main.\n\n    THIS CLASS INHERITS nn.Module AND THE TRAINERS CALL train()/eval(). The other\n    scorer classes are plain objects, so dropout inserted there would stay active\n    during evaluation; hasattr guards in train_arm/dev_ndcg/score_rows make the\n    mode switch a no-op for them and real for this one.\n\n    If dropout removes every answer of a query (possible at J=1), u is zero and\n    the score falls back to a function of x_dir alone, which is the right\n    degradation. Permutation invariant: shared phi, symmetric mean.\n    \"\"\"\n\n    name = \"deepsets\"\n\n    def __init__(self, device, hidden=4, p_drop=0.15):\n        super().__init__()\n        self.p_drop = float(p_drop)\n        self.phi = _nn.Sequential(_nn.Linear(2, hidden), _nn.GELU(),\n                                  _nn.Linear(hidden, hidden))\n        self.rho = _nn.Sequential(_nn.Linear(hidden + 1, hidden), _nn.GELU(),\n                                  _nn.Linear(hidden, 1))\n        self.logbeta = _nn.Parameter(_torch.zeros(()))\n        self.to(device)\n\n    def state(self):\n        return {\"sd\": {k: v.detach().cpu().tolist() for k, v in self.state_dict().items()},\n                \"beta\": float(self.logbeta.detach().exp().cpu()),\n                \"p_drop\": self.p_drop}\n\n    def load(self, st):\n        import torch\n        dev = self.logbeta.device\n        self.load_state_dict({k: torch.tensor(v, device=dev) for k, v in st[\"sd\"].items()})\n\n    def forward(self, H, mask, dense, pop):\n        import torch\n        x_dir, x_hyp, _ = _views(H, mask, dense, pop, self.logbeta.exp())\n        B, J, D = x_hyp.shape\n        m = mask\n        if self.training and self.p_drop > 0:\n            # One mask per (query, answer), shared across all D papers: the unit\n            # being dropped is the ANSWER, not a (paper, answer) cell.\n            keep = (torch.rand(B, J, device=m.device) >= self.p_drop).float()\n            m = m * keep\n        inp = torch.stack([x_hyp, x_dir.unsqueeze(1).expand(B, J, D)], -1)  # [B,J,D,2]\n        z = self.phi(inp) * m.view(B, J, 1, 1)                              # [B,J,D,h]\n        u = z.sum(1) / m.sum(1).clamp(min=1.0).view(B, 1, 1)                # masked mean\n        feat = torch.cat([x_dir.unsqueeze(-1), u], -1)                      # [B,D,h+1]\n        return self.rho(feat).squeeze(-1)\n\n\nclass SetMLPScorer(_nn.Module):\n    \"\"\"A small, automatic, permutation-invariant semantic scorer.\n\n    For every adjusted hypothetical-answer match x_j, one shared MLP produces a\n    learned representation.  Summing these representations is the standard\n    Deep Sets invariant; it is not a precomputed semantic feature.  A final\n    residual MLP maps the pooled set and direct query match to one paper score.\n\n    The first per-view channel is an identity path.  It gives gradients a stable\n    route and makes the initial model a strong direct-plus-pooled retriever.  All\n    output weights, the nonlinear set features, and beta remain trainable.  No\n    maximum, top-k statistic, rank position, or manually weighted operator input\n    is computed.\n    \"\"\"\n\n    name = \"setmlp\"\n\n    def __init__(self, device, hidden=8, p_drop=0.0):\n        super().__init__()\n        hidden = int(hidden)\n        assert hidden >= 2\n        self.hidden = hidden\n        self.p_drop = float(p_drop)\n        # h-1 nonlinear channels plus one unmodified identity channel.\n        self.phi = _nn.Sequential(\n            _nn.Linear(1, hidden), _nn.GELU(), _nn.Linear(hidden, hidden - 1))\n        self.linear = _nn.Linear(hidden + 1, 1, bias=False)\n        self.residual = _nn.Sequential(\n            _nn.Linear(hidden + 1, hidden), _nn.GELU(),\n            _nn.Dropout(0.10), _nn.Linear(hidden, 1))\n        self.logbeta = _nn.Parameter(_torch.zeros(()))\n\n        # Stable residual learning: epoch zero is direct + the identity set\n        # channel.  This is an initialisation only; every coefficient is learned.\n        with _torch.no_grad():\n            self.linear.weight.zero_()\n            self.linear.weight[0, 0] = 1.0       # direct query view\n            self.linear.weight[0, 1] = 1.0       # pooled identity view\n            self.residual[-1].weight.zero_()\n            self.residual[-1].bias.zero_()\n        self.to(device)\n\n    def state(self):\n        return {\"sd\": {k: v.detach().cpu().tolist() for k, v in self.state_dict().items()},\n                \"beta\": float(self.logbeta.detach().exp().cpu()),\n                \"hidden\": self.hidden, \"p_drop\": self.p_drop}\n\n    def load(self, st):\n        dev = self.logbeta.device\n        self.load_state_dict({k: _torch.tensor(v, device=dev) for k, v in st[\"sd\"].items()})\n\n    def forward(self, H, mask, dense, pop):\n        import torch\n        x_dir, x_hyp = _set_views(H, mask, dense, pop, self.logbeta.exp())\n        B, J, D = x_hyp.shape\n        m = mask\n        if self.training and self.p_drop > 0:\n            keep = (torch.rand(B, J, device=m.device) >= self.p_drop).float()\n            # Never erase the entire set: keep the first valid answer if a rare\n            # all-dropped row occurs.  The rule is independent of paper scores.\n            empty = (keep * m).sum(1) == 0\n            if empty.any():\n                first = m.float().argmax(1)\n                keep[empty, first[empty]] = 1.0\n            m = m * keep\n\n        nonlinear = self.phi(x_hyp.unsqueeze(-1))\n        per_view = torch.cat([x_hyp.unsqueeze(-1), nonlinear], -1)\n        pooled = (per_view * m.view(B, J, 1, 1)).sum(1)              # [B,D,h]\n\n        # Calibrate every learned pooled channel across this query's candidate\n        # corpus.  This prevents a high-variance channel from winning merely by\n        # scale and gives the final MLP comparable inputs.\n        pooled = ((pooled - pooled.mean(1, keepdim=True)) /\n                  (pooled.std(1, keepdim=True) + EPS))\n        feat = torch.cat([x_dir.unsqueeze(-1), pooled], -1)\n        return (self.linear(feat) + self.residual(feat)).squeeze(-1)\n\n\nclass DualSetMLPScorer(_nn.Module):\n    \"\"\"Set MLP with complementary additive and learned-selective pooling.\n\n    ``u_all`` accumulates evidence from every answer. ``u_sel`` uses a learned\n    positive temperature and a softmax *inside the set representation* to focus\n    on informative answers.  The final score is unconstrained: unlike view-attention as the\n    scorer itself, it is not a convex average and therefore does not cap\n    evidence accumulation.  Both pools are permutation invariant and neither\n    computes a handcrafted maximum.\n    \"\"\"\n\n    name = \"dualsetmlp\"\n\n    def __init__(self, device, hidden=8, p_drop=0.0):\n        super().__init__()\n        hidden = int(hidden)\n        assert hidden >= 2\n        self.hidden = hidden\n        self.p_drop = float(p_drop)\n        self.phi = _nn.Sequential(\n            _nn.Linear(1, hidden), _nn.GELU(), _nn.Linear(hidden, hidden - 1))\n        # A single positive temperature is enough to learn the continuum from\n        # broad averaging (small tau) to strongest-view selection (large tau).\n        # Monotonic selection generalises better than another free MLP here.\n        self.logtau = _nn.Parameter(_torch.tensor(math.log(5.0)))\n        width = 1 + 2 * hidden\n        self.linear = _nn.Linear(width, 1, bias=False)\n        self.residual = _nn.Sequential(\n            _nn.Linear(width, hidden), _nn.GELU(), _nn.Dropout(0.10),\n            _nn.Linear(hidden, 1))\n        self.logbeta = _nn.Parameter(_torch.zeros(()))\n        with _torch.no_grad():\n            self.linear.weight.zero_()\n            self.linear.weight[0, 0] = 1.0\n            self.linear.weight[0, 1] = 1.0\n            self.residual[-1].weight.zero_()\n            self.residual[-1].bias.zero_()\n        self.to(device)\n\n    def state(self):\n        return {\"sd\": {k: v.detach().cpu().tolist() for k, v in self.state_dict().items()},\n                \"beta\": float(self.logbeta.detach().exp().cpu()),\n                \"hidden\": self.hidden, \"p_drop\": self.p_drop}\n\n    def load(self, st):\n        dev = self.logbeta.device\n        self.load_state_dict({k: _torch.tensor(v, device=dev) for k, v in st[\"sd\"].items()})\n\n    def forward(self, H, mask, dense, pop):\n        import torch\n        x_dir, x_hyp = _set_views(H, mask, dense, pop, self.logbeta.exp())\n        B, J, D = x_hyp.shape\n        m = mask\n        if self.training and self.p_drop > 0:\n            keep = (torch.rand(B, J, device=m.device) >= self.p_drop).float()\n            empty = (keep * m).sum(1) == 0\n            if empty.any():\n                first = m.float().argmax(1)\n                keep[empty, first[empty]] = 1.0\n            m = m * keep\n\n        nonlinear = self.phi(x_hyp.unsqueeze(-1))\n        per_view = torch.cat([x_hyp.unsqueeze(-1), nonlinear], -1)\n        valid = m.view(B, J, 1)\n        u_all = (per_view * valid.unsqueeze(-1)).sum(1)\n\n        logits = self.logtau.exp() * x_hyp\n        logits = logits.masked_fill(valid == 0, -float(\"inf\"))\n        attn = torch.softmax(logits, 1)\n        u_sel = (attn.unsqueeze(-1) * per_view).sum(1)\n\n        def corpus_z(u):\n            return (u - u.mean(1, keepdim=True)) / (u.std(1, keepdim=True) + EPS)\n\n        feat = torch.cat([x_dir.unsqueeze(-1), corpus_z(u_all), corpus_z(u_sel)], -1)\n        return (self.linear(feat) + self.residual(feat)).squeeze(-1)\n\n\nclass AttentionScorer:\n    \"\"\"The thesis method (Ch.6 Eq 6.10-6.12), exactly as written.\n\n        l_v = f_phi([x_v, t_v])         one shared MLP, t=0 direct, t=1 hypothetical\n        a_v = softmax over ALL J+1 views (padded answers masked to -inf)\n        s   = sum_v a_v x_v\n\n    No temperature, no warm start, no separate channels: this arm exists to test\n    the written method as written. Views are normalised by _views(), i.e. the\n    shared-scale refinement (centre each answer, one scale per query) that the\n    method adopted in place of per-view standardisation.\n\n    Known structural properties, stated so the result is readable:\n      * the score is a weighted average, so it cannot exceed the paper's own\n        largest view -- evidence cannot accumulate across views;\n      * averaging J partly-independent views shrinks the hypothetical mass's\n        spread well below x_dir's, so the direct view tends to dominate.\n\n    Permutation invariant: one MLP scores every view, softmax and the weighted\n    sum are symmetric. The direct view is always valid, so no row is fully masked.\n    \"\"\"\n\n    name = \"attention\"\n\n    def __init__(self, device, hidden=8):\n        import torch\n        import torch.nn as nn\n        self.mlp = nn.Sequential(nn.Linear(2, hidden), nn.GELU(),\n                                 nn.Linear(hidden, 1)).to(device)\n        self.logbeta = nn.Parameter(torch.zeros((), device=device))\n\n    def parameters(self):\n        return list(self.mlp.parameters()) + [self.logbeta]\n\n    def state(self):\n        return {\"mlp\": {k: v.detach().cpu().tolist() for k, v in self.mlp.state_dict().items()},\n                \"beta\": float(self.logbeta.detach().exp().cpu())}\n\n    def load(self, st):\n        import torch\n        dev = self.logbeta.device\n        self.mlp.load_state_dict({k: torch.tensor(v, device=dev)\n                                  for k, v in st[\"mlp\"].items()})\n        with torch.no_grad():\n            self.logbeta.copy_(torch.tensor(math.log(st[\"beta\"]), device=dev))\n\n    def __call__(self, H, mask, dense, pop):\n        import torch\n        x_dir, x_hyp, _ = _views(H, mask, dense, pop, self.logbeta.exp())\n        B, J, D = x_hyp.shape\n        dev = mask.device\n        views = torch.cat([x_dir.unsqueeze(1), x_hyp], 1)            # [B, 1+J, D]\n        vmask = torch.cat([torch.ones(B, 1, device=dev), mask], 1)\n        t = torch.cat([torch.zeros(1, device=dev),\n                       torch.ones(J, device=dev)]).view(1, 1 + J, 1)\n        inp = torch.stack([views, t.expand(B, 1 + J, D)], -1)        # [B, 1+J, D, 2]\n        logit = self.mlp(inp).squeeze(-1)                            # [B, 1+J, D]\n        logit = logit.masked_fill(vmask.unsqueeze(-1) == 0, -float(\"inf\"))\n        a = torch.softmax(logit, 1)\n        return (a * views).sum(1)\n\n\nclass SortedMLPScorer:\n    \"\"\"One MLP on the SORTED vector of view scores. No sum, no max, no attention.\n\n        input  = [ x_dir , sort_desc(x_hyp_1..J) padded to jmax , n_valid/jmax ]\n        s      = MLP(input)\n\n    NOTHING IS HANDCRAFTED. Sorting is not a summary statistic, it is a\n    canonical ordering: it makes the input permutation invariant by construction\n    while throwing away nothing. The MLP then learns whatever function of the\n    score distribution it wants -- how much the best answer counts, whether the\n    second one matters, whether the gap between them matters, how many answers\n    need to fire. `sum` and `max` are not computed anywhere.\n\n    IT GENERALISES SUMMARY-BASED SCORING, WHICH IS NOT THE SAME AS CONTAINING IT.\n    `current` is w0 x_dir + w1 (sum_j x_hyp) + w2 (max_j x_hyp), and on this input\n    the sum is the sum of the sorted coordinates, the max is the first sorted\n    coordinate, and x_dir is coordinate 0 -- so a linear readout gets close. But\n    `_views` centres each hypothetical answer SEPARATELY before the shared scale,\n    so the max channel here is the largest CENTRED value, which is not the same\n    quantity `current` computes from raw matches. The honest claim is that this\n    arm exposes the complete ordered match profile instead of two fixed summaries\n    of it, not that it reproduces the baseline exactly.\n\n    NO SPREAD PROBLEM. Every coordinate gets its own free weight, so the model can\n    set any balance between the direct match and the answers. Both previous arms\n    failed on exactly this: gated summed the answers and they overwhelmed x_dir,\n    attention averaged them and x_dir overwhelmed them.\n\n    Padded slots sort to the end and are zeroed, and the valid count is supplied as\n    a feature, so a query with three answers and one with eight are distinguishable\n    without the padding masquerading as evidence.\n    \"\"\"\n\n    name = \"mlp\"\n\n    def __init__(self, device, jmax, hidden=16):\n        import torch\n        import torch.nn as nn\n        self.jmax = int(jmax)\n        self.net = nn.Sequential(nn.Linear(self.jmax + 2, hidden), nn.GELU(),\n                                 nn.Linear(hidden, 1)).to(device)\n        self.logbeta = nn.Parameter(torch.zeros((), device=device))\n\n    def parameters(self):\n        return list(self.net.parameters()) + [self.logbeta]\n\n    def state(self):\n        return {\"net\": {k: v.detach().cpu().tolist() for k, v in self.net.state_dict().items()},\n                \"jmax\": self.jmax,\n                \"beta\": float(self.logbeta.detach().exp().cpu())}\n\n    def load(self, st):\n        import torch\n        dev = self.logbeta.device\n        self.net.load_state_dict({k: torch.tensor(v, device=dev)\n                                  for k, v in st[\"net\"].items()})\n        with torch.no_grad():\n            self.logbeta.copy_(torch.tensor(math.log(st[\"beta\"]), device=dev))\n\n    def __call__(self, H, mask, dense, pop):\n        import torch\n        x_dir, x_hyp, _ = _views(H, mask, dense, pop, self.logbeta.exp())\n        B, J, D = x_hyp.shape\n        dev = x_hyp.device\n\n        # Invalid answers sort to the end, then get zeroed by position. Comparing\n        # against the fill value to find them would be fragile; the count is exact.\n        fill = torch.finfo(x_hyp.dtype).min / 4\n        xs, _ = torch.sort(x_hyp.masked_fill(mask.unsqueeze(-1) == 0, fill),\n                           dim=1, descending=True)\n        nv = mask.sum(1)                                            # [B]\n        xs = xs * (torch.arange(J, device=dev).view(1, J) < nv.view(B, 1)).float().unsqueeze(-1)\n\n        # One fixed input width, so a split whose Jmax differs cannot change the\n        # layer shape. Truncating keeps the HIGHEST scores, which is the useful end.\n        if J < self.jmax:\n            xs = torch.cat([xs, torch.zeros(B, self.jmax - J, D, device=dev)], 1)\n        elif J > self.jmax:\n            xs = xs[:, :self.jmax]\n\n        feat = torch.cat([x_dir.unsqueeze(-1),                       # [B, D, 1]\n                          xs.permute(0, 2, 1),                       # [B, D, jmax]\n                          (nv.view(B, 1, 1) / self.jmax).expand(B, D, 1)], -1)\n        return self.net(feat).squeeze(-1)                            # [B, D]\n\n\nclass GatedScorer:\n    \"\"\"Adaptive gated view pooling. Independent sigmoid gates, no softmax.\"\"\"\n\n    name = \"gated\"\n\n    def __init__(self, device, hidden=8):\n        import torch.nn as nn\n        self.mlp = nn.Sequential(nn.Linear(2, hidden), nn.GELU(),\n                                 nn.Linear(hidden, 1)).to(device)\n        import torch\n        self.logbeta = nn.Parameter(torch.zeros((), device=device))\n\n    def parameters(self):\n        return list(self.mlp.parameters()) + [self.logbeta]\n\n    def state(self):\n        return {\"mlp\": {k: v.detach().cpu().tolist() for k, v in self.mlp.state_dict().items()},\n                \"beta\": float(self.logbeta.detach().exp().cpu())}\n\n    def load(self, st):\n        import torch\n        self.mlp.load_state_dict({k: torch.tensor(v, device=self.logbeta.device)\n                                  for k, v in st[\"mlp\"].items()})\n        with torch.no_grad():\n            self.logbeta.copy_(torch.tensor(math.log(st[\"beta\"]),\n                                            device=self.logbeta.device))\n\n    def __call__(self, H, mask, dense, pop):\n        import torch\n        x_dir, x_hyp, _ = _views(H, mask, dense, pop, self.logbeta.exp())\n        B, J, D = x_hyp.shape\n        views = torch.cat([x_dir.unsqueeze(1), x_hyp], 1)            # [B, 1+J, D]\n        vmask = torch.cat([torch.ones(B, 1, device=mask.device), mask], 1)\n        t = torch.cat([torch.zeros(1, device=mask.device),\n                       torch.ones(J, device=mask.device)]).view(1, 1 + J, 1)\n        inp = torch.stack([views, t.expand(B, 1 + J, D)], -1)        # [B, 1+J, D, 2]\n        g = torch.sigmoid(self.mlp(inp).squeeze(-1))                 # [B, 1+J, D]\n        # SUM, not a weighted average: gates do not sum to 1, so a paper that\n        # matches several views well accumulates past one that spikes on a single\n        # view. A softmax here would cap every score at its own largest view.\n        return (g * views * vmask.unsqueeze(-1)).sum(1)\n\n\n# --------------------------------------------------------------------------\n# stage 3: the corrected multi-gold loss, and dev nDCG@10\n# --------------------------------------------------------------------------\ndef operator_loss(scores, gold_idx, gold_val):\n    \"\"\"cargo_operator.py's objective, reproduced exactly.\n\n        -torch.log_softmax(S_op, 1)[qi, gi].mean()\n\n    i.e. the mean over all (query, gold) PAIRS of -log_softmax(s)[gold], with the\n    denominator running over the whole corpus. THIS IS THE DEFAULT, so the\n    `current` arm here reproduces the operator as it is actually fitted everywhere\n    else in the project and the comparison changes only the architecture.\n\n    It differs from `multigold_loss` in two ways at once, which is worth knowing\n    when reading a --loss ablation:\n      1. a query's other golds stay inside each gold's denominator;\n      2. weighting is per (query, gold) pair, so a 10-gold query counts ten times\n         as much as a 1-gold one.\n    \"\"\"\n    import torch\n    lp = torch.log_softmax(scores, 1).gather(1, gold_idx)\n    return -(lp * gold_val).sum() / gold_val.sum().clamp_min(1.0)\n\n\ndef multigold_loss(scores, gold_idx, gold_val):\n    \"\"\"-log( exp(s_g) / (exp(s_g) + sum_{d not gold} exp(s_d)) ), averaged.\n\n    Golds are removed from EACH OTHER's denominator, then averaged within a query\n    and equally across queries so a 10-gold query does not outweigh a 1-gold one.\n    \"\"\"\n    import torch\n    # scatter_ADD, not scatter_. Padded gold slots carry index 0, so a query whose\n    # first real gold IS document 0 would have two writes racing at the same cell\n    # and the -inf could be overwritten by the padding's value, silently leaving a\n    # gold in its own denominator. Accumulating avoids the collision entirely.\n    acc = torch.zeros_like(scores)\n    acc.scatter_add_(1, gold_idx, gold_val)\n    neg = scores.masked_fill(acc > 0, -float(\"inf\"))\n    neg_lse = torch.logsumexp(neg, 1)                            # [B]\n    sg = scores.gather(1, gold_idx)                              # [B, G]\n    per_gold = torch.logaddexp(sg, neg_lse.unsqueeze(1)) - sg\n    per_query = (per_gold * gold_val).sum(1) / gold_val.sum(1).clamp_min(1.0)\n    return per_query.mean()\n\n\ndef ndcg_at(order, golds, k):\n    g = set(golds)\n    if not g:\n        return 0.0\n    dcg = sum(1 / math.log2(r + 2) for r, d in enumerate(order[:k]) if d in g)\n    idcg = sum(1 / math.log2(r + 2) for r in range(min(len(g), k)))\n    return dcg / idcg if idcg else 0.0\n\n\ndef batches(n, bs, shuffle=False, rng=None):\n    idx = np.arange(n)\n    if shuffle:\n        rng.shuffle(idx)\n    for i in range(0, n, bs):\n        yield idx[i:i + bs]\n\n\ndef score_rows(model, data, rows, device, qbatch, torch, pop=None):\n    \"\"\"Score a set of queries against the whole corpus, in query minibatches.\"\"\"\n    if hasattr(model, \"eval\"):\n        model.eval()          # dropout must NOT fire at evaluation time\n    out = np.empty((len(rows), data[\"H\"].shape[-1]), np.float32)\n    pop = Pop.from_data(data, device) if pop is None else pop\n    for s in range(0, len(rows), qbatch):\n        sel = rows[s:s + qbatch]\n        H = torch.as_tensor(np.asarray(data[\"H\"][sel], np.float32), device=device)\n        mk = torch.as_tensor(data[\"mask\"][sel].astype(np.float32), device=device)\n        dn = torch.as_tensor(np.asarray(data[\"dense\"][sel], np.float32), device=device)\n        out[s:s + qbatch] = model(H, mk, dn, pop).detach().float().cpu().numpy()\n    return out\n\n\ndef dev_ndcg_multi(model, data, rows, gold_lists, device, qbatch, torch, ks=(10, 100),\n                   pop=None):\n    \"\"\"nDCG at several cutoffs from ONE scoring pass over the dev queries.\n\n    Scoring the dev set is the expensive part of an epoch (every query against the\n    whole corpus), so computing @10 and @100 with two calls would double it for a\n    number that comes free from the same ranking.\n    \"\"\"\n    with torch.no_grad():\n        sc = score_rows(model, data, rows, device, qbatch, torch, pop)\n    acc = {k: [] for k in ks}\n    kmax = max(ks)\n    for i, r in enumerate(rows):\n        if not gold_lists[r]:\n            continue\n        order = list(np.argsort(-sc[i])[:kmax])\n        for k in ks:\n            acc[k].append(ndcg_at(order[:k], gold_lists[r], k))\n    return tuple(float(np.mean(acc[k])) if acc[k] else 0.0 for k in ks)\n\n\ndef dev_ndcg(model, data, rows, gold_lists, device, qbatch, torch, k=10, pop=None):\n    with torch.no_grad():\n        sc = score_rows(model, data, rows, device, qbatch, torch, pop)\n    vals = []\n    for i, r in enumerate(rows):\n        if gold_lists[r]:\n            vals.append(ndcg_at(list(np.argsort(-sc[i])[:k]), gold_lists[r], k))\n    return float(np.mean(vals)) if vals else 0.0\n\n\ndef train_fixed(model, tr, fit_rows, gold_idx, gold_val, device, a, torch,\n                n_epochs, lr=None, seed=None):\n    \"\"\"Train for exactly n_epochs on fit_rows. No dev split, no early stopping.\n\n    Used for the FINAL fit of a k-fold run, where the epoch count has already been\n    chosen by the folds and there is no held-out data left to select on -- which is\n    the point: every train query is in this fit.\n    \"\"\"\n    pop = Pop.from_data(tr, device) if pop is None else pop\n    # Under joint training the popularity predictor is part of the optimised\n    # model, so its matrices belong in the decayed group like any other.\n    trainable = list(model.parameters()) + pop.parameters()\n    groups = [g for g in (\n        {\"params\": [p for p in trainable if p.ndim >= 2],\n         \"weight_decay\": a.weight_decay},\n        {\"params\": [p for p in trainable if p.ndim < 2],\n         \"weight_decay\": 0.0}) if g[\"params\"]]\n    opt = torch.optim.AdamW(groups, lr=a.lr if lr is None else lr)\n    gi = torch.as_tensor(gold_idx, device=device)\n    gv = torch.as_tensor(gold_val, device=device)\n    rng = np.random.default_rng(a.seed if seed is None else seed)\n    lossfn = operator_loss if a.loss == \"operator\" else multigold_loss\n    hist = []\n    for ep in range(n_epochs):\n        if hasattr(model, \"train\"):\n            model.train()\n        losses = []\n        for bi in batches(len(fit_rows), a.qbatch, True, rng):\n            sel = fit_rows[bi]\n            H = torch.as_tensor(np.asarray(tr[\"H\"][sel], np.float32), device=device)\n            mk = torch.as_tensor(tr[\"mask\"][sel].astype(np.float32), device=device)\n            dn = torch.as_tensor(np.asarray(tr[\"dense\"][sel], np.float32), device=device)\n            loss = lossfn(model(H, mk, dn, pop), gi[sel], gv[sel])\n            opt.zero_grad(); loss.backward(); opt.step()\n            losses.append(float(loss.detach()))\n        hist.append({\"epoch\": ep + 1, \"train_loss\": float(np.mean(losses))})\n        print(f\"  [{model.name}] final ep{ep + 1:3d}/{n_epochs} \"\n              f\"train {np.mean(losses):.4f}\", flush=True)\n    if hasattr(model, \"eval\"):\n        model.eval()\n    return hist\n\n\ndef kfold_epochs(build, tr, all_rows, gold_idx, gold_val, gold_lists, device, a,\n                 torch, lr=None, seed=None):\n    \"\"\"Choose the epoch count by K-fold CV over EVERY train query.\n\n    Each query serves in dev exactly once, so the selection signal spans the whole\n    train split rather than one 300-query slice, and every query still contributes\n    to fitting in K-1 of the K folds. Returns the median selected epoch, which is\n    then used for a final fit on all of the data.\n    \"\"\"\n    K = a.kfold\n    rng = np.random.default_rng(a.seed if seed is None else seed)\n    order = all_rows.copy()\n    rng.shuffle(order)\n    folds = np.array_split(order, K)\n    picked = []\n    for k in range(K):\n        dev_k = folds[k]\n        fit_k = np.concatenate([folds[j] for j in range(K) if j != k])\n        m = build()\n        _, _, hist, _, _ = train_arm(m, tr, fit_k, dev_k, gold_idx, gold_val,\n                                     gold_lists, device, a, torch, lr=lr, seed=seed)\n        key = \"dev_loss\" if a.select_on == \"loss\" else \"dev_ndcg@10\"\n        best_ep = (min(hist, key=lambda r: r[key]) if a.select_on == \"loss\"\n                   else max(hist, key=lambda r: r[key]))[\"epoch\"]\n        picked.append(best_ep)\n        print(f\"  [fold {k + 1}/{K}] fit {len(fit_k)} dev {len(dev_k)} \"\n              f\"-> epoch {best_ep}\")\n    E = int(np.median(picked))\n    print(f\"  [kfold] selected epochs per fold {picked} -> median {E}\")\n    return max(E, 1), picked\n\n\ndef dev_loss(model, data, rows, gold_idx, gold_val, device, qbatch, torch, lossfn,\n             pop=None):\n    \"\"\"The training objective evaluated on held-out dev queries.\n\n    WHY SELECT ON THIS RATHER THAN dev nDCG@10. nDCG@10 with binary relevance is a\n    STEP function: a query's score moves only when a gold crosses a rank boundary\n    inside the top 10, so most parameter updates change it by exactly zero and the\n    rest change it in jumps. Taking the best over ~30 epochs x 3 seeds of a chunky\n    signal on the same 300 queries selects the checkpoint that got luckiest on\n    those queries, not the one that generalises -- which is why the physics run\n    inverted: the arm with the WORST dev nDCG won on test, and the arm with the\n    best dev nDCG lost. The loss is continuous, moves every step, and is the\n    quantity actually being optimised. nDCG is still computed and reported.\n    \"\"\"\n    if hasattr(model, \"eval\"):\n        model.eval()\n    pop = Pop.from_data(data, device) if pop is None else pop\n    gi = torch.as_tensor(gold_idx, device=device)\n    gv = torch.as_tensor(gold_val, device=device)\n    tot, n = 0.0, 0\n    with torch.no_grad():\n        for s in range(0, len(rows), qbatch):\n            sel = rows[s:s + qbatch]\n            H = torch.as_tensor(np.asarray(data[\"H\"][sel], np.float32), device=device)\n            mk = torch.as_tensor(data[\"mask\"][sel].astype(np.float32), device=device)\n            dn = torch.as_tensor(np.asarray(data[\"dense\"][sel], np.float32), device=device)\n            # The loss fns average within a call, so weight by rows to recover the\n            # overall mean when the last minibatch is short.\n            tot += float(lossfn(model(H, mk, dn, pop), gi[sel], gv[sel])) * len(sel)\n            n += len(sel)\n    return tot / max(n, 1)\n\n\ndef train_arm(model, tr, fit_rows, dev_rows, gold_idx, gold_val, gold_lists,\n              device, a, torch, lr=None, seed=None, pop=None):\n    # WEIGHT DECAY ON WEIGHT MATRICES ONLY (ndim >= 2). Biases and the scalar\n    # calibration parameters are excluded deliberately: decaying them is not\n    # regularisation, it is a prior. log_tau -> 0 means tau -> 1, logbeta -> 0\n    # means beta -> 1, and shrinking `current`'s w toward 0 flattens the corpus\n    # softmax and inflates the loss the optimiser is trying to reduce. The stated\n    # purpose is to discourage sharp MLP functions, and this is the scoping that\n    # actually does that and nothing else.\n    pop = Pop.from_data(tr, device) if pop is None else pop\n    # Under joint training the popularity predictor is part of the optimised\n    # model, so its matrices belong in the decayed group like any other.\n    trainable = list(model.parameters()) + pop.parameters()\n    groups = [g for g in (\n        {\"params\": [p for p in trainable if p.ndim >= 2],\n         \"weight_decay\": a.weight_decay},\n        {\"params\": [p for p in trainable if p.ndim < 2],\n         \"weight_decay\": 0.0}) if g[\"params\"]]\n    opt = torch.optim.AdamW(groups, lr=a.lr if lr is None else lr)\n    gi = torch.as_tensor(gold_idx, device=device)\n    gv = torch.as_tensor(gold_val, device=device)\n    rng = np.random.default_rng(a.seed if seed is None else seed)\n    lossfn = operator_loss if a.loss == \"operator\" else multigold_loss\n    on_loss = a.select_on == \"loss\"\n    sel_k = 100 if a.select_on == \"ndcg100\" else 10\n    # Lower is better for loss, higher for nDCG, so track the score to BEAT in the\n    # sign the criterion wants and compare one way.\n    best_sel, best_state, stale, hist = float(\"inf\"), None, 0, []\n    best_nd, best_pop = 0.0, None\n    for ep in range(a.epochs):\n        # Real for nn.Module arms (dropout on/off), no-op for the plain classes.\n        if hasattr(model, \"train\"):\n            model.train()\n        losses = []\n        for bi in batches(len(fit_rows), a.qbatch, True, rng):\n            sel = fit_rows[bi]\n            H = torch.as_tensor(np.asarray(tr[\"H\"][sel], np.float32), device=device)\n            mk = torch.as_tensor(tr[\"mask\"][sel].astype(np.float32), device=device)\n            dn = torch.as_tensor(np.asarray(tr[\"dense\"][sel], np.float32), device=device)\n            loss = lossfn(model(H, mk, dn, pop), gi[sel], gv[sel])\n            aux = pop.aux_loss()\n            if not isinstance(aux, float):\n                loss = loss + a.pop_lambda * aux\n            opt.zero_grad()\n            loss.backward()\n            opt.step()\n            losses.append(float(loss.detach()))\n        if hasattr(model, \"eval\"):\n            model.eval()\n        dl = dev_loss(model, tr, dev_rows, gold_idx, gold_val, device, a.qbatch,\n                      torch, lossfn, pop)\n        nd, nd100 = dev_ndcg_multi(model, tr, dev_rows, gold_lists, device,\n                                   a.qbatch, torch, (10, 100), pop)\n        hist.append({\"epoch\": ep + 1, \"train_loss\": float(np.mean(losses)),\n                     \"dev_loss\": dl, \"dev_ndcg@10\": nd, \"dev_ndcg@100\": nd100})\n        # all three minimised after the sign flip\n        sel_now = dl if on_loss else -(nd100 if sel_k == 100 else nd)\n        flag = \"\"\n        if sel_now < best_sel - 1e-9:\n            best_sel, best_nd, stale, flag = sel_now, nd, 0, \" *\"\n            best_state = json.loads(json.dumps(model.state()))\n            best_pop = pop.state()          # None unless the predictor is joint\n        else:\n            stale += 1\n        print(f\"  [{model.name}] ep{ep + 1:3d} train {np.mean(losses):.4f} \"\n              f\"dev_loss {dl:.4f} nDCG@10 {nd:.4f} nDCG@100 {nd100:.4f} \"\n              f\"beta {math.exp(float(model.logbeta.detach())):.3f}{flag}\", flush=True)\n        if stale >= a.patience:\n            print(f\"  [{model.name}] early stop: {stale} evals without a \"\n                  f\"dev {a.select_on} gain\")\n            break\n    model.load(best_state)\n    # pop_tr and pop_te SHARE one predictor object, so restoring it here also\n    # restores the popularity the test split will be scored with.\n    pop.load_state(best_pop)\n    # Return the nDCG AT THE SELECTED CHECKPOINT, not the best nDCG seen. Those\n    # differ under loss selection, and reporting the max would reintroduce exactly\n    # the optimistic bias this change removes.\n    return best_nd, best_state, hist, best_sel, best_pop\n\n\ndef distill_operator(model, teacher, tr, fit_rows, device, a, torch, seed=None):\n    \"\"\"Initialise a raw-view neural scorer from the fitted operator ranking.\n\n    This is representation distillation, not an inference-time ensemble: the\n    teacher is used only on source-training queries.  The student never receives\n    the operator's sum or maximum as input and the teacher is absent at test.\n    \"\"\"\n    groups = [g for g in (\n        {\"params\": [p for p in model.parameters() if p.ndim >= 2],\n         \"weight_decay\": a.weight_decay},\n        {\"params\": [p for p in model.parameters() if p.ndim < 2],\n         \"weight_decay\": 0.0}) if g[\"params\"]]\n    opt = torch.optim.AdamW(groups, lr=a.distill_lr)\n    pop = Pop.from_data(tr, device)\n    rng = np.random.default_rng(a.seed if seed is None else seed)\n    teacher_was_training = getattr(teacher, \"training\", False)\n    if hasattr(teacher, \"eval\"):\n        teacher.eval()\n    if hasattr(model, \"eval\"):\n        model.eval()  # deterministic teacher matching; gradients remain enabled\n    for ep in range(a.distill_epochs):\n        losses = []\n        for bi in batches(len(fit_rows), a.qbatch, True, rng):\n            sel = fit_rows[bi]\n            H = torch.as_tensor(np.asarray(tr[\"H\"][sel], np.float32), device=device)\n            mk = torch.as_tensor(tr[\"mask\"][sel].astype(np.float32), device=device)\n            dn = torch.as_tensor(np.asarray(tr[\"dense\"][sel], np.float32), device=device)\n            with torch.no_grad():\n                target = teacher(H, mk, dn, pop)\n                target = ((target - target.mean(1, keepdim=True)) /\n                          (target.std(1, keepdim=True) + EPS))\n            pred = model(H, mk, dn, pop)\n            pred = ((pred - pred.mean(1, keepdim=True)) /\n                    (pred.std(1, keepdim=True) + EPS))\n            loss = torch.mean((pred - target) ** 2)\n            opt.zero_grad()\n            loss.backward()\n            opt.step()\n            losses.append(float(loss.detach()))\n        print(f\"  [distill] ep{ep + 1:3d} mse {np.mean(losses):.6f}\", flush=True)\n    if hasattr(teacher, \"train\") and teacher_was_training:\n        teacher.train()\n\n\ndef write_predictions(scores, data, path, topk=100):\n    recs = []\n    for i, q in enumerate(data[\"queries\"]):\n        order = np.argsort(-scores[i])[:topk]\n        recs.append({\"id\": q[\"id\"], \"stratum\": q.get(\"stratum\"),\n                     \"supporting_documents\": q.get(\"supporting_documents\", []),\n                     \"predictions\": {\"document\": [[data[\"doc_ids\"][j], float(scores[i, j])]\n                                                  for j in order]}})\n    json.dump(recs, open(path, \"w\"))\n    return path\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\"--model\", default=\"BAAI/bge-large-en-v1.5\")\n    # 0 = every train query left after the dev slice. The old default of 2,500\n    # came from cargo_operator.py, where it fitted FOUR parameters and more data\n    # bought nothing. The learned arms here have 34-200 parameters and their\n    # measured failure mode is overfitting, so capping the fit set is backwards:\n    # it discarded 2,645 CS queries and 6,869 TOMATO queries for no reason.\n    ap.add_argument(\"--train_fit\", type=int, default=0,\n                    help=\"fit queries after the dev slice; 0 = all remaining\")\n    # 300, not cargo_operator.py's 600. Dev is sliced FIRST, so on a small domain\n    # it decides how much fit data is left: matsci has 1,304 train queries, where\n    # 600 leaves 704 and 300 leaves 1,004. 300 still gives a stable nDCG@10.\n    ap.add_argument(\"--dev\", type=int, default=300)\n    ap.add_argument(\"--epochs\", type=int, default=30)\n    ap.add_argument(\"--patience\", type=int, default=5)\n    ap.add_argument(\"--lr\", type=float, default=3e-4)\n    ap.add_argument(\"--weight_decay\", type=float, default=1e-2,\n                    help=\"applied to MLP weight MATRICES only; see train_arm\")\n    # Per-arm override, unset by default so every arm trains at --lr. Kept\n    # because attention peaked at epoch 1 under 1e-3 while the others peaked at\n    # 4-6, so a separate rate may be wanted again.\n    ap.add_argument(\"--lr_attention\", type=float, default=None)\n    ap.add_argument(\"--hidden\", type=int, default=8)\n    ap.add_argument(\"--qbatch\", type=int, default=8, help=\"queries per minibatch\")\n    ap.add_argument(\"--seed\", type=int, default=0)\n    ap.add_argument(\"--out\", default=None)\n    ap.add_argument(\"--arms\", default=\"dense,current,attention,deepsets,mlp\",\n                    help=\"any of dense, current, attention, deepsets, setmlp, \"\n                         \"dualsetmlp, mlp, gated\")\n    ap.add_argument(\"--mlp_hidden\", type=int, default=16)\n    ap.add_argument(\"--ds_hidden\", type=int, default=4,\n                    help=\"compact deepsets phi/rho width (spec: 4)\")\n    ap.add_argument(\"--ds_dropout\", type=float, default=0.15,\n                    help=\"whole-answer dropout in the deepsets arm, training only\")\n    ap.add_argument(\"--set_hidden\", type=int, default=8,\n                    help=\"identity-preserving set-MLP width\")\n    ap.add_argument(\"--set_dropout\", type=float, default=0.0,\n                    help=\"whole-answer dropout in the setmlp arm, training only\")\n    ap.add_argument(\"--distill_operator\", default=None,\n                    help=\"fitted current-scorer JSON used only to initialise a set MLP\")\n    ap.add_argument(\"--distill_epochs\", type=int, default=0)\n    ap.add_argument(\"--distill_lr\", type=float, default=1e-3)\n    ap.add_argument(\"--seeds\", default=\"0\",\n                    help=\"comma-separated training seeds; each trained arm runs once \"\n                         \"per seed and keeps the best dev checkpoint. The fit/dev \"\n                         \"SPLIT stays fixed by --seed, so seeds vary only init, \"\n                         \"batch order and dropout.\")\n    # DEFAULT IS THE OPERATOR'S OWN OBJECTIVE, so the `current` arm reproduces the\n    # scorer as it is fitted elsewhere in the project and the only thing varying\n    # between arms is the architecture. `fixed` is the multi-gold correction:\n    # SIR-4 averages ~4 golds per query and the operator objective keeps them in\n    # each other's denominator, so it pushes a query's own labels apart. Run both\n    # to measure that rather than assume it.\n    ap.add_argument(\"--loss\", default=\"fixed\", choices=[\"operator\", \"fixed\"])\n    # DEPLOYABILITY OF THE POPULARITY TERM. `loo` needs the OTHER test queries'\n    # hypothetical answers, so it cannot score one query alone. `bank` replaces it\n    # with the train-answer bank (query-independent, one matmul per corpus at\n    # indexing time). `predicted` distils that bank into an MLP of the paper\n    # embedding, so inference is fully local. Default stays loo so every prior\n    # number reproduces.\n    ap.add_argument(\"--popularity\", default=\"loo\", choices=[\"loo\", \"bank\", \"predicted\"])\n    # The mlp arm's popularity is PART OF THE ARM, not a run-level axis: the whole\n    # proposal is a scorer that needs nothing but the paper embedding at inference.\n    # Set 0 to make it share --popularity with the other arms instead.\n    ap.add_argument(\"--mlp_popularity\", type=int, default=1,\n                    help=\"1 = the mlp arm uses its own learned popularity predictor\")\n    # TWO-STAGE vs JOINT. Two-stage fits the predictor to the bank targets, freezes\n    # it, then trains the scorer -- so a win is attributable to the scorer and the\n    # predictor keeps meaning \"general matchability\". Joint lets the retrieval\n    # gradient reach the predictor as well, anchored by --pop_lambda on the same\n    # log-MSE target; more expressive, less interpretable, and the predictor can\n    # drift into being extra scorer capacity if lambda is too small.\n    ap.add_argument(\"--mlp_pop_joint\", type=int, default=0,\n                    help=\"1 = train the popularity predictor jointly with the scorer\")\n    ap.add_argument(\"--pop_lambda\", type=float, default=1.0,\n                    help=\"weight on the matchability anchor under joint training\")\n    # Checkpoint/early-stopping criterion. `loss` is the default because dev\n    # nDCG@10 is a step function and selecting its max over ~90 evaluations on one\n    # 300-query slice picks luck, not generalisation. `ndcg` reproduces the old\n    # behaviour.\n    # SELECT ON THE METRIC, NOT THE LOSS. Measured on physics: dev loss and dev\n    # nDCG@10 move in OPPOSITE directions on the same held-out queries (attention\n    # seed 2, ep18 -> ep29: loss 5.8507 -> 5.6360 while nDCG 0.4329 -> 0.3976).\n    # The loss is InfoNCE over the whole corpus, so it is rewarded for separating\n    # the gold from negatives at rank 2000 that no metric sees; beta collapses\n    # (0.89 -> 0.37), switching off the popularity discount that suppresses\n    # generically attractive papers at the TOP. Global separation improves, head\n    # precision degrades. nDCG@10 is a chunky step function, which is the real\n    # problem, but the cure is a SMOOTHER VERSION OF THE METRIC:\n    #   ndcg    nDCG@10, the reported metric, chunky but aligned\n    #   ndcg100 nDCG@100, responds to a gold moving anywhere in the top 100, so\n    #           many more events per epoch, still head-weighted by the log discount\n    #   loss    kept only to reproduce the runs that exposed this\n    ap.add_argument(\"--select_on\", default=\"loss\",\n                    choices=[\"ndcg\", \"ndcg100\", \"loss\"],\n                    help=\"ndcg = nDCG@10 (the reported metric); ndcg100 = nDCG@100 \"\n                         \"(same metric, more resolution per epoch); loss = dev loss\")\n    # 0 = the historical single dev slice. K >= 2 runs K-fold CV over EVERY train\n    # query to choose the epoch count, then refits on ALL of them -- so no query is\n    # permanently held out of training, and the selection signal is the whole train\n    # split instead of a 300-query sample.\n    ap.add_argument(\"--kfold\", type=int, default=0)\n    ap.add_argument(\"--force_build\", action=\"store_true\")\n    ap.add_argument(\"--cache_only\", action=\"store_true\",\n                    help=\"fail rather than load an embedding model if an array is missing\")\n    ap.add_argument(\"--selection_only\", action=\"store_true\",\n                    help=\"train and select on development data without scoring test\")\n    add_dataset_arg(ap)\n    a = ap.parse_args()\n    set_dataset(a.dataset)\n    print(banner())\n\n    import torch\n    device = (\"cuda\" if torch.cuda.is_available()\n              else \"mps\" if torch.backends.mps.is_available() else \"cpu\")\n    torch.manual_seed(a.seed)\n    op = _op_module()\n    out = a.out or f\"{_ROOT}/sir4-retrieval/results/semantic_{a.dataset}\"\n    os.makedirs(out, exist_ok=True)\n    cache = f\"{_ROOT}/outputs/caches/semantic/{a.dataset}\"\n    print(f\"device={device} encoder={a.model}\\nout={out}\")\n\n    if a.cache_only:\n        # build_inputs reaches the encoder only when one of these arrays is\n        # absent.  Check here so a cache-only run fails before importing any\n        # embedding library or constructing a model.\n        for split in (\"train\", \"test\"):\n            _, _, queries, _ = op.load_split(split)\n            qkey = hashlib.md5(\"|\".join(q[\"id\"] for q in queries).encode()).hexdigest()[:8]\n            slug = op.model_slug(a.model)\n            qi = op.query_instruction(a.model)\n            required = [f\"{emb_dir()}/{split}_doc{slug}.npy\",\n                        f\"{emb_dir()}/{split}_query_{qkey}{slug}{op.qi_tag(qi)}.npy\",\n                        f\"{emb_dir()}/{split}_probe_{qkey}{slug}.npy\"]\n            missing = [p for p in required if not os.path.exists(p)]\n            assert not missing, \"cache-only run is missing:\\n  \" + \"\\n  \".join(missing)\n    tr = build_inputs(op, a.model, \"train\", cache, a.force_build)\n    te = build_inputs(op, a.model, \"test\", cache, a.force_build)\n\n    arms_req = [x.strip() for x in a.arms.split(\",\") if x.strip()]\n    # THE MLP ARM CARRIES ITS OWN LEARNED POPULARITY. It is not a separate axis:\n    # the proposal is one deployable scorer -- learned pooling AND a popularity\n    # discount predicted from the paper embedding alone, with no dependence on the\n    # other test queries. `current` keeps the leave-one-out term it has always\n    # used, so the comparison is proposal-vs-baseline as each is actually meant to\n    # be deployed. Both sources are computed once and stored side by side.\n    MLP_ARMS = {\"mlp\": \"joint\", \"mlp2s\": \"_pred\", \"mlpbank\": \"_bank\"}\n    if any(x in arms_req for x in MLP_ARMS) or a.popularity != \"loo\":\n        # TRAIN targets are already on disk: total_S is the sum over every train\n        # answer, so dividing by the answer count IS the bank mean for the train\n        # corpus. Only the TEST corpus needs a fresh matmul, against the SAME\n        # train bank -- no test query ever contributes to any popularity value.\n        slug = op.model_slug(a.model)\n        n_bank = int(tr[\"mask\"].sum())\n        p_tr = (tr[\"total_S\"] / max(n_bank, 1)).astype(np.float32)\n        bank_tr = p_tr.copy()          # the anchor target, before any prediction\n        bank_emb = np.load(f\"{emb_dir()}/train_probe_{tr['meta']['qkey']}{slug}.npy\"\n                           ).astype(np.float32)\n        assert bank_emb.shape[0] == n_bank, (\n            f\"train answer bank {bank_emb.shape[0]} rows != {n_bank} valid answers\")\n        de_te = np.load(f\"{emb_dir()}/test_doc{slug}.npy\").astype(np.float32)\n        assert de_te.shape[0] == len(te[\"doc_ids\"]), \"test doc embedding misaligned\"\n        p_te = bank_popularity(bank_emb, de_te)\n        print(f\"[pop] bank of {n_bank} train answers; train p in \"\n              f\"[{p_tr.min():.4f}, {p_tr.max():.4f}], test p in \"\n              f\"[{p_te.min():.4f}, {p_te.max():.4f}]\")\n        # KEEP THE BANK VECTORS. They are the anchor targets AND the `mlpbank`\n        # arm's popularity, so overwriting them with predictions below would lose\n        # the one configuration measured to beat the baseline.\n        bank_te = p_te.copy()\n        tr[\"pop_mode_bank\"] = te[\"pop_mode_bank\"] = \"bank\"\n        tr[\"pop_vec_bank\"], te[\"pop_vec_bank\"] = bank_tr, bank_te\n        pop_info = {\"mode\": a.popularity, \"bank_answers\": n_bank}\n        _fitted, pop_joint = False, None\n        if a.popularity == \"predicted\" or a.mlp_popularity:\n            _fitted = True\n            de_tr = np.load(f\"{emb_dir()}/train_doc{slug}.npy\").astype(np.float32)\n            assert de_tr.shape[0] == len(tr[\"doc_ids\"]), \"train doc embedding misaligned\"\n            gm, fit_info = fit_matchability(de_tr, p_tr, device, seed=a.seed)\n            pop_info[\"fit\"] = fit_info\n            with _torch.no_grad():\n                # The scorer sees PREDICTED popularity on BOTH splits, so its\n                # training-time input distribution matches inference exactly.\n                p_tr = gm(_torch.as_tensor(de_tr, device=device)).cpu().numpy()\n                p_te = gm(_torch.as_tensor(de_te, device=device)).cpu().numpy()\n            tr[\"pop_mode_pred\"] = te[\"pop_mode_pred\"] = \"predicted\"\n            tr[\"pop_vec_pred\"], te[\"pop_vec_pred\"] = p_tr, p_te\n            _torch.save(gm.state_dict(),\n                        f\"{out}/matchability_{a.dataset}{slug}.pt\")\n            # Joint mode rebuilds the predictor per seed, warm-started from this\n            # two-stage fit, so it begins at \"general matchability\" rather than at\n            # noise and any drift is attributable to the retrieval gradient.\n            pop_joint = {\"init\": {k: v.detach().cpu() for k, v in gm.state_dict().items()},\n                         \"dim\": de_tr.shape[1], \"hidden\": fit_info[\"hidden\"],\n                         \"de_tr\": de_tr, \"de_te\": de_te,\n                         \"target_tr\": bank_tr}\n        if a.popularity != \"loo\":\n            tr[\"pop_mode\"] = te[\"pop_mode\"] = a.popularity\n            tr[\"pop_vec\"], te[\"pop_vec\"] = p_tr, p_te\n        # pop_joint only exists if a predictor was fitted above; the joint arm is\n        # unavailable otherwise and falls back to the frozen vectors.\n        pop_joint = pop_joint if (a.mlp_pop_joint and _fitted) else None\n    else:\n        pop_info, pop_joint = {\"mode\": \"loo\"}, None\n    gidx, gval, glists = gold_matrix(tr[\"queries\"], tr[\"doc_ids\"])\n    te_glists = None if a.selection_only else gold_matrix(te[\"queries\"], te[\"doc_ids\"])[2]\n\n    # SAME SLICING RULE AS cargo_operator.py: dev first, then fit from what is\n    # left. matsci train holds 1,304 queries, so a 2,500-query fit set does not\n    # exist and the request is silently truncated -- print what actually happened.\n    Q = len(tr[\"queries\"])\n    rng = np.random.default_rng(a.seed)\n    perm = rng.permutation(Q)\n    dev_rows = perm[:a.dev]\n    fit_rows = perm[a.dev:] if a.train_fit <= 0 else perm[a.dev:a.dev + a.train_fit]\n    all_rows = perm                      # every train query, for --kfold\n    # Under --kfold the single fit/dev split is unused: the folds provide both, and\n    # the final model refits on all_rows. So an empty fit_rows is only a problem in\n    # the non-kfold path.\n    assert a.kfold >= 2 or len(fit_rows) > 0, (\n        f\"no fit queries left: {Q} train queries, --dev {a.dev}\")\n    if a.kfold < 2 and ((a.train_fit > 0 and len(fit_rows) < a.train_fit)\n                        or len(dev_rows) < a.dev):\n        print(f\"[split] REQUESTED fit={a.train_fit} dev={a.dev} but the train split \"\n              f\"holds {Q} queries -> using fit={len(fit_rows)} dev={len(dev_rows)}\")\n    # ONE input width across splits: the two splits can have different Jmax and a\n    # layer shaped for one would fail on the other, after training.\n    JFIX = max(tr[\"H\"].shape[1], te[\"H\"].shape[1])\n    print(f\"[split] jmax(train)={tr['H'].shape[1]} jmax(test)={te['H'].shape[1]} -> JFIX={JFIX}\")\n    if a.kfold >= 2:\n        print(f\"[split] {a.kfold}-fold CV over ALL {Q} train queries: each fold fits \"\n              f\"on ~{Q - Q // a.kfold} and devs on ~{Q // a.kfold}; the final model \"\n              f\"refits on all {Q}. test={len(te['queries'])}\")\n    else:\n        print(f\"[split] fit={len(fit_rows)} dev={len(dev_rows)} of {Q} train queries \"\n              f\"(seed {a.seed}); test={len(te['queries'])}\")\n\n    # Filenames are per-arm AND per-loss and never reused, so a rerun adds files\n    # rather than silently replacing another configuration's predictions. Without\n    # the loss in the name a legacy run would overwrite the corrected one and the\n    # two would be indistinguishable afterwards.\n    LSUF = \"_operatorloss\" if a.loss == \"operator\" else \"_fixedloss\"\n    # bank/predicted results must never overwrite the loo ones they are read against.\n    LSUF += \"\" if a.popularity == \"loo\" else f\"_{a.popularity}pop\"\n    TAG = {k: f\"{k}{LSUF}\" for k in\n           (\"current\", \"attention\", \"deepsets\", \"setmlp\", \"dualsetmlp\", \"gated\")}\n    TAG[\"dense\"] = \"dense\"          # untrained, so no loss belongs in its name\n    for k in (\"mlp\", \"mlp2s\", \"mlpbank\"):\n        TAG[k] = f\"{k}{LSUF}\"\n    seeds = [int(x) for x in str(a.seeds).split(\",\") if x.strip() != \"\"]\n    results, summary = {}, {}\n    for arm in [x.strip() for x in a.arms.split(\",\") if x.strip()]:\n        assert arm in TAG, f\"unknown arm {arm!r}; choose from {sorted(TAG)}\"\n        print(f\"\\n=== arm: {arm} ===\")\n\n        def build():\n            return (CurrentScorer(device) if arm == \"current\"\n                    else AttentionScorer(device, a.hidden) if arm == \"attention\"\n                    else DeepSetsScorer(device, a.ds_hidden, a.ds_dropout) if arm == \"deepsets\"\n                    else SetMLPScorer(device, a.set_hidden, a.set_dropout) if arm == \"setmlp\"\n                    else DualSetMLPScorer(device, a.set_hidden, a.set_dropout) if arm == \"dualsetmlp\"\n                    else DenseScorer(device) if arm == \"dense\"\n                    else SortedMLPScorer(device, JFIX, a.mlp_hidden) if arm in MLP_ARMS\n                    else GatedScorer(device, a.hidden))\n\n        # THE MLP ARM READS ITS OWN POPULARITY. Its learned predictor is part of\n        # the proposal, not a run-level axis, so it is selected per arm here while\n        # every other arm keeps whatever --popularity chose.\n        # EACH MLP VARIANT DIFFERS ONLY IN ITS POPULARITY SOURCE, so one run\n        # measures all three against the same baseline on the same split:\n        #   mlp      joint  -- predictor trained with the scorer (the proposal)\n        #   mlp2s    _pred  -- predictor fitted to the bank, then frozen\n        #   mlpbank  _bank  -- the measured bank mean, nothing learned\n        want = MLP_ARMS.get(arm, \"\")\n        joint_here = want == \"joint\" and pop_joint is not None\n        vk = \"\" if joint_here else (want if f\"pop_vec{want}\" in tr else \"\")\n\n        def make_pops():\n            \"\"\"Fresh Pop pair. Under joint mode the predictor is rebuilt per seed.\"\"\"\n            if not joint_here:\n                return Pop.from_data(tr, device, vk), Pop.from_data(te, device, vk)\n            gj = MatchabilityPredictor(pop_joint[\"dim\"], pop_joint[\"hidden\"]).to(device)\n            gj.load_state_dict({k: v.to(device) for k, v in pop_joint[\"init\"].items()})\n            dtr = _torch.as_tensor(pop_joint[\"de_tr\"], device=device)\n            dte = _torch.as_tensor(pop_joint[\"de_te\"], device=device)\n            tgt = _torch.as_tensor(pop_joint[\"target_tr\"], device=device)\n            # ONE predictor shared by both splits: the test popularity must come\n            # from the network that training produced, not a second copy.\n            return (Pop(\"joint\", predictor=gj, doc_emb=dtr, target=tgt),\n                    Pop(\"joint\", predictor=gj, doc_emb=dte))\n\n        pop_tr, pop_te = make_pops()\n\n        model = build()\n        npar = sum(p.numel() for p in model.parameters())\n        print(f\"  parameters: {npar}\"\n              + (\"  popularity: predictor, JOINT with the scorer\" if joint_here\n                 else \"  popularity: predictor, frozen (two-stage)\" if vk == \"_pred\"\n                 else \"  popularity: train-answer bank (measured, not learned)\" if vk == \"_bank\"\n                 else f\"  popularity: {tr.get('pop_mode', 'loo')}\"))\n        t0 = time.time()\n        per_seed = {}\n        # BEFORE the branch. An untrained arm never enters the seed loop, so an\n        # initialisation inside it leaves this name unbound for `dense` and the\n        # predictor-saving step below crashes on the very first arm.\n        best_pop_seed = None\n        if npar == 0:\n            # Nothing to fit. Its dev score is still recorded so the trained arms\n            # can be read against a fixed reference on the same queries.\n            print(\"  no parameters: scoring directly, no training\")\n            best = dev_ndcg(model, tr, dev_rows, glists, device, a.qbatch, torch,\n                            pop=pop_tr)\n            state, hist = model.state(), []\n            print(f\"  [dense] dev nDCG@10 {best:.4f}\")\n        else:\n            lr = a.lr_attention if (arm == \"attention\" and a.lr_attention) else a.lr\n            ndec = sum(p.numel() for p in model.parameters() if p.ndim >= 2)\n            print(f\"  lr: {lr}  weight_decay: {a.weight_decay} on {ndec} of {npar} params\"\n                  + (f\"  seeds: {seeds}\" if len(seeds) > 1 else \"\"))\n            # SEEDS VARY TRAINING ONLY (init, batch order, dropout). The fit/dev\n            # split is pinned by --seed, so every seed and every arm sees the same\n            # queries and the best-dev selection is a fair comparison.\n            # SEEDS ARE SELECTED ON THE SAME CRITERION AS EPOCHS. Picking the\n            # best-of-3 by dev nDCG while epochs are picked by dev loss would put\n            # the discarded step-function signal straight back in, one level up.\n            best_sel_seed, best, state, hist = float(\"inf\"), -1.0, None, []\n            for sd in seeds:\n                torch.manual_seed(sd)\n                # Rebuild for EVERY seed, including a one-seed invocation such\n                # as --seeds 2.  Otherwise that invocation silently retains the\n                # model constructed under --seed (normally zero).\n                model = build()\n                # REBUILD THE PREDICTOR TOO. Left outside the loop it would carry\n                # seed 0's trained weights into seed 1, so the seeds would not be\n                # independent and later ones would start pre-trained.\n                pop_tr, pop_te = make_pops()\n                if len(seeds) > 1:\n                    print(f\"  -- seed {sd} --\")\n                if arm in (\"setmlp\", \"dualsetmlp\") and a.distill_epochs > 0:\n                    assert a.distill_operator and os.path.exists(a.distill_operator), (\n                        \"--distill_epochs requires an existing --distill_operator JSON\")\n                    teacher = CurrentScorer(device)\n                    teacher.load(json.load(open(a.distill_operator)))\n                    print(f\"  distilling fitted operator for {a.distill_epochs} epochs; \"\n                          \"teacher is not used at inference\")\n                    distill_operator(model, teacher, tr, fit_rows, device, a, torch, seed=sd)\n                if a.kfold >= 2:\n                    # Folds choose the epoch count; the final fit then uses EVERY\n                    # train query. There is no held-out data left to select on,\n                    # which is the intended trade: all the data trains the model.\n                    E, picked = kfold_epochs(build, tr, all_rows, gidx, gval, glists,\n                                             device, a, torch, lr=lr, seed=sd)\n                    model = build()\n                    h = train_fixed(model, tr, all_rows, gidx, gval, device, a,\n                                    torch, E, lr=lr, seed=sd)\n                    st = json.loads(json.dumps(model.state()))\n                    b = dev_ndcg(model, tr, all_rows, glists, device, a.qbatch, torch)\n                    bsel, bpop = float(h[-1][\"train_loss\"]), pop_tr.state()\n                    per_seed[sd] = {\"kfold_epochs\": picked, \"epochs_used\": E,\n                                    \"train_ndcg@10_insample\": round(b, 4)}\n                else:\n                    b, st, h, bsel, bpop = train_arm(model, tr, fit_rows, dev_rows,\n                                                     gidx, gval, glists, device, a,\n                                                     torch, lr=lr, seed=sd, pop=pop_tr)\n                    # Name the field after what was actually selected on: bsel is\n                    # the loss under --select_on loss and MINUS the nDCG otherwise,\n                    # so a fixed \"dev_loss\" label printed the nDCG twice.\n                    per_seed[sd] = {\"dev_ndcg@10\": round(b, 4),\n                                    f\"selected_on_{a.select_on}\":\n                                        round(bsel if a.select_on == \"loss\" else -bsel, 4)}\n                if bsel < best_sel_seed - 1e-9:\n                    best_sel_seed, best, state, hist = bsel, b, st, h\n                    best_pop_seed, best_pop_te = bpop, pop_te\n            if len(seeds) > 1:\n                print(f\"  per-seed: {per_seed}\")\n                if a.kfold >= 2:\n                    # NO HELD-OUT DATA REMAINS, so a seed cannot be selected on\n                    # merit. Keeping the lowest FINAL TRAINING loss is a tie-break,\n                    # not model selection, and it is reported as such.\n                    print(\"  kfold: every train query is in the final fit, so no \"\n                          \"held-out selection is possible; seeds differ only by \"\n                          \"init/order and the lowest final train loss is kept\")\n                else:\n                    print(f\"  kept the seed with the best dev \"\n                          f\"{'loss' if a.select_on == 'loss' else 'nDCG@10'}\")\n            model.load(state)\n            if best_pop_seed is not None:\n                # Restore the WINNING seed's predictor into the Pop that test\n                # scoring uses, so both halves come from one checkpoint.\n                best_pop_te.load_state(best_pop_seed)\n                pop_te = best_pop_te\n        tag = TAG[arm]\n        json.dump(state, open(f\"{out}/params_semantic_{tag}_{a.dataset}.json\", \"w\"), indent=1)\n        # THE JOINT PREDICTOR IS HALF THE MODEL, SO IT HAS TO BE HALF THE CHECKPOINT.\n        # Only the pre-joint two-stage fit was ever written to disk, so once the\n        # process exited the trained popularity was gone and the params json on disk\n        # described a scorer paired with a predictor that no longer existed. Anything\n        # reloading this arm -- the graph fusion warm start, a rerun, a transfer --\n        # would silently get the wrong half. Written next to the scorer under a\n        # matching name so the pair cannot be separated by accident.\n        if best_pop_seed is not None:\n            _torch.save(best_pop_seed, f\"{out}/popnet_semantic_{tag}_{a.dataset}.pt\")\n            print(f\"[{arm}] saved joint popularity predictor -> \"\n                  f\"popnet_semantic_{tag}_{a.dataset}.pt\")\n        pred, test_ndcg = None, None\n        if not a.selection_only:\n            # pop_te, NOT the run-level default. Without it a joint run scored\n            # test with leave-one-out popularity and the trained predictor was\n            # never used at inference at all.\n            sc = score_rows(model, te, np.arange(len(te[\"queries\"])),\n                            device, a.qbatch, torch, pop=pop_te)\n            pred = write_predictions(\n                sc, te, f\"{out}/predictions_semantic_{tag}_{a.dataset}_test.json\")\n            test_ndcg = float(np.mean(\n                [ndcg_at(list(np.argsort(-sc[i])[:10]), te_glists[i], 10)\n                 for i in range(len(te_glists)) if te_glists[i]]))\n        results[arm] = {\"best_dev_ndcg@10\": best, \"params\": state, \"history\": hist,\n                        \"per_seed_dev_ndcg@10\": per_seed,\n                        \"predictions\": pred, \"minutes\": round((time.time() - t0) / 60, 2)}\n        summary[arm] = {\"dev_ndcg@10\": round(best, 4),\n                        \"test_ndcg@10_quick\": (None if test_ndcg is None\n                                                else round(test_ndcg, 4))}\n        destination = \"test deliberately not scored\" if pred is None else os.path.basename(pred)\n        print(f\"[{arm}] best dev nDCG@10 {best:.4f} -> {destination}\")\n\n    meta = {\"dataset\": a.dataset, \"encoder\": a.model, \"device\": device,\n            \"hyperparameters\": {k: getattr(a, k) for k in\n                                (\"train_fit\", \"dev\", \"epochs\", \"patience\", \"lr\",\n                                 \"hidden\", \"ds_hidden\", \"ds_dropout\", \"mlp_hidden\",\n                                 \"set_hidden\", \"set_dropout\",\n                                 \"distill_operator\", \"distill_epochs\", \"distill_lr\",\n                                 \"qbatch\", \"seed\", \"seeds\", \"loss\", \"popularity\",\n                                 \"select_on\", \"kfold\", \"mlp_popularity\",\n                                 \"mlp_pop_joint\", \"pop_lambda\",\n                                 \"lr_attention\", \"weight_decay\")},\n            \"popularity\": pop_info,\n            \"actual_split\": {\"fit\": int(len(fit_rows)), \"dev\": int(len(dev_rows)),\n                             \"train_queries\": int(Q), \"test_queries\": len(te[\"queries\"])},\n            \"loss\": (\"operator objective (cargo_operator.py): full-corpus denominator \"\n                     \"including the query's other golds, weighted per (query, gold) pair\"\n                     if a.loss == \"operator\" else\n                     \"multi-gold corrected: other golds excluded from each denominator, \"\n                     \"averaged within query then equally across queries\"),\n            \"arms\": results, \"quick_summary\": summary}\n    mp = f\"{out}/semantic_comparison_{a.dataset}{LSUF}.json\"\n    json.dump(meta, open(mp, \"w\"), indent=1)\n    print(f\"\\nwrote {mp}\")\n    print(\"\\nquick development selection\" +\n          (\" (test not scored):\" if a.selection_only else \"/test nDCG@10:\"))\n    for k, v in summary.items():\n        test_text = \"not scored\" if v[\"test_ndcg@10_quick\"] is None else f\"{v['test_ndcg@10_quick']:.4f}\"\n        print(f\"  {k:10} dev {v['dev_ndcg@10']:.4f}  test {test_text}\")\n\n\nif __name__ == \"__main__\":\n    main()\n", "sir4-retrieval/eval/report_domain_results.py": "\"\"\"Report query-level retrieval performance by ResearchBench discipline.\n\nThe input files are the ``--per-query-out`` mappings written by\n``eval/score_sir4.py``. Legacy files are losslessly normalised in memory because\nthey already store conventional MRR under ``mrr_best``: their old ``mrr`` becomes\n``mgrr``, while ``mrr_best`` becomes ``mrr``. Rankings are never recomputed.\n\nExample\n-------\npython3 eval/report_domain_results.py \\\n  --arm \"BGE=/path/bge_perquery.json\" \\\n  --arm \"SIR-4 Biology=/path/sir4_biology_perquery.json\" \\\n  --arm \"TOMATO-Star=/path/tomato_perquery.json\"\n\"\"\"\nfrom __future__ import annotations\n\nimport argparse\nimport json\nfrom collections import defaultdict\n\n\nMETRICS = (\n    (\"mrr\", \"MRR\"),\n    (\"ndcg@5\", \"nDCG@5\"),\n    (\"recall@5\", \"R@5\"),\n    (\"completeset@5\", \"CGS@5\"),\n)\n\n\ndef parse_arm(spec: str) -> tuple[str, str]:\n    if \"=\" not in spec:\n        raise argparse.ArgumentTypeError(\"arm must be LABEL=PATH\")\n    label, path = spec.split(\"=\", 1)\n    if not label.strip() or not path.strip():\n        raise argparse.ArgumentTypeError(\"arm must be LABEL=PATH\")\n    return label.strip(), path.strip()\n\n\ndef load_arm(path: str) -> dict[str, dict]:\n    rows = json.load(open(path))\n    if not isinstance(rows, dict) or not rows:\n        raise SystemExit(f\"{path}: expected a non-empty per-query mapping\")\n    legacy_mean, legacy_standard = 0, 0\n    for qid, row in rows.items():\n        if \"mgrr\" not in row and \"mrr_best\" in row:\n            row[\"mgrr\"] = row.get(\"mrr\", 0.0)\n            row[\"mrr\"] = row[\"mrr_best\"]\n            legacy_mean += 1\n        elif \"mgrr\" not in row and \"mrr_best\" not in row and \"mrr\" in row:\n            # The earliest scorer emitted only conventional first-relevant MRR.\n            # There is no separate all-gold quantity to preserve in this schema.\n            legacy_standard += 1\n        missing = [key for key, _ in METRICS if key not in row]\n        if missing:\n            raise SystemExit(f\"{path}: query {qid!r} lacks {missing}\")\n        if \"mrr\" not in row:\n            raise SystemExit(f\"{path}: query {qid!r} has no MRR value\")\n        if \"mrr_best\" in row and abs(row[\"mrr\"] - row[\"mrr_best\"]) > 1e-12:\n            raise SystemExit(f\"{path}: query {qid!r} does not use standard MRR\")\n        if not row.get(\"discipline\"):\n            raise SystemExit(f\"{path}: query {qid!r} has no ResearchBench discipline\")\n    if legacy_mean:\n        print(f\"{path}: normalised standard MRR from mrr_best for \"\n              f\"{legacy_mean:,} legacy rows\")\n    if legacy_standard:\n        print(f\"{path}: using the existing standard mrr field for \"\n              f\"{legacy_standard:,} early-schema rows\")\n    return rows\n\n\ndef aggregate(arms: list[tuple[str, dict[str, dict]]]) -> dict:\n    reference_label, reference = arms[0]\n    qids = set(reference)\n    for label, rows in arms[1:]:\n        if set(rows) != qids:\n            raise SystemExit(\n                f\"{label} and {reference_label} cover different query sets: \"\n                f\"{len(rows)} versus {len(reference)}\"\n            )\n        mismatch = [qid for qid in qids\n                    if rows[qid][\"discipline\"] != reference[qid][\"discipline\"]]\n        if mismatch:\n            raise SystemExit(f\"{label}: discipline mismatch for {mismatch[0]!r}\")\n\n    by_domain: dict[str, list[str]] = defaultdict(list)\n    for qid, row in reference.items():\n        by_domain[row[\"discipline\"]].append(qid)\n\n    result = {}\n    for domain in sorted(by_domain):\n        ids = by_domain[domain]\n        result[domain] = {\"n\": len(ids), \"arms\": {}}\n        for label, rows in arms:\n            result[domain][\"arms\"][label] = {\n                key: sum(rows[qid][key] for qid in ids) / len(ids)\n                for key, _ in METRICS\n            }\n\n        # SAME / CROSS WITHIN THE DISCIPLINE. score_sir4 records each query's\n        # slice memberships, so this is a partition of the ids above, not a\n        # rescore. Taken from the reference arm because every arm ranked the same\n        # queries and the slices are a property of the query, not the ranking.\n        for slc in (\"same\", \"cross\"):\n            sub = [q for q in ids if slc in (reference[q].get(\"slices\") or [])]\n            if not sub:\n                continue\n            result[domain].setdefault(\"slices\", {})[slc] = {\n                \"n\": len(sub),\n                \"arms\": {label: {key: sum(rows[q][key] for q in sub) / len(sub)\n                                 for key, _ in METRICS}\n                         for label, rows in arms},\n            }\n    return result\n\n\ndef markdown(result: dict, labels: list[str]) -> str:\n    lines = [\n        \"### Performance by ResearchBench discipline\",\n        \"\",\n        \"| ResearchBench domain | n | Model | \"\n        + \" | \".join(label for _, label in METRICS) + \" |\",\n        \"|---|--:|---|\" + \"--:|\" * len(METRICS),\n    ]\n    for domain, block in result.items():\n        for index, label in enumerate(labels):\n            values = block[\"arms\"][label]\n            lines.append(\n                f\"| {domain if index == 0 else ''} | \"\n                f\"{block['n'] if index == 0 else ''} | {label} | \"\n                + \" | \".join(f\"{values[key]:.4f}\" for key, _ in METRICS)\n                + \" |\"\n            )\n    lines += [\"\", \"MRR is the reciprocal rank of the first relevant paper. \"\n              \"CGS treats ResearchBench's accepted gold list as one complete set.\"]\n\n    # Table 7.5's shape: same and cross paired under each metric, one block per\n    # discipline. Built here rather than reshaped by hand, because transcribing\n    # eight numbers per arm is where a transcription error would enter unnoticed.\n    if any(\"slices\" in block for block in result.values()):\n        lines += [\"\", \"### Same vs cross within each discipline\", \"\"]\n        header = \" | \".join(f\"{lab} Same | {lab} Cross\" for _, lab in METRICS)\n        lines += [f\"| domain | Model | n same | n cross | {header} |\",\n                  \"|---|---|--:|--:|\" + \"--:|\" * (2 * len(METRICS))]\n        for domain, block in result.items():\n            sl = block.get(\"slices\") or {}\n            if not sl:\n                continue\n            ns = sl.get(\"same\", {}).get(\"n\", 0)\n            nc = sl.get(\"cross\", {}).get(\"n\", 0)\n            for index, label in enumerate(labels):\n                cells = []\n                for key, _ in METRICS:\n                    for slc in (\"same\", \"cross\"):\n                        v = sl.get(slc, {}).get(\"arms\", {}).get(label, {}).get(key)\n                        cells.append(\"--\" if v is None else f\"{v:.4f}\")\n                lines.append(\n                    f\"| {domain if index == 0 else ''} | {label} | \"\n                    f\"{ns if index == 0 else ''} | {nc if index == 0 else ''} | \"\n                    + \" | \".join(cells) + \" |\")\n        lines += [\"\", \"`same` and `cross` describe whether the gold inspiration \"\n                  \"comes from the target's own field. Cross counts are small in \"\n                  \"several disciplines; read those columns as indicative.\"]\n    return \"\\n\".join(lines)\n\n\ndef main() -> int:\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\"--arm\", action=\"append\", type=parse_arm, required=True,\n                        help=\"LABEL=PATH; repeat in the desired display order\")\n    parser.add_argument(\"--md-out\")\n    parser.add_argument(\"--json-out\")\n    args = parser.parse_args()\n\n    if len(args.arm) < 2:\n        raise SystemExit(\"provide at least two --arm arguments\")\n    labels = [label for label, _ in args.arm]\n    if len(labels) != len(set(labels)):\n        raise SystemExit(\"arm labels must be unique\")\n\n    arms = [(label, load_arm(path)) for label, path in args.arm]\n    result = aggregate(arms)\n    table = markdown(result, labels)\n    print(table)\n    if args.md_out:\n        open(args.md_out, \"w\").write(table + \"\\n\")\n        print(f\"\\nwrote {args.md_out}\")\n    if args.json_out:\n        json.dump(result, open(args.json_out, \"w\"), indent=1)\n        print(f\"wrote {args.json_out}\")\n    return 0\n\n\nif __name__ == \"__main__\":\n    raise SystemExit(main())\n", "sir4-retrieval/transfer/paired_bootstrap.py": "\"\"\"\npaired_bootstrap.py -- paired comparison of two arms over the same queries.\n\nWHY PAIRED. Both arms rank the same 1,367 queries over byte-identical cached\ncandidate graphs. That pairing is the most valuable property of the design and an\nunpaired test throws it away: query difficulty varies enormously here (2 to 6 golds,\n12 disciplines), and unpaired tests charge that variance against the difference you\ncare about. The paired bootstrap resamples QUERIES, so per-query difficulty cancels.\n\nWHY BOOTSTRAP RATHER THAN A t-TEST. MRR, recall@k and completeset@k are bounded,\nheavily tied and nowhere near normal -- completeset@k is 0/1, recall@k takes about\nfour distinct values at 2.35 golds per query. A bootstrap makes no distributional\nclaim; it just resamples.\n\nNULL RESULTS ARE RESULTS. This prints the CI whether or not it crosses zero, and says\nso in words. An interval spanning zero after 1,367 paired queries is a finding about\nthe size of the effect, not a failed run to be re-cut until it clears.\n\nUsage\n-----\n    python3 transfer/paired_bootstrap.py \\\n        --a out/sir4_biology_perquery.json --a-name \"SIR-4 Biology\" \\\n        --b out/tomato_perquery.json       --b-name \"TOMATO-Star\" \\\n        --slice all --metrics mrr,recall@5,completeset@5\n    # restrict to the predeclared strict zero-shot subset\n    python3 transfer/paired_bootstrap.py ... --exclude-disciplines Biology,\"Cell Biology\",Chemistry\n\"\"\"\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport os\nimport random\n\n# SUPERSEDED, kept only so `--exclude-disciplines strict` reproduces the original\n# definition for the write-up's comparison. Dropping ResearchBench's Biology, Cell\n# Biology and Chemistry buckets leaves 990 queries, but 177 of them read biomedical:\n# the buckets do not describe content (Math is 29% biomedical, Law 27%). The live\n# definition is the frozen `strict_zeroshot` list in subsets.json, written by\n# transfer/declare_subsets.py before any model was run. Use --subset.\nSTRICT_EXCLUDE = (\"Biology\", \"Cell Biology\", \"Chemistry\")\nSUBSETS = \"kg-construction/data/researchbench_test/subsets.json\"\n\n\ndef load(path: str) -> dict:\n    d = json.load(open(path))\n    if not isinstance(d, dict):\n        raise SystemExit(f\"{path}: expected the --per-query-out mapping, got {type(d).__name__}\")\n    legacy = 0\n    for qid, row in d.items():\n        # score_sir4.py historically wrote mean gold reciprocal rank as `mrr`,\n        # but also retained conventional first-relevant MRR as `mrr_best`.\n        # Normalise old files in memory so a disconnected Colab runtime can reuse\n        # its saved per-query results without rerunning model inference.\n        if \"mgrr\" not in row and \"mrr_best\" in row:\n            row[\"mgrr\"] = row.get(\"mrr\", 0.0)\n            row[\"mrr\"] = row[\"mrr_best\"]\n            legacy += 1\n        if \"mrr_best\" in row and abs(row.get(\"mrr\", 0.0) - row[\"mrr_best\"]) > 1e-12:\n            raise SystemExit(f\"{path}: query {qid!r} has ambiguous MRR semantics\")\n    if legacy:\n        print(f\"{path}: normalised standard MRR from mrr_best for \"\n              f\"{legacy:,} legacy rows\")\n    return d\n\n\ndef main() -> int:\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\"--a\", required=True, help=\"per-query json for arm A (the claim)\")\n    ap.add_argument(\"--b\", required=True, help=\"per-query json for arm B (the comparator)\")\n    ap.add_argument(\"--a-name\", default=\"A\")\n    ap.add_argument(\"--b-name\", default=\"B\")\n    ap.add_argument(\"--metrics\", default=\"mrr,hits@1,hits@5,recall@1,recall@5,\"\n                                         \"recall@10,recall@20,completeset@5,completeset@10\")\n    ap.add_argument(\"--slice\", default=\"all\",\n                    help=\"restrict to queries whose `slices` list contains this\")\n    ap.add_argument(\"--subset\", default=None,\n                    help=\"name of a frozen subset in subsets.json, e.g. strict_zeroshot\")\n    ap.add_argument(\"--subsets-file\", default=None)\n    ap.add_argument(\"--exclude-disciplines\", default=None,\n                    help=f\"SUPERSEDED, for comparison only; 'strict' = drop {list(STRICT_EXCLUDE)}\")\n    ap.add_argument(\"--iters\", type=int, default=10_000)\n    ap.add_argument(\"--seed\", type=int, default=0)\n    ap.add_argument(\"--json-out\", default=None)\n    a = ap.parse_args()\n\n    A, B = load(a.a), load(a.b)\n\n    # An arm missing queries is a silent denominator change, which is the exact bug\n    # class this experiment's validity checks exist to catch. Refuse rather than\n    # quietly intersect.\n    only_a, only_b = set(A) - set(B), set(B) - set(A)\n    if only_a or only_b:\n        raise SystemExit(\n            f\"arms cover different queries: {len(only_a)} only in {a.a_name}, \"\n            f\"{len(only_b)} only in {a.b_name}. Both arms must score every query \"\n            f\"(zero-seed queries included, never dropped).\")\n\n    ex, keep, label = (), None, \"all queries\"\n    if a.subset:\n        # Read the frozen list. Never recompute it here: a subset that can be\n        # recomputed at report time is a subset that can be tuned at report time.\n        here = os.path.dirname(os.path.abspath(__file__))\n        path = a.subsets_file or os.path.join(\n            os.path.dirname(os.path.dirname(here)), SUBSETS)\n        if not os.path.exists(path):\n            raise SystemExit(f\"no frozen subsets at {path}; \"\n                             f\"run transfer/declare_subsets.py first\")\n        subs = json.load(open(path))[\"subsets\"]\n        if a.subset not in subs:\n            raise SystemExit(f\"{a.subset!r} is not declared; have {list(subs)}\")\n        keep, label = set(subs[a.subset]), f\"subset={a.subset}\"\n    elif a.exclude_disciplines:\n        ex = (STRICT_EXCLUDE if a.exclude_disciplines.strip().lower() == \"strict\"\n              else tuple(s.strip() for s in a.exclude_disciplines.split(\",\")))\n        label = f\"excluding {list(ex)}\"\n\n    qids = [q for q in sorted(A)\n            if a.slice in A[q].get(\"slices\", [\"all\"])\n            and A[q].get(\"discipline\") not in ex\n            and (keep is None or q in keep)]\n    if not qids:\n        raise SystemExit(f\"no queries left after slice={a.slice!r} {label}\")\n    if keep is not None and len(qids) < len(keep):\n        print(f\"note: {len(keep) - len(qids)} of the subset's queries are absent \"\n              f\"from the scored arms (slice filter, or never scored)\\n\")\n\n    metrics = [m.strip() for m in a.metrics.split(\",\")]\n    metrics = [m for m in metrics if m in A[qids[0]] and m in B[qids[0]]]\n\n    print(f\"\\n### {a.a_name}  vs  {a.b_name}\")\n    print(f\"slice={a.slice}  {label}  n={len(qids)}  bootstrap={a.iters:,}\\n\")\n    print(f\"| metric | {a.a_name} | {a.b_name} | diff | 95% CI | verdict |\")\n    print(\"|---|--:|--:|--:|---|---|\")\n\n    rng = random.Random(a.seed)\n    # One resampled index set per iteration, SHARED across metrics, so the columns\n    # of the table are mutually consistent rather than each telling its own story.\n    draws = [[rng.randrange(len(qids)) for _ in range(len(qids))] for _ in range(a.iters)]\n\n    out = {}\n    for m in metrics:\n        da = [A[q][m] for q in qids]\n        db = [B[q][m] for q in qids]\n        diffs = [x - y for x, y in zip(da, db)]\n        obs = sum(diffs) / len(diffs)\n        boot = sorted(sum(diffs[i] for i in idx) / len(idx) for idx in draws)\n        lo, hi = boot[int(.025 * a.iters)], boot[int(.975 * a.iters)]\n        crosses = lo <= 0 <= hi\n        verdict = \"no difference\" if crosses else (\n            f\"**{a.a_name} better**\" if obs > 0 else f\"**{a.b_name} better**\")\n        print(f\"| {m} | {sum(da)/len(da):.4f} | {sum(db)/len(db):.4f} | \"\n              f\"{obs:+.4f} | [{lo:+.4f}, {hi:+.4f}] | {verdict} |\")\n        out[m] = {\"a\": sum(da) / len(da), \"b\": sum(db) / len(db), \"diff\": obs,\n                  \"ci95\": [lo, hi], \"crosses_zero\": crosses, \"n\": len(qids)}\n\n    print(f\"\\npaired bootstrap over {len(qids)} queries, resampling queries.\")\n    print(\"A CI spanning zero means the difference is not resolvable at this sample \"\n          \"size. That is a result; report it.\")\n    if a.json_out:\n        json.dump({\"a_name\": a.a_name, \"b_name\": a.b_name, \"slice\": a.slice,\n                   \"subset\": a.subset, \"excluded\": list(ex), \"n\": len(qids), \"iters\": a.iters,\n                   \"metrics\": out}, open(a.json_out, \"w\"), indent=1)\n        print(f\"\\nwrote {a.json_out}\")\n    return 0\n\n\nif __name__ == \"__main__\":\n    raise SystemExit(main())\n", "kg-construction/eval/cargo_operator.py": "\"\"\"\neval/operator.py \u2014 the CARGO dense OPERATOR on TOMATO (faithful port of\narchive/make_cargo_operator_notebook.py, sections 5-7).\n\n    dense = q . d                          (selected encoder; query instruction on queries)\n    S     = sum_probes [cos(probe, d)]+    (probe-sum)\n    M     = max_probes [cos(probe, d)]+    (probe-max)\n    dem   = leave-one-out probe-sum popularity   (label-free anti-hub denominator)\n    S_op  = w0 . z(dense) + w1 . z(S / dem^b) + w2 . z(M / dem^b)\n\n(w0, w1, w2, b) are fit on a TRAIN slice by InfoNCE and selected on a held-out TRAIN dev slice,\nthen applied to TEST. We report dense retrieval and S_op side by side, so the delta isolates the\nprobe + anti-hub contribution over the selected dense encoder. NO graph, NO API key.\n\nEmbeddings are cached to outputs/caches/op_emb/ (encode once; re-fits are instant).\nReads probes from construct_v2/cache/probes_{train,test}.jsonl (run gen_probes.py first).\n\nRun (Mac solo, or Colab GPU \u2014 auto-detects device):\n    python eval/operator.py --train_fit 2500 --dev 600\n\"\"\"\nimport argparse, hashlib, json, os, sys\nimport numpy as np\nfrom tqdm import tqdm\n\n# CARGO_ROOT so this runs off-laptop (Colab unzips to /content/cargo). Unset,\n# it resolves to exactly the path this replaced.\n_ROOT = os.environ.get(\"CARGO_ROOT\") or os.path.expanduser(\"~/Desktop/CARGO\")\nBASE = f\"{_ROOT}/kg-construction\"\nBGE_QI = \"Represent this sentence for searching relevant passages: \"\n# The Qwen3-Embedding model card's own example task, verbatim. It replaced a\n# hand-written task description that named cross-domain transfer explicitly.\n# Reason: BGE is run with its stock model-card instruction, so a tuned Qwen3\n# instruction made the two dense baselines non-comparable, and any margin over\n# Qwen3 partly measured prompt engineering rather than method.\nQWEN_QI = (\"Instruct: Given a web search query, retrieve relevant passages that \"\n           \"answer the query\\nQuery:\")\nDEFAULT_MODEL = \"BAAI/bge-large-en-v1.5\"\nKS = [1, 5, 10, 25, 50, 100]\n\n# Corpus + cache resolution lives in one module so the pipeline scripts cannot\n# drift. Default dataset \"tomato\" reproduces every path this replaced. The\n# embedding cache is the dangerous one: `{split}_doc.npy` carries no corpus\n# name, so a stale TOMATO matrix would load into a SIR-4 run and pass the only\n# check `cached_encode` makes (row count).\nsys.path.insert(0, _ROOT)\nfrom cargo_paths import (add_dataset_arg, banner, corpus_dir, emb_dir,  # noqa: E402\n                         probes_path, set_dataset)\n\n\ndef model_slug(name):\n    \"\"\"Return a cache suffix so embeddings from different encoders cannot mix.\"\"\"\n    if name == DEFAULT_MODEL:\n        return \"\"\n    import re\n    return \"_\" + re.sub(r\"[^a-z0-9]+\", \"-\", name.lower()).strip(\"-\")\n\n\ndef query_instruction(name):\n    \"\"\"Use the retrieval instruction intended for the selected encoder.\"\"\"\n    return QWEN_QI if \"qwen\" in name.lower() else BGE_QI\n\n\ndef qi_tag(instruct):\n    \"\"\"Instruction fingerprint, for the QUERY cache key only.\n\n    WITHOUT THIS, CHANGING AN INSTRUCTION DOES NOTHING. The query cache is keyed\n    on which query ids are in the split plus the model slug, and cached_encode\n    validates row count and nothing else -- so a new instruction reloads the\n    previous instruction's embeddings and reports them as the new result. Silent,\n    and indistinguishable from \"the instruction did not matter\".\n\n    Documents and probes are encoded with no instruction at all, so only the\n    query cache needs this.\n    \"\"\"\n    return \"_i\" + hashlib.md5(instruct.encode()).hexdigest()[:6]\n\n\ndef load_split(split):\n    \"\"\"Return doc_ids, doc_texts, queries(list of dict id/question/gold/stratum), probes{id:[...]}.\"\"\"\n    root = os.path.join(corpus_dir(split), \"raw\")\n    corpus = json.load(open(os.path.join(root, \"documents.json\")))           # {doc_id: text}\n    queries = json.load(open(os.path.join(root, f\"{split}.json\")))\n    probes = {}\n    for line in open(probes_path(split)):\n        r = json.loads(line); probes[r[\"id\"]] = r[\"probes\"]\n    return list(corpus), corpus, queries, probes\n\n\ndef cached_encode(model, texts, tag, instruct=\"\", batch=64, chunk=1500):\n    \"\"\"Encode in chunks, releasing MPS unified-memory buffers between chunks so the encoder\n    can't snowball into swap (the Apple-Silicon MPS accumulation bug). Cached to .npy.\"\"\"\n    import torch\n    p = os.path.join(emb_dir(), f\"{tag}.npy\")   # emb_dir() makes the dir\n    if os.path.exists(p):\n        e = np.load(p)\n        if e.shape[0] == len(texts):\n            print(f\"  [emb] loaded {tag}: {e.shape}\")\n            return e\n        print(f\"  [emb] {tag} stale ({e.shape[0]} != {len(texts)}), re-encoding\")\n    print(f\"  [emb] encoding {tag}: {len(texts)} texts (chunk={chunk}) ...\")\n    parts = []\n    for i in tqdm(range(0, len(texts), chunk), desc=f\"  enc {tag}\"):\n        sub = [instruct + t for t in texts[i:i + chunk]]\n        v = model.encode(sub, normalize_embeddings=True, batch_size=batch, show_progress_bar=False)\n        parts.append(np.asarray(v, dtype=np.float32))\n        if torch.backends.mps.is_available():\n            torch.mps.empty_cache()             # release GPU buffers so memory stays bounded\n    e = np.concatenate(parts, 0)\n    np.save(p, e)\n    print(f\"  [emb] saved {tag}: {e.shape}\")\n    return e\n\n\ndef signals(qe, de, pe, own, NQ):\n    \"\"\"dense/S/M/dem exactly as the operator notebook (section 5).\"\"\"\n    dense = qe @ de.T\n    ND = de.shape[0]\n    S = np.zeros((NQ, ND), np.float32); M = np.zeros((NQ, ND), np.float32)\n    for i in tqdm(range(NQ), desc=\"  signals\", leave=False):\n        idx = np.where(own == i)[0]\n        if len(idx):\n            H = np.clip(pe[idx] @ de.T, 0, None)          # [n_probes_i, ND], ReLU\n            S[i] = H.sum(0); M[i] = H.max(0)\n    dem = np.clip(S.sum(0, keepdims=True) - S, 1e-6, None)  # leave-one-out popularity\n    return dense.astype(np.float32), S, M, dem.astype(np.float32)\n\n\ndef build_signals(model, model_name, split, q_subset=None):\n    \"\"\"Encode (cached) and assemble signals for a split. q_subset = list of query dicts to use.\"\"\"\n    import hashlib\n    doc_ids, corpus, queries, probes = load_split(split)\n    if q_subset is not None:\n        queries = q_subset\n    qkey = hashlib.md5(\"|\".join(q[\"id\"] for q in queries).encode()).hexdigest()[:8]  # cache by WHICH queries\n    slug = model_slug(model_name)\n    docpos = {d: i for i, d in enumerate(doc_ids)}\n    de = cached_encode(model, [corpus[d] for d in doc_ids], f\"{split}_doc{slug}\")\n    qi = query_instruction(model_name)\n    qe = cached_encode(model, [q[\"question\"] for q in queries],\n                       f\"{split}_query_{qkey}{slug}{qi_tag(qi)}\", instruct=qi)\n    flat, own = [], []\n    for i, q in enumerate(queries):\n        for pr in probes.get(q[\"id\"], []):\n            flat.append(pr); own.append(i)\n    pe = cached_encode(model, flat, f\"{split}_probe_{qkey}{slug}\")\n    own = np.array(own)\n    gold_idx = [[docpos[g] for g in (q.get(\"supporting_documents\") or []) if g in docpos] for q in queries]\n    strat = [q.get(\"stratum\") for q in queries]\n    sig = signals(qe, de, pe, own, len(queries))\n    return sig, gold_idx, strat\n\n\ndef ndcg10(order, gold):\n    g = set(gold)\n    if not g:\n        return 0.0\n    dcg = sum(1 / np.log2(r + 2) for r, d in enumerate(order[:10]) if d in g)\n    idcg = sum(1 / np.log2(r + 2) for r in range(min(len(g), 10)))\n    return dcg / idcg if idcg else 0.0\n\n\ndef evaluate(score, gold_idx, strat, name):\n    order = np.argsort(-score, 1)\n    buck = {\"all\": [], \"same\": [], \"cross\": []}\n    for i, gs in enumerate(gold_idx):\n        if not gs:\n            continue\n        o = list(order[i][:200]); g = set(gs)\n        row = ({k: len(set(o[:k]) & g) / len(g) for k in KS}, ndcg10(o, gs))\n        buck[\"all\"].append(row)\n        if strat[i] in (\"same\", \"cross\"):\n            buck[strat[i]].append(row)\n    print(f\"\\n=== {name} ===\")\n    print(f\"{'stratum':6} \" + \" \".join(f'R@{k:<4}' for k in KS) + \" nDCG  n\")\n    out = {}\n    for s in (\"all\", \"cross\", \"same\"):\n        r = buck[s]\n        if not r:\n            continue\n        mr = {k: 100 * np.mean([x[0][k] for x in r]) for k in KS}\n        nd = 100 * np.mean([x[1] for x in r])\n        out[s] = mr\n        print(f\"{s:6} \" + \" \".join(f'{mr[k]:5.1f}' for k in KS) + f\" {nd:4.1f} {len(r)}\")\n    return out\n\n\ndef main():\n    import torch, torch.nn as nn\n    from sentence_transformers import SentenceTransformer\n\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\"--model\", default=DEFAULT_MODEL)\n    ap.add_argument(\"--train_fit\", type=int, default=2500, help=\"train queries used to fit the 4 params\")\n    ap.add_argument(\"--dev\", type=int, default=600, help=\"held-out train queries for model selection\")\n    ap.add_argument(\"--epochs\", type=int, default=401)\n    ap.add_argument(\"--seed\", type=int, default=0)\n    ap.add_argument(\"--cpu\", action=\"store_true\", help=\"force CPU encoding (steady, no MPS swap blowup)\")\n    add_dataset_arg(ap)\n    a = ap.parse_args()\n    set_dataset(a.dataset)\n    print(banner())\n\n    dev_t = \"cuda\" if torch.cuda.is_available() else (\"mps\" if torch.backends.mps.is_available() else \"cpu\")\n    enc_dev = \"cpu\" if a.cpu else dev_t\n    print(f\"compute={dev_t} | encode={enc_dev} | model={a.model}\")\n    model = SentenceTransformer(a.model, device=enc_dev)\n    model.max_seq_length = 512\n\n    # ---- TRAIN: sample fit + dev queries (shared train corpus), build signals ----\n    _, _, train_q, _ = load_split(\"train\")\n    rng = np.random.default_rng(a.seed); rng.shuffle(train_q)\n    dev_q = train_q[:a.dev]\n    fit_q = train_q[a.dev:a.dev + a.train_fit]\n    print(f\"\\n[train] fit={len(fit_q)} dev={len(dev_q)} queries (corpus shared)\")\n    (fD, fS, fM, fDe), fgold, _ = build_signals(model, a.model, \"train\", q_subset=fit_q)\n    (dD, dS, dM, dDe), dgold, _ = build_signals(model, a.model, \"train\", q_subset=dev_q)\n\n    # ---- operator (4 params), InfoNCE on fit, select on dev nDCG@10 ----\n    DEVt = dev_t\n    T = lambda x: torch.tensor(x, device=DEVt)\n    def zr(X): return (X - X.mean(1, keepdim=True)) / (X.std(1, keepdim=True) + 1e-6)\n    class Operator(nn.Module):\n        def __init__(s):\n            super().__init__(); s.logbeta = nn.Parameter(torch.zeros(1))\n            s.w = nn.Parameter(torch.tensor([1., 1., 0.2]))\n        def forward(s, dense, ssum, smax, dem):\n            degb = dem.clamp_min(1e-6) ** torch.exp(s.logbeta)\n            return s.w[0] * zr(dense) + s.w[1] * zr(ssum / degb) + s.w[2] * zr(smax / degb)\n\n    fDt, fSt, fMt, fDet = T(fD), T(fS), T(fM), T(fDe)\n    dDt, dSt, dMt, dDet = T(dD), T(dS), T(dM), T(dDe)\n    qi, gi = [], []\n    for i, gs in enumerate(fgold):\n        for g in gs:\n            qi.append(i); gi.append(g)\n    qi, gi = torch.tensor(qi, device=DEVt), torch.tensor(gi, device=DEVt)\n\n    m = Operator().to(DEVt); opt = torch.optim.Adam(m.parameters(), lr=0.05)\n    best, bsd, stale = -1, None, 0\n    # Stop once the dev score stops improving. The 401 gradient steps are cheap\n    # (four parameters), but each evaluation scores 600 dev queries against the\n    # whole corpus and argsorts each one in Python -- that is the real cost, and\n    # it runs 21 times. The fit typically plateaus by ~ep 40. Set OP_PATIENCE=0\n    # to disable and run the full schedule.\n    PATIENCE = int(os.environ.get(\"OP_PATIENCE\", \"3\"))\n    for ep in range(a.epochs):\n        m.train(); opt.zero_grad()\n        loss = -torch.log_softmax(m(fDt, fSt, fMt, fDet), 1)[qi, gi].mean()\n        loss.backward(); opt.step()\n        if ep % 20 == 0:\n            with torch.no_grad():\n                nd = np.mean([ndcg10(list((-m(dDt, dSt, dMt, dDet).cpu().numpy()[i]).argsort()), dgold[i])\n                              for i in range(len(dgold)) if dgold[i]])\n            tag = \"\"\n            if nd > best:\n                best = nd; bsd = {k: v.clone() for k, v in m.state_dict().items()}; tag = \"*\"\n                stale = 0\n            else:\n                stale += 1\n            print(f\"ep{ep:3d} loss {loss.item():.3f} dev nDCG@10 {nd:.3f} \"\n                  f\"beta {torch.exp(m.logbeta).item():.2f} w {m.w.detach().cpu().numpy().round(2)} {tag}\")\n            if PATIENCE and stale >= PATIENCE:\n                print(f\"[early stop] ep{ep}: no dev gain for {stale} evals \"\n                      f\"({stale * 20} epochs); keeping best {best:.3f}\")\n                break\n    m.load_state_dict(bsd)\n    print(f\"\\nLEARNED beta={torch.exp(m.logbeta).item():.3f} w={m.w.detach().cpu().numpy().round(3)} best dev {best:.3f}\")\n\n    # SAVE the fitted parameters. They used to be printed and nothing else, while\n    # FusionGraphReasoner warm-starts w and beta from hardcoded W_INIT/BETA_INIT\n    # fitted on TOMATO. So a SIR-4 fusion run began from TOMATO's calibration and\n    # this fit was thrown away. Written per-dataset so one corpus can never read\n    # another's, same contract as cargo_paths.\n    # ALSO SCOPED BY ENCODER. w and beta are fitted against one encoder's score\n    # distributions, so a Qwen3 fit and a BGE fit are different calibrations of the\n    # same four parameters. Sharing a filename would let a refit destroy the values\n    # that already-trained checkpoints were warm-started from, with nothing to show\n    # it happened. Empty slug for the default encoder keeps existing paths intact.\n    sfx = (\"\" if a.dataset == \"tomato\" else f\"_{a.dataset}\") + model_slug(a.model)\n    params = {\"dataset\": a.dataset, \"encoder\": a.model,\n              \"w\": [round(float(x), 6) for x in m.w.detach().cpu().numpy()],\n              \"beta\": round(float(torch.exp(m.logbeta).item()), 6),\n              \"dev_ndcg10\": round(float(best), 6),\n              \"train_fit\": a.train_fit, \"dev\": a.dev}\n    pp = os.path.join(BASE, \"eval\", f\"operator_params{sfx}.json\")\n    json.dump(params, open(pp, \"w\"), indent=1)\n    print(f\"[params] wrote {pp}\")\n    print(f\"[params] fusion warm start -> W_INIT={tuple(params['w'])} BETA_INIT={params['beta']}\")\n\n    # ---- TEST: signals on test corpus, apply operator, score by stratum ----\n    print(\"\\n[test] building signals ...\")\n    (teD, teS, teM, teDe), tegold, testrat = build_signals(model, a.model, \"test\")\n    with torch.no_grad():\n        S_op = m(T(teD), T(teS), T(teM), T(teDe)).cpu().numpy()\n    summary = {}\n    dense_name = f\"{a.model} (dense)\"\n    operator_name = f\"Operator ({a.model}; probes + anti-hub)\"\n    summary[dense_name] = evaluate(teD, tegold, testrat, dense_name)\n    summary[operator_name] = evaluate(S_op, tegold, testrat, operator_name)\n    out = os.path.join(BASE, \"eval\", f\"operator_results{sfx}.json\")\n    json.dump(summary, open(out, \"w\"), indent=2)\n    # save per-query operator rankings (for downstream operator-seeded PPR)\n    te_doc_ids, _, te_q, _ = load_split(\"test\")\n    preds = []\n    for i, q in enumerate(te_q):\n        order = np.argsort(-S_op[i])[:100]\n        preds.append({\"id\": q[\"id\"], \"stratum\": q.get(\"stratum\"),\n                      \"supporting_documents\": q.get(\"supporting_documents\", []),\n                      \"predictions\": {\"document\": [[te_doc_ids[j], float(S_op[i, j])] for j in order]}})\n    json.dump(preds, open(os.path.join(BASE, \"eval\", f\"predictions_operator{sfx}.json\"), \"w\"))\n    print(f\"wrote eval/predictions_operator{sfx}.json  encoder={a.model}\")\n    bo = summary[operator_name]; bg = summary[dense_name]\n    print(f\"\\nHEADLINE cross R@10:  operator {bo['cross'][10]:.1f}  vs  dense {bg['cross'][10]:.1f}  \"\n          f\"(HyDE 25.3) | wrote {out}\")\n\n\nif __name__ == \"__main__\":\n    main()\n", "kg-construction/experiments/probe_greasoner/precompute_operator_components.py": "\"\"\"\nprecompute_operator_components.py \u2014 cache the operator's RAW ingredients (not the final score) so the\noperator's weights w=[w0,w1,w2] and exponent beta can be LEARNED jointly inside FusionGraphReasoner.\n\nThe operator score is   S_op = w0 z(dense) + w1 z(S/dem^beta) + w2 z(M/dem^beta),  where\n  dense = q . d            (query-doc similarity in the selected encoder space)\n  S, M  = probe-sum / probe-max similarity   (HyDE bridge probes vs docs)\n  dem   = total_S - S      (leave-one-out anti-hub popularity; total_S = S.sum over queries)\nCaching dense, S, M and total_S lets the wrapper recompute S_op live with LEARNABLE w, beta (dem is\nderived as total_S - S). All arrays are aligned to the graph's nodes.csv document order.\n\nSaves data/<graph>/operator_components.npz {dense,S,M: float16 [Q x n_doc], total_S: float32 [n_doc],\nquery_ids:[...]}. Local, no API (reuses the op_emb caches). Mirrors precompute_operator_scores.py.\n  python experiments/probe_greasoner/precompute_operator_components.py --graph tomato_train_v16sc --split train\n  python experiments/probe_greasoner/precompute_operator_components.py --graph tomato_test_v16sc  --split test\n\"\"\"\nimport argparse, csv, hashlib, json, os, sys\nimport numpy as np\nfrom tqdm import tqdm\n\n# CARGO_ROOT so this runs off-laptop (Colab unzips to /content/cargo). Unset,\n# it resolves to exactly the path this replaced.\n_ROOT = os.environ.get(\"CARGO_ROOT\") or os.path.expanduser(\"~/Desktop/CARGO\")\nBASE = f\"{_ROOT}/kg-construction\"\nsys.path.insert(0, BASE)\n# One resolver for corpus + caches; default \"tomato\" reproduces every legacy path.\nsys.path.insert(0, _ROOT)\nfrom cargo_paths import add_dataset_arg, banner, corpus_dir, emb_dir, probes_path, set_dataset  # noqa: E402\ncsv.field_size_limit(10 ** 7)\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\"--graph\", required=True, help=\"graph dir (for doc-node order), e.g. tomato_train_v16sc\")\n    ap.add_argument(\"--split\", default=\"train\", choices=[\"train\", \"test\"])\n    ap.add_argument(\"--model\", default=\"BAAI/bge-large-en-v1.5\",\n                    help=\"sentence-transformer used for dense and probe similarities\")\n    add_dataset_arg(ap)\n    a = ap.parse_args()\n    set_dataset(a.dataset)\n    print(banner())\n\n    s1 = f\"{BASE}/data/{a.graph}/processed/stage1\"\n    docnodes = [n[\"name\"] for n in csv.DictReader(open(f\"{s1}/nodes.csv\")) if n[\"type\"] == \"document\"]\n    corpus = json.load(open(f\"{corpus_dir(a.split)}/raw/documents.json\"))\n    doc_ids = list(corpus)\n    d2i = {d: i for i, d in enumerate(doc_ids)}\n    assert all(n in d2i for n in docnodes), \"graph has document nodes not in documents.json\"\n    col = [d2i[name] for name in docnodes]              # corpus order -> nodes.csv doc order\n    raw = json.load(open(f\"{corpus_dir(a.split)}/raw/{a.split}.json\"))\n    qids = [q[\"id\"] for q in raw]\n    qkey = hashlib.md5(\"|\".join(qids).encode()).hexdigest()[:8]\n\n    probes = {json.loads(l)[\"id\"]: json.loads(l)[\"probes\"]\n              for l in open(probes_path(a.split))}\n\n    # ENCODE ON MISS, rather than assuming a previous run happened to use the\n    # same query subset. The embedding cache is keyed by an md5 of the query\n    # ids, and cargo_operator.py only ever encodes its FIT and DEV samples\n    # (e.g. 30 + 10 of 50, or 2,500 + 600 of 5,445). This script needs ALL of\n    # them, so its hash never matched and the load died with a bare\n    # FileNotFoundError naming a hash that nothing had produced.\n    import importlib.util\n    spec = importlib.util.spec_from_file_location(\"op\", f\"{BASE}/eval/cargo_operator.py\")\n    op = importlib.util.module_from_spec(spec)\n    spec.loader.exec_module(op)                          # same encoder, instruction and cache key\n    slug = op.model_slug(a.model)\n    qi    = op.query_instruction(a.model)\n    qslug = f\"{slug}{op.qi_tag(qi)}\"        # the instruction is part of what queries ARE\n    dpath = f\"{emb_dir()}/{a.split}_doc{slug}.npy\"\n    qpath = f\"{emb_dir()}/{a.split}_query_{qkey}{qslug}.npy\"\n    ppath = f\"{emb_dir()}/{a.split}_probe_{qkey}{slug}.npy\"\n    if not all(os.path.exists(p) for p in (dpath, qpath, ppath)):\n        from sentence_transformers import SentenceTransformer\n        print(f\"[op-comp] {a.model} full-split embeddings absent for {a.split} \"\n              f\"({len(qids)} queries); encoding once\")\n        model = SentenceTransformer(a.model)\n        model.max_seq_length = 512\n        op.cached_encode(model, [corpus[d] for d in doc_ids], f\"{a.split}_doc{slug}\")\n        op.cached_encode(model, [q[\"question\"] for q in raw],\n                         f\"{a.split}_query_{qkey}{qslug}\", instruct=qi)\n        flat = [pr for q in raw for pr in probes.get(q[\"id\"], [])]\n        op.cached_encode(model, flat, f\"{a.split}_probe_{qkey}{slug}\")\n\n    de = np.load(dpath).astype(np.float32)\n    qe = np.load(qpath).astype(np.float32)\n    pe = np.load(ppath).astype(np.float32)\n    own = []\n    for q in raw:\n        own += [q[\"id\"]] * len(probes.get(q[\"id\"], []))\n    own = np.array(own)\n\n    Q, D = len(raw), len(doc_ids)\n    print(f\"[op-comp] {Q} queries x {D} docs; computing raw components ...\", flush=True)\n    print(f\"[op-comp] dense: one {Q}x{len(qe[0])} @ {len(de[0])}x{D} matmul ...\", flush=True)\n    dense = qe @ de.T                                    # [Q, D]\n    S = np.zeros((Q, D), np.float32); M = np.zeros((Q, D), np.float32)\n    # The probe loop is the long pole: one matmul per query against the whole\n    # corpus. It ran completely silently, so a 5,445-query train split looked\n    # indistinguishable from a hang for several minutes.\n    for i, q in enumerate(tqdm(raw, desc=\"[op-comp] probes\", unit=\"q\")):\n        idx = np.where(own == q[\"id\"])[0]\n        if len(idx):\n            H = np.clip(pe[idx] @ de.T, 0, None); S[i] = H.sum(0); M[i] = H.max(0)\n\n    # reorder columns to nodes.csv document order (so they align with graph.nodes_by_type['document'])\n    dense = dense[:, col]; S = S[:, col]; M = M[:, col]\n    total_S = S.sum(0)                                   # [D] popularity (for dem = total_S - S)\n\n    # MODEL-SCOPED, because the components ARE the encoder's output. Writing every\n    # encoder to one filename would let a Qwen3 run silently clobber the BGE\n    # components that already-trained checkpoints were calibrated against -- and the\n    # ResearchBench transfer is consuming exactly those right now. The slug is empty\n    # for the default encoder, so existing paths are unchanged.\n    out = f\"{BASE}/data/{a.graph}/operator_components{slug}.npz\"\n    # Record the instruction, not just the encoder. `dense` is the only array the\n    # instruction touches (probes and documents are encoded without one), so a\n    # components file built under a different instruction is a different baseline\n    # wearing the same filename. Consumers can now check instead of assuming.\n    np.savez_compressed(out,\n                        dense=dense.astype(np.float16), S=S.astype(np.float16), M=M.astype(np.float16),\n                        total_S=total_S.astype(np.float32), query_ids=np.array(qids),\n                        encoder=np.array(a.model), query_instruct=np.array(qi))\n    sz = os.path.getsize(out) / 1e6\n    print(f\"[op-comp] saved {out}  ({sz:.0f} MB)  dense/S/M={dense.shape} f16, total_S={total_S.shape} f32\")\n    print(f\"[op-comp] env:  OPERATOR_COMPONENTS{'_TEST' if a.split=='test' else ''}={out}\")\n\n\nif __name__ == \"__main__\":\n    main()\n", "kg-construction/experiments/probe_greasoner/precompute_semantic_components.py": "\"\"\"\nprecompute_semantic_components.py \u2014 cache what the LEARNED semantic scorer needs inside the\ngraph fusion, aligned to the graph's nodes.csv document order.\n\nThe operator version of this script (precompute_operator_components.py) caches S and M, the two\nhandcrafted summaries of the hypothetical-answer matches. The sorted-MLP does not use summaries:\nit reads the whole ordered match profile, so it needs the FULL per-answer matrix\n\n    H  [Q, Jmax, n_doc]   float16   ReLU(cos(hypothetical answer j, document d))\n\nplus the direct query-document similarity, the answer-validity mask, and the document embeddings\nthat the popularity predictor p_hat(d) = softplus(g(E(d))) reads.\n\nH IS NOT COPIED. semantic_scorer.py already materialises exactly this matrix as a memmap under\noutputs/caches/semantic/<dataset>/, and at CS scale it is 1.76 GB \u2014 duplicating it per graph would\ncost more disk than every other artefact in the project combined, and would introduce a second copy\nthat can go stale. This script instead records the memmap's PATH and the column permutation `col`\nthat maps corpus document order to nodes.csv document order, and the fusion applies `col` to the\nfew rows it slices per batch. Everything small (dense, mask, doc_emb, total_S) is reordered here\nand stored outright, because those are cheap and reordering them per batch would not be.\n\nRun it AFTER the semantic scorer (notebook section 5d), which is what builds the memmap and trains\nthe checkpoint the fusion warm-starts from. If the memmap is absent this rebuilds it from the\ncached embeddings; if the embeddings are absent too it will encode, which is the slow path.\n\n  python experiments/probe_greasoner/precompute_semantic_components.py \\\n      --dataset sir4_physics --model /content/qwen3 --graph sir4_physics_train_v16sc --split train\n\"\"\"\nimport argparse\nimport csv\nimport hashlib\nimport importlib.util\nimport json\nimport os\nimport sys\n\nimport numpy as np\n\n# CARGO_ROOT so this runs off-laptop (Colab unzips to /content/cargo). Unset, it\n# resolves to exactly the path this replaced.\n_ROOT = os.environ.get(\"CARGO_ROOT\") or os.path.expanduser(\"~/Desktop/CARGO\")\nBASE = f\"{_ROOT}/kg-construction\"\nsys.path.insert(0, BASE)\nsys.path.insert(0, _ROOT)\nfrom cargo_paths import add_dataset_arg, banner, corpus_dir, emb_dir, set_dataset  # noqa: E402\n\ncsv.field_size_limit(10 ** 7)\n\n\ndef _load(name, path):\n    spec = importlib.util.spec_from_file_location(name, path)\n    mod = importlib.util.module_from_spec(spec)\n    spec.loader.exec_module(mod)\n    return mod\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\"--graph\", required=True, help=\"graph dir supplying nodes.csv document order\")\n    ap.add_argument(\"--split\", default=\"train\", choices=[\"train\", \"test\"])\n    ap.add_argument(\"--model\", default=\"BAAI/bge-large-en-v1.5\")\n    ap.add_argument(\"--force\", action=\"store_true\", help=\"rebuild H even if cached\")\n    add_dataset_arg(ap)\n    a = ap.parse_args()\n    set_dataset(a.dataset)\n    print(banner())\n\n    op = _load(\"op\", f\"{BASE}/eval/cargo_operator.py\")\n    sem = _load(\"sem\", f\"{_ROOT}/sir4-retrieval/eval/semantic_scorer.py\")\n    slug = op.model_slug(a.model)\n\n    # ---- the graph's document order -------------------------------------------------\n    s1 = f\"{BASE}/data/{a.graph}/processed/stage1\"\n    docnodes = [n[\"name\"] for n in csv.DictReader(open(f\"{s1}/nodes.csv\")) if n[\"type\"] == \"document\"]\n    corpus = json.load(open(f\"{corpus_dir(a.split)}/raw/documents.json\"))\n    doc_ids = list(corpus)\n    d2i = {d: i for i, d in enumerate(doc_ids)}\n    assert all(n in d2i for n in docnodes), \"graph has document nodes not in documents.json\"\n    col = np.asarray([d2i[name] for name in docnodes], dtype=np.int64)   # corpus -> nodes.csv\n\n    # ---- the semantic scorer's own inputs, in CORPUS order ---------------------------\n    cache = f\"{_ROOT}/outputs/caches/semantic/{a.dataset}\"\n    data = sem.build_inputs(op, a.model, a.split, cache, force=a.force)\n    meta = data[\"meta\"]\n    Q, Jmax, D = data[\"H\"].shape\n    assert D == len(doc_ids), f\"H has {D} document columns, corpus has {len(doc_ids)}\"\n    assert len(data[\"queries\"]) == Q\n\n    # The memmap path is reconstructed from the SAME key build_inputs used, so a cache\n    # written under a different encoder or a reordered corpus cannot be picked up here.\n    tag = f\"{a.split}_{meta['qkey']}_{meta['dockey']}{slug}\"\n    h_path = f\"{cache}/semantic_H_{tag}.f16\"\n    assert os.path.exists(h_path), f\"H memmap missing: {h_path}\"\n\n    # ---- document embeddings, for the popularity predictor ---------------------------\n    de = np.load(f\"{emb_dir()}/{a.split}_doc{slug}.npy\").astype(np.float32)\n    assert de.shape[0] == D, f\"doc embeddings {de.shape[0]} rows != {D} documents\"\n\n    qids = [q[\"id\"] for q in data[\"queries\"]]\n    raw_qids = [q[\"id\"] for q in json.load(open(f\"{corpus_dir(a.split)}/raw/{a.split}.json\"))]\n    assert qids == raw_qids, \"build_inputs query order differs from the raw split order\"\n    assert meta[\"qkey\"] == hashlib.md5(\"|\".join(qids).encode()).hexdigest()[:8]\n\n    out = f\"{BASE}/data/{a.graph}/semantic_components{slug}.npz\"\n    np.savez_compressed(\n        out,\n        # reordered to nodes.csv document order, exactly like the operator components\n        dense=np.asarray(data[\"dense\"])[:, col].astype(np.float16),\n        doc_emb=de[col].astype(np.float16),\n        total_S=np.asarray(data[\"total_S\"])[col].astype(np.float32),\n        mask=np.asarray(data[\"mask\"]),                       # [Q, Jmax] bool, order-independent\n        # H stays where semantic_scorer.py put it; the fusion applies `col` per batch\n        col=col, h_path=np.array(h_path), h_shape=np.array([Q, Jmax, D]),\n        query_ids=np.array(qids), Jmax=np.array(Jmax),\n        encoder=np.array(a.model), query_instruct=np.array(meta[\"query_instruct\"]),\n        dockey=np.array(meta[\"dockey\"]),\n    )\n    sz = os.path.getsize(out) / 1e6\n    print(f\"[sem-comp] saved {out}  ({sz:.0f} MB)\")\n    print(f\"[sem-comp]   dense {(Q, len(col))} f16 | doc_emb {(len(col), de.shape[1])} f16 \"\n          f\"| mask {(Q, Jmax)} bool\")\n    print(f\"[sem-comp]   H referenced at {h_path} ({os.path.getsize(h_path)/1e6:.0f} MB, not copied)\")\n    print(f\"[sem-comp] env:  SEMANTIC_COMPONENTS{'_TEST' if a.split == 'test' else ''}={out}\")\n\n\nif __name__ == \"__main__\":\n    main()\n"}""")
ov = f"{DRIVE}/code_overlay"
if os.path.isdir(ov):
    shutil.copytree(ov, CARGO_ROOT, dirs_exist_ok=True); print("applied Drive code_overlay")
for rel, src in OVERLAY.items():
    p = f"{CARGO_ROOT}/{rel}"
    os.makedirs(os.path.dirname(p), exist_ok=True); open(p, "w").write(src)
print(f"installed {len(OVERLAY)} repo scripts captured 2026-09-08 08:55")
sys.path.insert(0, CARGO_ROOT)
import cargo_paths as cp
for d in DATASETS:
    for g in [SPEC[d]["frame"]] + ([SPEC[d]["openie"]] if SPEC[d]["openie"] else []):
        ok = os.path.exists(f"{DATA_ROOT}/{g}/processed/stage1/nodes.csv")
        print(f"  {'ok ' if ok else 'MISSING'}  {d:13} graph {g}")


## 2b. Qwen3-Embedding (cached on Drive)

In [ ]:
# Only runs when cell 1 found a scan or a path search still to do; drawing the figures from
# cached interpretations needs neither the engine nor Qwen3.
if NEED_ENGINE:
    import os, shutil
    QDIR = "/content/qwen3"
    DRIVE_QWEN = f"{DRIVE}/qwen3-embedding-0.6b"
    BASE = "https://huggingface.co/Qwen/Qwen3-Embedding-0.6B/resolve/main"
    TOK = os.environ.get("HF_TOKEN", "")
    AUTH = f'-H "Authorization: Bearer {TOK}"' if TOK else ""
    NEED = ["model.safetensors","config.json","config_sentence_transformers.json","modules.json",
            "tokenizer.json","tokenizer_config.json","vocab.json","merges.txt","1_Pooling/config.json"]

    def ready(d):
        return (all(os.path.exists(f"{d}/{f}") for f in NEED)
                and os.path.getsize(f"{d}/model.safetensors") > 1_000_000_000
                and os.path.getsize(f"{d}/tokenizer.json") > 11_000_000)

    if not ready(QDIR) and ready(DRIVE_QWEN):
        print("restoring Qwen3 from Drive cache ..."); shutil.copytree(DRIVE_QWEN, QDIR, dirs_exist_ok=True)

    if not ready(QDIR):
        os.makedirs(f"{QDIR}/1_Pooling", exist_ok=True)
        # 1) big weight via aria2c (run ONCE — a repeat can delete the finished file)
        if not (os.path.exists(f"{QDIR}/model.safetensors") and os.path.getsize(f"{QDIR}/model.safetensors") > 1_000_000_000):
            os.system("apt-get -qq install -y aria2")
            os.system(f'aria2c -x16 -s16 -k1M --max-tries=5 --retry-wait=2 --file-allocation=none '
                      f'{AUTH} -d {QDIR} -o model.safetensors "{BASE}/model.safetensors"')
        # 2) small files bypass Xet — plain curl
        for f in ["config.json","config_sentence_transformers.json","modules.json",
                  "tokenizer_config.json","vocab.json","merges.txt","1_Pooling/config.json"]:
            os.system(f'curl -sSL -f {AUTH} "{BASE}/{f}" -o "{QDIR}/{f}"')
        # 3) tokenizer.json (11 MB, also Xet) — retry until a request lands on the good CDN
        for i in range(20):
            os.system(f"rm -f {QDIR}/tokenizer.json")
            os.system(f'curl -sSL -f {AUTH} "{BASE}/tokenizer.json" -o {QDIR}/tokenizer.json')
            if os.path.exists(f"{QDIR}/tokenizer.json") and os.path.getsize(f"{QDIR}/tokenizer.json") > 11_000_000:
                print(f"tokenizer.json ok on try {i+1}"); break
        assert ready(QDIR), "Qwen3 incomplete — re-run this cell (aria2c may need another pass)"
        os.makedirs(os.path.dirname(DRIVE_QWEN), exist_ok=True)
        shutil.copytree(QDIR, DRIVE_QWEN, dirs_exist_ok=True); print("cached Qwen3 to Drive")

    # load by LOCAL PATH — never by hub name again
    from sentence_transformers import SentenceTransformer
    _m = SentenceTransformer(QDIR)
    print("Qwen3 loaded offline:", _m.encode(["test"], normalize_embeddings=True).shape)  # (1, 1024)
    del _m
else:
    print('skipped: every scan and path file is already on Drive, so the engine is not needed')


## 3. Engine

In [ ]:
# Only runs when cell 1 found a scan or a path search still to do; drawing the figures from
# cached interpretations needs neither the engine nor Qwen3.
if NEED_ENGINE:
    import os, sys, torch
    !rm -rf /content/gfm-rag
    !cd /content && unzip -q {DRIVE}/gfm-rag-adapted.zip
    !pip install -q --no-deps -e /content/gfm-rag
    # TORCH IS DELIBERATELY NOT IN THIS LIST. Colab ships a torch/torchvision pair built
    # against each other, and asking pip for `torch` can move torch off the version its
    # torchvision was compiled for. THAT mismatch is what produced
    #   ImportError: cannot import name 'VideoReader'
    # from datasets' torch formatter. gfmrag installs --no-deps, so nothing here needs a
    # torch newer than the host image's.
    !pip install -q torch-geometric sentence-transformers transformers hydra-core omegaconf \
                  easydict ninja faiss-cpu pymetis wandb tqdm numpy pandas python-dotenv \
                  langchain-community 2>&1 | tail -3

    # REPAIR, NEVER REMOVE. This used to be `pip uninstall -y torchvision`, on the reasoning
    # that gfmrag does not use it. That reasoning has expired: current transformers resolves
    # PreTrainedModel through a lazy module that imports torchvision, so deleting it turns
    # every `from transformers import ...` into
    #   ModuleNotFoundError: Could not import module 'PreTrainedModel'
    # and takes sentence-transformers, and therefore every encoder in this notebook, down with
    # it. Verify the pair instead, and only intervene if it is actually broken.
    import torch
    try:
        import torchvision
        from transformers import PreTrainedModel          # the import that has to work
        print(f"torch {torch.__version__} | torchvision {torchvision.__version__} | transformers ok")
    except Exception as _e:
        # STOP, DO NOT SELF-HEAL. The obvious repair, `pip install torchvision`, resolves to
        # the LATEST torchvision and drags torch up with it: measured on 2026-08-17 it took a
        # stock runtime from torch 2.11.0+cu128 to 2.13.0+cu130, a 2 GB download that rebuilt
        # the whole CUDA stack, broke Colab's cudf/cuml/raft pins, and left the running kernel
        # holding the OLD torch. Silently re-pinning CUDA under a training run is far worse
        # than refusing, because the damage only surfaces as `torch.cuda.is_available()` going
        # False, or as numerics nobody can reproduce.
        #
        # A matched pair is what the stock image already ships. Getting back to it is one
        # menu action, and no pip incantation is more reliable than that.
        raise RuntimeError(
            f"torchvision/transformers are broken in this runtime "
            f"({type(_e).__name__}: {_e}).\n"
            f"torch here is {torch.__version__}.\n"
            f"FIX: Runtime > Disconnect and DELETE runtime (not 'Restart session' -- a restart "
            f"keeps whatever pip did), then run this notebook from the top. The stock image "
            f"ships a matched torch/torchvision pair and this cell installs neither, so the "
            f"check above will pass with no downloads.\n"
            f"Do NOT `pip install torchvision` to get past this: it upgrades torch and CUDA "
            f"underneath you."
        ) from _e

    # THE OTHER HALF OF THE TORCHVISION STORY. datasets' torch formatter runs
    # `from torchvision.io import VideoReader` whenever torchvision is importable, and
    # current torchvision has REMOVED VideoReader. That import sits on the training
    # dataloader's hot path, so with torchvision present every training run dies on its
    # first batch. Uninstalling torchvision was the old workaround and now breaks
    # transformers instead (see above), so the remaining move is to patch datasets ON
    # DISK: guard the import, skip the isinstance when it is unavailable. The training
    # subprocess re-imports datasets from disk, so the patch reaches it with no restart.
    import re as _re
    import datasets.formatting.torch_formatter as _dtf
    _p = _dtf.__file__
    _s = open(_p).read()
    if "VideoReader = None" in _s:
        print("datasets torch formatter already patched")
    else:
        _s2 = _re.sub(r'^( *)from torchvision\.io import VideoReader$',
                      lambda m: (f"{m.group(1)}try:\n{m.group(1)}    from torchvision.io import VideoReader\n"
                                 f"{m.group(1)}except Exception:\n{m.group(1)}    VideoReader = None"),
                      _s, flags=_re.M)
        _s2 = _s2.replace("isinstance(value, VideoReader)",
                          "(VideoReader is not None and isinstance(value, VideoReader))")
        assert _s2 != _s, "VideoReader import not found -- datasets layout changed, patch by hand"
        open(_p, "w").write(_s2)
        print("patched datasets torch formatter:", _p)
    # Replay the exact failing path in a fresh interpreter: torchvision imported (that is
    # what arms the buggy branch), then a torch-formatted Dataset read.
    import subprocess as _sp
    _r = _sp.run([sys.executable, "-c",
                  "import torchvision, datasets\n"
                  "d = datasets.Dataset.from_dict({'x': [1, 2, 3]}).with_format('torch')\n"
                  "print('datasets formatter ok:', d[:2]['x'])"],
                 capture_output=True, text=True)
    print(_r.stdout.strip())
    assert _r.returncode == 0, _r.stderr[-2000:]

    def soft_import(path, line, fallback):
        t = open(path).read()
        if f"try:\n    {line}" not in t:
            open(path, "w").write(t.replace(line, f"try:\n    {line}\nexcept Exception:\n    {fallback}"))
    soft_import("/content/gfm-rag/gfmrag/text_emb_models/__init__.py",
                "from .qwen3_model import Qwen3TextEmbModel", "Qwen3TextEmbModel = None")
    # pylate/ColBERT entity-linker is unused by SFT training; make its import non-fatal so the
    # training subprocess (fresh Python re-imports gfmrag) does not crash on `import pylate`.
    soft_import("/content/gfm-rag/gfmrag/graph_index_construction/entity_linking_model/__init__.py",
                "from .colbert_el_model import ColbertELModel", "ColbertELModel = None")
    # 4c. LLM-OpenIE model imports langchain_community (ChatOllama/ChatLlamaCpp); the installed version
    #     dropped ChatOllama. SFT training never builds an index, so make this import non-fatal.
    soft_import("/content/gfm-rag/gfmrag/graph_index_construction/openie_model/__init__.py",
                "from .llm_openie_model import LLMOPENIEModel", "LLMOPENIEModel = None")
    # 4d. same for the LLM-NER model (separate __init__, separate import line).
    soft_import("/content/gfm-rag/gfmrag/graph_index_construction/ner_model/__init__.py",
                "from .llm_ner_model import LLMNERModel", "LLMNERModel = None")
    # the adapted zip strips config/wandb/ (and sometimes config/text_emb_model/) -> create before writing
    for _cd in ["/content/gfm-rag/gfmrag/workflow/config/wandb",
                "/content/gfm-rag/gfmrag/workflow/config/text_emb_model"]:
        os.makedirs(_cd, exist_ok=True)
    open("/content/gfm-rag/gfmrag/workflow/config/wandb/default.yaml", "w").write(
        'enabled: false\nlog_model: false\nproject: "gfm-rag"\nentity: null\nname: null\ngroup: null\ntags: []\nnotes: ""\n')
    open("/content/gfm-rag/gfmrag/workflow/config/text_emb_model/qwen3_st.yaml", "w").write(
        '_target_: gfmrag.text_emb_models.BaseTextEmbModel\n'
        'text_emb_model_name: /content/qwen3\nnormalize: True\nbatch_size: 32\n'   # LOCAL path, not hub name (see cell 3)
        'query_instruct: "Instruct: Given a scientific research problem or open need, retrieve papers whose method, mechanism, or technique could be borrowed as inspiration, including transfers from other domains.\\nQuery: "\n'
        'passage_instruct: null\nmodel_kwargs: null\n')
    sys.path.insert(0, "/content/gfm-rag")
    # stub the unused pylate/ColBERT dep so the IN-KERNEL gfmrag import does not crash
    import types as _t
    for _m in ["pylate", "pylate.indexes", "pylate.models", "pylate.retrieve"]:
        sys.modules.setdefault(_m, _t.ModuleType(_m))
    class _D:
        def __init__(self, *a, **k): pass
    sys.modules["pylate.indexes"].PLAID = _D; sys.modules["pylate.models"].ColBERT = _D; sys.modules["pylate.retrieve"].ColBERT = _D
    from gfmrag.models.gfm_reasoner import GraphReasoner
    assert torch.cuda.is_available(), "Use an A100/high-RAM GPU"
    print("G-Reasoner OK |", torch.cuda.get_device_name(0))

    # ---- PATCH: per-epoch STRATIFIED metrics ----
    # Training runs in a subprocess, so we edit the source: wrap trainer.evaluate() to add
    # per-slice document_hits@k/mrr keys. _log_metrics then prints them EACH EPOCH in the same
    # format as the aggregate lines. Toggle with STRAT_EVAL=0; BGE split from STRAT_BGE.
    STF = "/content/gfm-rag/gfmrag/workflow/sft_training.py"
    _src = open(STF).read()
    if "_evaluate_stratified" not in _src:
        _inject = "\n".join([
            "    # --- injected: per-epoch stratified eval (ZERO extra forward pass) ---",
            "    # A forward hook records each eval query's gold-document rank during evaluate()'s",
            "    # existing pass; we then add per-slice keys to the metrics dict (logged as usual).",
            "    import os as _o, json as _j",
            "    from collections import defaultdict as _dd",
            "    _rec = []",
            "    _recording = {'on': False}",
            "    def _hook(_module, _inp, _out):",
            "        if not _recording['on'] or len(_inp) < 2:",
            "            return",
            "        try:",
            "            _g, _b = _inp[0], _inp[1]",
            "            _did = _g.nodes_by_type['document']",
            "            _dp = _out[:, _did]",
            "            _tgt = _b['target_nodes_mask'][:, _did].bool()",
            "            _rk = _dp.argsort(dim=-1, descending=True).argsort(dim=-1)",
            "            _ids = _b['id']",
            "            for _qi in range(_dp.shape[0]):",
            "                _pos = _tgt[_qi].nonzero(as_tuple=True)[0]",
            "                if len(_pos):",
            "                    _r = int(_rk[_qi, _pos].min().item()) + 1",
            "                    _q = _ids[_qi]",
            "                    _q = _q.item() if hasattr(_q, 'item') else _q",
            "                    _rec.append((_q, _r))",
            "        except Exception:",
            "            pass",
            "    trainer.model.register_forward_hook(_hook)",
            "    _orig_evaluate = trainer.evaluate",
            "    def _evaluate_stratified():",
            "        _rec.clear(); _recording['on'] = True",
            "        m = _orig_evaluate()",
            "        _recording['on'] = False",
            "        if _o.environ.get('STRAT_EVAL','1') != '1':",
            "            return m",
            "        try:",
            "            _name = _o.environ.get('STRAT_NAME','eval')",
            "            _tj = _o.environ.get('STRAT_TEST','')",
            "            _bp = _o.environ.get('STRAT_BGE','')",
            "            _meta = {q['id']: q for q in _j.load(open(_tj))} if _tj and _o.path.exists(_tj) else {}",
            "            _BGE = {r['id']: r for r in _j.load(open(_bp))} if _bp and _o.path.exists(_bp) else {}",
            "            def _brank(qid, g):",
            "                for i,(d,_s) in enumerate(_BGE.get(qid,{}).get('predictions',{}).get('document',[]),1):",
            "                    if d==g: return i",
            "                return 10**9",
            "            _sl = _dd(lambda: _dd(list)); _seen = set()",
            "            for _q,_r in _rec:",
            "                if _q in _seen: continue",
            "                _seen.add(_q)",
            "                _mq = _meta.get(_q, {})",
            "                _gg = _mq.get('supporting_documents') or []",
            "                _gd = _gg[0] if isinstance(_gg,list) and _gg else _gg",
            "                _st = 'same' if _mq.get('stratum')=='same' else 'cross'",
            "                _sim = 'dissim' if _brank(_q,_gd)>100 else 'sim'",
            "                for _nm in [_st,_sim] + (['cross+dissim'] if (_st=='cross' and _sim=='dissim') else []):",
            "                    _d = _sl[_nm]",
            "                    for _k in (1,5,10): _d['hits@'+str(_k)].append(float(_r<=_k))",
            "                    _d['mrr'].append(1.0/_r if _r<=100 else 0.0)",
            "            for _nm,_d in _sl.items():",
            "                _n = len(_d['mrr']) or 1",
            "                for _c in ('hits@1','hits@5','hits@10','mrr'):",
            "                    m[_name+'/document_'+_c+'/'+_nm] = sum(_d[_c])/_n",
            "        except Exception as _e:",
            "            print('[stratified] skipped:', _e)",
            "        return m",
            "    trainer.evaluate = _evaluate_stratified",
            "    trainer.train()",
        ])
        _src = _src.replace("    trainer.train()", _inject, 1)
        open(STF, "w").write(_src)
        print("patched sft_training.py -> per-epoch stratified eval")
    else:
        print("stratified patch already applied")
    # ---- PATCH: tqdm shows per-component RUNNING-AVERAGE losses (bce / pcr / mse / total) ----
    # Patches base_trainer.py ON DISK so the training SUBPROCESS shows every loss, not just the total.
    _bt = "/content/gfm-rag/gfmrag/trainers/base_trainer.py"
    _bs = open(_bt).read()
    _OLD = 'progress_bar.set_postfix(loss=step_metrics.get("loss", 0.0))'
    _NEW = ('_names = {"bce_loss": "bce", "pcr_loss": "pcr", "mse_loss": "mse", "loss": "tot"}\n'
            '                progress_bar.set_postfix({_names.get(k, k): f"{np.mean(v):.3f}" for k, v in epoch_metrics.items()})')
    if "_names.get(k, k)" in _bs:
        print("base_trainer.py already patched (per-component postfix present)")
    elif _OLD in _bs:
        open(_bt, "w").write(_bs.replace(_OLD, _NEW))
        print("patched base_trainer.py -> tqdm shows running-average bce/pcr/mse/tot")
    else:
        print("WARN: postfix line not found in base_trainer.py (file may have changed); inspect ~line 424")
else:
    print('skipped: every scan and path file is already on Drive, so the engine is not needed')


## 3b. Fusion sources (with interpret())

In [ ]:
# Only runs when cell 1 found a scan or a path search still to do; drawing the figures from
# cached interpretations needs neither the engine nor Qwen3.
if NEED_ENGINE:
    # === write the CARGO-fusion files into the fork (generated from the repo copies 2026-09-08 08:55) ===
    # Runs AFTER the graph baselines on purpose: they trained on the stock engine.
    import json, os
    FILES = json.loads(r'''{"/content/gfm-rag/gfmrag/models/fusion_reasoner.py": "\"\"\"\nfusion_reasoner.py \u2014 CARGO fusion: v16sc G-Reasoner + operator, EVERYTHING learned jointly in one run.\n\nWraps the standard GraphReasoner. The operator is recomputed LIVE from its raw ingredients so its\nweights and exponent are trainable (matches the interim report: beta and the fusion weights are learned):\n\n    S_op      = w0 z(dense) + w1 z(S / dem^beta) + w2 z(M / dem^beta)   # dem = total_S - S (anti-hub)\n    fused_doc = z(S_op) + gamma_q * relu( z(graph_doc) )\n    gamma_q   = softplus( gate(coverage) )                             # per-query gate >= 0\n\nTWO FUSION FORMS, selected by FUSION_FORM (default 'additive' = the arithmetic above,\nbit-identical to every run made before the option existed).\n\n    FUSION_FORM=mixture   fused_doc = log( (1 - a_q) p_s + a_q p_g )\n                          p_s = softmax( z(S_op) )            # tau_s fixed at 1\n                          p_g = softmax( z(graph_doc)/tau_g ) # tau_g learned in [0.25, 4]\n                          a_q = a_max * sigmoid( router(phi_q) )\n\nThe additive form cannot promote a document, at any gamma it actually learns: a z-score\nover this corpus tops out near 5-10 and the converged gamma is 0.036, so the graph's\nlargest possible contribution to any document is ~0.36 z-units against a semantic top-50\nspread of 1-3. It can reorder neighbours, never rescue a gold the semantic scorer missed,\nwhich is exactly the cross-domain case. Mixing calibrated DISTRIBUTIONS instead of scores\nmakes the response exponential, so the graph's few confident documents can move hundreds\nof places, while a flat p_g leaves the ranking exactly unchanged rather than adding noise\nto every document. The cost is that the graph's confident MISTAKES are promoted just as\nhard, and the graph ranking is substantially weaker overall, so a concentrated p_g puts\nlarge mass on wrong documents on many queries; a_max bounds that. Full detail, including\nwhy tau_s is frozen and why tau_g is bounded rather than free, in __init__.\n\nTrained end-to-end from one ranking loss on fused_doc:\n  - the GNN weights (the v16sc graph reasoner, from scratch),\n  - the gate (when to trust the graph), and\n  - the operator scalars w=[w0,w1,w2] and beta.\nThe BGE encoder that produced dense/S/M is frozen (we only learn the handful of combination scalars).\nw, beta are warm-started at the fitted values and learn at the base LR (op_lr_scale=1.0) \u2014 4 params on\na near-convex objective, so they converge fast. relu => the graph can only promote a doc (aggregate floor).\n\nIngredients come from env OPERATOR_COMPONENTS (train) / OPERATOR_COMPONENTS_TEST (test): npz with\n{dense,S,M: float16 [Q x n_doc] in nodes.csv doc order, total_S: float32 [n_doc], query_ids:[...]}.\nQuery ids are split-unique, so both tables are merged and looked up by batch['id'].\n\nMULTI-CORPUS TRAINING. Either variable also accepts a COMMA-SEPARATED list of npz\npaths, which is what joint Physics+Biology training needs: the trainer walks a list\nof graphs and each carries its own corpus, so `dense` has a different document-column\ncount per graph (physics 10,349 vs biology 15,588) and one merged table cannot serve\nboth. Each file becomes its own table, and because query ids are unique across SIR-4\ndatasets the existing id -> (tag, row) lookup already routes a batch to the right one.\nGraphDatasetLoader keeps one graph resident at a time, so a batch never spans two\ncorpora and the single-tag assert in _operator still holds.\n\nTHE SEMANTIC CHANNEL IS SELECTABLE (`semantic:` in the config).\n\n  \"operator\" (default)  the handcrafted scorer above. Unchanged, bit-identical.\n  \"mlp\"                 the LEARNED scorer from semantic_scorer.py: one MLP reading the\n                        SORTED vector of per-answer match scores, with a popularity\n                        discount predicted from the document embedding alone.\n\nOnly the semantic channel changes. The gate, the relu floor, the fusion arithmetic and the\nhard-negative mining are identical either way, so a run pair isolates \"handcrafted vs learned\nsemantic scorer\" with the graph half held fixed. The learned scorer is warm-started from a\ntrained 5d checkpoint (SEMANTIC_CKPT + SEMANTIC_POPNET) and keeps training under the fusion's\nranking loss unless semantic_train=False.\n\nWHY THE POPULARITY PREDICTOR AND NOT THE LEAVE-ONE-OUT TERM. The operator's anti-hub\ndenominator is total_S - S, i.e. a document's popularity measured from the OTHER queries'\nhypothetical answers in the same split. Inside the fusion that is the same transductive\ndependency it has always been. The learned scorer replaces it with a function of the\ndocument's own embedding, so the fused model can score one query against an unseen corpus.\n\nIngredients come from env SEMANTIC_COMPONENTS / SEMANTIC_COMPONENTS_TEST (see\nprecompute_semantic_components.py). The per-answer matrix H is NOT copied into those files:\nthey carry the memmap's path plus the corpus->nodes.csv column permutation, and the rows for\none batch are sliced and permuted on demand.\n\"\"\"\nimport json\nimport math\nimport os\n\nimport numpy as np\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F  # noqa: N812\n\nfrom gfmrag.models.gfm_reasoner import GraphReasoner\n\nW_INIT = (1.05, 1.05, 0.25)   # fitted operator fusion weights\nBETA_INIT = 0.95              # fitted anti-hub exponent\n\n\nclass FusionGraphReasoner(nn.Module):\n    def __init__(self, entity_model, feat_dim, gamma_init=0.5, gate_hidden=8,\n                 op_lr_scale=1.0, semantic=\"operator\", semantic_train=True,\n                 cqig=False, cqig_lam=0.1, cqig_layers=None, cqig_norm=None,\n                 cqig_rho=1e-3, cqig_grad_I=None, cqig_op=None, cqig_mu=None, **kwargs):\n        # The operator was first proposed as `cqig_mode`. Accepted as an alias rather than\n        # left to fall through **kwargs into GraphReasoner, where it would be a silent\n        # no-op and the arm would train as the default operator under the other one's name.\n        if \"cqig_mode\" in kwargs:\n            cqig_op = kwargs.pop(\"cqig_mode\") if cqig_op is None else cqig_op\n        super().__init__()\n        self.base = GraphReasoner(entity_model, feat_dim, **kwargs)\n        assert semantic in (\"operator\", \"mlp\"), f\"unknown semantic channel {semantic!r}\"\n        self.semantic = semantic\n\n        # --- Cross-Query Informativeness Gating (off by default, see cqig.py) ---\n        # Attaches a forward pre-hook to each conv layer. With cqig=False nothing is\n        # registered at all, so the ungated arm is bit-identical.\n        #\n        # The three knobs that define an arm of the controlled experiment come from the\n        # ENVIRONMENT when the config leaves them unset, for the same reason every other\n        # run knob does: a Hydra override of a string like \"3-6\" has to survive quoting\n        # through subprocess, an env var does not.\n        # --- CCMP responsibility head (off by default) --------------------------\n        # One projection per layer because self.dims may differ layer to layer, then a\n        # SHARED trunk plus a layer embedding, so the predictor is one function of\n        # (state, remaining depth) rather than L unrelated predictors. Built here, not\n        # lazily on first forward, because a module created after the optimiser is\n        # constructed never receives a gradient step and the arm would silently run as\n        # its own control.\n        _em = self.base.entity_model\n        # --- Routing baselines for the CCMP comparison (off by default) ----------\n        # ROUTE=astar  A*Net-style: the SAME responsibility head CCMP uses, but trained\n        #              only through the ranking loss (no continuation targets) and applied\n        #              as a hard top-K selection over the reached frontier per layer.\n        # ROUTE=attn   RED-GNN-style: query-conditioned attention over each receiver's\n        #              incoming edges, trained only through the ranking loss.\n        # Either is a REPLACEMENT for CCMP, never an addition, so the three arms differ\n        # in exactly one thing: where the routing signal comes from.\n        _route = os.environ.get(\"ROUTE\", \"\").strip().lower()\n        assert _route in (\"\", \"none\", \"astar\", \"attn\"), f\"unknown ROUTE={_route!r}\"\n        _route = \"\" if _route == \"none\" else _route\n        _ccmp_on = os.environ.get(\"CCMP\", \"0\") == \"1\"\n        assert not (_ccmp_on and _route), \"ROUTE replaces CCMP; unset one of CCMP / ROUTE\"\n        _em.route_mode = _route\n        if _ccmp_on or _route == \"astar\":\n            # RNG STATE SAVED AND RESTORED AROUND THIS BLOCK. Constructing these modules\n            # draws from the global generator, so every parameter initialised AFTER this\n            # point -- the fusion gate, the router, the operator scalars -- would start\n            # from different values in the CCMP arm than in the control. The pair would\n            # then differ in the loss AND in the initialisation, and the control's own\n            # reruns already move 0.0118 nDCG@5.\n            _rng = torch.get_rng_state()\n            _dims = list(_em.dims)[:-1]\n            _h = int(os.environ.get(\"CCMP_HID\", \"64\"))\n            _em.resp_proj = nn.ModuleList([nn.Linear(_d, _h) for _d in _dims])\n            _em.resp_emb = nn.Embedding(len(_dims), _h)\n            nn.init.zeros_(_em.resp_emb.weight)\n            _em.resp_head = nn.Sequential(nn.ReLU(), nn.Linear(_h, _h),\n                                          nn.ReLU(), nn.Linear(_h, 1))\n            # Output bias at 0 => yhat starts at 0.5 => a mean-normalised gate starts at\n            # exactly 1.0 at every node, so the CCMP arm's epoch 0 is the control's epoch 0\n            # and any difference later is the loss, not a different initialisation.\n            nn.init.zeros_(_em.resp_head[-1].bias)\n            nn.init.zeros_(_em.resp_head[-1].weight)\n            _em.resp_gate = os.environ.get(\"CCMP_GATE\", \"1\") == \"1\"\n            _em.resp_gate_norm = os.environ.get(\"CCMP_GATE_NORM\", \"1\") == \"1\"\n            _em.resp_eta = float(os.environ.get(\"CCMP_ETA\", \"0.5\"))\n            torch.set_rng_state(_rng)\n            if _route == \"astar\":\n                _em.route_k = int(os.environ.get(\"ROUTE_K\", \"1024\"))\n                _em.resp_gate = True\n                print(f\"[route] mode=astar: same head ({len(_dims)} layers, hid={_h}), \"\n                      f\"hard top-{_em.route_k} of the reached frontier per layer, priority \"\n                      f\"trained by the ranking loss only (no CCMP targets)\", flush=True)\n            else:\n                print(f\"[ccmp] responsibility head on {len(_dims)} layers, hid={_h}, \"\n                      f\"gate={_em.resp_gate} mean_norm={_em.resp_gate_norm} \"\n                      f\"eta={_em.resp_eta}\", flush=True)\n        else:\n            _em.resp_proj = None\n            _em.resp_gate = False\n        if _route == \"attn\":\n            # Same RNG discipline as the CCMP head, same hidden width, and a zero-initialised\n            # output layer so attention starts UNIFORM: after degree normalisation every\n            # edge weight is exactly 1 and epoch 0 is the control's epoch 0.\n            _rng = torch.get_rng_state()\n            _dims = list(_em.dims)[:-1]\n            _h = int(os.environ.get(\"ROUTE_HID\", os.environ.get(\"CCMP_HID\", \"64\")))\n            _em.attn_node = nn.ModuleList([nn.Linear(_d, _h) for _d in _dims])\n            _em.attn_rel = nn.Linear(int(_em.dims[0]), _h)\n            _em.attn_query = nn.Linear(int(_em.dims[0]), _h)\n            _em.attn_emb = nn.Embedding(len(_dims), _h)\n            nn.init.zeros_(_em.attn_emb.weight)\n            _em.attn_out = nn.Linear(_h, 1)\n            nn.init.zeros_(_em.attn_out.weight)\n            nn.init.zeros_(_em.attn_out.bias)\n            _em.attn_norm = os.environ.get(\"ROUTE_ATTN_NORM\", \"1\") == \"1\"\n            torch.set_rng_state(_rng)\n            print(f\"[route] mode=attn: query-conditioned edge attention on {len(_dims)} layers \"\n                  f\"(hid={_h}, degree-normalised={_em.attn_norm}), trained by the ranking \"\n                  f\"loss only\", flush=True)\n        else:\n            _em.attn_node = None\n\n        self.cqig = None\n        if cqig:\n            from gfmrag.models.cqig import CQIGate\n            spec = cqig_layers if cqig_layers is not None else os.environ.get(\"CQIG_LAYERS\")\n            norm = cqig_norm if cqig_norm is not None else os.environ.get(\"CQIG_NORM\", \"layer\")\n            grad = (cqig_grad_I if cqig_grad_I is not None\n                    else os.environ.get(\"CQIG_GRAD_I\", \"0\") == \"1\")\n            op = cqig_op if cqig_op is not None else os.environ.get(\"CQIG_OP\", \"gate\")\n            mu = cqig_mu if cqig_mu is not None else os.environ.get(\"CQIG_MU\", \"ref\")\n            self.cqig = CQIGate(self.base.entity_model.layers, lam_init=cqig_lam,\n                                rho=cqig_rho, gate_layers=spec, norm=norm,\n                                grad_through_I=grad, op=op, mu=mu)\n            print(f\"[cqig] attached to {self.cqig.n_layers} layers; gating \"\n                  f\"{sorted(i + 1 for i in self.cqig.gate_layers)} (1-indexed) \"\n                  f\"= positions {sorted(self.cqig.gate_layers)}; norm={norm!r} \"\n                  f\"lam_init={cqig_lam} grad_through_I={grad} op={self.cqig.op!r} \"\n                  f\"mu={self.cqig.mu_source!r}\")\n            if self.cqig.mu_zero:\n                print(\"[cqig] mu=zero: THIS IS THE ABLATION, NOT THE METHOD. mu is forced to \"\n                      \"0, so I = ||h||^2/s is an activation-MAGNITUDE gate and the reference \"\n                      \"bank contributes nothing. The references are still run and discarded \"\n                      \"so the two arms differ in mu alone. If this reproduces the method's \"\n                      \"score, the bank was never load-bearing.\")\n            if self.cqig.mu_zero and self.cqig.centring:\n                raise AssertionError(\n                    \"cqig mu='zero' with a centring op is a literal no-op: the operator \"\n                    \"subtracts c*mu and mu is 0, so the model is the ungated reasoner. \"\n                    \"Ablate mu against op='gate', which is the arm whose result is in doubt.\")\n            if self.cqig.centring:\n                print(f\"[cqig] op={self.cqig.op!r}: the state is CENTRED, h - c*mu, not \"\n                      f\"scaled. c is \"\n                      + (\"lam (constant, query-independent; alpha and dtau get no \"\n                         \"gradient in this arm)\" if self.cqig.op == \"centre-fixed\"\n                         else \"1-g, so an uninformative node loses more of its background\")\n                      + \". Nodes no reference reached have mu=0 and are left ALONE here, \"\n                        \"where a gating arm damps them hardest.\")\n\n        # --- per-query gate: gamma_q = softplus(gate(phi_q)) ---\n        #\n        # ROUTER FEATURES ARE NOW SHARED BY BOTH FUSION FORMS. This used to be read only\n        # inside the mixture branch, so on an ADDITIVE arm `FUSION_ROUTER` was accepted,\n        # logged, and never used: the gate was nn.Linear(1, ...) on the semantic top-5\n        # mean alone. Every additive run recorded before this change was therefore a `cov`\n        # run whatever its flag said, and the matsci two-stage 2x2's legacy-vs-phi5 axis\n        # compared two identical configurations (measured: -0.0004 / +0.0021 / -0.0019 /\n        # -0.0038 against a 0.0118 noise floor, i.e. the null the wiring predicts).\n        #\n        # `full` is the default because a gate on semantic confidence alone cannot express\n        # the one thing worth conditioning on: whether the GRAPH is worth listening to on\n        # this query. Set FUSION_ROUTER=cov to reproduce any pre-existing additive run.\n        self.router_feats = os.environ.get(\"FUSION_ROUTER\", \"full\").lower()\n        assert self.router_feats in (\"full\", \"cov\"), self.router_feats\n        n_feat = 5 if self.router_feats == \"full\" else 1\n        self.gate = nn.Sequential(\n            nn.Linear(n_feat, gate_hidden), nn.ReLU(), nn.Linear(gate_hidden, 1)\n        )\n        # INIT IS UNCHANGED BY n_feat. The last layer's weight is zeroed, so gamma_q is\n        # exactly softplus(bias) = gamma_init for every query at step 0 no matter how many\n        # features feed it. A `full` arm and a `cov` arm therefore start from the identical\n        # ranking and diverge only as the gate learns, which is what makes them a pair.\n        nn.init.zeros_(self.gate[-1].weight)\n        nn.init.constant_(self.gate[-1].bias, math.log(math.expm1(max(gamma_init, 1e-3))))\n        self._gamma_lowp_note = False\n\n        # --- FUSION FORM: additive z-scores (default) or a mixture of calibrated\n        #     distributions (FUSION_FORM=mixture) ------------------------------------\n        #\n        # WHY A SECOND FORM AT ALL. The additive form cannot promote a document, at any\n        # gamma it actually learns. A z-score over a 15k-60k document corpus tops out\n        # near 5-10 even for the graph's single most confident paper, and the converged\n        # gamma is 0.036, so the largest contribution the graph can make to ANY document\n        # is about 0.36 z-units. The semantic channel's own spread across its top 50 is\n        # 1-3 z-units. The graph is therefore structurally unable to lift a paper from\n        # outside the top 50 into the top 5 however certain it is; it can only reorder\n        # documents the semantic scorer already placed next to each other. That is a\n        # property of the functional form, not of the gate, the gating authority lam, or\n        # the bf16 freeze, and it caps the measured contribution at +0.017 nDCG@5.\n        #\n        # THE MIXTURE. Combine calibrated distributions instead of scores:\n        #\n        #     log p_s = z(s_op)      - logsumexp(z(s_op))        # tau_s FIXED at 1\n        #     log p_g = z(g)/tau_g   - logsumexp(z(g)/tau_g)\n        #     a_q     = a_max * sigmoid(router(phi_q))           # in [0, a_max]\n        #     fused   = log( (1-a_q) p_s + a_q p_g )\n        #\n        # Four properties, each of which the additive form lacks:\n        #\n        #  1. EXPONENTIAL RESPONSE. p_g for a document the graph ranks first out of 60k\n        #     is O(0.1); p_s in the semantic tail is O(1e-5). Even at a_q = 0.05 the\n        #     second term dominates and that document moves hundreds of places. This is\n        #     the only mechanism by which a 0.27-nDCG channel can rescue a gold the\n        #     semantic scorer missed outright, which is the cross-domain case.\n        #  2. AN UNINFORMATIVE GRAPH IS EXACTLY INERT. A flat p_g is a constant, and\n        #     logaddexp(const, log(1-a) + log p_s) is strictly increasing in p_s, so the\n        #     ranking does not move AT ALL -- not approximately, exactly. The weak-channel\n        #     problem is handled by the algebra instead of by keeping gamma small. tau_g\n        #     controls how sharp the graph's vote is relative to the semantic's, which is\n        #     the one quantity that matters, and it is the only new scalar.\n        #  3. MIXTURE, NOT PRODUCT. A product of experts (log p_s + b log p_g) would let\n        #     the weak channel VETO a document the semantic scorer got right, driving its\n        #     score toward zero. A mixture can only ADD mass, so no document's score ever\n        #     falls, a_q -> 0 recovers the semantic ranking and a_max bounds the worst case.\n        #     READ THAT AS A STATEMENT ABOUT SCORES, NOT RANKS. Rank is zero-sum: a\n        #     confidently-wrong graph promotes other documents OVER a correct one and\n        #     demotes it just as effectively as a veto would. Measured: a gold at semantic\n        #     rank 1 falls to rank 3 when the graph is certain about six wrong documents,\n        #     its own score gain being exactly 0.0 while theirs are 0.13 to 9.75. The\n        #     additive form's relu is non-negative too, so this floor is NOT what\n        #     distinguishes the two forms; a_max and property 2 are.\n        #  4. GRADIENT FROM THE RANKING LOSS EVERYWHERE. relu(z(g)) zeroes the FUSED\n        #     loss's gradient to the GNN for every document below the graph's own mean --\n        #     precisely the buried golds. Under the mixture the graph softmax couples the\n        #     whole corpus, so every document gets a gradient from the final ranking loss,\n        #     attenuated rather than deleted. NOTE the precise claim: the graph is not\n        #     gradient-starved today, because AUX_W=1.0 runs a separate graph-only\n        #     contrastive on _raw_doc_z that never passes through the relu. What changes\n        #     is that the RANKING loss itself now reaches every graph logit, instead of\n        #     the graph learning about its buried documents only from the auxiliary term.\n        #\n        # THE RISK, STATED PLAINLY. Property 1 works just as well on the graph's confident\n        # MISTAKES. nDCG@5 = 0.27 does NOT establish a top-1 error rate -- with ~1.9 golds\n        # per query it is not a Hits@1 measurement and nothing here should be read as one\n        # -- but the graph ranking is substantially weaker overall, so a concentrated p_g\n        # will put large mass on wrong documents on many queries. The additive form is\n        # safe because it is inert; this one is useful because it is not. a_max is the\n        # only thing bounding that, which is why it defaults to 0.5 rather than 1.0 and\n        # why a_q is learned per query rather than fixed. Whether the rescues outnumber\n        # the false promotions is the empirical question the run exists to answer.\n        #\n        # tau_s IS DEFINED AND FROZEN AT 1, NOT LEARNED. The training loss is already\n        # `logsumexp(lineup) - gold`, a softmax cross-entropy, so a learnable tau_s would\n        # be softmaxing a softmax: shrinking it sharpens log p_s, which lowers the loss\n        # whenever the gold already leads, without improving any ranking. Frozen at 1 it\n        # buys something instead: log p_s = z(s_op) - const, cross-entropy is shift\n        # invariant per query, so AT INITIALISATION (a_q small) this arm's loss is\n        # numerically the same objective the additive arm trains under. No LR retuning,\n        # and the two arms stay comparable.\n        self.fusion_form = os.environ.get(\"FUSION_FORM\", \"additive\").lower()\n        assert self.fusion_form in (\"additive\", \"mixture\"), (\n            f\"FUSION_FORM={self.fusion_form!r}; expected 'additive' or 'mixture'\")\n        self._fusion_note = \"\"\n        if self.fusion_form == \"mixture\":\n            self.a_max = float(os.environ.get(\"FUSION_AMAX\", \"0.5\"))\n            a_init = float(os.environ.get(\"FUSION_AINIT\", \"0.1\"))\n            assert 0.0 < a_init < self.a_max <= 1.0, (\n                f\"need 0 < FUSION_AINIT ({a_init}) < FUSION_AMAX ({self.a_max}) <= 1\")\n            # a_q must not start AT zero: sigmoid'(-inf) = 0 and the router would get no\n            # gradient to tell queries apart with. 0.1 is small enough that epoch 1 is\n            # still essentially the semantic ranking.\n            # router_feats and n_feat are set once above, for BOTH forms; the mixture just\n            # builds a second head on the same input.\n            self.router = nn.Sequential(\n                nn.Linear(n_feat, gate_hidden), nn.ReLU(), nn.Linear(gate_hidden, 1)\n            )\n            nn.init.zeros_(self.router[-1].weight)\n            nn.init.constant_(self.router[-1].bias,\n                              math.log(a_init / (self.a_max - a_init)))\n            # tau_g = exp(log_tau_g), so it stays positive without a clamp. log_tau_g = 0\n            # is tau_g = 1, i.e. the graph's vote starts exactly as sharp as z(g) makes it.\n            #\n            # tau_g IS BOUNDED, NOT FREE. Two separate problems, only one of which is\n            # already handled:\n            #\n            #   SCALE IDENTIFIABILITY -- handled, and this is why the graph channel is\n            #   z-scored before it gets here. softmax(c*g / (c*tau_g)) = softmax(g/tau_g),\n            #   so with RAW graph logits the readout could inflate its own scale while\n            #   tau_g grew to match and the pair would not be separately identifiable.\n            #   z() is exactly scale-invariant (z(c*g) = z(g)), so that degeneracy cannot\n            #   arise. Feeding raw logits here would reintroduce it, and would also expose\n            #   tau_g to the GNN's score scale drifting across epochs.\n            #\n            #   OVERCONFIDENCE -- NOT handled by z(), and this is the reason for the\n            #   bounds. Shrinking tau_g sharpens p_g, which lowers the training\n            #   cross-entropy on any query whose gold already leads, WITHOUT the graph\n            #   ranking any better. That is the same degenerate direction that tau_s is\n            #   frozen to avoid, and z-scoring does nothing about it. tau_g is therefore\n            #   parameterised as tau_min + (tau_max - tau_min) * sigmoid(tau_hat), so the\n            #   sharpest and flattest the graph's vote can get are both fixed in advance.\n            #   The default [0.25, 4] is a factor of 4 either side of neutral.\n            #\n            # The stronger version of this is post-hoc calibration: train the reasoner,\n            # freeze its readout, then fit tau_g and the router on held-out SOURCE queries\n            # and freeze both for target inference, which makes tau_g a genuine\n            # calibration parameter instead of one more jointly-optimised scale. That\n            # costs an extra stage; the bounds are the cheap version of the same\n            # protection. FUSION_TAUG=<float> freezes tau_g outright if that stage is\n            # ever run and its fitted value is known.\n            self.tau_min = float(os.environ.get(\"FUSION_TAU_MIN\", \"0.25\"))\n            self.tau_max = float(os.environ.get(\"FUSION_TAU_MAX\", \"4.0\"))\n            assert 0 < self.tau_min < self.tau_max, (self.tau_min, self.tau_max)\n            tg = os.environ.get(\"FUSION_TAUG\", \"learn\").lower()\n            if tg == \"learn\":\n                # tau_hat such that tau_g starts at exactly 1: the graph's vote begins\n                # neither sharpened nor flattened relative to z(g).\n                f = (1.0 - self.tau_min) / (self.tau_max - self.tau_min)\n                assert 0.0 < f < 1.0, (\n                    f\"FUSION_TAU_MIN/MAX = [{self.tau_min}, {self.tau_max}] must bracket \"\n                    f\"tau_g = 1, or the arm cannot start at the neutral sharpness\")\n                self.tau_g_hat = nn.Parameter(torch.tensor(math.log(f / (1.0 - f))))\n            else:  # frozen at a given value, for the ablation or a calibrated fit\n                v = float(tg)\n                assert self.tau_min <= v <= self.tau_max, (\n                    f\"FUSION_TAUG={v} is outside [{self.tau_min}, {self.tau_max}]\")\n                f = (v - self.tau_min) / (self.tau_max - self.tau_min)\n                self.register_buffer(\"tau_g_hat\", torch.tensor(math.log(f / (1.0 - f))))\n            print(f\"[fusion] form=mixture: fused = log((1-a_q) p_s + a_q p_g), \"\n                  f\"a_max={self.a_max} a_init={a_init} router={self.router_feats!r} \"\n                  f\"({n_feat} features) tau_s=1 (frozen) tau_g={tg}. The diagnostic \"\n                  f\"column printed as 'gamma' is now a_q, the mixing weight in \"\n                  f\"[0, {self.a_max}] -- NOT the additive arm's gamma, and the two are \"\n                  f\"not on the same scale.\")\n        else:\n            # SAY WHICH ROUTER RAN. The additive arm used to print nothing about the\n            # router, which is how FUSION_ROUTER stayed dead in this path unnoticed\n            # across a whole 2x2. A run that does not log the knob it varied cannot be\n            # audited from its own console output.\n            print(f\"[fusion] form=additive: fused = z(s_op) + gamma_q * relu(z(graph)), \"\n                  f\"gamma_q = softplus(gate(phi_q)), router={self.router_feats!r} \"\n                  f\"({n_feat} feature{'s' if n_feat > 1 else ''}), gamma_init={gamma_init}. \"\n                  + (\"phi_q = [semantic confidence, semantic peakedness, graph confidence, \"\n                     \"graph peakedness, channel agreement].\" if n_feat > 1 else\n                     \"phi_q = the semantic top-5 mean alone (the pre-2026-08-16 behaviour).\"))\n\n        # --- trainable operator scalars (warm-started at fitted values; base LR via op_lr_scale) ---\n        # effective value = init + op_lr_scale * delta, delta starts at 0. With Adam this makes the\n        # operator learn at op_lr_scale x the graph/gate LR (1.0 = same rate).\n        self.op_lr_scale = float(op_lr_scale)\n        self.register_buffer(\"w_init\", torch.tensor(W_INIT, dtype=torch.float32))\n        self.w_delta = nn.Parameter(torch.zeros(3))\n        self.beta_delta = nn.Parameter(torch.zeros(()))\n\n        # --- operator raw ingredients (CPU float16; rows moved to GPU per batch) ---\n        self._dense: dict[str, torch.Tensor] = {}\n        self._S: dict[str, torch.Tensor] = {}\n        self._M: dict[str, torch.Tensor] = {}\n        self._totS: dict[str, torch.Tensor] = {}\n        self._row: dict[str, tuple] = {}\n        seen_paths: dict[str, str] = {}          # abspath -> tag already holding it\n        for split, ev in (((\"train\", \"OPERATOR_COMPONENTS\"), (\"test\", \"OPERATOR_COMPONENTS_TEST\"))\n                          if semantic == \"operator\" else ()):\n            # Not loaded under semantic='mlp'. dense+S+M is 660 MB of resident CPU\n            # memory on CS, and nothing in that arm reads it.\n            raw = os.environ.get(ev)\n            if not raw:\n                continue\n            paths = [x.strip() for x in raw.split(\",\") if x.strip()]\n            for j, p in enumerate(paths):\n                # THE SAME FILE FOR BOTH VARIABLES IS THE ZERO-SHOT IDIOM. Predict runs\n                # have one corpus and set OPERATOR_COMPONENTS=OPERATOR_COMPONENTS_TEST=x,\n                # which must stay legal: the duplicate check below exists to catch two\n                # DIFFERENT tables claiming the same query ids (a real routing bug), not\n                # one table registered twice (the same rows either way).\n                ap = os.path.abspath(p)\n                if ap in seen_paths:\n                    print(f\"[fusion] {ev}[{j}] is the same file as '{seen_paths[ap]}', reusing it\")\n                    continue\n                # One table per file. The tag stays \"train\"/\"test\" for the single-file\n                # case so existing runs and checkpoints are byte-identical; only a list\n                # introduces the suffixed tags.\n                tag = split if len(paths) == 1 else f\"{split}#{j}\"\n                seen_paths[ap] = tag\n                d = np.load(p, allow_pickle=True)\n                self._dense[tag] = torch.from_numpy(np.asarray(d[\"dense\"], dtype=np.float16))\n                self._S[tag] = torch.from_numpy(np.asarray(d[\"S\"], dtype=np.float16))\n                self._M[tag] = torch.from_numpy(np.asarray(d[\"M\"], dtype=np.float16))\n                self._totS[tag] = torch.from_numpy(np.asarray(d[\"total_S\"], dtype=np.float32))\n                # A query id appearing in two tables would make routing order-dependent\n                # and silently score half the batch against the wrong corpus.\n                dup = [str(q) for q in d[\"query_ids\"] if str(q) in self._row]\n                assert not dup, (\n                    f\"{p}: {len(dup)} query ids already claimed by another components \"\n                    f\"table (e.g. {dup[:3]}); tables must cover disjoint query sets\")\n                for i, q in enumerate(d[\"query_ids\"]):\n                    self._row[str(q)] = (tag, i)\n                print(f\"[fusion] loaded {ev}[{j}] as '{tag}': dense/S/M \"\n                      f\"{tuple(self._dense[tag].shape)} ({len(d['query_ids'])} queries) \"\n                      f\"{os.path.basename(p)}\")\n        if semantic == \"operator\":\n            assert self._row, (\"no operator components: set OPERATOR_COMPONENTS / \"\n                               \"OPERATOR_COMPONENTS_TEST\")\n        else:\n            self._init_semantic(semantic_train)\n\n        self._raw_doc = None   # graph-alone doc scores, cached each forward for the aux loss\n        self._doc_ids = None\n        # [B, n_doc] detached SEMANTIC scores, cached for hard-negative mining. Named\n        # _s_op for back-compatibility with the trainer and the routed reasoner, but it\n        # holds whichever channel `semantic` selected, not necessarily the operator.\n        self._s_op = None\n\n    # ------------------------------------------------------------------ precision\n    def _apply(self, *args, **kwargs):\n        \"\"\"Keep the gamma gate's weights in float32 through `model.to(dtype=...)`.\n\n        SAME BUG AS CQIGate._apply, DIFFERENT MODULE. The trainer casts the whole model\n        with `model.to(dtype=torch.bfloat16)` (utils/setup_training.py). A bfloat16 value\n        has a 2^-8 relative ULP, so any parameter above |w| ~= 256*lr = 0.128 at lr 5e-4\n        has every AdamW step rounded straight back, permanently, since round-to-nearest\n        keeps no remainder.\n\n        `self.gate[-1].bias` is initialised at log(expm1(gamma_init)) = -4.6 for the\n        configured gamma_init=0.01. That is 36x the freeze line, so gamma's output bias\n        could never move at all, and gate[0]'s weights are drawn from\n        U(-1/sqrt(8), 1/sqrt(8)) = U(-0.354, 0.354), so most of those froze as well.\n\n        The measured consequence: gamma stops dead at epoch 4 in EVERY arm and holds to\n        four decimals for the next seven epochs while the graph channel's own nDCG still\n        moves by 0.017. Whatever gamma reached in those four epochs is the run's final\n        answer, because fused = z(s_op) + gamma*relu(z(graph)) and gamma is the only term\n        deciding how loud the graph is. matsci: gamma froze at 0.0279 -> 0.5207,\n        0.0359 -> 0.5340, 0.0288 -> 0.5218. The score is a function of the freeze point.\n\n        This makes the lam=0.9 result unsafe to attribute to lam: graph-channel nDCG and\n        informativeness AUC are the same in all three arms, so what lam changed was where\n        gamma happened to stop, not the quality of the gating. CQIG_ALLOW_LOWP=1\n        reproduces the old frozen behaviour for both this and the CQIG scalars.\n\n        EVERY FUSION-HEAD PARAMETER IS PINNED, not just the gate. The mixture arm's\n        router carries the same -1.39 output bias and its tau_g would drift past the\n        freeze line within an epoch, so pinning the gate alone would reintroduce exactly\n        this bug under a new name. These are a few dozen scalars in total; float32 for\n        all of them costs nothing measurable.\n        \"\"\"\n        heads = [(\"gate\", self.gate)]\n        # THE CCMP HEAD SITS ON THE SAME CLIFF AS GAMMA. The resp_head trunk initialises\n        # at +-0.125 against a 256*lr = 0.128 freeze line, so roughly half its weights are\n        # one small drift away from never updating again -- the exact failure that decided\n        # every CQIG arm. Empty in the control, where these attributes are None.\n        _rm = self.base.entity_model\n        # ROUTE=attn's head is on the same cliff, and `_route_attention` runs it with\n        # autocast off on float32 inputs, so leaving it in bfloat16 is not a precision\n        # loss but a crash: \"mat1 and mat2 must have the same dtype\" on the first step\n        # (tomato attn smoke, 2026-09-07). Empty unless ROUTE=attn.\n        for _t in (\"resp_proj\", \"resp_emb\", \"resp_head\",\n                   \"attn_node\", \"attn_rel\", \"attn_query\", \"attn_emb\", \"attn_out\"):\n            if getattr(_rm, _t, None) is not None:\n                heads.append((_t, getattr(_rm, _t)))\n        if getattr(self, \"fusion_form\", \"additive\") == \"mixture\":\n            heads.append((\"router\", self.router))\n            if isinstance(getattr(self, \"tau_g_hat\", None), nn.Parameter):\n                # `self` with recurse=False is tau_g_hat AND the operator scalars\n                # w_delta/beta_delta. Pinning those too is deliberate: w_delta starts at\n                # 0 but drifts, and W_INIT is 1.05, so they sit on the same cliff. Only\n                # reached under form=mixture, so the additive arm is untouched.\n                heads.append((\"fusion\", self))\n        saved = {f\"{tag}.{n}\": p.detach().clone().float()\n                 for tag, mod in heads\n                 for n, p in (mod.named_parameters(recurse=False)\n                              if mod is self else mod.named_parameters())}\n        out = super()._apply(*args, **kwargs)\n        if os.environ.get(\"CQIG_ALLOW_LOWP\", \"0\") == \"1\":\n            return out\n        pinned = []\n        for tag, mod in heads:\n            it = (mod.named_parameters(recurse=False) if mod is self\n                  else mod.named_parameters())\n            for n, p in it:\n                k = f\"{tag}.{n}\"\n                if k in saved and p.is_floating_point() and p.dtype != torch.float32:\n                    was = p.dtype\n                    p.data = saved[k].to(device=p.device)\n                    pinned.append((k, was))\n        if pinned and not self._gamma_lowp_note:\n            self._gamma_lowp_note = True\n            print(f\"[fusion] fusion head pinned to float32 against a {pinned[0][1]} model \"\n                  f\"cast: {', '.join(n for n, _ in pinned)}. The output bias starts at \"\n                  f\"{float(self.gate[-1].bias.flatten()[0]):.3f}, far above the \"\n                  f\"|w| > 256*lr freeze line, so in {pinned[0][1]} gamma stopped moving \"\n                  f\"at epoch ~4 and the run's score was fixed by whatever it reached.\")\n        return out\n\n    def _router_phi(self, sz, gz, cov):\n        \"\"\"The per-query router input, shared by the additive gate and the mixture router.\n\n        ONE DEFINITION, TWO CONSUMERS. These features were written for the mixture and\n        lived inside it, which is how the additive arm ended up accepting FUSION_ROUTER\n        and ignoring it. Both forms now call this, so \"router=full\" means the same five\n        numbers whichever fusion is running and the two forms stay comparable.\n\n        `cov` (the semantic top-5 mean) is feature 1 under `full` and the whole vector\n        under `cov`, so the ablation is a strict narrowing rather than a different input.\n\n        Divisors are NOMINAL, not tuned: a top-5 mean of a z-score over a corpus this size\n        runs about 3-6 and a top1-to-top5 margin about 1-3, so these put all five features\n        on the same O(1) footing and stop the [0,1] overlap feature from starting with a\n        hundredth of the gradient of the others. They are constants, not parameters, so\n        nothing here can be fitted to the test set.\n        \"\"\"\n        if self.router_feats == \"cov\":\n            return cov\n        k = min(5, sz.shape[1])\n        st, gt = sz.topk(k, dim=-1).values, gz.topk(k, dim=-1).values\n        s_cov, g_cov = st.mean(-1, keepdim=True), gt.mean(-1, keepdim=True)\n        ko = min(10, sz.shape[1])\n        si = sz.topk(ko, dim=-1).indices                        # [B, ko]\n        gi = gz.topk(ko, dim=-1).indices\n        # agreement: what fraction of the semantic top-10 the graph also ranks top-10.\n        # Low overlap with a confident graph is the only configuration in which the\n        # graph has something to say that the semantic channel has not already said.\n        ov = (si.unsqueeze(2) == gi.unsqueeze(1)).any(-1).float().mean(-1, keepdim=True)\n        return torch.cat([s_cov / 5.0,                          # semantic confidence\n                          (st[:, :1] - s_cov) / 2.0,            # semantic peakedness\n                          g_cov / 5.0,                          # graph confidence\n                          (gt[:, :1] - g_cov) / 2.0,            # graph peakedness\n                          ov], dim=-1)                          # channel agreement\n\n    def _fuse_mixture(self, sz, gz, cov):\n        \"\"\"log( (1-a_q) p_s + a_q p_g ) from the two standardised channels.\n\n        sz, gz : [B, n_doc] z-scored semantic and graph document scores (float32).\n        cov    : [B, 1] the additive arm's router feature, reused as feature 1.\n        returns (a_q [B], fused [B, n_doc]).\n        \"\"\"\n        # ROUTER FEATURES. The additive gate saw one number, the semantic top-5 mean, so\n        # it could tell whether the SEMANTIC scorer was confident but had no way to know\n        # whether the graph was worth listening to on this query. Under a mixture the\n        # flat-graph case is already handled exactly by the algebra, so these features are\n        # not load-bearing for safety -- they are what lets a_q open further on the\n        # queries where the graph is peaked AND disagrees, which is where a promote-only\n        # channel can actually change the answer. FUSION_ROUTER=cov drops back to the one\n        # feature if the richer input ever needs to be ablated out.\n        #\n        # Divisors are NOMINAL, not tuned: a top-5 mean of a z-score over a corpus this\n        # size runs about 3-6 and a top1-to-top5 margin about 1-3, so these put all five\n        # features on the same O(1) footing and stop the [0,1] overlap feature from\n        # starting with a hundredth of the gradient of the others. They are constants, not\n        # parameters, so nothing here can be fitted to the test set.\n        phi = self._router_phi(sz, gz, cov)\n        # AUTOCAST OFF, same reason as the additive gate: under AMP an nn.Linear emits\n        # bfloat16 whatever its parameter dtype, and the output bias sits at -1.39 where\n        # a bfloat16 ULP is 0.005, which is noise of the same order as the quantity being\n        # learned. Pinning the weights in `_apply` fixes the master copy, not this.\n        with torch.autocast(device_type=sz.device.type, enabled=False):\n            a = self.a_max * torch.sigmoid(self.router(phi.float())).squeeze(-1).float()\n        # FUSION_AFIX=<float> REPLACES THE ROUTER WITH A CONSTANT. The first mixture run\n        # (matsci, a_max=0.5) is why this exists: a_q climbed to 0.371 and spanned only\n        # [0.309, 0.437] over 331 queries, i.e. the router did not route, it just opened.\n        # It opened because it is trained on the TRAINING loss, where the GNN fits its own\n        # queries far better than the 0.27 nDCG@5 it scores at test, so it calibrated to a\n        # graph quality that does not exist at inference. Fused came out at 0.4265 against\n        # a 0.5178 semantic channel: -0.091, exactly the predicted damage at that weight.\n        # A constant a removes the train/test mismatch and makes the arm a one-parameter\n        # sweep, which is the only honest way to find out whether ANY a > 0 helps here.\n        if os.environ.get(\"FUSION_AFIX\", \"\") != \"\":\n            a = torch.full_like(a, float(os.environ[\"FUSION_AFIX\"]))\n        # DELIBERATELY OUTSIDE THE BRANCH ABOVE. tau_g used to be assigned inside it, so\n        # the router path -- i.e. every mixture arm run so far -- reached the division\n        # below with the name unbound. It has not fired yet only because those runs\n        # predate the FUSION_AFIX block; the next router run would raise UnboundLocalError.\n        tau_g = self.tau_min + (self.tau_max - self.tau_min) * torch.sigmoid(\n            self.tau_g_hat.float())\n        # tau_s is 1 and not learned, so log p_s = sz - logsumexp(sz) exactly (see the\n        # note in __init__). The per-query logsumexp is a constant within a query, and\n        # both the ranking and the cross-entropy loss are invariant to it, which is why\n        # this arm starts from the same objective the additive arm trains under.\n        lg = gz / tau_g.clamp(min=1e-3)\n        log_ps = sz - torch.logsumexp(sz, dim=-1, keepdim=True)\n        log_pg = lg - torch.logsumexp(lg, dim=-1, keepdim=True)\n        # FUSION_TOPK=<int> CONFINES THE GRAPH TO THE SEMANTIC HEAD. This is the one\n        # structural difference between the two forms, and it is why the additive arm wins\n        # despite its arithmetic ceiling. gamma*relu(z(g)) can move a document a few places\n        # WITHIN the semantic head and can essentially never lift one out of the deep tail;\n        # it is a safe local reordering. The mixture is the opposite: every document with\n        # a*p_g > p_s is scored log a + log p_g, i.e. purely by graph order, so they arrive\n        # as a contiguous block whose depth is set by a. nDCG@5 has five slots, and the\n        # graph's own top-5 is worth 0.245 against the semantic channel's 0.517, so once\n        # that block reaches rank 5 the trade is losing by construction. Masking p_g outside\n        # the semantic top-K keeps the calibrated reweighting (which the additive form can\n        # only approximate, capped at 0.36 z-units) and drops the flooding.\n        #\n        # Out-of-set documents get logaddexp(l1a + log_ps, -inf) = l1a + log_ps exactly:\n        # the semantic order shifted by one per-query constant, so their relative order is\n        # untouched and in-set documents can only rise past them, never the reverse. The\n        # renormalise keeps a's meaning (\"fraction of the mixture's mass taken from the\n        # graph\") rather than silently shrinking it by the mass that was masked away.\n        _k = int(os.environ.get(\"FUSION_TOPK\", \"0\"))\n        if 0 < _k < sz.shape[1]:\n            keep = torch.zeros_like(sz, dtype=torch.bool)\n            keep.scatter_(1, sz.topk(_k, dim=-1).indices, True)\n            log_pg = log_pg.masked_fill(~keep, float(\"-inf\"))\n            log_pg = log_pg - torch.logsumexp(log_pg, dim=-1, keepdim=True)\n        # clamp_min on a, and log1p(-a) rather than log(1-a), so the two mixing logs stay\n        # finite if the router saturates at either end. a_max <= 1 keeps 1-a positive.\n        la = torch.log(a.clamp(min=1e-8))[:, None]\n        l1a = torch.log1p(-a.clamp(max=1.0 - 1e-6))[:, None]\n        fused = torch.logaddexp(l1a + log_ps, la + log_pg)\n        # ONE FLOAT32 CAVEAT, measured. d/dx log((1-a)e^x + a e^k) > 0, so a constant\n        # (uninformative) p_g preserves the semantic order EXACTLY in real arithmetic.\n        # In float32 it preserves it only until the a*p_g floor swamps the semantic term\n        # and two documents COLLIDE onto the same value -- a tie, not a reorder. Measured\n        # on a 15,588-document corpus at a_q=0.1: top-300 bit-identical to the semantic\n        # order, first collision at rank ~3,100 between documents whose semantic z-scores\n        # differ by 4e-7, ~40 tied of 15,588. Far below anything reported (@5, @10, @100),\n        # so the invariance holds where it is read. It would NOT hold at recall@5000, and\n        # a metric that deep would need the tie broken by the semantic score.\n        with torch.no_grad():\n            self._fusion_note = (f\"mix a_q {a.mean().item():.4f} \"\n                                 f\"[{a.min().item():.4f},{a.max().item():.4f}] \"\n                                 f\"/{self.a_max:g}  tau_g {tau_g.item():.4f}\")\n        return a, fused\n\n    @staticmethod\n    def _z(x):\n        return (x - x.mean(-1, keepdim=True)) / (x.std(-1, keepdim=True) + 1e-6)\n\n    def _operator(self, ids, n_doc, device):\n        \"\"\"Recompute S_op live from cached ingredients with the current (trainable) w, beta.\"\"\"\n        order = [self._row[str(x.item() if hasattr(x, \"item\") else x)] for x in ids]\n        tag = order[0][0]\n        # One graph is resident per batch, so one table serves the whole batch. A\n        # mixed batch means the loader changed, and index_select below would silently\n        # read rows of the wrong corpus rather than fail.\n        assert all(t == tag for t, _ in order), (\n            f\"batch mixes operator tables {sorted({t for t, _ in order})}\")\n        idx = torch.tensor([r for _, r in order], dtype=torch.long)\n\n        dense = self._dense[tag].index_select(0, idx).to(device, torch.float32)\n        S = self._S[tag].index_select(0, idx).to(device, torch.float32)\n        M = self._M[tag].index_select(0, idx).to(device, torch.float32)\n        totS = self._totS[tag].to(device)                       # [n_doc]\n        assert dense.shape[1] == n_doc, f\"operator cols {dense.shape[1]} != {n_doc} doc nodes (alignment)\"\n\n        w = self.w_init + self.op_lr_scale * self.w_delta       # [3]\n        beta = (BETA_INIT + self.op_lr_scale * self.beta_delta).clamp(min=0.05)\n        dem = (totS.unsqueeze(0) - S).clamp(min=1e-6)           # [B, n_doc] leave-one-out popularity\n        degb = dem.pow(beta)\n        return w[0] * self._z(dense) + w[1] * self._z(S / degb) + w[2] * self._z(M / degb)\n\n    # ---------------------------------------------------------------- learned scorer\n    @staticmethod\n    def _load_semantic_module():\n        \"\"\"Import semantic_scorer.py from the repo rather than vendoring a copy of it.\n\n        The scorer, its normalisation and its popularity object are one design. A second\n        copy living here would drift from the one that produced the checkpoint, and that\n        drift would surface as a quietly different score rather than an import error.\n        \"\"\"\n        import importlib.util\n        import sys\n        root = os.environ.get(\"CARGO_ROOT\") or os.path.expanduser(\"~/Desktop/CARGO\")\n        path = f\"{root}/sir4-retrieval/eval/semantic_scorer.py\"\n        assert os.path.exists(path), (\n            f\"semantic scorer source not found at {path}; set CARGO_ROOT to the repo root\")\n        for p in (root, f\"{root}/kg-construction\"):\n            if p not in sys.path:\n                sys.path.insert(0, p)\n        spec = importlib.util.spec_from_file_location(\"cargo_semantic_scorer\", path)\n        mod = importlib.util.module_from_spec(spec)\n        spec.loader.exec_module(mod)\n        return mod\n\n    def _init_semantic(self, trainable):\n        sem = self._load_semantic_module()\n        self._sem = sem\n        self._sem_tab, self._sem_row, self._sem_last_tag = {}, {}, None\n        seen: dict[str, str] = {}\n        for split, ev in ((\"train\", \"SEMANTIC_COMPONENTS\"), (\"test\", \"SEMANTIC_COMPONENTS_TEST\")):\n            raw = os.environ.get(ev)\n            if not raw:\n                continue\n            for j, p in enumerate([x.strip() for x in raw.split(\",\") if x.strip()]):\n                ap = os.path.abspath(p)\n                if ap in seen:\n                    print(f\"[fusion] {ev}[{j}] is the same file as '{seen[ap]}', reusing it\")\n                    continue\n                tag = split if raw.count(\",\") == 0 else f\"{split}#{j}\"\n                seen[ap] = tag\n                d = np.load(p, allow_pickle=True)\n                Q, jmax, D = (int(x) for x in d[\"h_shape\"])\n                h_path = str(d[\"h_path\"])\n                assert os.path.exists(h_path), (\n                    f\"{p} references the per-answer matrix at {h_path}, which is missing. \"\n                    \"It lives in the semantic scorer's own cache, which does not survive a \"\n                    \"runtime reset; rerun section 5d or the precompute with --force.\")\n                col = np.asarray(d[\"col\"], dtype=np.int64)\n                nb = max(float(np.asarray(d[\"mask\"]).sum()), 1.0)\n                self._sem_tab[tag] = {\n                    # memmap: only the batch's rows and this graph's columns are ever read\n                    \"H\": np.memmap(h_path, np.float16, \"r\", shape=(Q, jmax, D)),\n                    \"col\": col,\n                    \"dense\": torch.from_numpy(np.asarray(d[\"dense\"], dtype=np.float16)),\n                    \"mask\": torch.from_numpy(np.asarray(d[\"mask\"]).astype(np.float32)),\n                    \"doc_emb\": torch.from_numpy(np.asarray(d[\"doc_emb\"], dtype=np.float32)),\n                    # the anchor target: total_S / (number of answers) IS the bank mean\n                    \"pop_target\": torch.from_numpy(\n                        np.asarray(d[\"total_S\"], dtype=np.float32) / nb),\n                    \"doc_emb_gpu\": None, \"pop_target_gpu\": None,\n                }\n                dup = [str(q) for q in d[\"query_ids\"] if str(q) in self._sem_row]\n                assert not dup, (f\"{p}: {len(dup)} query ids already claimed by another \"\n                                 f\"semantic components table (e.g. {dup[:3]})\")\n                for i, q in enumerate(d[\"query_ids\"]):\n                    self._sem_row[str(q)] = (tag, i)\n                print(f\"[fusion] loaded {ev}[{j}] as '{tag}': H {(Q, jmax, D)} -> {len(col)} \"\n                      f\"doc nodes ({Q} queries) {os.path.basename(p)}\")\n        assert self._sem_row, (\"semantic='mlp' needs SEMANTIC_COMPONENTS / \"\n                               \"SEMANTIC_COMPONENTS_TEST from precompute_semantic_components.py\")\n\n        # --- the trained scorer and its predictor, as ONE warm start ---\n        # Loading the scorer from a joint run without its predictor would pair a trained\n        # readout with an untrained popularity, which is a model that never existed.\n        ck, pn = os.environ.get(\"SEMANTIC_CKPT\", \"\"), os.environ.get(\"SEMANTIC_POPNET\", \"\")\n        assert ck and os.path.exists(ck), (\n            \"semantic='mlp' needs SEMANTIC_CKPT: a params_semantic_mlp_*.json from section 5d\")\n        assert pn and os.path.exists(pn), (\n            \"semantic='mlp' needs SEMANTIC_POPNET: the popnet_semantic_mlp_*.pt written \"\n            \"alongside it by a joint run (--mlp_pop_joint 1)\")\n        st = json.load(open(ck))\n        scorer = sem.SortedMLPScorer(\"cpu\", int(st[\"jmax\"]), hidden=len(st[\"net\"][\"0.weight\"]))\n        scorer.load(st)\n        psd = torch.load(pn, map_location=\"cpu\")\n        p_hidden, p_dim = psd[\"net.0.weight\"].shape\n        popnet = sem.MatchabilityPredictor(int(p_dim), int(p_hidden))\n        popnet.load_state_dict(psd)\n\n        # Registered as attributes so the optimiser sees them and .to(device) moves them.\n        # These are the SAME objects the scorer holds, so updates propagate both ways.\n        self.sem_net, self.sem_logbeta, self.sem_popnet = scorer.net, scorer.logbeta, popnet\n        self._sem_scorer = scorer\n        if not trainable:\n            for prm in self.sem_net.parameters():\n                prm.requires_grad_(False)\n            self.sem_logbeta.requires_grad_(False)\n            for prm in self.sem_popnet.parameters():\n                prm.requires_grad_(False)\n        print(f\"[fusion] semantic='mlp' warm start: jmax={st['jmax']} beta={st['beta']:.4f} \"\n              f\"popnet {p_dim}->{p_hidden}->1, trainable={trainable}\")\n\n    def _semantic(self, ids, n_doc, device):\n        \"\"\"Score with the learned sorted-MLP, recomputed live so it keeps training.\"\"\"\n        order = [self._sem_row[str(x.item() if hasattr(x, \"item\") else x)] for x in ids]\n        tag = order[0][0]\n        assert all(t == tag for t, _ in order), (\n            f\"batch mixes semantic tables {sorted({t for t, _ in order})}\")\n        tab, rows = self._sem_tab[tag], [r for _, r in order]\n        self._sem_last_tag = tag\n\n        # Slice rows from disk, then permute to nodes.csv document order. The resident\n        # cost is B x Jmax x n_doc, not the whole 0.1-1.8 GB matrix.\n        H = torch.from_numpy(\n            np.asarray(tab[\"H\"][rows])[:, :, tab[\"col\"]].astype(np.float32)).to(device)\n        idx = torch.tensor(rows, dtype=torch.long)\n        dense = tab[\"dense\"].index_select(0, idx).to(device, torch.float32)\n        mask = tab[\"mask\"].index_select(0, idx).to(device, torch.float32)\n        assert H.shape[-1] == n_doc, (\n            f\"semantic cols {H.shape[-1]} != {n_doc} doc nodes (alignment)\")\n\n        if tab[\"doc_emb_gpu\"] is None or tab[\"doc_emb_gpu\"].device != device:\n            tab[\"doc_emb_gpu\"] = tab[\"doc_emb\"].to(device, torch.float32)\n            tab[\"pop_target_gpu\"] = tab[\"pop_target\"].to(device, torch.float32)\n        pop = self._sem.Pop(\"joint\", predictor=self.sem_popnet, doc_emb=tab[\"doc_emb_gpu\"])\n        return self._sem_scorer(H, mask, dense, pop)\n\n    # ---------------------------------------------------------------- CQIG calibration\n    def _cqig_run_refs(self, graph, batches, dev, count=False):\n        for b in batches:\n            b = {k: (v.to(dev) if torch.is_tensor(v) else v) for k, v in b.items()}\n            self.base(graph, b)\n            if count:\n                # COUNT QUERIES, NOT BATCHES. `len(b[\"id\"])` counted whatever the loader\n                # happened to pack, so a bank of 16 items meant 16 queries at train batch\n                # size 1 and 123 at eval batch size 8, and the two calibrations were not\n                # comparable. Re-linked reference batches carry no \"id\" at all.\n                self.cqig.note_reference_batch(int(b[\"question_embeddings\"].shape[0]))\n\n    def cqig_calibrate(self, graph, graph_key, ref_batches=None, device=None, rounds=1):\n        \"\"\"Calibrate THIS graph from the fixed reference problems, in two passes per round.\n\n        Pass \"mean\" accumulates mu. Pass \"dev\" measures ||h - mu||^2 directly, which gives\n        both the variance -- exactly zero for a node whose state does not move, unlike the\n        one-pass identity -- and the deviations tau_ref is the median of.\n\n        `rounds` > 1 repeats both passes with the previous round's gates active. That\n        matters only when more than one layer is gated: a layer's own gate cannot change\n        its own input, so a single gated layer is already exact at rounds=1.\n\n        Unlabelled and target-query-free: this is the indexing-time pass that gives an\n        unseen graph its own reference statistics. Statistics are per graph because mu has\n        one row per node, so the training graph's mu is not even shape-compatible with a\n        different corpus. `ref_batches` defaults to the bank stored on the gate, which is\n        what makes a reloaded checkpoint able to calibrate a corpus it has never seen.\n        \"\"\"\n        assert self.cqig is not None, \"cqig is not enabled on this model\"\n        batches = ref_batches if ref_batches is not None else self.cqig.ref_bank\n        assert batches, (\"no reference bank: the trainer sets one during training and it \"\n                         \"is carried in the checkpoint; cannot calibrate without it\")\n        dev = device or next(self.parameters()).device\n        was_training = self.training\n        self.eval()\n        report = {}\n        with torch.no_grad():\n            for r in range(max(1, int(rounds))):\n                gated = r > 0\n                for phase in (\"mean\", \"dev\"):\n                    self.cqig.start_calibration(graph_key, phase=phase,\n                                                apply_existing_gates=gated)\n                    self._cqig_run_refs(graph, batches, dev, count=True)\n                    if phase == \"mean\":\n                        self.cqig.finish_mean()\n                    else:\n                        report = self.cqig.finish_calibration()\n                for row in report.values():\n                    row[\"round\"] = r + 1\n        if was_training:\n            self.train()\n        self.cqig.use_graph(graph_key)\n        self.cqig.mode = \"gate\"\n        return report\n\n    def cqig_use_graph(self, graph_key):\n        \"\"\"Point the gate at an already-calibrated graph, e.g. back to train after eval.\"\"\"\n        assert self.cqig is not None and self.cqig.calibrated(graph_key), (\n            f\"graph {graph_key!r} has not been calibrated \"\n            f\"(known: {self.cqig.known_graphs() if self.cqig else []})\")\n        self.cqig.use_graph(graph_key)\n        self.cqig.mode = \"gate\"\n\n    def semantic_aux_loss(self):\n        \"\"\"Log-space anchor holding p_hat near measured popularity. Mirrors Pop.aux_loss.\n\n        Without it the fusion's ranking gradient is free to repurpose the predictor as extra\n        scorer capacity, exactly as it would in the standalone run. Uses the tag of the last\n        forward, which is the corpus the current batch came from.\n        \"\"\"\n        tab = self._sem_tab.get(self._sem_last_tag) if self.semantic == \"mlp\" else None\n        if tab is None or tab[\"doc_emb_gpu\"] is None:\n            return 0.0\n        eps = 1e-6\n        p = self.sem_popnet(tab[\"doc_emb_gpu\"])\n        return ((torch.log(eps + p) - torch.log(eps + tab[\"pop_target_gpu\"])) ** 2).mean()\n\n    def forward(self, graph, batch, entities_weight=None):\n        # The frontier A^(l) is expanded inside bellmanford, which sees the edge index but\n        # not the batch. Handed over here rather than threaded through GraphReasoner's\n        # signature, which every other reasoner shares.\n        if getattr(self.base.entity_model, \"resp_proj\", None) is not None:\n            self.base.entity_model._ccmp_seeds = batch.get(\"start_nodes_mask\")\n        g = self.base(graph, batch, entities_weight)            # [B, N] per-node scores\n        doc = graph.nodes_by_type[\"document\"].to(g.device)      # nodes.csv document order\n        gdoc = g.index_select(1, doc).float()                   # [B, n_doc] graph-alone doc scores\n\n        # THE ONLY LINE THAT DIFFERS BETWEEN THE TWO ARMS. Everything below -- gate,\n        # relu floor, fusion arithmetic, hard-negative mining -- is shared, so a run\n        # pair isolates the semantic scorer with the graph half held fixed.\n        s_op = (self._operator(batch[\"id\"], doc.numel(), g.device)\n                if self.semantic == \"operator\"\n                else self._semantic(batch[\"id\"], doc.numel(), g.device))\n        # IS A HIGH INFORMATIVENESS ACTUALLY ON THE GOLD PAPERS? Measurement only, and only\n        # while the trainer has recording switched on (around evaluate()). Nothing here\n        # feeds back into the statistics, so it is not a transductive path -- but it is the\n        # test that decides whether the gate's premise holds at all, so it needs the labels\n        # and the hard negatives, which exist only here.\n        if self.cqig is not None and self.cqig.recording():\n            tgt = batch[\"target_nodes_mask\"] if \"target_nodes_mask\" in batch else None\n            self.cqig.note_alignment(tgt, doc, s_op.detach())\n        # THE GATE READS THE STANDARDISED SCORE, like the fusion does. On the raw score\n        # the semantic scorer could add a constant or scale everything up and change\n        # gamma_q while leaving z(s_op) and its own ranking untouched, so the gate would\n        # be keyed to a quantity with no semantic content. That is live for the learned\n        # arm in particular: an MLP's output scale is free, unlike the operator's\n        # warm-started w. After z(), a peaked top-5 genuinely means a confident scorer.\n        sz = self._z(s_op).float()\n        cov = sz.topk(min(5, sz.shape[1]), dim=-1).values.mean(-1, keepdim=True)\n        # .float() on both operands, explicitly. Under AMP autocast the gate and the\n        # learned scorer are nn.Linear and emit bfloat16 regardless of input dtype, and\n        # a bfloat16 sum would round away a graph contribution of order gamma=0.01 for\n        # the same reason the bfloat16 output write did. This currently lands in float32\n        # by accident, via promotion from `gz`; relying on that is one config change away\n        # from silently losing the effect.\n        # AUTOCAST OFF FOR THE GATE. Pinning the weights to float32 (see `_apply`) is not\n        # sufficient on its own: under AMP an nn.Linear is autocast to bfloat16 whatever\n        # its parameter dtype, so the pre-activation would be quantised at an ULP of\n        # 4.6 * 2^-8 = 0.018 around the -4.6 output bias, roughly 0.6% of gamma itself.\n        # The master weights would still update in float32, but gamma would carry\n        # avoidable noise. This gate is a 1->8->1 MLP, so float32 here costs nothing.\n        gz = self._z(gdoc).float()                              # what the ranking uses\n        if self.fusion_form == \"mixture\":\n            gamma, fused = self._fuse_mixture(sz, gz, cov)\n        else:\n            # phi_q, not cov. Under FUSION_ROUTER=cov this is exactly the old one-feature\n            # input; under the default `full` the gate additionally sees graph confidence,\n            # graph peakedness and channel agreement, so it can condition gamma on whether\n            # the GRAPH is worth listening to rather than only on semantic confidence.\n            # Needs `gz`, which is why this sits after the gz line above and not with cov.\n            with torch.autocast(device_type=cov.device.type, enabled=False):\n                phi = self._router_phi(sz, gz, cov)\n                gamma = F.softplus(self.gate(phi.float())).squeeze(-1).float()  # [B] per-query gate\n            # FUSION_GAMMAFIX=<float>, the additive twin of FUSION_AFIX, and the only way\n            # to find out whether the 0.5340 arm is a result or a coincidence. That number\n            # came from a run whose gamma was bf16-FROZEN at 0.0359; the fp32-learnable\n            # rerun froze at 0.0288 and scored 0.5218, and 0.012 nDCG@5 on n=331 is inside\n            # what two training runs differ by anyway. Pinning gamma at 0.0359 makes the\n            # comparison paired instead of accidental. Everything the mixture is being\n            # asked to beat rests on that one number reproducing.\n            if os.environ.get(\"FUSION_GAMMAFIX\", \"\") != \"\":\n                gamma = torch.full_like(gamma, float(os.environ[\"FUSION_GAMMAFIX\"]))\n            fused = sz + gamma[:, None] * torch.relu(gz)\n\n        # FLOAT32 OUT. `g` is bfloat16, whose epsilon near 1.0 is 2^-8 = 0.0039 -- and\n        # the graph's contribution is gamma_q * relu(z(graph)), which at the gate's 0.01\n        # init is the same order. Writing the fused score back into a bfloat16 tensor\n        # therefore rounds away most of the graph bonus and creates ranking ties, i.e.\n        # it destroys precisely the signal being measured. clone() first so a float32\n        # `g` is never mutated in place.\n        out = g.clone().float()\n        out[:, doc] = fused\n        self._raw_doc = gdoc          # RAW graph scores: mining and diagnostics (both top-k, scale-free)\n        # STANDARDISED graph scores: what the auxiliary loss must train, because it is\n        # what the fused ranking consumes. z() is scale-invariant, so a loss on the raw\n        # score has a free direction -- multiply every score by a constant and the loss\n        # falls while the fused ranking does not move at all. Kept pre-relu on purpose:\n        # relu would zero the gradient for exactly the golds sitting below the graph\n        # mean, which are the ones that most need lifting.\n        #\n        # DELIBERATELY z(g) UNDER BOTH FORMS, not lg = z(g)/tau_g. Cross-entropy is\n        # shift-invariant, so handing it lg would differ from gz only by the 1/tau_g\n        # factor, i.e. it would couple the auxiliary loss's temperature to a fusion\n        # parameter and give tau_g a gradient path that has nothing to do with fusion\n        # quality. Keeping gz makes the graph-alone objective IDENTICAL in the additive\n        # and mixture arms, so a run pair isolates the fusion form and nothing else.\n        self._raw_doc_z = gz\n        self._doc_ids = doc\n        # The SELECTED scorer, so switching `semantic` also switches which documents the\n        # contrastive loss makes the graph beat. One variable, every consumer.\n        self._s_op = s_op.detach()\n        # The gate, cached for diagnostics. Whether gamma_q ever leaves its 0.01 init is\n        # the first thing to know about a fusion run: if it does not, the fused score IS\n        # the semantic score and no loss change can matter.\n        #\n        # UNDER form=mixture THIS IS a_q, THE MIXING WEIGHT IN [0, a_max], NOT gamma. The\n        # eval hook prints it in the same column, so a mixture run's \"gamma mean\" is on a\n        # different scale from every additive run's and the two must not be read off the\n        # same axis. a_q ~ 0.1 is its initialisation, not a stall.\n        self._gamma = gamma.detach()\n        return out\n", "/content/gfm-rag/gfmrag/models/cqig.py": "\"\"\"\ncqig.py \u2014 Cross-Query Informativeness Gating.\n\nA node earns the right to send a message by responding to THIS query differently from how\nthe nodes of this graph TYPICALLY respond to unrelated scientific problems. A node that\nanswers every query the same way is broadcasting background graph activity, and under sum\naggregation that broadcast reaches every one of its neighbours identically, so it shifts a\nwhole domain rather than discriminating within it.\n\n    mu_v^l    = (1/M) sum_m h_{a_m v}^l                       reference mean,     PER NODE\n    V_v^l     = (1/(M-1)) sum_m || h_{a_m v}^l - mu_v ||^2    reference variance, PER NODE\n    s^l       = median_{v : V_v^l > delta} V_v^l              reference scale,    PER LAYER\n    I_qv^l    = || h_qv^l - mu_v^l ||^2 / (s^l + eps)\n    tau_l(G)  = tau_ref_l(G) + dtau_l                         graph term + LEARNED term\n    g_qv^l    = 1 - lam_l [ 1 - sigmoid( alpha_l ( log(1+I) - tau_l(G) ) ) ]\n    m~        = g_qv^l * m_{v->u}^l                           op=\"gate\", the default\n\nWHY THE DENOMINATOR IS A LAYER-WIDE SCALE AND NOT THE NODE'S OWN VARIANCE. Two dead ends\ncame first and both are kept as ablations, because the difference between them IS the\nhypothesis:\n\n  norm=\"energy\"  divide by E_v = mean ||h||^2. Dead on arrival under\n                 `use_ent_emb: early-late-fusion`: h_qv is dominated by the node's static\n                 entity embedding and the query-dependent part is ~1% of it in norm, so\n                 I ~ 1e-4 for EVERY node and the sigmoid never leaves its midpoint.\n  norm=\"node\"    divide by the node's OWN variance V_v. Numerically alive (I ~ 1) but it\n                 cancels the very comparison the method is built on: a generic node that\n                 always moves by 0.01 and a discriminative node that always moves by 1.0\n                 both come out at 0.01/0.01 = 1.0/1.0 = 1. Every node looks equally\n                 informative and the gate is a near-constant rescaling.\n  norm=\"layer\"   divide by ONE robust scale for the whole layer. A weak generic response is\n                 now small, a strong query-specific response is large, and a node that\n                 never responds stays at zero. This is the only form in which I is\n                 comparable BETWEEN nodes, which is what \"this node is informative\" has to\n                 mean. Default.\n\nWHY CALIBRATION IS TWO PASSES AND NOT ONE. The one-pass identity E||h-mu||^2 = E||h||^2 -\n||mu||^2 is a catastrophic cancellation here. The deviation is ~1e-4 of the energy under\nearly-late fusion, float32 carries ~1e-7 relative, and the accumulation itself rounds. At\nthe real proportions (D=1024, deviation 1% of the state norm) that leaves a node which\nnever responds with a variance up to 4e-3 of a genuine one -- ragged, node-dependent, and\nclose enough to a real response to survive any sane cut. The second pass computes\n||h - mu||^2 directly: the residue drops to ~1e-10 of the working scale and, more useful\nstill, becomes UNIFORM across the unreachable part of the graph, so one threshold removes\nall of it. It costs one more forward over M references.\n\nWHY THE THRESHOLD IS MEASURED, NOT DERIVED. The same second pass records the deviations\nthemselves, so tau_ref is the median of log(1+I) over the (node, reference query) pairs\nthat respond at all. Deriving it from V instead would compare a per-query deviation against\na mean-over-queries deviation, which sit at systematically different points of the same\ndistribution. Measuring puts the operating point exactly at \"a typical reference query's\ntypical responding node\", so dtau is a learned offset in interpretable units. Both the\nvariance and the recorded deviations are corrected to leave-one-out, since mu is built from\nthe same M references: V uses the M-1 denominator, and a stored deviation is scaled by\n(M/(M-1))^2, which is exactly ||h_m - mu_{-m}||^2. A query the bank has never seen then\nlands in the distribution tau_ref was measured on.\n\nWHY I IS DETACHED FROM THE AUTOGRAD GRAPH BY DEFAULT. With a gradient path through I the\nGNN can open every gate at once by drifting h away from mu -- mu is only recomputed every\nCQIG_RECAL steps, so that direction is free, it raises I for every query equally, and it\ndestroys the mechanism while lowering the loss. Detached, the gate is a modulation whose\nshape (lam, alpha, dtau) is still learned, and h still receives gradient through the\nproduct h*g. Set grad_through_I=True to restore the differentiable version as an ablation.\nDetaching also lets I be computed in float32 for free, which is worth having but is not a\ncure. h is bfloat16 under autocast and the deviation is a small fraction of it, so most of\neach component's significant digits are gone before the gate sees them. Measured against\nfloat64 at D=1024: with the deviation at 1% of the state norm, doing the subtraction and\nthe sum in float32 gives ~2.6% median error on I against ~5.2% for the all-bfloat16 path\nthe earlier version used -- a factor of about two across M and across deviation ratios from\n0.3% to 3%. The remainder is h's OWN quantisation and cannot be recovered here. What that\nerror scales with is the deviation ratio, not the arithmetic: at 0.3% of norm even the\nfloat32 path carries ~29%, at 3% it carries ~0.3%. If a layer's states are nearly identical\nacross queries, I is noise there whatever the dtype, which is what `responding` in the\ncalibration report is for.\n\nWHY A FORWARD PRE-HOOK AND NOT A FORK OF THE MESSAGE PASSING. The layer computes\n`message = input_j * relation_j` for DistMult, which is LINEAR in the source state, and the\nboundary condition arrives as a separate argument while the residual is added outside the\nlayer. Scaling `input` before the call is therefore exactly equivalent to scaling every\nmessage leaving that node, and touches neither the boundary nor the shortcut.\n\nTHREE OPERATORS, ONE COEFFICIENT (`op`). The same pre-hook and the same statistic support a\nsecond family, selected by `op`. Write d_qv = h_qv - mu_v for the query-specific residual.\nBecause a DistMult message is LINEAR in the source state and the relation embedding is a\nfunction of the graph's relation text alone -- `rel_mlp(graph.rel_attr)` is expanded across\nthe batch and never sees the query -- E_a[ m(h_av, r) ] = m(mu_v, r) holds EXACTLY, so\n\"propagate the residual and retain a kappa-fraction of the background\" is a subtraction on\nthe input and the hook implements it exactly rather than approximately:\n\n    m(d_qv, r) + kappa m(mu_v, r) = m( h_qv - (1 - kappa) mu_v , r )\n\n    op=\"gate\"          h_qv * g_qv                 scale the whole state, the original\n    op=\"centre\"        h_qv - (1 - g_qv) mu_v      subtract background in proportion to\n                                                   how UNinformative the node is\n    op=\"centre-fixed\"  h_qv - lam_l mu_v           the same subtraction at a constant rate\n\nwith kappa_qv = g_qv, so one coefficient covers all of it and lam_l = 0 recovers the ungated\nreasoner exactly under every op. The ladder is nested: fixed centring is adaptive centring\nwith the coefficient frozen, and `op=\"centre-fixed\"` at lam=1 is full centring, residual only.\n\nWHAT ACTUALLY DIFFERS BETWEEN GATING AND CENTRING, given both are driven by the same g:\n\n  * DIRECTION. Gating is a scalar multiple, so the message keeps its direction and only its\n    magnitude moves. Centring removes a component, so the direction moves too. At full\n    damping a gated node sends nothing while a centred node sends its residual d_qv. Whether\n    silence or the residual is better is an empirical question about what a generic node's\n    small d_qv contains -- encoder noise, or a weak but real cross-domain cue -- and it is\n    the one thing the pair of arms genuinely tests.\n  * WHICH NODES ARE TOUCHED. A node no reference query reaches has mu_v = 0 EXACTLY, so\n    centring leaves it alone, while gating sees I = 0 and damps it to the floor. When seed\n    coverage is partial these are not small print: they are opposite treatments of whatever\n    fraction of the graph the references never entered, which `mu_zero_frac` reports.\n  * QUERY-CONDITIONING. mu_v does not depend on the query, so `op=\"centre-fixed\"` subtracts a\n    query-INDEPENDENT background from every node. It still changes rankings -- a node fed by\n    many broadly-active neighbours loses more -- but it is a hub correction, not\n    query-conditioned reasoning, and alpha and dtau receive no gradient in that arm. Only\n    the adaptive coefficient makes the operator a function of the query.\n  * SIZE OF THE EDIT. Under `use_ent_emb: early-late-fusion` mu is dominated by the node's\n    static entity embedding, so at a given lam the subtraction can be a far larger\n    perturbation than the multiplication. `edit_rel` in the live report measures it directly\n    as ||h~ - h|| / ||h||, which is also the number to read against `layer_norm: yes`: the\n    norm renormalises much of a magnitude change away and leaves the direction change, so a\n    centring arm should be expected to act through the direction.\n\nGATING MORE THAN ONE LAYER. Calibration of layer l assumes the state entering it was\nproduced by the same computation at inference. With layers upstream of l gated, that holds\nonly if their gates were also active while l was calibrated. `rounds=1` calibrates every\ngated layer with no gate active anywhere, which is exact for a single gated layer -- a\nlayer's own gate cannot change its own input -- and leaves the upstream mismatch in place\nfor a multi-layer arm. `rounds=2` repeats both passes with the round-1 gates live, which\nremoves it. Anything above 2 is a fixed-point iteration and has not been needed.\n\nSTATISTICS ARE PER GRAPH AND MUST BE SELECTED BEFORE A FORWARD. mu has one row per node, so\nthe train graph's mu is not even shape-compatible with the test graph's states. `_stats` is\nkeyed by a caller-supplied graph key and `use_graph(key)` must be called before any gated\nforward. Reading statistics for the wrong graph raises rather than broadcasting.\n\nZERO-SHOT, AND WHAT THAT ACTUALLY REQUIRES. The reference bank is stored in a\nGRAPH-INDEPENDENT form: a frozen question embedding plus the entity-linked start nodes as\nNAMES with their attachment weights. `materialise(node2id, num_nodes)` re-links those same\nsource problems against any target graph, exactly as the indexer does\n(`start_mask[node2id[name]] = weight`), and returns both the rebuilt batches and the\ncoverage of the source seeds in the target vocabulary. Every graph is therefore calibrated\nwith the SAME M individual reference problems, and nothing is ever read from the target\ncorpus's own query set. A reloaded checkpoint carries the bank in its state dict, so an\nunseen graph needs one unlabelled calibration pass at indexing time and nothing else.\n\nEXACT NAME LOOKUP IS NOT ENOUGH, AND THE FAILURE IS SILENT. Re-linking by string equality\nassumes the target graph spells its concepts the way the source graph does. It does not:\non sir4_physics the source seeds resolved 501/501 against the train graph and 25/501\nagainst the TEST graph, same domain, different papers. mu was then estimated from\nreferences that barely entered the graph, and nothing failed loudly -- coverage was a\nprinted number, not an error. `link=\"semantic\"` fixes the linker rather than the bank:\n\n    exact match if the name exists; otherwise take the K nearest target nodes to the\n    seed's FROZEN source embedding, reject any below a source-selected cosine floor,\n    and split the seed's attachment weight across the survivors by softmax(cos / T).\n\nThe seed's total starting weight is preserved, so a semantically linked reference injects\nexactly as much probability mass as an exactly linked one and the two are comparable. This\nis still zero-shot: the reference questions are fixed on source data, the encoder is frozen,\nthe floor is chosen from source-side statistics, the target's labels and queries are never\ntouched, and nothing is fine-tuned. What changes is only that a reference problem can now\nenter a target graph that does not happen to use the source's exact phrasing.\n\nlam_l = 0 recovers the ungated reasoner exactly, so the ablation is one scalar.\n\"\"\"\nimport math\nimport os\n\nimport torch\nimport torch.nn as nn\n\n# A normalised response below this counts as \"this node did not respond to this query\".\n# I is ~1 for a typical responding node by construction, so 1e-6 is six orders below the\n# working range. With the deviation computed directly it only ever catches exact zeros.\nRESP = 1e-6\n\n# Bound on the transient float32 buffer used to compute ||h - mu||^2 without ever\n# subtracting in bfloat16. ~256 MB.\n_CHUNK_BYTES = 1 << 28\n\n# Bound on the reference deviations kept for the threshold and the gate report. Whole\n# rows are kept, never a flattened subsample, so a per-node scale stays aligned.\n_SAMPLE_ELEMS = 8_000_000\n\n# Target nodes per chunk when matching seeds by cosine. A seed set is a few hundred rows\n# and a graph is a few hundred thousand, so the full product is ~0.5 GB in float32 and is\n# built one slice at a time instead.\n_LINK_CHUNK = 65_536\n\n\ndef topk_cosine(q, node_emb, k, chunk=_LINK_CHUNK):\n    \"\"\"Top-k cosine of each row of `q` against every row of `node_emb`.\n\n    Both are L2-normalised here rather than by the caller, so a caller cannot pass one\n    normalised and one not and get a silently rescaled similarity. Chunked over target\n    nodes: the full [seeds, nodes] product is what makes this expensive, and it is never\n    needed all at once.\n    \"\"\"\n    qn = q.float()\n    qn = qn / qn.norm(dim=-1, keepdim=True).clamp_min(1e-6)\n    n = int(node_emb.shape[0])\n    k = max(1, min(int(k), n))\n    best_c = torch.full((qn.shape[0], k), -2.0, device=qn.device)\n    best_i = torch.zeros((qn.shape[0], k), dtype=torch.long, device=qn.device)\n    for a in range(0, n, chunk):\n        blk = node_emb[a:a + chunk].to(qn.device).float()\n        blk = blk / blk.norm(dim=-1, keepdim=True).clamp_min(1e-6)\n        c = qn @ blk.T                                       # [S, chunk]\n        kk = min(k, c.shape[1])\n        cv, ci = torch.topk(c, kk, dim=1)\n        best_c = torch.cat([best_c, cv], 1)\n        best_i = torch.cat([best_i, ci + a], 1)\n        best_c, order = torch.topk(best_c, k, dim=1)\n        best_i = torch.gather(best_i, 1, order)\n    return best_c, best_i\n\n\ndef _inv_softplus(y: float) -> float:\n    return math.log(math.expm1(y))\n\n\ndef _logit(p: float) -> float:\n    return math.log(p / (1.0 - p))\n\n\ndef _describe(t, qs=(0.1, 0.5, 0.9)):\n    \"\"\"min / quantiles / max / mean / std of a tensor, as plain floats.\n\n    min, max, mean and std are exact over the whole tensor; only the quantiles fall back\n    to a random subsample, because torch.quantile has a hard size ceiling.\n    \"\"\"\n    keys = [f\"p{int(round(q * 100))}\" for q in qs]\n    if t is None or t.numel() == 0:\n        return dict({\"n\": 0, \"min\": 0.0, \"max\": 0.0, \"mean\": 0.0, \"std\": 0.0},\n                    **{k: 0.0 for k in keys})\n    t = t.detach().float().flatten()\n    out = {\"n\": int(t.numel()), \"min\": float(t.min()), \"max\": float(t.max()),\n           \"mean\": float(t.mean()),\n           \"std\": float(t.std(unbiased=False)) if t.numel() > 1 else 0.0}\n    s = t\n    if s.numel() > 1_000_000:\n        s = s[torch.randperm(s.numel(), device=s.device)[:1_000_000]]\n    qv = torch.quantile(s, torch.tensor(list(qs), device=s.device, dtype=s.dtype))\n    for k, v in zip(keys, qv):\n        out[k] = float(v)\n    return out\n\n\ndef parse_layers(spec, n_layers):\n    \"\"\"Resolve a gated-layer specification to 0-indexed layer positions.\n\n    NUMBERS ARE 1-INDEXED, matching how the layers are talked about: with six layers,\n    \"6\" is the last one and \"3-6\" is the second half. Accepted forms:\n\n        None / \"\" / \"last\"     the final layer only (default, exact at rounds=1)\n        \"all\"                  every layer\n        \"6\" / \"3-6\" / \"3,5,6\"  1-indexed layer numbers, ranges inclusive\n        [3, 4, 5, 6]           the same, as a list\n\n    Returns 0-indexed positions, which is what the hook and the checkpoint use.\n    \"\"\"\n    if spec is None:\n        return {n_layers - 1}\n    if isinstance(spec, str):\n        s = spec.strip().strip(\"'\\\"\").lower()\n        if s in (\"\", \"none\", \"null\", \"last\"):\n            return {n_layers - 1}\n        if s == \"all\":\n            return set(range(n_layers))\n        nums = set()\n        for part in s.replace(\" \", \"\").split(\",\"):\n            if not part:\n                continue\n            if \"-\" in part.lstrip(\"-\"):\n                a, b = part.split(\"-\", 1)\n                lo, hi = int(a), int(b)\n                assert lo <= hi, f\"cqig layer range {part!r} runs backwards\"\n                nums.update(range(lo, hi + 1))\n            else:\n                nums.add(int(part))\n    else:\n        nums = {int(x) for x in spec}\n    assert nums, f\"cqig layer spec {spec!r} selects no layers\"\n    bad = sorted(x for x in nums if not 1 <= x <= n_layers)\n    assert not bad, (f\"cqig layer spec {spec!r} names layer(s) {bad}, but the model has \"\n                     f\"{n_layers} layers, numbered 1..{n_layers} (1-indexed)\")\n    return {x - 1 for x in nums}\n\n\nclass CQIGate(nn.Module):\n    \"\"\"Per-layer informativeness gate, attached to an entity model's conv layers.\n\n    `mode` is WHAT THE HOOK IS DOING RIGHT NOW and `op` is WHICH OPERATOR the arm runs.\n    They are independent and the names are close, so they are spelled out here:\n\n    mode: \"off\"        pass through untouched; the model is bit-identical to the ungated one\n          \"calibrate\"  accumulate reference statistics for the active graph (two phases)\n          \"gate\"       compute I against the active graph's statistics and edit the state\n\n    op:   \"gate\"          h * g                 the original, and the default\n          \"centre\"        h - (1 - g) * mu      adaptive centring\n          \"centre-fixed\"  h - lam * mu          query-independent centring\n\n    mu:   \"ref\"           mu from the reference bank; the method\n          \"zero\"          mu forced to 0; THE ABLATION, see mu_zero below\n    \"\"\"\n\n    OPS = (\"gate\", \"centre\", \"centre-fixed\")\n    MUS = (\"ref\", \"zero\")\n\n    def __init__(self, layers, lam_init=0.1, alpha_init=1.0, rho=1e-3, eps=1e-12,\n                 gate_layers=None, norm=\"layer\", grad_through_I=False, op=\"gate\",\n                 mu=\"ref\"):\n        super().__init__()\n        self.n_layers = len(layers)\n        self.gate_layers = parse_layers(gate_layers, self.n_layers)\n        self.rho, self.eps = rho, eps\n        assert norm in (\"layer\", \"node\", \"energy\"), (\n            f\"unknown cqig norm {norm!r}; one of 'layer' (default), 'node', 'energy'\")\n        self.norm = norm\n        self.grad_through_I = bool(grad_through_I)\n        op = str(op or \"gate\").strip().lower()\n        assert op in self.OPS, (\n            f\"unknown cqig op {op!r}; one of {', '.join(map(repr, self.OPS))}. This is the \"\n            f\"OPERATOR (what is done to the state), not `mode` (what the hook is doing).\")\n        self.op = op\n        self.centring = op != \"gate\"\n        mu = str(mu or \"ref\").strip().lower()\n        assert mu in self.MUS, f\"unknown cqig mu {mu!r}; one of {', '.join(map(repr, self.MUS))}\"\n        self.mu_source = mu\n        self.mu_zero = mu == \"zero\"\n\n        # lam in [0,1] via sigmoid, alpha > 0 via softplus.\n        # lam_init is deliberately NOT 0: at exactly 0 the sigmoid derivative is negligible\n        # and the gate could never learn to open, which would present as \"it did nothing\".\n        self.lam_raw = nn.Parameter(torch.full((self.n_layers,), _logit(lam_init)))\n        self.alpha_raw = nn.Parameter(torch.full((self.n_layers,), _inv_softplus(alpha_init)))\n        # LEARNED OFFSET ONLY. The absolute threshold is graph-specific and recomputed at\n        # every calibration; a single learned tau was silently overwritten by each recal,\n        # so it never actually learned anything.\n        self.dtau = nn.Parameter(torch.zeros(self.n_layers))\n\n        # The reference bank travels with the model, in a form that is not tied to any\n        # graph, so a reloaded checkpoint can calibrate a corpus it has never seen.\n        self.ref_bank = None                  # list[dict]: {\"qemb\", \"seeds\", \"id\"}\n        self.ref_meta = {}\n\n        self.mode = \"off\"\n        self._stats: dict = {}                # graph_key -> {layer: {mu, scale, ...}}\n        self._active = None                   # graph_key currently selected\n        self._acc: dict = {}                  # layer -> running sums for the current phase\n        self._mu: dict = {}                   # layer -> mu, between the two phases\n        self._energy: dict = {}               # layer -> mean ||h||^2, for norm=\"energy\"\n        self._acc_key, self._n_ref, self._m_ref = None, 0, 0\n        self._phase = \"mean\"\n        self._calib_gate = False              # apply existing gates while calibrating\n        self._live = None                     # layer -> running gate histogram, when on\n        self._live_bins = 256\n        self._align: dict = {}                # layer -> gold-vs-hard-negative tally\n        self._edit: dict = {}                 # layer -> ||h~ - h|| / ||h|| tally\n        self.last_I: dict = {}                # layer -> [B, N] I of the last forward\n        self._handles = [\n            layer.register_forward_pre_hook(self._make_hook(i))\n            for i, layer in enumerate(layers)\n        ]\n        self.last_gate = {}\n        self._lowp_note = False\n\n    # ------------------------------------------------------------------ precision\n    def _apply(self, *args, **kwargs):\n        \"\"\"Keep lam_raw, alpha_raw and dtau in float32 through `model.to(dtype=...)`.\n\n        THIS IS A CORRECTNESS FIX, NOT AN OPTIMISATION. The trainer casts the whole model\n        with `model.to(dtype=torch.bfloat16)` (utils/setup_training.py), which catches these\n        three scalars. A bfloat16 value has a 2^-8 relative ULP, so half an ULP is\n        |w| * 2^-9 .. |w| * 2^-8, while an AdamW step is at most `lr`. At lr 5e-4 every\n        parameter above |w| ~= 0.128 has its update rounded straight back to the old value\n        on EVERY step, and because round-to-nearest keeps no remainder it never accumulates.\n\n        lam_raw = logit(0.9) = 2.197 and alpha_raw = inv_softplus(1) = 0.541 are both above\n        that line, so neither ever learned. The logs are their own proof: float32 init would\n        report lam exactly 0.9000000 and alpha exactly 1.0000000, and what every run to date\n        actually reported was lam = 0.9023438 = sigmoid(bf16(2.197)) and alpha = 1.00104 =\n        softplus(bf16(0.541)). Those digits ARE the rounding error. dtau starts at 0 where\n        bfloat16 resolves finely so it did move, but it stalls hard at |dtau| = 0.25, where\n        half an ULP overtakes lr again.\n\n        `.to()`, `.cuda()`, `.float()` and DDP all funnel through `_apply`, so refusing the\n        cast here covers every call site and travels with this file alone. Device moves are\n        unaffected: the saved copy is restored onto the parameter's NEW device. The values\n        kept are the pre-cast float32 ones, so lam_init is exactly logit(0.9) again rather\n        than its bfloat16 neighbour.\n\n        Nothing downstream needs a matching change. I is already float32 out of `_sqdist`,\n        and both operator branches cast the coefficient back themselves -- `g.to(h.dtype)`\n        in `_apply_op` and the per-chunk `.to(h.dtype)` in `_centre`. So the gate is\n        computed in float32 and only the coefficient crosses into bfloat16, which is what\n        mixed precision was supposed to be doing. Set CQIG_ALLOW_LOWP=1 to reproduce the old\n        frozen-scalar runs.\n        \"\"\"\n        saved = {n: p.detach().clone().float()\n                 for n, p in self.named_parameters(recurse=False)}\n        out = super()._apply(*args, **kwargs)\n        if os.environ.get(\"CQIG_ALLOW_LOWP\", \"0\") == \"1\":\n            return out\n        pinned = []\n        for n, p in self.named_parameters(recurse=False):\n            if n in saved and p.is_floating_point() and p.dtype != torch.float32:\n                was = p.dtype\n                p.data = saved[n].to(device=p.device)\n                pinned.append((n, was))\n        if pinned and not self._lowp_note:\n            self._lowp_note = True\n            print(f\"[cqig] gate scalars pinned to float32 against a \"\n                  f\"{pinned[0][1]} model cast: {', '.join(n for n, _ in pinned)}. \"\n                  f\"These are learnable; in {pinned[0][1]} an lr-5e-4 step is below half \"\n                  f\"an ULP for lam_raw and alpha_raw, so they would never move.\")\n        return out\n\n    # ------------------------------------------------------------------ parameters\n    def lam(self, i):\n        return torch.sigmoid(self.lam_raw[i])\n\n    def alpha(self, i):\n        return nn.functional.softplus(self.alpha_raw[i])\n\n    def tau(self, i, key=None):\n        st = self._stats.get(key if key is not None else self._active) or {}\n        return st.get(i, {}).get(\"tau_ref\", 0.0) + float(self.dtau[i].detach())\n\n    def _g_of(self, i, I, tau):\n        return 1.0 - self.lam(i) * (\n            1.0 - torch.sigmoid(self.alpha(i) * (torch.log1p(I) - tau))\n        )\n\n    @staticmethod\n    def _g_scalars(I, lam, alpha, tau):\n        \"\"\"The same gate from plain floats, for reporting off-device.\"\"\"\n        return 1.0 - lam * (1.0 - torch.sigmoid(alpha * (torch.log1p(I) - tau)))\n\n    # ------------------------------------------------------------------ graph selection\n    def use_graph(self, key):\n        \"\"\"Select which graph's statistics the gate reads. Required before a gated forward.\"\"\"\n        self._active = key\n\n    def calibrated(self, key):\n        return key in self._stats\n\n    def known_graphs(self):\n        return sorted(self._stats)\n\n    # ------------------------------------------------------------------ reference bank\n    def set_reference_bank(self, refs, meta=None):\n        \"\"\"Store M source reference problems in a GRAPH-INDEPENDENT form.\n\n        Each ref is {\"qemb\": [1, D] float tensor, \"seeds\": [(name, weight), ...], \"id\": str}\n        and optionally \"seed_emb\": [len(seeds), D], the FROZEN source-graph embedding of each\n        seed name. Names alone re-link only where the target spells a concept identically;\n        the embeddings are what let `link=\"semantic\"` find \"thin-film growth\" from \"thin film\n        deposition\". They are source-side artefacts of a frozen encoder, so carrying them\n        costs one float32 row per seed (~1.8 MB at M=16) and concedes nothing about zero-shot.\n        \"\"\"\n        bank = []\n        for r in refs:\n            e = r[\"qemb\"].detach().float().cpu()\n            assert e.dim() == 2 and e.shape[0] == 1, (\n                f\"a reference must be ONE query, got question_embeddings of shape \"\n                f\"{tuple(e.shape)}; batches of queries are not references\")\n            seeds = [(str(n), float(w)) for n, w in r[\"seeds\"]]\n            se = r.get(\"seed_emb\")\n            if se is not None:\n                se = se.detach().float().cpu()\n                # ROW k OF seed_emb IS SEED k. If that correspondence slips, every semantic\n                # link is to the wrong concept and nothing downstream can detect it.\n                assert se.dim() == 2 and se.shape[0] == len(seeds), (\n                    f\"seed_emb has {tuple(se.shape)} rows for {len(seeds)} seeds; they are \"\n                    f\"positionally paired and must be built together\")\n            bank.append({\"qemb\": e, \"seeds\": seeds, \"seed_emb\": se, \"id\": r.get(\"id\")})\n        assert len(bank) >= 2, (\n            f\"a reference bank of {len(bank)} cannot give a cross-query variance; \"\n            f\"raise CQIG_M\")\n        self.ref_bank = bank\n        self.ref_meta = dict(meta or {})\n\n    def materialise(self, node2id, num_nodes, device=None, dtype=None, batch_size=1,\n                    node_emb=None, link=\"exact\", link_k=3, link_temp=0.05, link_floor=0.0):\n        \"\"\"Re-link the stored source problems against THIS graph's vocabulary.\n\n        `link=\"exact\"` is the operation the indexer performs when it builds start_nodes_mask:\n        `start_mask[node2id[name]] = weight`, silently dropping names the graph does not\n        contain. That silent drop is the failure mode: 5% coverage and 100% coverage produce\n        the same shapes and the same absence of errors.\n\n        `link=\"semantic\"` keeps every exact match and rescues the rest. A seed the target\n        does not spell is embedded (its frozen source embedding is in the bank), matched to\n        its `link_k` nearest target nodes, filtered at `link_floor`, and its weight split\n        across the survivors by softmax(cos / link_temp). The SPLIT IS NORMALISED OVER THE\n        SURVIVORS, so the seed contributes exactly its original weight whether it landed on\n        one node or three, and an exactly linked seed and a semantically linked one inject\n        the same mass. `link_floor` comes from the source side (see the trainer), so no\n        target statistic sets it.\n\n        Returns (batches, coverage). Coverage separates exact from semantic hits, because\n        an aggregate that mixes them cannot answer the question the arm exists to ask.\n\n        Every graph gets the SAME M problems, so `coverage[\"queries\"]` is M on every graph\n        and two graphs' statistics are computed from equal-sized reference sets.\n        \"\"\"\n        assert self.ref_bank, \"no reference bank; call set_reference_bank first\"\n        assert link in (\"exact\", \"semantic\"), f\"unknown cqig link mode {link!r}\"\n        semantic = link == \"semantic\"\n        if semantic:\n            assert node_emb is not None, (\n                \"link='semantic' needs the TARGET graph's node embeddings (graph.x); it is \"\n                \"None, so this graph was indexed without node features\")\n            assert int(node_emb.shape[0]) == int(num_nodes), (\n                f\"node_emb has {int(node_emb.shape[0])} rows for a {num_nodes}-node graph\")\n            missing = [r[\"id\"] for r in self.ref_bank if r.get(\"seed_emb\") is None]\n            assert not missing, (\n                f\"{len(missing)} reference(s) carry no seed embeddings, so they cannot be \"\n                f\"linked semantically. The bank was frozen before this feature existed; \"\n                f\"retrain, or run this arm with CQIG_LINK=exact\")\n        masks, embs, total, empty = [], [], 0, 0\n        n_exact = n_sem = 0\n        sem_cos, sem_deg = [], []\n        for r in self.ref_bank:\n            mask = torch.zeros(num_nodes, dtype=torch.float32)\n            hit, pend = 0, []\n            for k, (name, w) in enumerate(r[\"seeds\"]):\n                j = node2id.get(name)\n                if j is not None and 0 <= int(j) < num_nodes:\n                    # ACCUMULATE. Exact hits have distinct ids so this matches the old\n                    # assignment, but a semantic link can land on an already-seeded node\n                    # and overwriting there would quietly destroy the other seed's weight.\n                    mask[int(j)] += w\n                    n_exact += 1\n                    hit += 1\n                elif semantic:\n                    pend.append((k, w))\n            if pend:\n                # Onto the GRAPH's device, not the bank's: the bank is a few hundred CPU\n                # rows and the graph is hundreds of thousands, so moving the small side is\n                # the only direction that does not stream the whole graph through host RAM.\n                q = r[\"seed_emb\"][[k for k, _ in pend]].to(node_emb.device)\n                cos, idx = topk_cosine(q, node_emb, link_k)\n                cos, idx = cos.cpu(), idx.cpu()\n                for t, (_, w) in enumerate(pend):\n                    keep = cos[t] >= link_floor\n                    if not bool(keep.any()):\n                        continue                      # below the floor: a real miss, not a link\n                    c, ix = cos[t][keep], idx[t][keep]\n                    # Softmax over the SURVIVORS, times w: total attachment weight preserved.\n                    wt = torch.softmax(c / max(float(link_temp), 1e-6), dim=0) * w\n                    mask.index_add_(0, ix, wt)\n                    n_sem += 1\n                    hit += 1\n                    sem_cos.append(float(c.max()))\n                    sem_deg.append(int(ix.numel()))\n            total += len(r[\"seeds\"])\n            empty += int(hit == 0)\n            masks.append(mask)\n            embs.append(r[\"qemb\"])\n        bs = max(1, int(batch_size))\n        batches = []\n        for a in range(0, len(masks), bs):\n            b = {\"question_embeddings\": torch.cat(embs[a:a + bs], 0).clone(),\n                 \"start_nodes_mask\": torch.stack(masks[a:a + bs], 0)}\n            if device is not None:\n                b = {k: v.to(device) for k, v in b.items()}\n            if dtype is not None:\n                b = {k: (v.to(dtype) if v.is_floating_point() else v) for k, v in b.items()}\n            batches.append(b)\n        found = n_exact + n_sem\n        med = sorted(sem_cos)[len(sem_cos) // 2] if sem_cos else float(\"nan\")\n        cov = {\"queries\": len(masks),\n               \"batches\": len(batches),\n               \"seeds_found\": found,\n               \"seeds_total\": total,\n               \"seed_coverage\": (found / total) if total else 0.0,\n               # KEPT SEPARATE ON PURPOSE. The semantic arm's whole claim is that it raises\n               # coverage, so an aggregate that hides which half moved cannot test it.\n               \"seeds_exact\": n_exact,\n               \"seeds_semantic\": n_sem,\n               \"seeds_unmatched\": total - found,\n               \"exact_coverage\": (n_exact / total) if total else 0.0,\n               \"link\": link,\n               \"link_k\": int(link_k) if semantic else 0,\n               \"link_temp\": float(link_temp) if semantic else 0.0,\n               \"link_floor\": float(link_floor) if semantic else 0.0,\n               \"sem_cos_median\": med,\n               \"sem_nodes_per_seed\": (sum(sem_deg) / len(sem_deg)) if sem_deg else 0.0,\n               \"queries_with_no_seed\": empty}\n        return batches, cov\n\n    # ------------------------------------------------------------------ calibration\n    def start_calibration(self, key, phase=\"mean\", apply_existing_gates=False):\n        \"\"\"Begin one reference pass for `key`.\n\n        phase=\"mean\" accumulates sum h and sum ||h||^2; phase=\"dev\" accumulates\n        sum ||h - mu||^2 and keeps a sample of the per-query deviations. Both passes must\n        see the same M references, which finish_calibration checks.\n\n        apply_existing_gates: run the already-calibrated gates while re-accumulating. This\n        is what makes a second round consistent when more than one layer is gated -- see\n        the module docstring. It is a no-op on the first round, when nothing is calibrated.\n        \"\"\"\n        assert phase in (\"mean\", \"dev\"), f\"unknown calibration phase {phase!r}\"\n        if phase == \"dev\":\n            assert self._mu, \"phase 'dev' needs phase 'mean' to have finished first\"\n        else:\n            self._mu, self._energy = {}, {}\n        self.mode, self._acc, self._acc_key, self._n_ref = \"calibrate\", {}, key, 0\n        self._phase = phase\n        self._active = key\n        self._calib_gate = bool(apply_existing_gates) and key in self._stats\n        if phase == \"mean\" and not self._calib_gate:\n            # DROP THE OUTGOING STATISTICS BEFORE BUILDING THEIR REPLACEMENT. mu is one\n            # float32 row per node, which on the CS train graph is 950 MB per gated layer;\n            # holding the old set alive through a recalibration doubles the peak for no\n            # reason, since nothing reads it until the new one is installed. Kept only when\n            # apply_existing_gates asked for it, which is the round-2 path.\n            self._stats.pop(key, None)\n\n    def note_reference_batch(self, b):\n        self._n_ref += b\n\n    def finish_mean(self):\n        \"\"\"Close phase \"mean\": turn the running sums into mu (and the energy).\"\"\"\n        assert self._n_ref > 0, \"no reference queries were run\"\n        assert self._acc, \"calibration collected nothing; are any layers in gate_layers?\"\n        # div_, not div: the accumulator IS the mean once divided, and nothing else holds a\n        # reference to it. Allocating a second [N, D] float32 here would be another 950 MB\n        # per gated layer on the CS graph, at the moment memory is already at its peak.\n        #\n        # mu=\"zero\" IS THE ABLATION THAT DECIDES WHETHER THE REFERENCE BANK DOES ANYTHING.\n        # With mu = 0 the statistic degenerates to I = ||h||^2 / s, an activation-MAGNITUDE\n        # gate, and CQIG's whole claim -- that a node earns its message by responding to\n        # THIS query unlike it responds to unrelated ones -- is gone. If this arm reproduces\n        # the method's score, the reference bank was never load-bearing and the mechanism is\n        # magnitude gating under another name.\n        #\n        # The mean pass still RUNS and its result is discarded. That wastes M forwards, and\n        # it is the point: the two arms then differ in the value of mu and in nothing else,\n        # not in code path, not in RNG consumption. Zeroed in place, so no extra memory, and\n        # every quantity derived downstream (V, the layer scale s, tau_ref) is recomputed\n        # consistently against mu = 0 rather than being inherited from a real one.\n        self._mu = {i: (a[\"sum_h\"].zero_() if self.mu_zero else a[\"sum_h\"].div_(self._n_ref))\n                    for i, a in self._acc.items()}\n        self._energy = {i: a[\"sum_e\"].div_(self._n_ref) for i, a in self._acc.items()}\n        self._m_ref, self._acc, self.mode = self._n_ref, {}, \"off\"\n        return len(self._mu)\n\n    def finish_calibration(self):\n        \"\"\"Close phase \"dev\": variance, layer scale, threshold, and what the gate will do.\n\n        s is the MEDIAN reference variance over responding nodes, found in two steps: a\n        median over V > 0 fixes the order of magnitude, then nodes more than six orders\n        below it are dropped and the median is retaken. A single median over V > 0 would be\n        pulled down by whatever residue the barely-reachable part of the graph leaves.\n        \"\"\"\n        M = self._n_ref\n        assert M >= 2, f\"the variance needs at least 2 reference queries, got {M}\"\n        assert M == self._m_ref, (\n            f\"the mean pass saw {self._m_ref} reference queries and the deviation pass \"\n            f\"{M}; the two must be the same set\")\n        assert self._acc, \"the deviation pass collected nothing\"\n        loo = (M / (M - 1.0)) ** 2         # in-sample deviation -> leave-one-out deviation\n        report, per_layer = {}, {}\n        for i, a in self._acc.items():\n            mu = self._mu[i]\n            V = a[\"sum_sq\"] / (M - 1.0)                    # unbiased cross-query variance\n            # WHICH NODES COUNT AS RESPONDING, AND WHY THE CUT IS ANCHORED AT THE TOP.\n            # mu is a mean of M float32 values, so a node whose state never moves still\n            # lands ~1e-10 of the working scale away from it -- uniformly, on every\n            # unreachable node. Most of a KG is unreachable from any one query's seeds, so\n            # a cut anchored at a MEDIAN over V > 0 sits inside that dust cloud, keeps\n            # every node \"live\", and hands back a scale ten orders too small. The dust is\n            # many orders below any real response and the response range spans at most a\n            # few, so anchoring six orders under the largest variance separates them with\n            # room on both sides.\n            vmax = float(V.max()) if V.numel() else 0.0\n            n_pos = int((V > 0).sum())\n            live = (V > RESP * vmax) if vmax > 0 else (V > 0)\n            s = float(V[live].median()) if int(live.sum()) else 0.0\n            n_live = int(live.sum())\n            vbar = float(V[live].mean()) if n_live else 0.0\n            degenerate = not (s > 0)\n            if self.norm == \"layer\":\n                # ONE robust scale for the whole layer, so I is comparable BETWEEN nodes.\n                # s == 0 means no node responded to any reference query; 1.0 keeps I finite\n                # and the report says so rather than dividing by eps.\n                scale = torch.full((1, 1), (s if s > 0 else 1.0) + self.eps,\n                                   device=V.device, dtype=torch.float32)\n            elif self.norm == \"node\":                      # ablation: self-normalising\n                scale = (V + (self.rho * vbar + self.eps)).unsqueeze(0)\n            else:                                          # ablation: the original energy\n                scale = (self._energy[i] + (self.rho + self.eps)).unsqueeze(0)\n            per_layer[i] = {\"mu\": mu, \"scale\": scale, \"live\": live, \"V\": V, \"s\": s}\n\n            # --- what a reference query actually scores here, and what the gate does ----\n            sample = torch.cat(a[\"rows\"], 0) * loo         # [rows, N] on cpu\n            I = sample / scale.detach().to(sample.device)\n            resp = I[I > RESP]\n            if resp.numel() >= 100:\n                tau_ref = float(torch.log1p(resp).median())\n                tau_src = \"reference queries\"\n            else:\n                # Too few responding pairs for a median to mean anything: fall back to the\n                # variance, which is the same quantity averaged over queries.\n                iota = (V.unsqueeze(0) / scale).flatten()\n                tau_ref = float(torch.log1p(iota[live]).median()) if n_live else 0.0\n                tau_src = f\"reference variance (only {int(resp.numel())} responding pairs)\"\n            per_layer[i][\"tau_ref\"] = tau_ref\n            per_layer[i][\"tau_src\"] = tau_src\n            lam, alpha = float(self.lam(i).detach()), float(self.alpha(i).detach())\n            tau = tau_ref + float(self.dtau[i].detach())\n            gv = self._g_scalars(I, lam, alpha, tau)\n            # WHAT THE ACTIVE OP MULTIPLIES BY, not what \"gate\" would have. Reporting g\n            # under a centring arm would describe an operator that arm does not run.\n            cv = (gv if self.op == \"gate\" else\n                  torch.full_like(gv, lam) if self.op == \"centre-fixed\" else 1.0 - gv)\n            # mu_v = 0 EXACTLY is \"no reference query ever reached this node\". A centring op\n            # cannot touch those nodes at all while a gating op damps them hardest, so this\n            # fraction is how much of the graph the two arms treat oppositely.\n            mu_zero = int((mu.norm(dim=-1) == 0).sum())\n            report[i] = {\n                \"nodes\": int(V.numel()), \"responding\": n_pos, \"live\": n_live,\n                \"responding_frac\": n_live / max(int(V.numel()), 1),\n                \"scale\": s, \"var_max\": vmax, \"var_mean_live\": vbar,\n                \"var_q\": _describe(V[live]) if n_live else _describe(None),\n                \"I\": _describe(I), \"gate\": _describe(gv),\n                \"op\": self.op, \"coef\": _describe(cv),\n                \"mu_source\": self.mu_source, \"mu_zero\": mu_zero,\n                \"mu_zero_frac\": mu_zero / max(int(V.numel()), 1),\n                \"gate_unreached\": float(self._g_scalars(\n                    torch.zeros(()), lam, alpha, tau)),\n                \"resp_frac\": float(resp.numel()) / max(int(I.numel()), 1),\n                \"sample_queries\": int(sample.shape[0]),\n                \"degenerate\": degenerate, \"lam\": lam, \"alpha\": alpha, \"tau\": tau,\n                \"tau_ref\": tau_ref, \"tau_src\": tau_src, \"M\": M,\n            }\n        self._stats[self._acc_key] = per_layer\n        self._active = self._acc_key\n        self._acc, self._mu, self._energy = {}, {}, {}\n        self._acc_key, self.mode, self._calib_gate = None, \"off\", False\n        return report\n\n    # ------------------------------------------------------------------ live measurement\n    def reset_live(self, bins=1024):\n        \"\"\"Start accumulating the gate distribution over REAL forwards, and the\n        gold-vs-hard-negative alignment. Both are measurement only: nothing is calibrated\n        from them and no target query or label enters the statistics.\"\"\"\n        self._live, self._align, self._edit, self.last_I = {}, {}, {}, {}\n        self._live_bins = int(bins)\n\n    def stop_live(self):\n        self._live, self._align, self._edit, self.last_I = None, {}, {}, {}\n\n    def recording(self):\n        return self._live is not None\n\n    def _note_live(self, i, g):\n        a = self._live.get(i)\n        if a is None:\n            a = self._live[i] = {\n                \"n\": 0,\n                \"sum\": torch.zeros((), device=g.device, dtype=torch.float64),\n                \"sqs\": torch.zeros((), device=g.device, dtype=torch.float64),\n                \"min\": torch.full((), float(\"inf\"), device=g.device),\n                \"max\": torch.full((), float(\"-inf\"), device=g.device),\n                \"hist\": torch.zeros(self._live_bins, device=g.device),\n            }\n        f = g.flatten().float()\n        a[\"n\"] += int(f.numel())\n        a[\"sum\"] += f.sum().double()\n        a[\"sqs\"] += (f * f).sum().double()\n        a[\"min\"] = torch.minimum(a[\"min\"], f.min())\n        a[\"max\"] = torch.maximum(a[\"max\"], f.max())\n        a[\"hist\"] += torch.histc(f, bins=self._live_bins, min=0.0, max=1.0)\n\n    def _note_edit(self, i, hh, h):\n        \"\"\"Running ||h~ - h|| / ||h|| over nodes, for whichever op is active.\n\n        Measured on the states themselves rather than derived from the coefficient, because\n        the two ops move a state by different amounts at the same coefficient and the point\n        of the number is to compare them. Nodes with a numerically zero state are dropped\n        rather than clamped: most of a KG is unreachable from one query's seeds, and a ratio\n        of 0/eps there would dominate the mean with an artefact.\n        \"\"\"\n        a = self._edit.get(i)\n        if a is None:\n            a = self._edit[i] = {\n                \"n\": 0,\n                \"sum\": torch.zeros((), device=h.device, dtype=torch.float64),\n                \"max\": torch.zeros((), device=h.device),\n                \"moved\": 0,\n            }\n        hn = h.float().norm(dim=-1)                              # [B, N]\n        dn = (hh.float() - h.float()).norm(dim=-1)\n        live = hn > 0\n        if not bool(live.any()):\n            return\n        rel = dn[live] / hn[live]\n        a[\"n\"] += int(rel.numel())\n        a[\"sum\"] += rel.sum().double()\n        a[\"max\"] = torch.maximum(a[\"max\"], rel.max())\n        a[\"moved\"] += int((dn[live] > 0).sum())\n\n    def note_alignment(self, target_mask, doc_idx, doc_scores, k=50):\n        \"\"\"Is a high I actually concentrated on the gold papers?\n\n        The hypothesis behind the gate is that informative nodes lie on the paths that\n        matter. This is its most direct falsifiable form: for every query, compare I at the\n        GOLD document nodes against I at the `k` documents the semantic scorer ranks\n        highest among the non-golds -- the same hard negatives the training objective uses.\n        The statistic is the AUC, i.e. the probability that a random gold outscores a random\n        hard negative in informativeness, with ties counted as half. 0.5 is no alignment,\n        and no amount of tuning lam/alpha/tau can rescue a gate whose statistic sits there.\n        \"\"\"\n        if self._live is None or not self.last_I or target_mask is None:\n            return\n        with torch.no_grad():\n            tgt = target_mask.index_select(1, doc_idx.to(target_mask.device)).bool()\n            for i, I in self.last_I.items():\n                Id = I.index_select(1, doc_idx.to(I.device)).float()\n                a = self._align.setdefault(\n                    i, {\"wins\": 0.0, \"n\": 0, \"gold\": 0.0, \"neg\": 0.0})\n                for b in range(min(Id.shape[0], tgt.shape[0])):\n                    gold = tgt[b].nonzero(as_tuple=False).flatten()\n                    if gold.numel() == 0:\n                        continue\n                    s = doc_scores[b].detach().float().clone()\n                    s[gold] = float(\"-inf\")\n                    kk = int(min(k, s.numel() - gold.numel()))\n                    if kk <= 0:\n                        continue\n                    neg = s.topk(kk).indices\n                    gi = Id[b].index_select(0, gold.to(Id.device)).unsqueeze(1)\n                    ni = Id[b].index_select(0, neg.to(Id.device)).unsqueeze(0)\n                    a[\"wins\"] += float((gi > ni).float().mean()\n                                       + 0.5 * (gi == ni).float().mean())\n                    a[\"n\"] += 1\n                    a[\"gold\"] += float(gi.mean())\n                    a[\"neg\"] += float(ni.mean())\n\n    def live_report(self):\n        \"\"\"Gate distribution over the real forwards, plus the alignment AUC.\"\"\"\n        out = {}\n        for i, a in (self._live or {}).items():\n            n = max(a[\"n\"], 1)\n            mean = float(a[\"sum\"]) / n\n            var = max(float(a[\"sqs\"]) / n - mean * mean, 0.0)\n            lo, hi = float(a[\"min\"]), float(a[\"max\"])\n            h = a[\"hist\"]\n            cdf = torch.cumsum(h, 0) / max(float(h.sum()), 1.0)\n            edges = (torch.arange(self._live_bins, device=h.device).float() + 0.5) \\\n                / self._live_bins\n            qv = {}\n            for q in (0.1, 0.5, 0.9):\n                j = int(torch.searchsorted(cdf, torch.tensor(q, device=h.device)).clamp(\n                    max=self._live_bins - 1))\n                # CLAMPED INTO THE OBSERVED RANGE. min and max are exact while the\n                # quantiles are bin centres, so an unclamped p10 can print below the min\n                # and read as a bug in the gate rather than in the histogram.\n                qv[f\"p{int(q * 100)}\"] = min(max(float(edges[j]), lo), hi)\n            al = self._align.get(i)\n            ok = bool(al and al[\"n\"])\n            ed = self._edit.get(i)\n            out[i] = {\"n\": a[\"n\"], \"min\": float(a[\"min\"]), \"max\": float(a[\"max\"]),\n                      \"mean\": mean, \"std\": var ** 0.5, **qv,\n                      \"op\": self.op,\n                      \"auc\": (al[\"wins\"] / al[\"n\"]) if ok else float(\"nan\"),\n                      \"auc_n\": (al[\"n\"] if al else 0),\n                      \"I_gold\": (al[\"gold\"] / al[\"n\"]) if ok else float(\"nan\"),\n                      \"I_neg\": (al[\"neg\"] / al[\"n\"]) if ok else float(\"nan\"),\n                      \"edit_rel_mean\": (float(ed[\"sum\"]) / max(ed[\"n\"], 1)) if ed else 0.0,\n                      \"edit_rel_max\": float(ed[\"max\"]) if ed else 0.0,\n                      # What fraction of live nodes the operator moved AT ALL. Under a\n                      # centring op this is the coverage question in its bluntest form:\n                      # mu_v = 0 means the node is untouched however uninformative it is.\n                      \"edit_frac\": (ed[\"moved\"] / max(ed[\"n\"], 1)) if ed else 0.0}\n        return out\n\n    # ------------------------------------------------------------------ the hook\n    @staticmethod\n    def _chunk(n_rows, d):\n        return max(1024, int(_CHUNK_BYTES // max(n_rows * d * 4, 1)))\n\n    def _accumulate_mean(self, i, h):\n        \"\"\"Running sum_h and sum_||h||^2, in float32 and in chunks over the node axis.\n\n        FLOAT32 ACCUMULATORS. Under autocast h is bfloat16, whose ~8-bit mantissa would\n        lose most of a 64-term running sum. Chunked because the float32 upcast of a\n        [B, N, D] state is 2 GB at B=8 on this graph.\n        \"\"\"\n        b, n, d = h.shape\n        acc = self._acc.setdefault(i, {\n            \"sum_h\": torch.zeros(n, d, device=h.device, dtype=torch.float32),\n            \"sum_e\": torch.zeros(n, device=h.device, dtype=torch.float32),\n        })\n        step = self._chunk(b, d)\n        for a0 in range(0, n, step):\n            a1 = min(a0 + step, n)\n            hd = h[:, a0:a1].detach().float()\n            acc[\"sum_h\"][a0:a1] += hd.sum(0)\n            acc[\"sum_e\"][a0:a1] += (hd * hd).sum(-1).sum(0)\n\n    def _accumulate_dev(self, i, h):\n        \"\"\"Running sum ||h - mu||^2, plus a bounded sample of the deviations themselves.\"\"\"\n        mu = self._mu.get(i)\n        if mu is None or mu.shape[0] != h.shape[1]:\n            return\n        sq = self._sqdist(h.detach(), mu)                  # [B, N] float32, exact\n        acc = self._acc.setdefault(i, {\n            \"sum_sq\": torch.zeros(h.shape[1], device=h.device, dtype=torch.float32),\n            \"rows\": [], \"kept\": 0})\n        acc[\"sum_sq\"] += sq.sum(0)\n        cap = max(1, _SAMPLE_ELEMS // max(h.shape[1], 1))\n        if acc[\"kept\"] < cap:\n            take = min(sq.shape[0], cap - acc[\"kept\"])\n            acc[\"rows\"].append(sq[:take].cpu())\n            acc[\"kept\"] += take\n\n    def _sqdist(self, h, mu):\n        \"\"\"||h - mu||^2 per node, in float32, chunked over the node axis.\n\n        The subtraction must not happen in bfloat16: the deviation is ~1% of the state\n        under early-late fusion and bfloat16 resolves ~0.4%, so a bfloat16 difference is\n        ~40% rounding noise. Chunking keeps the float32 temporary bounded.\n        \"\"\"\n        b, n, d = h.shape\n        out = torch.empty(b, n, device=h.device, dtype=torch.float32)\n        step = self._chunk(b, d)\n        for a0 in range(0, n, step):\n            a1 = min(a0 + step, n)\n            df = h[:, a0:a1].float() - mu[a0:a1].unsqueeze(0)\n            out[:, a0:a1] = (df * df).sum(-1)\n        return out\n\n    def _centre(self, h, mu, c):\n        \"\"\"h - c * mu per node, chunked over the node axis, never in float32 at full size.\n\n        `c` is the subtraction coefficient: [B, N] for an adaptive op, 0-dim for a fixed one.\n        Written naively as `h - c.unsqueeze(-1) * mu` this allocates a float32 [B, N, D]\n        because mu is float32 -- 1.9 GB at B=1 on the CS graph, twice the state it is\n        editing, and then another copy to cast back. Chunked, with the cast to h's dtype\n        INSIDE the chunk, the transient is bounded by _CHUNK_BYTES the same way _sqdist is.\n        The slice assignments are autograd-tracked copies into a fresh tensor, so gradient\n        still reaches h, lam, alpha and dtau; the loop covers 0..n so nothing is left\n        uninitialised.\n        \"\"\"\n        b, n, d = h.shape\n        out = torch.empty_like(h)\n        step = self._chunk(b, d)\n        fixed = c.dim() == 0\n        for a0 in range(0, n, step):\n            a1 = min(a0 + step, n)\n            cc = (c if fixed else c[:, a0:a1].unsqueeze(-1)).to(h.dtype)\n            out[:, a0:a1] = h[:, a0:a1] - cc * mu[a0:a1].to(h.dtype).unsqueeze(0)\n        return out\n\n    def _coef(self, i, g):\n        \"\"\"The coefficient the active op multiplies by, given the gate value.\n\n        Returned as the quantity actually applied, so the report and the live histogram\n        describe what happened rather than what would have happened under `op=\"gate\"`.\n        \"\"\"\n        if self.op == \"gate\":\n            return g\n        if self.op == \"centre-fixed\":\n            return self.lam(i)                 # 0-dim: query-independent by construction\n        return 1.0 - g                         # \"centre\": damp the background, not the state\n\n    def _apply_op(self, i, h, st, I, detach_dtau=False):\n        \"\"\"Edit the state under the active op. Returns (h~, g), g for the live report.\"\"\"\n        dt = float(self.dtau[i].detach()) if detach_dtau else self.dtau[i]\n        g = self._g_of(i, I, st[\"tau_ref\"] + dt)\n        if self.op == \"gate\":\n            return h * g.to(h.dtype).unsqueeze(-1), g\n        # mu is the statistic being subtracted, so a centring op is a NO-OP wherever the\n        # references never reached (mu_v = 0 exactly), which is the opposite of what gating\n        # does there. See the module docstring; mu_zero_frac reports how much of the graph\n        # that is.\n        return self._centre(h, st[\"mu\"], self._coef(i, g)), g\n\n    def _stats_for(self, i, h):\n        assert self._active is not None, (\n            \"cqig is in gate mode with no active graph; call use_graph(key) first\")\n        st = self._stats.get(self._active)\n        assert st is not None and i in st, (\n            f\"cqig has no statistics for graph {self._active!r} at layer {i}; \"\n            f\"calibrate this graph before scoring it (known: {self.known_graphs()})\")\n        # A wrong-graph lookup would otherwise fail as a bare broadcast error deep in\n        # autograd. Say which graph and which shapes.\n        assert st[i][\"mu\"].shape[0] == h.shape[1], (\n            f\"cqig statistics for graph {self._active!r} have {st[i]['mu'].shape[0]} nodes \"\n            f\"but this forward has {h.shape[1]}; the wrong graph's statistics are active\")\n        return st[i]\n\n    def _make_hook(self, i):\n        def hook(_module, args):\n            # args = (input, query, boundary, edge_index, edge_type, size, edge_weight)\n            if i not in self.gate_layers or self.mode == \"off\":\n                return None                    # not gated: never calibrated either\n            h = args[0]                                    # [B, N, D] = h^(l)\n            if self.mode == \"calibrate\":\n                with torch.no_grad():\n                    if self._phase == \"mean\":\n                        self._accumulate_mean(i, h)\n                    else:\n                        self._accumulate_dev(i, h)\n                    if not self._calib_gate:\n                        return None                        # pass through unchanged\n                    st = (self._stats.get(self._active) or {}).get(i)\n                    if st is None or st[\"mu\"].shape[0] != h.shape[1]:\n                        return None\n                    I = self._sqdist(h, st[\"mu\"]) / st[\"scale\"]\n                    hh, _ = self._apply_op(i, h, st, I, detach_dtau=True)\n                return (hh,) + tuple(args[1:])\n            if self.mode != \"gate\":\n                return None\n            st = self._stats_for(i, h)\n            # DETACHED BY DEFAULT: see the module docstring. A gradient path through I is a\n            # free direction that opens every gate at once, and detaching also lets the\n            # float32 chunked subtraction cost nothing in stored activations.\n            if self.grad_through_I:\n                I = self._sqdist(h, st[\"mu\"]) / st[\"scale\"]\n            else:\n                with torch.no_grad():\n                    I = self._sqdist(h, st[\"mu\"]) / st[\"scale\"]\n            # Only the messages are edited. The boundary condition is a separate argument\n            # and the residual is added outside the layer, so both are left alone. The\n            # coefficient is cast back to h's dtype inside the op: a float32 one would\n            # silently promote the whole state and double every gated layer's activations.\n            hh, g = self._apply_op(i, h, st, I)\n            # KEPT AS A TENSOR. float() here is a device sync on every gated layer of every\n            # training step, which at four gated layers is four stalls per step for a\n            # number only summary() ever reads.\n            self.last_gate[i] = g.detach().mean()\n            if self._live is not None:\n                with torch.no_grad():\n                    self._note_live(i, self._coef(i, g).detach().expand_as(g))\n                    self.last_I[i] = I.detach()\n                    # HOW BIG THE EDIT ACTUALLY IS, relative to the state it edits. Under\n                    # early-late fusion h is dominated by the static entity embedding, so\n                    # the same lam is a very different perturbation under the two ops, and\n                    # `layer_norm: yes` renormalises much of a magnitude change away. This\n                    # is the number that says whether the operator did anything at all.\n                    self._note_edit(i, hh.detach(), h.detach())\n            return (hh,) + tuple(args[1:])\n        return hook\n\n    # ------------------------------------------------------------------ checkpointing\n    def get_extra_state(self):\n        return {\"ref_bank\": self.ref_bank, \"ref_meta\": self.ref_meta,\n                \"gate_layers\": sorted(self.gate_layers), \"norm\": self.norm,\n                \"op\": self.op, \"mu\": self.mu_source}\n\n    def set_extra_state(self, state):\n        if not state:\n            return\n        # STATISTICS BELONG TO A (WEIGHTS, GRAPH) PAIR, SO NEW WEIGHTS INVALIDATE THEM.\n        # `load_best_model_at_end` defaults to true, so the final evaluate() and predict()\n        # run on the BEST checkpoint while _stats still held mu from the LAST epoch's\n        # weights -- and the recalibration guard, which keys on the training step, saw no\n        # reason to refresh them. The reported numbers would have been the best model\n        # gated by a different model's reference statistics. Clearing here forces one\n        # calibration pass on the next inference, which is the cheap and correct answer.\n        if self._stats:\n            print(f\"[cqig] checkpoint loaded: dropping reference statistics for \"\n                  f\"{self.known_graphs()}; they belong to the previous weights\")\n        self._stats, self._active = {}, None\n        self.ref_bank = state.get(\"ref_bank\")\n        self.ref_meta = state.get(\"ref_meta\", {})\n        if state.get(\"gate_layers\"):\n            want = set(state[\"gate_layers\"])\n            if want != self.gate_layers:\n                print(f\"[cqig] checkpoint gates layers {sorted(want)} (0-indexed), the \"\n                      f\"config asked for {sorted(self.gate_layers)}; using the checkpoint's\")\n            self.gate_layers = want\n        if state.get(\"norm\"):\n            if state[\"norm\"] != self.norm:\n                print(f\"[cqig] checkpoint was normalised by {state['norm']!r}, the config \"\n                      f\"asked for {self.norm!r}; using the checkpoint's\")\n            self.norm = state[\"norm\"]\n        # THE OPERATOR IS PART OF THE TRAINED MODEL, not a scoring-time choice: lam, alpha\n        # and dtau were fitted under one of them. Absent on a checkpoint written before this\n        # existed, which can only have been \"gate\".\n        op = state.get(\"op\", \"gate\")\n        if op != self.op:\n            print(f\"[cqig] checkpoint was trained with op={op!r}, the config asked for \"\n                  f\"{self.op!r}; using the checkpoint's\")\n        self.op, self.centring = op, op != \"gate\"\n        # Same argument for mu: an ablated model recalibrating with a real mu at scoring\n        # time would report the method's number under the ablation's name.\n        mu = state.get(\"mu\", \"ref\")\n        if mu != self.mu_source:\n            print(f\"[cqig] checkpoint was trained with mu={mu!r}, the config asked for \"\n                  f\"{self.mu_source!r}; using the checkpoint's\")\n        self.mu_source, self.mu_zero = mu, mu == \"zero\"\n\n    # ------------------------------------------------------------------ diagnostics\n    def n_reference(self):\n        return self._n_ref\n\n    def stats_bytes(self):\n        return sum(v[\"mu\"].numel() * v[\"mu\"].element_size()\n                   for st in self._stats.values() for v in st.values())\n\n    def summary(self):\n        return {\n            \"gate_layers_1indexed\": sorted(i + 1 for i in self.gate_layers),\n            \"op\": self.op,\n            \"mu\": self.mu_source,\n            \"norm\": self.norm,\n            \"grad_through_I\": self.grad_through_I,\n            \"lam\": [round(float(self.lam(i).detach()), 4) for i in sorted(self.gate_layers)],\n            \"alpha\": [round(float(self.alpha(i).detach()), 4) for i in sorted(self.gate_layers)],\n            \"dtau\": [round(float(self.dtau[i].detach()), 4) for i in sorted(self.gate_layers)],\n            \"active_graph\": self._active,\n            \"calibrated_graphs\": self.known_graphs(),\n            \"reference_queries\": len(self.ref_bank or []),\n            \"stats_mb\": round(self.stats_bytes() / 2**20, 1),\n            \"mean_gate\": {k + 1: round(float(v), 4)\n                          for k, v in sorted(self.last_gate.items())},\n        }\n\n    def remove(self):\n        for h in self._handles:\n            h.remove()\n        self._handles = []\n", "/content/gfm-rag/gfmrag/models/ultra/models.py": "# mypy: ignore-errors\nimport os\nimport torch\nfrom torch import autograd, nn\n\nfrom . import layers\nfrom .base_nbfnet import BaseNBFNet\n\n\nclass EntityNBFNet(BaseNBFNet):\n    \"\"\"Neural Bellman-Ford Network for entity prediction.\"\"\"\n\n    def __init__(self, input_dim, hidden_dims, num_relation=1, **kwargs):\n        # dummy num_relation = 1 as we won't use it in the NBFNet layer\n        super().__init__(input_dim, hidden_dims, num_relation, **kwargs)\n        self.return_hidden = kwargs.get(\"return_hidden\", False)\n        self.layers = nn.ModuleList()\n        for i in range(len(self.dims) - 1):\n            self.layers.append(\n                layers.GeneralizedRelationalConv(\n                    self.dims[i],\n                    self.dims[i + 1],\n                    num_relation,\n                    self.dims[0],\n                    self.message_func,\n                    self.aggregate_func,\n                    self.layer_norm,\n                    self.activation,\n                    dependent=False,\n                    project_relations=True,\n                )\n            )\n\n        feature_dim = (\n            sum(hidden_dims) if self.concat_hidden else hidden_dims[-1]\n        ) + input_dim\n        if not self.return_hidden:\n            self.mlp = nn.Sequential()\n            mlp = []\n            for i in range(self.num_mlp_layers - 1):\n                mlp.append(nn.Linear(feature_dim, feature_dim))\n                mlp.append(nn.ReLU())\n            mlp.append(nn.Linear(feature_dim, 1))\n            self.mlp = nn.Sequential(*mlp)\n\n    def bellmanford(self, data, h_index, r_index, separate_grad=False):\n        batch_size = len(r_index)\n\n        # initialize queries (relation types of the given triples)\n        query = self.query[torch.arange(batch_size, device=r_index.device), r_index]\n        index = h_index.unsqueeze(-1).expand_as(query)\n\n        # initial (boundary) condition - initialize all node states as zeros\n        boundary = torch.zeros(\n            batch_size, data.num_nodes, self.dims[0], device=h_index.device\n        )\n        # by the scatter operation we put query (relation) embeddings as init features of source (index) nodes\n        boundary.scatter_add_(1, index.unsqueeze(1), query.unsqueeze(1))\n\n        size = (data.num_nodes, data.num_nodes)\n        edge_weight = torch.ones(data.num_edges, device=h_index.device)\n\n        hiddens = []\n        edge_weights = []\n        layer_input = boundary\n\n        for layer in self.layers:\n            # for visualization\n            if separate_grad:\n                edge_weight = edge_weight.clone().requires_grad_()\n\n            # Bellman-Ford iteration, we send the original boundary condition in addition to the updated node states\n            hidden = layer(\n                layer_input,\n                query,\n                boundary,\n                data.edge_index,\n                data.edge_type,\n                size,\n                edge_weight,\n            )\n            if self.short_cut and hidden.shape == layer_input.shape:\n                # residual connection here\n                hidden = hidden + layer_input\n            hiddens.append(hidden)\n            edge_weights.append(edge_weight)\n            layer_input = hidden\n\n        # original query (relation type) embeddings\n        node_query = query.unsqueeze(1).expand(\n            -1, data.num_nodes, -1\n        )  # (batch_size, num_nodes, input_dim)\n        if self.concat_hidden:\n            output = torch.cat(hiddens + [node_query], dim=-1)\n        else:\n            output = torch.cat([hiddens[-1], node_query], dim=-1)\n\n        return {\n            \"node_feature\": output,\n            \"edge_weights\": edge_weights,\n        }\n\n    def forward(self, data, relation_representations, batch):\n        h_index, t_index, r_index = batch.unbind(-1)\n\n        # initial query representations are those from the relation graph\n        self.query = relation_representations\n\n        # initialize relations in each NBFNet layer (with uinque projection internally)\n        for layer in self.layers:\n            layer.relation = relation_representations\n\n        # if self.training:\n        # Edge dropout in the training mode\n        # here we want to remove immediate edges (head, relation, tail) from the edge_index and edge_types\n        # to make NBFNet iteration learn non-trivial paths\n        # data = self.remove_easy_edges(data, h_index, t_index, r_index)\n\n        shape = h_index.shape\n        # turn all triples in a batch into a tail prediction mode\n        h_index, t_index, r_index = self.negative_sample_to_tail(\n            h_index, t_index, r_index, num_direct_rel=data.num_relations // 2\n        )\n        assert (h_index[:, [0]] == h_index).all()\n        assert (r_index[:, [0]] == r_index).all()\n\n        # message passing and updated node representations\n        output = self.bellmanford(\n            data, h_index[:, 0], r_index[:, 0]\n        )  # (num_nodes, batch_size, feature_dim\uff09\n        feature = output[\"node_feature\"]\n        index = t_index.unsqueeze(-1).expand(-1, -1, feature.shape[-1])\n        # extract representations of tail entities from the updated node states\n        feature = feature.gather(\n            1, index\n        )  # (batch_size, num_negative + 1, feature_dim)\n\n        # probability logit for each tail node in the batch\n        # (batch_size, num_negative + 1, dim) -> (batch_size, num_negative + 1)\n        score = self.mlp(feature).squeeze(-1)\n        return score.view(shape)\n\n\nclass QueryNBFNet(EntityNBFNet):\n    \"\"\"\n    The entity-level reasoner for UltraQuery-like complex query answering pipelines.\n\n    This class extends EntityNBFNet to handle query-specific reasoning in knowledge graphs.\n    Key differences from EntityNBFNet include:\n\n    1. Initial node features are provided during forward pass rather than read from triples batch\n    2. Query comes from outer loop\n    3. Returns distribution over all nodes (assuming t_index covers all nodes)\n\n    Attributes:\n        layers: List of neural network layers for message passing\n        short_cut: Boolean flag for using residual connections\n        concat_hidden: Boolean flag for concatenating hidden states\n        mlp: Multi-layer perceptron for final scoring\n        num_beam: Beam size for path search\n        path_topk: Number of top paths to return\n\n    Methods:\n        bellmanford(data, node_features, query, separate_grad=False):\n            Performs Bellman-Ford message passing iterations.\n            Args:\n                data: Graph data object containing edge information\n                node_features: Initial node representations\n                query: Query representation\n                separate_grad: Whether to track gradients separately for edges\n            Returns:\n                dict: Contains node features and edge weights\n\n        forward(data, node_features, relation_representations, query):\n            Main forward pass of the model.\n            Args:\n                data: Graph data object\n                node_features: Initial node features\n                relation_representations: Representations for relations\n                query: Query representation\n            Returns:\n                torch.Tensor: Scores for each node\n\n        visualize(data, sample, node_features, relation_representations, query):\n            Visualizes reasoning paths for given entities.\n            Args:\n                data: Graph data object\n                sample: Dictionary containing entity masks\n                node_features: Initial node features\n                relation_representations: Representations for relations\n                query: Query representation\n            Returns:\n                dict: Contains paths and weights for target entities\n    \"\"\"\n\n    def _route_attention(self, li, h, query, layer, data, edge_weight):\n        \"\"\"RED-GNN-style query-conditioned attention, one weight per (query, edge).\n\n        logit[b, e] = a . tanh(W_n h_b[src(e)] + W_r r_b[type(e)] + W_q q_b + emb[layer])\n        alpha       = softmax of logit over every receiver's INCOMING edges, per query\n        weight      = alpha * in_degree(receiver)        (ROUTE_ATTN_NORM=1, default)\n\n        The degree factor makes the weights average 1 over each receiver's incoming\n        edges, so the aggregation keeps the control's scale and a uniform attention is\n        exactly the ungated layer (the output layer is zero-initialised, so that is\n        epoch 0). ROUTE_ATTN_NORM=0 gives RED-GNN's raw softmax, which also changes the\n        aggregation from a sum to a weighted mean and is therefore a second variable.\n        Trained by the ranking loss only. Runs in fp32 like the CCMP head.\n        \"\"\"\n        from torch_geometric.utils import softmax as _pyg_softmax\n        src, dst = data.edge_index[0], data.edge_index[1]\n        B = h.shape[0]\n        with torch.autocast(device_type=h.device.type, enabled=False):\n            zn = self.attn_node[li](h.float())                          # [B, N, H]\n            rel = layer.relation\n            if isinstance(rel, nn.Embedding):\n                rel = rel.weight\n            if rel.dim() == 2:\n                rel = rel.unsqueeze(0).expand(B, -1, -1)\n            zr = self.attn_rel(rel.float())                             # [B, R, H]\n            zq = self.attn_query(query.float()).unsqueeze(1)            # [B, 1, H]\n            ze = (zn.index_select(1, src) + zr.index_select(1, data.edge_type) + zq\n                  + self.attn_emb.weight[li].float())                   # [B, E, H]\n            logit = self.attn_out(torch.tanh(ze)).squeeze(-1)           # [B, E]\n            alpha = _pyg_softmax(logit.t().contiguous(), dst, num_nodes=data.num_nodes)  # [E, B]\n            if getattr(self, \"attn_norm\", True):\n                deg = torch.bincount(dst, minlength=data.num_nodes).to(alpha.dtype)\n                alpha = alpha * deg.index_select(0, dst).unsqueeze(-1)\n            w = alpha.t().contiguous()                                  # [B, E]\n            w = w * edge_weight.float().unsqueeze(0)\n        return w\n\n    def bellmanford(self, data, node_features, query, separate_grad=False):\n        import torch.distributed as dist\n\n        dist_context = getattr(data, \"dist_context\", None)\n        is_dist = dist_context is not None and dist.is_initialized()\n        boundary_mode = getattr(data, \"boundary_mode\", False)\n\n        size = (data.num_nodes, data.num_nodes)\n        edge_weight = torch.ones(data.num_edges, device=query.device, dtype=query.dtype)\n\n        hiddens = []\n        edge_weights = []\n\n        if is_dist and boundary_mode:\n            # ------------------------------------------------------------------\n            # Boundary-only AllGather (METIS partition)\n            # ------------------------------------------------------------------\n            # Scalable variant: only communicate boundary source node states\n            # instead of the full hidden tensor.  The compact tensor layout is\n            # [local_nodes | boundary_nodes] with edges pre-remapped to this\n            # space by partition_graph_metis().\n            # ------------------------------------------------------------------\n            rank, world_size = dist_context\n            num_nodes = data.num_nodes\n            local_nodes = data.local_nodes\n            boundary_nodes = data.boundary_nodes\n            compact_size = data.compact_size\n            node2part = data.node2part\n            local_N = local_nodes.shape[0]\n            boundary_N = boundary_nodes.shape[0]\n\n            # Collect local_nodes from all ranks for scatter-back.\n            all_local_nodes_list = [None] * world_size\n            dist.all_gather_object(all_local_nodes_list, local_nodes.cpu())\n            all_local_N = [len(ns) for ns in all_local_nodes_list]\n            max_local_N = max(all_local_N)\n\n            def _scatter_allgather(local_t):\n                \"\"\"AllGather local outputs and scatter to global positions.\"\"\"\n                B, loc_n, D = local_t.shape\n                if loc_n < max_local_N:\n                    pad = local_t.new_zeros(B, max_local_N - loc_n, D)\n                    padded = torch.cat([local_t, pad], dim=1).contiguous()\n                else:\n                    padded = local_t.contiguous()\n                chunks = [torch.zeros_like(padded) for _ in range(world_size)]\n                dist.all_gather(chunks, padded)\n                # Scatter each rank's results to correct global positions.\n                output = local_t.new_zeros(B, num_nodes, D)\n                for r in range(world_size):\n                    r_nodes = all_local_nodes_list[r].to(local_t.device)\n                    r_data = chunks[r][:, : all_local_N[r], :]\n                    output[:, r_nodes, :] = r_data\n                return output\n\n            # Precompute boundary exchange info (once per forward).\n            boundary_owners = node2part[boundary_nodes]\n            all_boundary_nodes_list = [None] * world_size\n            dist.all_gather_object(all_boundary_nodes_list, boundary_nodes.cpu())\n\n            local_nodes_set = set(local_nodes.cpu().tolist())\n            send_indices = {}\n            for r in range(world_size):\n                if r == rank:\n                    continue\n                their_boundary = set(all_boundary_nodes_list[r].tolist())\n                needed_from_us = their_boundary & local_nodes_set\n                if needed_from_us:\n                    needed_t = torch.tensor(sorted(needed_from_us), device=query.device)\n                    idx = torch.searchsorted(local_nodes, needed_t)\n                    send_indices[r] = idx\n\n            recv_indices = {}\n            for r in range(world_size):\n                if r == rank:\n                    continue\n                mask = boundary_owners == r\n                if mask.any():\n                    recv_indices[r] = mask.nonzero(as_tuple=True)[0]\n\n            # Initial local hidden states and compact boundary condition.\n            local_layer_input = node_features[:, local_nodes, :].clone()\n            compact_boundary = torch.cat([\n                node_features[:, local_nodes, :],\n                node_features[:, boundary_nodes, :],\n            ], dim=1)\n\n            compact_edge_size = (compact_size, compact_size)\n\n            for layer in self.layers:\n                if separate_grad:\n                    edge_weight = edge_weight.clone().requires_grad_()\n\n                # Exchange boundary states.\n                B, _, D = local_layer_input.shape\n                boundary_hidden = local_layer_input.new_zeros(B, boundary_N, D)\n\n                send_data = {}\n                for r, idx in send_indices.items():\n                    send_data[r] = local_layer_input[:, idx, :].contiguous()\n\n                all_send_data = [None] * world_size\n                dist.all_gather_object(all_send_data, send_data)\n\n                for r in range(world_size):\n                    if r == rank:\n                        continue\n                    if rank in all_send_data[r]:\n                        states = all_send_data[r][rank].to(query.device)\n                        boundary_hidden[:, recv_indices[r], :] = states\n\n                compact_input = torch.cat(\n                    [local_layer_input, boundary_hidden], dim=1\n                )\n\n                hidden = layer(\n                    compact_input,\n                    query,\n                    compact_boundary,\n                    data.edge_index,\n                    data.edge_type,\n                    compact_edge_size,\n                    edge_weight,\n                )\n\n                local_hidden = hidden[:, :local_N, :]\n                if self.short_cut and local_hidden.shape == local_layer_input.shape:\n                    local_hidden = local_hidden + local_layer_input\n                hiddens.append(local_hidden)\n                edge_weights.append(edge_weight)\n                local_layer_input = local_hidden\n\n            node_query_local = (\n                query.unsqueeze(1).expand(-1, local_N, -1).contiguous()\n            )\n            if self.concat_hidden:\n                local_output = torch.cat(hiddens + [node_query_local], dim=-1)\n            else:\n                local_output = torch.cat([hiddens[-1], node_query_local], dim=-1)\n\n            output = _scatter_allgather(local_output)\n\n        elif is_dist:\n            # ------------------------------------------------------------------\n            # Distributed split-graph inference\n            # ------------------------------------------------------------------\n            # Strategy (mathematically exact):\n            #   1. Each rank owns nodes [local_start, local_end) and the edges\n            #      whose *target* falls in that slice (set by partition_graph_edges).\n            #   2. Before each layer: AllGather local hidden states \u00e2\u2020\u2019 full (B,N,D).\n            #   3. Run the layer with the full source states but local-only edges.\n            #      The layer output is correct at local target positions; non-local\n            #      positions contain boundary-only values and are discarded.\n            #   4. Slice to local portion, apply residual, store.\n            #   5. After all layers: AllGather the local concatenated output once\n            #      to reconstruct the full result on every rank.\n            # ------------------------------------------------------------------\n            rank, world_size = dist_context\n            num_nodes = data.num_nodes\n            base_N = (num_nodes + world_size - 1) // world_size  # ceiling division\n            local_start = rank * base_N\n            local_end = min((rank + 1) * base_N, num_nodes)\n            local_N = local_end - local_start\n\n            # Collect each rank's actual local_N once (for uneven last partition).\n            local_N_t = torch.tensor(local_N, device=query.device)\n            all_local_N_list = [torch.zeros_like(local_N_t) for _ in range(world_size)]\n            dist.all_gather(all_local_N_list, local_N_t)\n            all_local_N = [int(x.item()) for x in all_local_N_list]\n            max_local_N = max(all_local_N)  # == base_N\n\n            def _allgather(local_t: torch.Tensor) -> torch.Tensor:\n                \"\"\"AllGather (B, local_N, D) across ranks into (B, N, D).\n\n                Handles the case where the last rank may have fewer nodes than\n                the others by zero-padding to max_local_N before gathering,\n                then slicing each chunk to its actual size before concatenation.\n                \"\"\"\n                B, loc_n, D = local_t.shape\n                if loc_n < max_local_N:\n                    pad = local_t.new_zeros(B, max_local_N - loc_n, D)\n                    padded = torch.cat([local_t, pad], dim=1).contiguous()\n                else:\n                    padded = local_t.contiguous()\n                chunks = [torch.zeros_like(padded) for _ in range(world_size)]\n                dist.all_gather(chunks, padded)\n                return torch.cat(\n                    [chunks[r][:, : all_local_N[r], :] for r in range(world_size)],\n                    dim=1,\n                )  # (B, N, D)\n\n            # Local slice of the initial boundary / layer input.\n            local_layer_input = node_features[:, local_start:local_end, :].clone()\n\n            for layer in self.layers:\n                if separate_grad:\n                    edge_weight = edge_weight.clone().requires_grad_()\n\n                # AllGather \u00e2\u2020\u2019 full source-node states on each rank.\n                global_input = _allgather(local_layer_input)  # (B, N, D)\n\n                # Layer forward with full input but local-target edges.\n                # hidden[v] is correct for v in [local_start, local_end);\n                # non-local positions contain boundary-only values (discarded).\n                hidden = layer(\n                    global_input,\n                    query,\n                    node_features,  # boundary: full (B, N, D), same on all ranks\n                    data.edge_index,\n                    data.edge_type,\n                    size,\n                    edge_weight,\n                )\n\n                # Slice to local, apply residual, then store local hidden.\n                local_hidden = hidden[:, local_start:local_end, :]  # (B, local_N, D)\n                if self.short_cut and local_hidden.shape == local_layer_input.shape:\n                    local_hidden = local_hidden + local_layer_input\n                hiddens.append(local_hidden)\n                edge_weights.append(edge_weight)\n                local_layer_input = local_hidden\n\n            # Concatenate local hidden slices (+ local node_query) then AllGather.\n            node_query_local = (\n                query.unsqueeze(1).expand(-1, local_N, -1).contiguous()\n            )  # (B, local_N, input_dim)\n            if self.concat_hidden:\n                local_output = torch.cat(hiddens + [node_query_local], dim=-1)\n            else:\n                local_output = torch.cat([hiddens[-1], node_query_local], dim=-1)\n            # local_output: (B, local_N, out_dim)\n\n            output = _allgather(local_output)  # (B, N, out_dim)\n\n        else:\n            # ------------------------------------------------------------------\n            # Standard single-process path (unchanged)\n            # ------------------------------------------------------------------\n            layer_input = node_features\n            # --- CCMP: contrastive continuation message passing -------------------\n            # A node's predicted responsibility scales its OUTGOING messages. Scaling\n            # `layer_input` does exactly that: every message leaving v is computed from\n            # layer_input[v]. `node_features` is left alone on purpose -- that is the\n            # query's own seed injection (the boundary condition), not a routing choice.\n            #\n            # THE GATE IS MEAN-NORMALISED, and that is a deliberate departure from\n            # d + (1-d)*yhat as specified. That form lies in [d, 1], so it can only\n            # ATTENUATE, and the factors compound over L layers: even a perfect gate\n            # emitting 0.9 everywhere leaves a 6-hop route at 0.53 and a 3-hop route at\n            # 0.73. That is a short-path prior applied on top of a model whose measured\n            # failure is a hop2->hop3 rank cliff, i.e. it pushes the wrong way. Dividing\n            # by the mean keeps the relative selectivity, which is the whole mechanism,\n            # and drops the depth-dependent global shrink, which is an artefact.\n            # CCMP_GATE_NORM=0 restores the attenuating form for the ablation.\n            _rp = []\n            _rp_stat = []\n            _heads = getattr(self, \"resp_proj\", None)\n            # STRUCTURAL REACH, not \"the state is nonzero\". Early-late entity fusion puts a\n            # static text embedding on EVERY node before propagation, so a nonzero-state\n            # test is true almost everywhere from layer 1 and the \"activated set\" was in\n            # practice the whole graph. Then the gate's mean was a graph-wide mean and the\n            # normalisation said nothing about what the query had reached.\n            _reach = None\n            _seed = getattr(self, \"_ccmp_seeds\", None)\n            # Path interpretation (trainer.interpret) asks for the per-layer frontier so a\n            # node's responsibility can be read against the frontier mean it was normalised\n            # by. Off by default; costs one [B, N] clone per layer when on.\n            self._reach_layers = []\n            if _heads is not None and _seed is not None:\n                if getattr(self, \"_ccmp_adj_key\", None) != id(data):\n                    _ei = data.edge_index\n                    _r2 = torch.cat([_ei[0], _ei[1]])\n                    _c2 = torch.cat([_ei[1], _ei[0]])\n                    self._ccmp_adj = torch.sparse_coo_tensor(\n                        torch.stack([_r2, _c2]),\n                        torch.ones(_r2.numel(), device=_ei.device),\n                        (data.num_nodes, data.num_nodes)).coalesce()\n                    self._ccmp_adj_key = id(data)\n                _reach = (_seed.to(layer_input.device) > 0).float()\n            for _li, layer in enumerate(self.layers):\n                if getattr(self, \"_keep_reach\", False):\n                    self._reach_layers.append(None if _reach is None else _reach.detach().clone())\n                if _heads is not None:\n                    # Pinning the weights to fp32 is not sufficient on its own:\n                    # under AMP an nn.Linear emits bf16 whatever its parameter dtype.\n                    # The projection stays autocast (big matmul into a 64-dim\n                    # bottleneck); the trunk and the sigmoid run in fp32, which is where\n                    # the gate's precision actually lives -- bf16's ULP near 1.0 is\n                    # 0.0039 and the gate only spans about [0.5, 1.5] at eta=0.5.\n                    _z = (_heads[_li](layer_input).float()\n                          + self.resp_emb.weight[_li].float())\n                    with torch.autocast(device_type=_z.device.type, enabled=False):\n                        _yh = torch.sigmoid(self.resp_head(_z).squeeze(-1))  # [B, N]\n                    _rp.append(_yh)\n                    if getattr(self, \"route_mode\", \"\") == \"astar\":\n                        # A*Net-STYLE HARD ROUTING. Keep the top-K reached nodes by the\n                        # head's priority and drop the rest of the frontier; kept nodes\n                        # are weighted by priority / mean(priority over kept) so the head\n                        # receives a gradient from the ranking loss (A*Net multiplies\n                        # selected messages by the priority for the same reason) and so\n                        # the selection is mean-preserving like CCMP's gate. yhat starts\n                        # at 0.5 everywhere, so at initialisation this is a pure top-K.\n                        # Off the frontier the gate is 1, exactly as for CCMP.\n                        _k = int(getattr(self, \"route_k\", 1024))\n                        _act = (_reach if _reach is not None\n                                else torch.ones_like(_yh)).to(_yh.dtype)\n                        _sc = torch.where(_act.bool(), _yh, torch.full_like(_yh, -1.0))\n                        _top = _sc.topk(min(_k, _sc.shape[-1]), dim=-1).indices\n                        _keep = torch.zeros_like(_yh).scatter_(-1, _top, 1.0) * _act\n                        _num = _yh * _keep\n                        _den = (_num.sum(-1, keepdim=True)\n                                / _keep.sum(-1, keepdim=True).clamp(min=1.0)).clamp(min=1e-6)\n                        _gt = torch.where(_keep.bool(), _num / _den, torch.zeros_like(_yh))\n                        if _reach is not None:\n                            _gt = torch.where(_reach.bool(), _gt, torch.ones_like(_gt))\n                        with torch.no_grad():\n                            _sel = _gt[_keep.bool()]\n                            if _sel.numel():\n                                _rp_stat.append((float(_sel.mean()), float(_sel.max()),\n                                                 float(_sel.quantile(0.95))))\n                        _msg = layer_input * _gt.unsqueeze(-1).to(layer_input.dtype)\n                    elif getattr(self, \"resp_gate\", False):\n                        _eta = getattr(self, \"resp_eta\", 1.0)\n                        _num = 1e-6 + _yh\n                        if getattr(self, \"resp_gate_norm\", True):\n                            # NORMALISE FIRST, THEN INTERPOLATE. mean(ybar) = 1 by\n                            # construction, so mean(g) = (1-eta) + eta = 1 for EVERY eta:\n                            # the strength knob and the mean-preservation are independent,\n                            # and eta = 0 recovers the ungated GNN exactly. Doing it the\n                            # other way round (floor, then divide by the mean) also gives\n                            # mean 1 but makes the floor a range-compression whose meaning\n                            # changes with the spread of yhat.\n                            #\n                            # The mean is over the ACTIVATED set: nodes the query has\n                            # actually reached at this layer. Averaging over all ~60k nodes\n                            # would divide by a number dominated by unreached nodes, and\n                            # the gate would scale with how far propagation has spread\n                            # rather than with what it found.\n                            _act = (_reach if _reach is not None\n                                    else (layer_input.detach().abs().sum(-1) > 0).float()\n                                    ).to(_num.dtype)\n                            _den = ((_num * _act).sum(-1, keepdim=True)\n                                    / _act.sum(-1, keepdim=True).clamp(min=1.0))\n                            _num = _num / _den.clamp(min=1e-6)\n                        _gt = (1.0 - _eta) + _eta * _num\n                        # GATE ONLY WHAT IS SUPERVISED. The loss trains yhat on A^(l)\n                        # alone, so off the frontier the prediction is whatever the head\n                        # happens to emit -- and under early fusion those nodes still hold\n                        # static text embeddings and still send messages. Applying an\n                        # unsupervised gate to them lets CCMP perturb the graph in a\n                        # direction no gradient ever checked. Gate 1 there leaves the\n                        # background computation exactly as the control computes it.\n                        if _reach is not None:\n                            _gt = torch.where(_reach.bool(), _gt, torch.ones_like(_gt))\n                        # Unbounded above: the normaliser is a MEAN, so if most reached\n                        # nodes sit near 0 a few can be amplified hard, and six layers\n                        # compound it. Recorded rather than clipped -- a clip would hide\n                        # the instability instead of showing it.\n                        with torch.no_grad():\n                            _sel = _gt[_reach.bool()] if _reach is not None else _gt\n                            if _sel.numel():\n                                _rp_stat.append((float(_sel.mean()), float(_sel.max()),\n                                                 float(_sel.quantile(0.95))))\n                        # GATE THE MESSAGES ONLY. `_raw` is what the residual adds back.\n                        # Overwriting layer_input here would put the gate on the residual\n                        # too, i.e. h^(l+1) = Conv(g h) + g h instead of Conv(g h) + h, so\n                        # the scaling would compound through all six layers and the method\n                        # would be re-weighting states rather than routing messages --\n                        # which is not what CCMP claims to do.\n                        _msg = layer_input * _gt.unsqueeze(-1).to(layer_input.dtype)\n                    else:\n                        _msg = layer_input\n                    if _reach is not None:\n                        # AUTOCAST OFF. torch.sparse.mm is on the autocast list, so under\n                        # bf16 AMP both operands are cast and there is no\n                        # addmm_sparse_cuda kernel for BFloat16 -- it raises rather than\n                        # falling back. The frontier is pure topology, so nothing here\n                        # should be cast or differentiated in the first place.\n                        with torch.no_grad(), torch.autocast(\n                                device_type=_reach.device.type, enabled=False):\n                            _nx = torch.sparse.mm(\n                                self._ccmp_adj, _reach.t().contiguous().float()).t()\n                            _reach = ((_nx > 0) | (_reach > 0)).float()\n                else:\n                    _msg = layer_input\n                if separate_grad:\n                    edge_weight = edge_weight.clone().requires_grad_()\n\n                # RED-GNN-STYLE EDGE ATTENTION (ROUTE=attn): per-query weights on every\n                # edge, computed from the sender state, the relation and the query. Passed\n                # as a [B, E] edge_weight, which routes the layer through the unfused\n                # message/aggregate path (the rspmm kernel takes one weight per edge).\n                _ew = edge_weight\n                if getattr(self, \"attn_node\", None) is not None and \\\n                        os.environ.get(\"ROUTE_CKPT\", \"1\") == \"1\" and torch.is_grad_enabled():\n                    # GRADIENT CHECKPOINT THE ATTENTION LAYER. The unfused path keeps every\n                    # [B, E+N, D] intermediate (gathered senders, gathered relations, the\n                    # product, the boundary concat, the weighted copy) alive for backward:\n                    # ~1.5 GB each in bf16 on TOMATO's 253k-edge train graph, x6 layers, which\n                    # is how the attn smoke reached 75 GB on an 80 GB A100 (8 Sep) while the\n                    # fused arms need a fraction of that. Recompute each layer's forward in\n                    # backward instead: peak memory becomes one layer's transient, the\n                    # numbers are identical, and the forward is paid twice. The attention\n                    # logits are inside the checkpoint so their [B, E, H] tensors are not kept\n                    # either. ROUTE_CKPT=0 restores the plain path.\n                    from torch.utils.checkpoint import checkpoint as _ckpt\n\n                    def _attn_layer(_x, _m, _q, _nf, _w, _layer=layer, _l=_li):\n                        _w2 = self._route_attention(_l, _x, _q, _layer, data, _w)\n                        return _layer(_m, _q, _nf, data.edge_index, data.edge_type, size, _w2)\n\n                    hidden = _ckpt(_attn_layer, layer_input, _msg, query, node_features,\n                                   edge_weight, use_reentrant=False)\n                else:\n                    if getattr(self, \"attn_node\", None) is not None:\n                        _ew = self._route_attention(_li, layer_input, query, layer, data, edge_weight)\n                    # Bellman-Ford iteration, we send the original boundary condition in addition to the updated node states\n                    hidden = layer(\n                        _msg,\n                        query,\n                        node_features,\n                        data.edge_index,\n                        data.edge_type,\n                        size,\n                        _ew,\n                    )\n                if self.short_cut and hidden.shape == layer_input.shape:\n                    # residual connection here. UNGATED `layer_input`, not `_msg`: the\n                    # gate belongs on the messages this node sends, not on the state it\n                    # keeps.\n                    hidden = hidden + layer_input\n                hiddens.append(hidden)\n                edge_weights.append(edge_weight)\n                layer_input = hidden\n            # Cached for the loss. A list of [B, N] per layer, or empty when CCMP is off,\n            # so a control run pays nothing and the attribute always exists.\n            self._resp_pred = _rp\n            self._resp_gstat = _rp_stat\n\n            # original query (relation type) embeddings\n            node_query = query.unsqueeze(1).expand(\n                -1, data.num_nodes, -1\n            )  # (batch_size, num_nodes, input_dim)\n            if self.concat_hidden:\n                output = torch.cat(hiddens + [node_query], dim=-1)\n            else:\n                output = torch.cat([hiddens[-1], node_query], dim=-1)\n\n        return {\n            \"node_feature\": output,\n            \"edge_weights\": edge_weights,\n        }\n\n    def forward(self, data, node_features, relation_representations, query):\n        # initialize relations in each NBFNet layer (with uinque projection internally)\n        for layer in self.layers:\n            layer.relation = relation_representations\n\n        # we already did traversal_dropout in the outer loop of UltraQuery\n        # if self.training:\n        #     # Edge dropout in the training mode\n        #     # here we want to remove immediate edges (head, relation, tail) from the edge_index and edge_types\n        #     # to make NBFNet iteration learn non-trivial paths\n        #     data = self.remove_easy_edges(data, h_index, t_index, r_index)\n\n        # node features arrive in shape (bs, num_nodes, dim)\n        # NBFNet needs batch size on the first place\n        output = self.bellmanford(\n            data, node_features, query\n        )  # (num_nodes, batch_size, feature_dim\uff09\n        if self.return_hidden:\n            return output[\"node_feature\"]\n        else:\n            score = self.mlp(output[\"node_feature\"]).squeeze(-1)  # (bs, num_nodes)\n            # return only the score\n            return score\n\n    def visualize(self, data, sample, node_features, relation_representations, query):\n        for layer in self.layers:\n            layer.relation = relation_representations\n\n        output = self.bellmanford(\n            data, node_features, query, separate_grad=True\n        )  # (num_nodes, batch_size, feature_dim\uff09\n        node_feature = output[\"node_feature\"]\n        edge_weights = output[\"edge_weights\"]\n        question_entities_mask = sample[\"start_nodes_mask\"]\n        target_entities_mask = sample[\"target_nodes_mask\"]\n        query_entities_index = question_entities_mask.nonzero(as_tuple=True)[1]\n        target_entities_index = target_entities_mask.nonzero(as_tuple=True)[1]\n\n        paths_results = {}\n        for t_index in target_entities_index:\n            index = (\n                t_index.unsqueeze(0)\n                .unsqueeze(0)\n                .unsqueeze(-1)\n                .expand(-1, -1, node_feature.shape[-1])\n            )\n            feature = node_feature.gather(1, index).squeeze(0)\n            score = self.mlp(feature).squeeze(-1)\n\n            edge_grads = autograd.grad(score, edge_weights, retain_graph=True)\n            distances, back_edges = self.beam_search_distance(\n                data, edge_grads, query_entities_index, t_index, self.num_beam\n            )\n            paths, weights = self.topk_average_length(\n                distances, back_edges, t_index, self.path_topk\n            )\n            paths_results[t_index.item()] = (paths, weights)\n        return paths_results\n", "/content/gfm-rag/gfmrag/trainers/fusion_trainer.py": "\"\"\"\nfusion_trainer.py \u2014 SFTTrainer for the CARGO fusion (FusionGraphReasoner / RoutedFusionReasoner).\n\nTwo objectives, selected by env FUSION_OBJECTIVE:\n\n- \"bce_pcr\" (default, legacy): the config losses (bce + pcr) act on the FUSED document scores, plus a\n  small graph-alone ListCE aux (weight AUX_W). This is the run whose graph stayed inert (aux frozen at\n  ~log N): the operator already satisfies bce/pcr on the easy queries, so the GNN never gets a gradient.\n\n- \"hardneg\" (the report's Eq 3.9 objective): a softmax cross-entropy over a per-query lineup whose\n  negatives are the OPERATOR'S OWN top-ranked hubs \u2014 {gold} u operator-top-`HARDNEG_HUB` u `HARDNEG_RAND`\n  random docs. Ranking the gold above the docs the operator already loves can only be done with the graph,\n  so this is what actually teaches the graph to fix the operator's cross-domain misses. It is applied to\n  BOTH the fused score (trains gate/router + operator scalars + graph jointly) AND the graph-alone score\n  (weight AUX_W \u2014 trains the GNN DIRECTLY, so it still learns even when the fusion is initialised\n  near-operator and the fused-path gradient into the graph is tiny).\n\nLOSS V2 \u2014 three flag-gated, dissim-targeted edits to the hardneg objective.\nAll default OFF, giving bit-identical legacy behavior:\n\n1. PER_GOLD=1 \u2014 per-gold contrastive. Legacy pools golds in one logsumexp, which is a\n   soft-max: one easy gold satisfies the query and a buried dissim gold free-rides with\n   ~zero gradient. Per-gold, EVERY gold must individually beat the lineup:\n       L = sum_g w_g * [ logsumexp(negs u {g}) - s_g ] / sum_g w_g\n2. MISS_W_AUX=1 / MISS_W_FUSED=1 \u2014 miss-weighting: w_g = log1p(operator rank of gold g),\n   capped at MISS_W_CAP (default 8), normalised by the weight sum so the loss scale is\n   stable. Concentrates gradient on the golds the operator buries (68% of dissim golds\n   sit past rank 100). Recommended always-on for the graph-alone aux (no gate/router in\n   that path); on the FUSED term only under convex routing \u2014 with the per-query additive\n   gate, amplifying the gradient of graph-noisy queries is what taught the gate backwards.\n   With PER_GOLD=0 the pooled query term is weighted by its WORST-ranked gold.\n3. HARDNEG_GRAPH=K \u2014 graph-mined negatives: the graph-alone top-K (detached, golds\n   removed) join the lineup, so the contrastive also pushes DOWN docs the graph\n   over-scores (PPR domain-hub flooding) instead of only pushing golds up past operator\n   hubs. Early in training the GNN is ~random so these are just extra random negatives;\n   the term becomes self-adversarial as the graph learns.\n\nOnly train_step is overridden; evaluate()/predict() are inherited unchanged and already consume the\nfused score, because the fusion lives inside the model's forward().\n\"\"\"\nimport os\nimport json\n\nimport torch\n\nfrom gfmrag.losses import ListCELoss\nfrom gfmrag.models import cqig as cqig_mod\nfrom gfmrag.models.ultra import query_utils\n\nfrom .sft_trainer import SFTTrainer\n\n\ndef _emb(batch):\n    \"\"\"A single [1, D] question vector for a batch of any size.\"\"\"\n    e = batch[\"question_embeddings\"].float()\n    return e.reshape(-1, e.shape[-1]).mean(0, keepdim=True)\n\n\nclass FusionSFTTrainer(SFTTrainer):\n    def __init__(self, *args, **kwargs):\n        super().__init__(*args, **kwargs)\n        self._aux_loss_fn = ListCELoss()\n        self._aux_w = float(os.environ.get(\"AUX_W\", \"0.1\"))\n        self._objective = os.environ.get(\"FUSION_OBJECTIVE\", \"bce_pcr\")\n        self._hn_hub = int(os.environ.get(\"HARDNEG_HUB\", \"50\"))    # operator-top-K hubs per query\n        self._hn_rand = int(os.environ.get(\"HARDNEG_RAND\", \"50\"))  # random negatives per query\n        # Anchor weight for the learned semantic scorer's popularity predictor. Matches\n        # --pop_lambda in semantic_scorer.py so the fusion continues training the model\n        # under the objective it was selected under, not a different one.\n        self._sem_pop_lambda = float(os.environ.get(\"SEM_POP_LAMBDA\", \"1.0\"))\n        # --- Cross-Query Informativeness Gating (inert unless the model has a gate) ---\n        self._cqig_m = int(os.environ.get(\"CQIG_M\", \"16\"))            # reference queries\n        self._cqig_pool_size = int(os.environ.get(\"CQIG_POOL\", \"64\"))  # pool to choose from\n        self._cqig_recal = int(os.environ.get(\"CQIG_RECAL\", \"1000\"))   # steps; 0 = calibrate once\n        # Calibration rounds. 1 is exact for a single gated layer; 2 re-runs the whole\n        # calibration with the first round's gates active, which is what removes the stale\n        # centering when several layers are gated. Costs one more pair of reference passes.\n        self._cqig_rounds = int(os.environ.get(\"CQIG_ROUNDS\", \"1\"))\n        # Queries per reference forward. 1 is the safe default: the calibration state is\n        # [B, N, D] and the graphs here have ~60k nodes.\n        self._cqig_ref_batch = int(os.environ.get(\"CQIG_REF_BATCH\", \"1\"))\n        # Hard negatives per query for the gold-vs-negative informativeness AUC. Matches\n        # HARDNEG_HUB so the diagnostic asks about the same documents the loss does.\n        self._cqig_hn_k = int(os.environ.get(\"CQIG_HN_K\", os.environ.get(\"HARDNEG_HUB\", \"50\")))\n        # How the frozen source references are re-linked to a target graph.\n        #   exact     string equality on entity names, i.e. what the indexer does. Correct\n        #             on the source graph and near-useless off it: physics resolved 501/501\n        #             on its train graph and 25/501 on its own test graph.\n        #   semantic  exact first, then the K nearest target nodes to the seed's frozen\n        #             source embedding, above a source-selected cosine floor, weighted by\n        #             softmax(cos / T) and normalised so the seed's total weight is\n        #             unchanged. Still zero-shot: frozen encoder, source-chosen floor, no\n        #             target label or query touched.\n        self._cqig_link = os.environ.get(\"CQIG_LINK\", \"exact\").strip().lower()\n        assert self._cqig_link in (\"exact\", \"semantic\"), (\n            f\"CQIG_LINK={self._cqig_link!r}; expected 'exact' or 'semantic'\")\n        self._cqig_link_k = int(os.environ.get(\"CQIG_LINK_K\", \"3\"))\n        # Softmax temperature over cosines. At 0.05 a 0.1 gap in cosine is a ~7x weight\n        # ratio: sharp enough that the best match dominates, soft enough that a genuine\n        # near-tie splits rather than being decided by encoder noise.\n        self._cqig_link_temp = float(os.environ.get(\"CQIG_LINK_T\", \"0.05\"))\n        # The rejection floor. \"auto\" measures it on the SOURCE graph (see _cqig_link_floor);\n        # a number pins it. Either way it is fixed before any target graph is seen.\n        self._cqig_link_min = os.environ.get(\"CQIG_LINK_MIN\", \"auto\").strip().lower()\n        self._cqig_link_q = float(os.environ.get(\"CQIG_LINK_Q\", \"0.5\"))\n        # --- loss v2 flags (all default to exact legacy behavior) ---\n        self._per_gold = os.environ.get(\"PER_GOLD\", \"0\") == \"1\"\n        self._hn_graph = int(os.environ.get(\"HARDNEG_GRAPH\", \"0\"))       # graph-top-K negatives\n        self._miss_w_fused = os.environ.get(\"MISS_W_FUSED\", \"0\") == \"1\"\n        self._miss_w_aux = os.environ.get(\"MISS_W_AUX\", \"0\") == \"1\"\n        self._miss_w_cap = float(os.environ.get(\"MISS_W_CAP\", \"8.0\"))\n        # --- RESID_PRIOR: residual supervision against the graph's own structural prior ---\n        #\n        # MEASURED PROBLEM. A zero-parameter random walk on these graphs scores 0.245/0.273\n        # nDCG@5; the trained 6-layer GNN scores 0.213-0.273. The graph-alone contrastive\n        # asks the GNN to rank golds above operator hubs, which personalised PageRank from\n        # the same seeds largely does already, so the term is near-satisfied at init and the\n        # surviving gradient points back at the structure the walk exploits. The learned\n        # component is close to free-lunch topology.\n        #\n        # THE EDIT. Restrict each gold's negatives to the documents the PRIOR already ranks\n        # ABOVE it. Documents the prior ordered correctly contribute exactly zero gradient,\n        # so the loss can only be reduced by capacity the walk does not have. This is\n        # orthogonal to MISS_W_*, which reweights by the SEMANTIC channel's rank and\n        # therefore never told the graph anything about what the graph already knew.\n        #\n        # Costs one cached PPR per query: T sparse mat-muls, computed once and reused for\n        # every later epoch, against 6 dense GNN layers per step.\n        self._resid_prior = os.environ.get(\"RESID_PRIOR\", \"0\") == \"1\"\n        self._resid_k = int(os.environ.get(\"RESID_K\", \"200\"))       # cap on |negatives|\n        self._resid_T = int(os.environ.get(\"RESID_T\", \"3\"))         # walk length\n        self._resid_alpha = float(os.environ.get(\"RESID_ALPHA\", \"0.15\"))   # restart prob\n        # Golds the prior already ranks first have an EMPTY negative set and would vanish\n        # from the loss entirely, leaving the GNN unconstrained on everything it gets right.\n        # They keep a small term against the ordinary lineup instead.\n        self._resid_anchor = float(os.environ.get(\"RESID_ANCHOR\", \"0.1\"))\n        self._ppr_A = None          # row-normalised transition, one per graph\n        self._ppr_A_key = None\n        self._prior_cache = {}      # (graph key, query id) -> document prior, float16 CPU\n\n        # --- CCMP: contrastive continuation message passing ------------------------\n        self._ccmp = os.environ.get(\"CCMP\", \"0\") == \"1\"\n        self._ccmp_w = float(os.environ.get(\"CCMP_W\", \"0.1\"))     # lambda_r\n        self._ccmp_neg = int(os.environ.get(\"CCMP_NEG\", \"64\"))    # |H^sem_q|\n        # Over-fetch semantic errors before applying the graph-reachability filter.\n        # CCMP_POOL=256 with CCMP_NEG=64 is the paper setting: it preserves semantic\n        # hardness while giving the graph four times as many candidates from which to\n        # find errors it can actually reach within the GNN horizon.\n        self._ccmp_pool = max(\n            self._ccmp_neg, int(os.environ.get(\"CCMP_POOL\", \"256\"))\n        )\n        # Select the two sides independently. CCMP_M remains a backwards-compatible\n        # total cap for old runs; when supplied alone it is split evenly. New runs use\n        # 512 gold-favouring + 512 error-favouring nodes per layer.\n        _legacy_m = os.environ.get(\"CCMP_M\")\n        _legacy_side = max(1, int(_legacy_m) // 2) if _legacy_m else 512\n        self._ccmp_m_pos = int(os.environ.get(\"CCMP_M_POS\", str(_legacy_side)))\n        self._ccmp_m_neg = int(os.environ.get(\"CCMP_M_NEG\", str(_legacy_side)))\n        self._ccmp_cmin = float(os.environ.get(\"CCMP_CMIN\", \"0.2\"))\n        # RESIDUAL CCMP. Default OFF, so every run made before this is byte-identical.\n        # ON: supervision is restricted to the golds the semantic scorer has NOT already\n        # resolved, U_q = {g in G_q : exists d not in G_q with s_sem(q,d) >= s_sem(q,g)},\n        # and the negatives to the reachable errors that outrank them. Queries with\n        # U_q empty are supervised to leave propagation unchanged (g_qv = 1) instead of\n        # being asked to improve a ranking that is already correct. Nothing changes at\n        # inference: the head still predicts from node states alone.\n        self._ccmp_resid = os.environ.get(\"CCMP_RESIDUAL\", \"0\") == \"1\"\n        # Weight of the identity (neutral) bucket in the class-balanced mean. 0 drops the\n        # identity supervision and keeps only the restriction to unresolved golds, which\n        # separates the two halves of the change.\n        self._ccmp_id_w = float(os.environ.get(\"CCMP_IDENTITY_W\", \"1.0\"))\n        self._ccmp_P = None\n        self._ccmp_key = None\n        # The selected endpoint set and its continuation targets are fixed on first use,\n        # then cached. This keeps CCMP's supervision stationary while the semantic and\n        # graph scorers continue to train.\n        self._ccmp_cache = {}        # (graph key, query id) -> (idx, y, c, meta) CPU\n        if self._ccmp:\n            print(\n                f\"[ccmp targets] reachable semantic errors: pool={self._ccmp_pool} \"\n                f\"keep={self._ccmp_neg}; graph fallback on shortfall; \"\n                f\"nodes/layer=+{self._ccmp_m_pos}/-{self._ccmp_m_neg}\"\n                + (f\"; RESIDUAL on (identity_w={self._ccmp_id_w:g})\"\n                   if self._ccmp_resid else \"; residual off\"),\n                flush=True,\n            )\n\n    def build_lineups(self, target_doc, s_op, g_mine=None):\n        \"\"\"One candidate lineup per query, built ONCE and reused by both losses.\n\n        The fused and graph-alone terms are meant to face the SAME lineup, so that the\n        only difference between them is which score is being trained. Building the\n        negatives inside each call gave them the same hard negatives (top-k is\n        deterministic) but freshly drawn randoms, which is not the design and adds\n        variance for nothing.\n        \"\"\"\n        B, n_doc = target_doc.shape\n        dev = target_doc.device\n        out = []\n        for b in range(B):\n            pos = target_doc[b].nonzero(as_tuple=True)[0]\n            if pos.numel() == 0:\n                out.append(None)\n                continue\n            # A UNION, NOT A CONCATENATION. The semantic and graph top-K overlap heavily\n            # and torch.randint samples WITH replacement, so cat() put the same paper in\n            # the denominator two or three times and silently gave it two or three times\n            # the negative weight -- worst for exactly the papers both rankers over-score.\n            taken = torch.zeros(n_doc, dtype=torch.bool, device=dev)\n            taken[pos] = True\n            # Over-fetch by the most that could already be claimed, so a fresh top-k is\n            # always available without a .item() sync inside the training loop.\n            budget = pos.numel() + self._hn_hub + max(self._hn_graph, 0)\n\n            def _top_fresh(scores, k):\n                \"\"\"Top-k of `scores` not already claimed; marks what it returns.\"\"\"\n                if scores is None or k <= 0:\n                    return None\n                idx = scores.topk(min(k + budget, n_doc)).indices\n                idx = idx[~taken[idx]][:k]\n                taken[idx] = True\n                return idx\n\n            negs = [x for x in (\n                _top_fresh(s_op[b], self._hn_hub),          # the semantic scorer's own hubs\n                _top_fresh(g_mine[b] if g_mine is not None else None, self._hn_graph),\n            ) if x is not None and x.numel() > 0]\n            # EXACTLY `_hn_rand` DISTINCT random non-golds, rather than \"50 draws minus\n            # whatever collided\", which quietly delivered fewer. A permutation is exact;\n            # at 1k-20k documents it costs far less than the forward pass it rides on.\n            if self._hn_rand > 0:\n                pool = torch.randperm(n_doc, device=dev)\n                pool = pool[~taken[pool]][: self._hn_rand]\n                if pool.numel():\n                    negs.append(pool)\n            out.append(torch.cat(negs) if negs else pos.new_empty(0))\n        return out\n\n    def _contrastive_hardneg(self, doc_scores, target_doc, s_op, g_mine=None, miss_w=False,\n                             lineups=None):\n        \"\"\"Contrastive over {gold(s)} u operator-top-K hubs [u graph-top-K] u random negatives.\n\n        doc_scores : [B, n_doc] scores to train (fused or graph-alone), require grad.\n        target_doc : [B, n_doc] {0,1} gold mask over document nodes.\n        s_op       : [B, n_doc] operator scores (detached) \u2014 mines hard-negative hubs + miss weights.\n        g_mine     : [B, n_doc] graph-alone scores (DETACHED) \u2014 mines HARDNEG_GRAPH extra negatives.\n        miss_w     : weight each gold by log1p(its operator rank), capped at MISS_W_CAP.\n        \"\"\"\n        B, n_doc = doc_scores.shape\n        dev = doc_scores.device\n        loss = doc_scores.new_zeros(())\n        wsum = 0.0\n        for b in range(B):\n            pos = target_doc[b].nonzero(as_tuple=True)[0]\n            if pos.numel() == 0:\n                continue\n            # Built once per step and shared by both losses. Falls back to building its\n            # own only when called without one, which keeps older callers working.\n            neg = (lineups[b] if lineups is not None\n                   else self.build_lineups(target_doc, s_op, g_mine)[b])\n            neg_logits = doc_scores[b, neg]\n            # per-gold miss weights: how badly does the operator rank each gold?\n            if miss_w:\n                ranks = (s_op[b].unsqueeze(0) > s_op[b, pos].unsqueeze(1)).sum(1).float() + 1.0\n                w = torch.log1p(ranks).clamp(max=self._miss_w_cap)\n            else:\n                w = torch.ones(pos.numel(), device=dev)\n            if self._per_gold:\n                # every gold must individually beat the lineup (no free-riding behind a sibling)\n                for j in range(pos.numel()):\n                    lg = torch.cat([neg_logits, doc_scores[b, pos[j : j + 1]]])\n                    loss = loss + w[j] * (torch.logsumexp(lg, 0) - doc_scores[b, pos[j]])\n                    wsum += float(w[j])\n            else:\n                # legacy pooled multi-positive; miss_w weights the query by its worst-ranked gold\n                logits = torch.cat([doc_scores[b, pos], neg_logits])\n                log_z = torch.logsumexp(logits, 0)\n                log_pos = torch.logsumexp(logits[: pos.numel()], 0)\n                wq = float(w.max())\n                loss = loss + wq * (log_z - log_pos)\n                wsum += wq\n        return loss / max(wsum, 1e-6)\n\n    # ------------------------------------------------- RESID_PRIOR: the structural prior\n    def _prior_doc(self, graph, batch, doc_ids):\n        \"\"\"Personalised PageRank over the KG from this query's seeds, restricted to documents.\n\n        Parameter-free and detached: this is the baseline the GNN has to beat, so nothing\n        about it may depend on anything the GNN learns. Cached per (graph, seed set), which\n        makes it a first-epoch cost only. Cached on CPU in float16 because the train graph\n        holds 1304 queries x 4676 documents and that is 12 MB rather than 24.\n        \"\"\"\n        dev = batch[\"start_nodes_mask\"].device\n        N = graph.num_nodes\n        gkey = f\"{id(graph)}:{N}\"\n        if self._ppr_A_key != gkey:\n            ei = graph.edge_index\n            # SYMMETRISED. The KG is directed, and a walk that can only travel head->tail\n            # cannot reach a document from an entity the document mentions, which is the\n            # dominant path in this corpus. Both directions, then row-normalise.\n            r = torch.cat([ei[0], ei[1]])\n            c = torch.cat([ei[1], ei[0]])\n            deg = torch.zeros(N, device=dev).index_add_(\n                0, r, torch.ones(r.numel(), device=dev))\n            w = 1.0 / deg.clamp(min=1.0)[r]\n            self._ppr_A = torch.sparse_coo_tensor(\n                torch.stack([c, r]), w, (N, N), device=dev).coalesce()\n            self._ppr_A_key = gkey\n            self._prior_cache.clear()\n\n        seeds = batch[\"start_nodes_mask\"].float()                    # [B, N]\n        out = seeds.new_zeros(seeds.shape[0], doc_ids.numel())\n        for b in range(seeds.shape[0]):\n            idx = seeds[b].nonzero(as_tuple=True)[0]\n            key = (gkey, tuple(idx.tolist()))\n            hit = self._prior_cache.get(key)\n            if hit is None:\n                x0 = seeds[b : b + 1]\n                s = x0.sum()\n                if float(s) <= 0:\n                    # no seed resolved: a uniform prior, which makes EVERY document a\n                    # negative for every gold. Skipped by the loss instead.\n                    hit = torch.zeros(doc_ids.numel(), dtype=torch.float16)\n                else:\n                    # AUTOCAST OFF and no grad. train_step runs under bfloat16 AMP and\n                    # sparse.mm has no bf16 kernel on every build; a crash here would cost\n                    # a whole run. The prior is data, not a learned quantity, so neither\n                    # autocast nor autograd has any business touching it.\n                    with torch.autocast(device_type=x0.device.type, enabled=False), \\\n                            torch.no_grad():\n                        x0 = (x0 / s).float()\n                        x = x0\n                        for _ in range(self._resid_T):\n                            x = ((1.0 - self._resid_alpha)\n                                 * torch.sparse.mm(self._ppr_A, x.t()).t()\n                                 + self._resid_alpha * x0)\n                    hit = x[0, doc_ids].detach().half().cpu()\n                self._prior_cache[key] = hit\n            out[b] = hit.to(dev).float()\n        return out\n\n    def _contrastive_resid(self, doc_scores, target_doc, prior_doc, lineups):\n        \"\"\"Negatives = the documents the PRIOR already ranks above this gold.\n\n        A gold the walk already places above every non-gold has nothing to fix and drops\n        out (except for the anchor); a gold the walk buries under 300 documents is asked to\n        climb past the worst of them. The loss therefore measures only what the GNN adds\n        to the topology, which is the quantity the run is trying to move.\n        \"\"\"\n        B = doc_scores.shape[0]\n        loss = doc_scores.new_zeros(())\n        wsum, n_gold, n_touch, n_neg = 0.0, 0, 0, 0\n        for b in range(B):\n            pos = target_doc[b].nonzero(as_tuple=True)[0]\n            if pos.numel() == 0:\n                continue\n            pr = prior_doc[b]\n            if float(pr.max()) <= 0:\n                continue                        # unseeded query: the prior says nothing\n            for j in range(pos.numel()):\n                g = pos[j]\n                n_gold += 1\n                above = (pr > pr[g]).nonzero(as_tuple=True)[0]\n                if above.numel():\n                    # sibling golds are not negatives, whatever the prior thinks of them\n                    above = above[~torch.isin(above, pos)]\n                if above.numel() == 0:\n                    if self._resid_anchor > 0 and lineups is not None and lineups[b] is not None:\n                        lg = torch.cat([doc_scores[b, lineups[b]], doc_scores[b, g : g + 1]])\n                        loss = loss + self._resid_anchor * (\n                            torch.logsumexp(lg, 0) - doc_scores[b, g])\n                        wsum += self._resid_anchor\n                    continue\n                if above.numel() > self._resid_k:\n                    # keep the prior's HIGHEST-scored offenders. Those are the documents\n                    # actually occupying the top of the ranking the gold has to enter;\n                    # the tail of a 3000-long set is noise and would dominate by count.\n                    above = above[pr[above].topk(self._resid_k).indices]\n                n_touch += 1\n                n_neg += int(above.numel())\n                lg = torch.cat([doc_scores[b, above], doc_scores[b, g : g + 1]])\n                loss = loss + (torch.logsumexp(lg, 0) - doc_scores[b, g])\n                wsum += 1.0\n        stats = {\n            # what fraction of golds the prior actually gets wrong, i.e. how much of the\n            # data this loss can even see. If it is near zero the walk is already right\n            # and there is nothing to learn; if it is near one the prior is useless here.\n            \"resid_cover\": n_touch / max(n_gold, 1),\n            \"resid_negs\": n_neg / max(n_touch, 1),\n        }\n        return loss / max(wsum, 1e-6), stats\n\n    # ------------------------------------------------------------------- CCMP targets\n    def _ccmp_hit(self, onehot, h):\n        \"\"\"Pr[a walk from v reaches this column's target within h hops], in [0, 1].\n\n        Absorbing recurrence x <- 1_T + (1 - 1_T) P x. NOT sum_t P^t 1_T, which is the\n        expected number of visits: unbounded, and it over-counts nodes sitting on short\n        cycles, so it is not the probability the ratio is supposed to be a ratio of.\n        \"\"\"\n        # SELF-GUARDING. Correct today only because the one caller wraps it, and there is\n        # no bf16 addmm_sparse_cuda kernel: under autocast this raises rather than falling\n        # back, so a second caller would be a crash, not a slow path.\n        with torch.no_grad(), torch.autocast(\n                device_type=onehot.device.type, enabled=False):\n            x = onehot.clone().float()\n            keep = 1.0 - x\n            for _ in range(h):\n                x = onehot.float() + keep * torch.sparse.mm(self._ccmp_P.float(), x)\n        return x\n\n    def _ccmp_targets(self, graph, batch, doc_ids, s_op, g_mine=None):\n        \"\"\"(idx, y, c, meta) per query, with idx/y/c each [L, M]. The endpoint\n        set is selected on the query's first occurrence and then cached, making CCMP's\n        intermediate supervision stationary while the two retrieval channels train.\n\n        PER-TARGET COLUMNS, REDUCED PER SIDE. Seeding B+ from |G| nodes and B- from |H|\n        nodes -- the form as originally written -- makes y ~ |G|/(|G|+|H|) ~ 0.01 and\n        c = |2y-1| ~ 1 at essentially EVERY node, so the objective collapses to \"predict\n        0, confidently\", the gate shuts globally and delta is the only thing propagating.\n        Reducing each side separately is what makes a hub land at y ~ 0.5, c ~ 0, which\n        is the ambiguity the confidence weight exists to express.\n\n        ONE TARGET PER (q, l, v), over the gold UNION. yhat carries no gold index, so a\n        per-gold target would only ever be fit to its c-weighted mean over golds -- i.e.\n        exactly the \"one easy gold satisfies the supervision\" failure that per-gold\n        computation is supposed to prevent.\n        \"\"\"\n        dev = doc_ids.device\n        N = graph.num_nodes\n        gkey = f\"{id(graph)}:{N}\"\n        if self._ccmp_key != gkey:\n            ei = graph.edge_index\n            r = torch.cat([ei[0], ei[1]])\n            c = torch.cat([ei[1], ei[0]])\n            deg = torch.zeros(N, device=dev).index_add_(\n                0, r, torch.ones(r.numel(), device=dev))\n            self._ccmp_P = torch.sparse_coo_tensor(\n                torch.stack([r, c]), 1.0 / deg.clamp(min=1.0)[r],\n                (N, N), device=dev).coalesce()\n            self._ccmp_key = gkey\n            self._ccmp_cache.clear()\n        L = len(getattr(self.model.base.entity_model, \"resp_proj\", []) or [])\n        tgt = batch[\"target_nodes_mask\"]\n        seeds = batch[\"start_nodes_mask\"]\n        out = []\n        for b in range(tgt.shape[0]):\n            gp = tgt[b, doc_ids].nonzero(as_tuple=True)[0]\n            if gp.numel() == 0 or float(seeds[b].sum()) <= 0:\n                out.append(None)\n                continue\n            # KEYED ON THE QUERY ID. Golds + seed COUNT is not a query identity: two\n            # queries sharing golds and seed count would collide, and the negatives are\n            # the SEMANTIC scorer's top-K, which differ per query, so the second query\n            # would silently train against the first one's distractors.\n            key = (gkey, str(batch[\"id\"][b]))\n            hit = self._ccmp_cache.get(key)\n            if hit is None:\n                with torch.autocast(device_type=dev.type, enabled=False), torch.no_grad():\n                    # STRUCTURAL REACH. A^(0) = the query's seeds, A^(l+1) = A^(l) u\n                    # N(A^(l)). Besides restricting layer-l supervision to A^(l), the\n                    # final `reach` identifies documents the graph can reach within the\n                    # complete L-layer horizon. An unreachable negative has B-=0 from\n                    # every query-conditioned state and therefore cannot teach CCMP which\n                    # route to suppress.\n                    reach = (seeds[b] > 0).float()\n                    reaches = []\n                    for _ in range(L):\n                        reaches.append(reach.clone())\n                        nxt = torch.sparse.mm(\n                            self._ccmp_P, reach.unsqueeze(1)\n                        ).squeeze(1)\n                        reach = ((nxt > 0) | (reach > 0)).float()\n                    reachable_doc = reach[doc_ids] > 0\n\n                    # Start from the scorer's most convincing errors, but retain only\n                    # errors the graph could propagate to in L layers. We over-fetch\n                    # CCMP_POOL (256 by default) and keep the first CCMP_NEG (64).\n                    gold_doc = torch.zeros(\n                        doc_ids.numel(), dtype=torch.bool, device=dev\n                    )\n                    gold_doc[gp] = True\n                    s1 = s_op[b].detach().float().clone()\n                    s1[gold_doc] = -torch.inf\n\n                    # WHICH GOLDS THE SEMANTIC SCORER HAS NOT RESOLVED. A gold is\n                    # unresolved when at least one non-gold scores at least as highly,\n                    # i.e. s_sem(q,g) <= max_{d not in G_q} s_sem(q,d). With CCMP_RESIDUAL\n                    # off, gp_use is every gold and no threshold is applied, which is the\n                    # historical behaviour exactly.\n                    gp_use, neutral = gp, False\n                    if self._ccmp_resid:\n                        max_neg = s1.max()\n                        unres = gp[s_op[b].detach().float()[gp] <= max_neg]\n                        if unres.numel() == 0:\n                            # Every gold already outranks every non-gold. There is no\n                            # error for the graph to correct, so CCMP is supervised to be\n                            # a no-op here rather than perturbing a correct ranking. Node\n                            # SELECTION is left untouched (below) so the represented set\n                            # is the same one the contrastive queries use; only the target\n                            # becomes neutral.\n                            neutral = True\n                        else:\n                            gp_use = unres\n                            # Negatives become the errors that stand between the query and\n                            # its missed golds: non-golds scoring at least as highly as the\n                            # lowest unresolved gold. Documents ranked below every missed\n                            # gold are not obstructing anything and carry no signal about\n                            # which route to suppress.\n                            thr = s_op[b].detach().float()[gp_use].min()\n                            s1[s1 < thr] = -torch.inf\n                    pool_k = min(self._ccmp_pool, s1.numel())\n                    sem_pool = s1.topk(pool_k).indices\n                    # torch.isfinite drops the -inf padding topk returns once the\n                    # residual threshold has masked most of the corpus.\n                    sem_ok = (reachable_doc[sem_pool] & ~gold_doc[sem_pool]\n                              & torch.isfinite(s1[sem_pool]))\n                    sem_reachable = sem_pool[sem_ok]\n                    neg_idx = sem_reachable[: self._ccmp_neg]\n                    n_sem = int(neg_idx.numel())\n\n                    # Some queries have fewer than 64 reachable documents among the\n                    # semantic top 256. Fill only the missing slots with the graph's\n                    # highest-ranked reachable errors. Never re-add a gold or a semantic\n                    # negative already selected. If the reachable subgraph itself holds\n                    # fewer than 64 non-golds, use the smaller honest endpoint set rather\n                    # than reintroducing unreachable papers.\n                    n_graph = 0\n                    missing = self._ccmp_neg - n_sem\n                    if missing > 0 and g_mine is not None:\n                        eligible = reachable_doc & ~gold_doc\n                        if self._ccmp_resid and not neutral:\n                            # The graph-score fallback respects the same threshold, or the\n                            # \"errors that outrank the missed golds\" definition would leak.\n                            eligible = eligible & torch.isfinite(s1)\n                        if neg_idx.numel() > 0:\n                            eligible[neg_idx] = False\n                        fill_k = min(missing, int(eligible.sum()))\n                        if fill_k > 0:\n                            g1 = g_mine[b].detach().float().clone()\n                            g1[~eligible] = -torch.inf\n                            graph_fill = g1.topk(fill_k).indices\n                            neg_idx = torch.cat([neg_idx, graph_fill])\n                            n_graph = int(graph_fill.numel())\n\n                    # A seed component can contain only gold documents. In that rare\n                    # case no valid negative continuation exists, so omitting CCMP for\n                    # this query is preferable to taking a mean over an empty B- side.\n                    if neg_idx.numel() == 0:\n                        out.append(None)\n                        continue\n\n                    # B+ is seeded from the UNRESOLVED golds under CCMP_RESIDUAL, and\n                    # from every gold otherwise.\n                    gi, ni = doc_ids[gp_use], doc_ids[neg_idx]\n                    oh = torch.zeros(N, gi.numel() + ni.numel(), device=dev)\n                    oh[gi, torch.arange(gi.numel(), device=dev)] = 1.0\n                    oh[ni, torch.arange(ni.numel(), device=dev) + gi.numel()] = 1.0\n                    # Supervision at layer l is restricted to A^(l), because\n                    # h^(l) at an unreached node carries no query information at all: it\n                    # is still the static entity embedding that early-late fusion put\n                    # there. Asking the head to predict a query-specific target from a\n                    # query-independent state is asking it to fit noise, and on this graph\n                    # it is nearly all of the sampled supervision.\n                    I, Y, C = [], [], []\n                    cand_pos = cand_neg = sel_pos = sel_neg = 0\n                    for l in range(L):\n                        x = self._ccmp_hit(oh, L - l)\n                        bp = x[:, :gi.numel()].mean(1)\n                        bn = x[:, gi.numel():].mean(1)\n                        y = bp / (bp + bn + 1e-6)\n                        # c = |B+ - B-| / (B+ + B-), NOT |2y - 1|.\n                        #\n                        # The two agree exactly wherever the node reaches something. They\n                        # differ on the case that dominates this graph: a node reaching\n                        # NEITHER set has B+ = B- = 0, so y = 0/eps = 0 and |2y-1| = 1.\n                        # The old form therefore labelled every unreachable node\n                        # \"confidently negative\" with full weight, and since the M nodes\n                        # are chosen by top-c, the supervision was almost entirely those\n                        # nodes: measured at 90% of all nodes passing at layer 1 rising to\n                        # 100% by layer 6, with ~0% gold-side. The loss reduced to \"predict\n                        # zero everywhere\", after which a mean-normalised gate returns ~1\n                        # and CCMP is a no-op. This form sends those nodes to c = 0.\n                        cc = (bp - bn).abs() / (bp + bn + 1e-6)\n                        cc = cc * reaches[l]\n                        # BALANCED NODE SELECTION. A single top-M over confidence was\n                        # dominated by error-favouring nodes (~93% on this graph). The\n                        # BCE was class-balanced afterwards, but the representation set\n                        # itself contained very few positive routes. Rank the two sides\n                        # independently so scarce gold-favouring states cannot be crowded\n                        # out before the loss sees them. y=0.5 is genuinely ambiguous and\n                        # occupies neither quota.\n                        pos_all = ((y > 0.5) & (cc > 0)).nonzero(\n                            as_tuple=True\n                        )[0]\n                        neg_all = ((y < 0.5) & (cc > 0)).nonzero(\n                            as_tuple=True\n                        )[0]\n                        kp = min(self._ccmp_m_pos, int(pos_all.numel()))\n                        kn = min(self._ccmp_m_neg, int(neg_all.numel()))\n                        cand_pos += int(pos_all.numel())\n                        cand_neg += int(neg_all.numel())\n                        sel_pos += kp\n                        sel_neg += kn\n                        pos_idx = (\n                            pos_all[cc[pos_all].topk(kp).indices]\n                            if kp > 0 else pos_all\n                        )\n                        neg_idx_nodes = (\n                            neg_all[cc[neg_all].topk(kn).indices]\n                            if kn > 0 else neg_all\n                        )\n                        idx = torch.cat([pos_idx, neg_idx_nodes])\n                        yy, csel = y[idx], cc[idx]\n                        if neutral:\n                            # IDENTITY SUPERVISION. A constant prediction across the\n                            # reached nodes makes the mean-normalised gate\n                            # ybar = (eps+yhat)/mean(eps+yhat) equal 1, hence\n                            # g = (1-eta) + eta*1 = 1 and propagation is unchanged. 0.5 is\n                            # the constant that also minimises the BCE at p = 0.5, and it\n                            # is the sentinel _ccmp_loss routes to the neutral bucket.\n                            # Full confidence, because \"do not move\" is not an ambiguous\n                            # instruction; padding stays at c = 0 and is still dropped.\n                            yy = torch.full_like(yy, 0.5)\n                            csel = torch.ones_like(csel)\n\n                        # Keep fixed [L, M_pos+M_neg] tensors for the CPU cache. Padding\n                        # has c=0 and is removed by CCMP_CMIN before the BCE, so it cannot\n                        # affect the loss or the reported supervised-node counts.\n                        cap = self._ccmp_m_pos + self._ccmp_m_neg\n                        pad = cap - idx.numel()\n                        if pad > 0:\n                            idx = torch.cat([\n                                idx,\n                                torch.zeros(pad, dtype=torch.long, device=dev),\n                            ])\n                            yy = torch.cat([yy, torch.full(\n                                (pad,), 0.5, dtype=y.dtype, device=dev\n                            )])\n                            csel = torch.cat([csel, torch.zeros(\n                                pad, dtype=cc.dtype, device=dev\n                            )])\n                        I.append(idx); Y.append(yy); C.append(csel)\n                hit = (torch.stack(I).cpu(), torch.stack(Y).half().cpu(),\n                       torch.stack(C).half().cpu(),\n                       # Counts make the reachability filter auditable in the existing\n                       # step and epoch logs without storing any document identifiers.\n                       (n_sem, n_graph, int(sem_reachable.numel()), pool_k,\n                        cand_pos, cand_neg, sel_pos, sel_neg,\n                        # residual diagnostics: is this query neutral, and how many of\n                        # its golds the semantic scorer left unresolved.\n                        bool(neutral), int(gp_use.numel()), int(gp.numel())))\n                self._ccmp_cache[key] = hit\n            out.append(hit)\n        return out\n\n    def _ccmp_loss(self, preds, targets):\n        \"\"\"Confidence-weighted BCE between yhat^(l)_v and the continuation target.\"\"\"\n        if not preds:\n            return None, {}\n        dev = preds[0].device\n        pn = preds[0].new_zeros(())      # positive-side numerator (y > 1/2)\n        nn_ = preds[0].new_zeros(())     # negative-side numerator\n        un_ = preds[0].new_zeros(())    # neutral / identity numerator (y == 1/2)\n        pd = nd = ud = 0.0\n        nsup, csum, npos = 0, 0.0, 0\n        nquery = n_sem = n_graph = n_neg = n_pool_reach = n_pool = 0\n        n_cand_pos = n_cand_neg = n_sel_pos = n_sel_neg = 0\n        n_neutral = n_unres_gold = n_gold = 0\n        for b, hit in enumerate(targets):\n            if hit is None:\n                continue\n            idx, y, c, meta = hit\n            # 11-tuple under CCMP_RESIDUAL, 8-tuple on a cache written before it. Reading\n            # both keeps a warm _ccmp_cache from an earlier cell usable.\n            (sem_count, graph_count, pool_reach, pool_count,\n             cand_pos, cand_neg, sel_pos, sel_neg) = meta[:8]\n            is_neutral, n_used, n_all = (meta[8:] if len(meta) >= 11\n                                         else (False, 0, 0))\n            n_neutral += int(bool(is_neutral))\n            n_unres_gold += (0 if is_neutral else n_used)\n            n_gold += n_all\n            nquery += 1\n            n_sem += sem_count\n            n_graph += graph_count\n            n_neg += sem_count + graph_count\n            n_pool_reach += pool_reach\n            n_pool += pool_count\n            n_cand_pos += cand_pos\n            n_cand_neg += cand_neg\n            n_sel_pos += sel_pos\n            n_sel_neg += sel_neg\n            for l in range(min(len(preds), idx.shape[0])):\n                ii = idx[l].to(dev)\n                yy = y[l].to(dev).float()\n                cc = c[l].to(dev).float()\n                keep = cc > self._ccmp_cmin\n                if not bool(keep.any()):\n                    continue\n                ii, yy, cc = ii[keep], yy[keep], cc[keep]\n                p = preds[l][b, ii].float().clamp(1e-6, 1 - 1e-6)\n                bce = -(yy * p.log() + (1 - yy) * (1 - p).log())\n                # CLASS-BALANCED. Even after the confidence fix the reached targets run\n                # ~93% negative on this graph, so an unbalanced mean is minimised by\n                # \"predict 0 everywhere\" -- the same degenerate solution by a slower route.\n                # Each side is normalised by its own confidence mass and the two are\n                # averaged, so the head cannot buy the loss down by collapsing.\n                # THREE BUCKETS, NOT TWO. y == 0.5 is the identity sentinel written by\n                # _ccmp_targets for queries the semantic scorer already resolved. Left in\n                # the negative bucket it would read as \"predict 0\", which is the collapse\n                # the class balance exists to prevent, and it would drag the gate down on\n                # exactly the same-field queries this change is meant to protect.\n                # Ordinary targets never land on 0.5: pos_all/neg_all are strict.\n                u = yy == 0.5\n                m = yy > 0.5\n                if bool(m.any()):\n                    pn = pn + (cc[m] * bce[m]).sum(); pd += float(cc[m].sum())\n                    npos += int(m.sum())\n                neg = (~m) & (~u)\n                if bool(neg.any()):\n                    nn_ = nn_ + (cc[neg] * bce[neg]).sum(); nd += float(cc[neg].sum())\n                if bool(u.any()):\n                    un_ = un_ + (cc[u] * bce[u]).sum(); ud += float(cc[u].sum())\n                nsup += int(ii.numel())\n                csum += float(cc.sum())\n        # Mean over whichever buckets carry mass, each normalised by its own confidence.\n        # With no neutral targets this is 0.5*(pn/pd) + 0.5*(nn_/nd), identical to before.\n        # CCMP_IDENTITY_W=0 drops the identity term, isolating the restriction to\n        # unresolved golds from the instruction to stand still.\n        terms, weights = [], []\n        if pd > 0:\n            terms.append(pn / pd); weights.append(1.0)\n        if nd > 0:\n            terms.append(nn_ / nd); weights.append(1.0)\n        if ud > 0 and self._ccmp_id_w > 0:\n            terms.append(un_ / ud); weights.append(self._ccmp_id_w)\n        if not terms:\n            return None, {}\n        wsum = sum(weights)\n        loss = sum(w * t for w, t in zip(weights, terms)) / wsum\n        return loss, {\"ccmp_nodes\": nsup / max(len(targets), 1),\n                      \"ccmp_conf\": csum / max(nsup, 1),\n                      # share of supervised nodes on the gold side. If this is ~0 the\n                      # target has collapsed and nothing downstream can work.\n                      \"ccmp_pos\": npos / max(nsup, 1),\n                      \"ccmp_nodes_pos\": npos / max(nquery, 1),\n                      \"ccmp_nodes_neg\": (nsup - npos) / max(nquery, 1),\n                      # Pre-selection prevalence remains the collapse diagnostic now\n                      # that the selected representation set is deliberately balanced.\n                      \"ccmp_cand_pos\": n_cand_pos / max(nquery, 1),\n                      \"ccmp_cand_neg\": n_cand_neg / max(nquery, 1),\n                      \"ccmp_cand_pos_frac\": n_cand_pos / max(\n                          n_cand_pos + n_cand_neg, 1\n                      ),\n                      \"ccmp_sel_pos\": n_sel_pos / max(nquery, 1),\n                      \"ccmp_sel_neg\": n_sel_neg / max(nquery, 1),\n                      \"ccmp_neg_sem\": n_sem / max(nquery, 1),\n                      \"ccmp_neg_graph\": n_graph / max(nquery, 1),\n                      \"ccmp_neg_count\": n_neg / max(nquery, 1),\n                      \"ccmp_pool_reach\": n_pool_reach / max(n_pool, 1),\n                      # RESIDUAL DIAGNOSTICS. Read these first on a residual run.\n                      # ccmp_resolved is the share of queries the semantic scorer already\n                      # ranks perfectly, i.e. the share on which CCMP is now a supervised\n                      # no-op. If it is ~0 the change cannot do anything and the arm is\n                      # the old CCMP; if it is ~1 nothing is being corrected.\n                      # ccmp_unres_frac is the share of golds still contested on the\n                      # remaining queries, i.e. how much narrower B+ has become.\n                      \"ccmp_resolved\": n_neutral / max(nquery, 1),\n                      \"ccmp_unres_gold\": n_unres_gold / max(nquery - n_neutral, 1),\n                      \"ccmp_unres_frac\": n_unres_gold / max(n_gold, 1)}\n\n    # ------------------------------------------------------------------ CQIG plumbing\n    #\n    # PROTOCOL. One bank of M source problems is chosen ONCE, from the training queries, and\n    # stored on the model in a graph-independent form (frozen question embedding + start-node\n    # NAMES). Every graph, including graphs the model has never seen, is calibrated by\n    # re-linking those SAME problems against its own vocabulary. Nothing is ever read from\n    # the target corpus's query set, so the evaluation is not transductive.\n    @staticmethod\n    def _cqig_key(task_dataset):\n        \"\"\"Identify a graph. NOT the node count: two corpora could share one.\"\"\"\n        return getattr(task_dataset, \"name\", None) or f\"graph@{id(task_dataset)}\"\n\n    def _cqig_gate(self):\n        m = getattr(self, \"model\", None)\n        return getattr(m, \"cqig\", None) if m is not None else None\n\n    @staticmethod\n    def _cqig_vocab(ds, which):\n        v = getattr(ds, f\"cqig_{which}\", None)\n        assert v, (\n            f\"task dataset {getattr(ds, 'name', ds)!r} carries no {which}; CQIG re-links its \"\n            f\"reference problems by entity NAME, so it needs the graph's vocabulary. \"\n            f\"_create_task_dataset attaches it.\")\n        return v\n\n    @staticmethod\n    def _cqig_node_emb(ds):\n        \"\"\"This graph's frozen node-text embeddings, or None if it was indexed without them.\n\n        `graph.x` is what the indexer wrote by encoding every node name with the same frozen\n        text encoder the questions went through, so a seed name and a target node name are\n        already in one space and no encoder has to be carried to inference time.\n        \"\"\"\n        graph = getattr(ds, \"graph\", None)\n        return getattr(graph, \"x\", None) if graph is not None else None\n\n    @staticmethod\n    def _cqig_refs_from_batch(batch, id2node, node_emb=None):\n        \"\"\"Split a loader batch into PORTABLE per-query references.\n\n        Portable means nothing in it is sized to, or indexed by, this graph: the question\n        embedding is frozen text, and the seeds are the entity names the indexer looked up\n        in node2id, recovered here from the mask through id2node together with their\n        attachment weights. One entry per QUERY, never per batch.\n\n        `node_emb` is the SOURCE graph's node features. When given, each seed also carries\n        its frozen embedding, which is what the semantic linker matches against a target\n        vocabulary. It is a property of the seed's text under a frozen encoder, not of the\n        source graph's topology, so it travels as cleanly as the name does.\n        \"\"\"\n        e = batch[\"question_embeddings\"].detach().float().cpu()\n        m = batch[\"start_nodes_mask\"].detach().float().cpu()\n        ids = batch.get(\"id\")\n        out = []\n        for k in range(e.shape[0]):\n            nz = torch.nonzero(m[k], as_tuple=False).flatten().tolist()\n            rows = [j for j in nz if j in id2node]\n            seeds = [(id2node[j], float(m[k, j])) for j in rows]\n            se = None\n            if node_emb is not None and rows:\n                # Same order as `seeds`; set_reference_bank asserts on that pairing.\n                se = node_emb[torch.tensor(rows, dtype=torch.long)].detach().float().cpu()\n            out.append({\"qemb\": e[k:k + 1].clone(), \"seeds\": seeds, \"seed_emb\": se,\n                        \"id\": (str(ids[k]) if ids is not None else None)})\n        return out\n\n    def _cqig_link_floor(self, refs, node_emb):\n        \"\"\"Choose the semantic linker's rejection threshold from SOURCE data only.\n\n        The question the floor has to answer is \"how similar are two node names that mean\n        the same thing, in THIS encoder's geometry?\", and that is answerable without any\n        target corpus. For every reference seed, take its nearest OTHER node in the source\n        graph: those are the encoder's own near-synonyms, at the scale this graph's\n        vocabulary actually produces. The floor is a quantile of that distribution.\n\n        Deliberately not a round number like 0.7. A hand-set constant is a hyperparameter\n        tuned on whatever corpus it was first tried on, and it would not survive a change of\n        encoder; this rescales with the encoder because it is measured in it.\n        \"\"\"\n        emb = torch.cat([r[\"seed_emb\"] for r in refs if r.get(\"seed_emb\") is not None], 0)\n        if emb.numel() == 0:\n            return 0.0, 0\n        # top-2: the first is the seed's own node at cosine 1, the second is its neighbour.\n        cos, _ = cqig_mod.topk_cosine(emb.to(node_emb.device), node_emb, 2)\n        nn_cos = cos[:, 1].float().cpu()\n        q = min(max(self._cqig_link_q, 0.0), 1.0)\n        return float(torch.quantile(nn_cos, q)), int(nn_cos.numel())\n\n    @staticmethod\n    def _cqig_farthest_point(refs, m):\n        \"\"\"Pick m maximally spread references by farthest-point on question embeddings.\"\"\"\n        embs = torch.cat([r[\"qemb\"] for r in refs], 0)\n        embs = embs / embs.norm(dim=-1, keepdim=True).clamp_min(1e-6)\n        picked = [0]\n        while len(picked) < min(m, len(refs)):\n            d = (1.0 - embs @ embs[picked].T).min(dim=1).values\n            d[torch.tensor(picked)] = -1.0\n            picked.append(int(d.argmax()))\n        return [refs[i] for i in picked]\n\n    def _cqig_select_reference(self, batch, ds):\n        \"\"\"Fill a pool of individual TRAINING queries, then freeze M of them as the bank.\n\n        The bank is handed to the MODEL, not kept on the trainer, so it rides in the\n        checkpoint and a reloaded model can calibrate a corpus the trainer never saw.\n        \"\"\"\n        g = self._cqig_gate()\n        if g is None:\n            return False\n        if g.ref_bank:\n            return True\n        id2node = self._cqig_vocab(ds, \"id2node\")\n        # SOURCE node features. Only needed for CQIG_LINK=semantic, but stored whenever the\n        # graph has them: the bank rides in the checkpoint, and a bank frozen without seed\n        # embeddings cannot be linked semantically later without retraining.\n        src_emb = self._cqig_node_emb(ds)\n        self._cqig_pool = getattr(self, \"_cqig_pool\", [])\n        self._cqig_pool.extend(self._cqig_refs_from_batch(batch, id2node, src_emb))\n        if len(self._cqig_pool) < self._cqig_pool_size:\n            return False\n        pool = self._cqig_pool[: self._cqig_pool_size]\n        refs = self._cqig_farthest_point(pool, self._cqig_m)\n        # EXACTLY M, OR SAY SO. Every graph is calibrated from this one bank, so M is the\n        # same on all of them by construction -- but only if the bank really holds M.\n        assert len(refs) == self._cqig_m, (\n            f\"asked for M={self._cqig_m} reference queries but the pool of \"\n            f\"{len(pool)} yielded {len(refs)}; raise CQIG_POOL above CQIG_M\")\n        # THE FLOOR IS FROZEN WITH THE BANK, on the source graph, before any target graph\n        # exists. Measuring it later against a target would make the threshold a function\n        # of the corpus being transferred to, which is exactly what zero-shot forbids.\n        floor, n_meas = 0.0, 0\n        if self._cqig_link == \"semantic\":\n            assert src_emb is not None, (\n                \"CQIG_LINK=semantic needs the source graph's node embeddings (graph.x), \"\n                \"which are absent; this dataset was indexed without node features\")\n            if self._cqig_link_min == \"auto\":\n                floor, n_meas = self._cqig_link_floor(refs, src_emb.to(self.device))\n            else:\n                floor = float(self._cqig_link_min)\n        g.set_reference_bank(refs, meta={\n            \"m\": len(refs), \"pool\": self._cqig_pool_size,\n            \"source\": self._cqig_key(ds),\n            \"selection\": \"farthest-point on question embeddings, per query\",\n            \"link\": self._cqig_link, \"link_k\": self._cqig_link_k,\n            \"link_temp\": self._cqig_link_temp, \"link_floor\": floor})\n        self._cqig_pool = []\n        seeds = [len(r[\"seeds\"]) for r in refs]\n        print(f\"[cqig] source bank frozen: M={len(refs)} individual training queries from \"\n              f\"'{self._cqig_key(ds)}' (pool {self._cqig_pool_size}); \"\n              f\"seeds/query min={min(seeds)} median={sorted(seeds)[len(seeds) // 2]} \"\n              f\"max={max(seeds)}; stored on the model\", flush=True)\n        if self._cqig_link == \"semantic\":\n            how = (f\"source nearest-neighbour cosine, q={self._cqig_link_q:g} over \"\n                   f\"{n_meas} seeds\" if self._cqig_link_min == \"auto\"\n                   else f\"pinned by CQIG_LINK_MIN={self._cqig_link_min}\")\n            print(f\"[cqig] link=semantic: K={self._cqig_link_k} T={self._cqig_link_temp:g} \"\n                  f\"floor={floor:.4f} [{how}]; seed weight is split by softmax(cos/T) over \"\n                  f\"the survivors, so each seed's total attachment weight is unchanged\",\n                  flush=True)\n        return True\n\n    def _cqig_calibrate(self, graph, ds, why):\n        \"\"\"Re-link the source bank to THIS graph, calibrate, then measure what the gate does.\"\"\"\n        g = self._cqig_gate()\n        key = self._cqig_key(ds)\n        node2id = self._cqig_vocab(ds, \"node2id\")\n        # The floor travels with the BANK, not with the trainer: a checkpoint calibrated on\n        # a new corpus must reject at the same threshold the bank was frozen under, even in\n        # a process where CQIG_LINK_MIN was never set.\n        meta = g.ref_meta or {}\n        batches, cov = g.materialise(\n            node2id, int(graph.num_nodes), device=self.device,\n            batch_size=self._cqig_ref_batch,\n            node_emb=(graph.x if self._cqig_link == \"semantic\" else None),\n            link=self._cqig_link, link_k=self._cqig_link_k,\n            link_temp=meta.get(\"link_temp\", self._cqig_link_temp),\n            link_floor=meta.get(\"link_floor\", 0.0))\n        # SAME PRECISION CONTEXT AS EVERY OTHER FORWARD IN THIS TRAINER. Training, eval and\n        # predict all wrap their forwards in autocast. The train-side calibration inherited\n        # that by accident, being called from inside the training loop; the inference-side\n        # call sits outside it, so float32 activations met bfloat16 weights.\n        with torch.amp.autocast(device_type=self.device.type, dtype=self.dtype,\n                                enabled=self.use_amp):\n            rep = self.model.cqig_calibrate(graph, key, ref_batches=batches,\n                                            device=self.device,\n                                            rounds=self._cqig_rounds)\n        print(f\"[cqig] {why}: graph='{key}' M={cov['queries']} queries \"\n              f\"({cov['batches']} forwards x {self._cqig_rounds} round(s) x 2 passes) \"\n              f\"seed-coverage={cov['seed_coverage']:.1%} \"\n              f\"({cov['seeds_found']}/{cov['seeds_total']}) \"\n              f\"queries_with_no_seed={cov['queries_with_no_seed']} \"\n              f\"stats={g.stats_bytes() / 2**20:.0f} MB\", flush=True)\n        # THE SPLIT, ALWAYS. `exact` is directly comparable to every run made before the\n        # semantic linker existed, so the two arms can be read against each other rather\n        # than against an aggregate that moved for an unstated reason.\n        if cov[\"link\"] == \"semantic\":\n            how = (f\"K={cov['link_k']} floor={cov['link_floor']:.4f}, median best cos \"\n                   f\"{cov['sem_cos_median']:.3f}, {cov['sem_nodes_per_seed']:.2f} nodes/seed\"\n                   if cov[\"seeds_semantic\"] else\n                   f\"K={cov['link_k']} floor={cov['link_floor']:.4f}, none needed\")\n            print(f\"[cqig]   linking: exact {cov['seeds_exact']} \"\n                  f\"({cov['exact_coverage']:.1%})  + semantic {cov['seeds_semantic']} \"\n                  f\"({how})  = {cov['seeds_found']}/{cov['seeds_total']}; \"\n                  f\"unmatched {cov['seeds_unmatched']}\", flush=True)\n        # EVERY GATED LAYER, not just the first. With a multi-layer arm the interesting\n        # failure is one layer doing the work and the rest sitting at a constant.\n        for j in sorted(rep):\n            r = rep[j]\n            v, iq, gq = r[\"var_q\"], r[\"I\"], r[\"gate\"]\n            print(f\"[cqig]   layer {j + 1}/{g.n_layers}: nodes={r['nodes']} \"\n                  f\"responding={r['live']} ({r['responding_frac']:.2%})  \"\n                  f\"V p10/p50/p90 = {v['p10']:.4e} {v['p50']:.4e} {v['p90']:.4e}  \"\n                  f\"scale={r['scale']:.4e}  tau_ref={r['tau_ref']:.4e} \"\n                  f\"[{r['tau_src']}]\", flush=True)\n            print(f\"[cqig]     I    min/p10/p50/p90/max = {iq['min']:.4e} {iq['p10']:.4e} \"\n                  f\"{iq['p50']:.4e} {iq['p90']:.4e} {iq['max']:.4e}  \"\n                  f\"std={iq['std']:.4e}  responded={r['resp_frac']:.2%} over \"\n                  f\"{r['sample_queries']} reference queries\", flush=True)\n            print(f\"[cqig]     g    min/p10/p50/p90/max = {gq['min']:.4e} {gq['p10']:.4e} \"\n                  f\"{gq['p50']:.4e} {gq['p90']:.4e} {gq['max']:.4e}  \"\n                  f\"std={gq['std']:.4e}  span={gq['max'] - gq['min']:.4e}\", flush=True)\n            # THE COEFFICIENT THE ARM ACTUALLY APPLIES. Identical to g under op=\"gate\", so\n            # the line is only printed when it would say something different -- and there it\n            # is the one that matters, because g is not what multiplies anything.\n            if r.get(\"mu_source\", \"ref\") == \"zero\":\n                # The ablation must be visible in the calibration block itself, not only in\n                # the startup banner, because that block is what gets pasted around.\n                print(f\"[cqig]     mu=ZERO ABLATION: mu is 0 on all {r['mu_zero']} nodes, \"\n                      f\"so I = ||h||^2/scale and the reference bank contributes nothing to \"\n                      f\"this arm. Its score is the magnitude-gate floor, not the method.\",\n                      flush=True)\n            if r.get(\"op\", \"gate\") != \"gate\":\n                cq = r[\"coef\"]\n                print(f\"[cqig]     op={r['op']}: h - c*mu, c min/p10/p50/p90/max = \"\n                      f\"{cq['min']:.4e} {cq['p10']:.4e} {cq['p50']:.4e} {cq['p90']:.4e} \"\n                      f\"{cq['max']:.4e}   mu=0 on {r['mu_zero']} nodes \"\n                      f\"({r['mu_zero_frac']:.2%}), which this op cannot touch at all\",\n                      flush=True)\n            print(f\"[cqig]     lam={r['lam']:.6e} alpha={r['alpha']:.6e} \"\n                  f\"tau={r['tau']:.6e} gate_at_I=0 {r['gate_unreached']:.4e}\", flush=True)\n            if r[\"degenerate\"]:\n                print(f\"[cqig]     WARNING: no node responded to any reference query at \"\n                      f\"layer {j + 1}; the scale fell back to 1.0 and I is meaningless.\",\n                      flush=True)\n            elif gq[\"max\"] - gq[\"min\"] < 1e-3:\n                print(f\"[cqig]     WARNING: the gate spans {gq['max'] - gq['min']:.2e} \"\n                      f\"across nodes. That is a near-constant rescaling, not a gate, and \"\n                      f\"this arm will read as a no-op whatever the final metrics say.\",\n                      flush=True)\n        if cov[\"seed_coverage\"] < 0.2:\n            fix = (\"\" if cov[\"link\"] == \"semantic\" else\n                   \" Exact name lookup is the likely cause, not the corpus: set \"\n                   \"CQIG_LINK=semantic to link the remainder by embedding.\")\n            print(f\"[cqig]   WARNING: only {cov['seed_coverage']:.1%} of the source seeds \"\n                  f\"resolve in '{key}'. The reference problems barely touch this graph, so \"\n                  f\"its statistics are not measuring a response to them.{fix}\", flush=True)\n        elif cov[\"link\"] == \"semantic\" and cov[\"exact_coverage\"] < 0.2:\n            print(f\"[cqig]   note: exact lookup alone would have reached \"\n                  f\"{cov['exact_coverage']:.1%} on '{key}'; the semantic linker carried it \"\n                  f\"to {cov['seed_coverage']:.1%}. That gap is what this arm is testing.\",\n                  flush=True)\n        gb = g.stats_bytes() / 2**30\n        if gb > 4.0:\n            print(f\"[cqig]   WARNING: the reference statistics now hold {gb:.1f} GiB of \"\n                  f\"GPU memory ({len(g.gate_layers)} gated layers x \"\n                  f\"{len(g.known_graphs())} graphs). mu is one float32 row per node per \"\n                  f\"gated layer, so gating fewer layers is the lever if this run runs out \"\n                  f\"of memory.\", flush=True)\n        return key\n\n    def _cqig_report_live(self, why):\n        \"\"\"What the gate did on the queries that were actually scored, plus the AUC that\n        decides whether informativeness has anything to do with relevance.\"\"\"\n        g = self._cqig_gate()\n        if g is None or not g.recording():\n            return\n        rep = g.live_report()\n        if not rep:\n            g.stop_live()\n            return\n        for j in sorted(rep):\n            r = rep[j]\n            what = \"gate\" if r.get(\"op\", \"gate\") == \"gate\" else f\"{r['op']} coef\"\n            print(f\"[cqig] {why} {what}, layer {j + 1}: n={r['n']:.4e}  \"\n                  f\"min/p10/p50/p90/max = {r['min']:.4e} {r['p10']:.4e} {r['p50']:.4e} \"\n                  f\"{r['p90']:.4e} {r['max']:.4e}  mean={r['mean']:.4e} \"\n                  f\"std={r['std']:.4e}  span={r['max'] - r['min']:.4e}  \"\n                  f\"lam={float(g.lam(j).detach()):.6e} \"\n                  f\"alpha={float(g.alpha(j).detach()):.6e} \"\n                  f\"tau={g.tau(j):.6e}\", flush=True)\n            # HOW BIG THE EDIT WAS, measured on the states. A coefficient far from 1 that\n            # moves the state by 1e-3 is a no-op wearing a large number, and with\n            # `layer_norm: yes` downstream that is the failure worth being able to see.\n            print(f\"[cqig]   edit ||h~-h||/||h||: mean={r['edit_rel_mean']:.4e} \"\n                  f\"max={r['edit_rel_max']:.4e} on {r['edit_frac']:.2%} of live nodes\",\n                  flush=True)\n            if r[\"auc_n\"]:\n                print(f\"[cqig]   informativeness AUC, gold vs semantic-top-\"\n                      f\"{self._cqig_hn_k}: {r['auc']:.4f} over {r['auc_n']} queries  \"\n                      f\"(mean I gold {r['I_gold']:.4e} vs negative {r['I_neg']:.4e})\",\n                      flush=True)\n                if abs(r[\"auc\"] - 0.5) < 0.02:\n                    print(f\"[cqig]   NOTE: an AUC of {r['auc']:.4f} says a high I is no \"\n                          f\"likelier on a gold paper than on a hard negative. The gate's \"\n                          f\"premise, not its calibration, is what is failing.\", flush=True)\n        g.stop_live()\n\n    def evaluate(self) -> dict:\n        \"\"\"Measure the gate on the real evaluation queries, around the inherited eval.\n\n        Reset before and report after, so the numbers belong to one evaluation rather than\n        accumulating across epochs. Purely observational: `reset_live` switches on a\n        histogram and a label-side AUC, neither of which is read by any calibration.\n        \"\"\"\n        g = self._cqig_gate()\n        if g is not None and g.ref_bank:\n            g.reset_live()\n        m = super().evaluate()\n        self._cqig_report_live(\"eval\")\n        return m\n\n    def predict(self) -> dict:\n        \"\"\"Same measurement around the pass that writes the predictions file.\n\n        The AUC stays empty here, because prediction batches carry no gold mask; the gate\n        distribution is the part that matters, since these are the scores the benchmark\n        numbers are computed from.\n        \"\"\"\n        g = self._cqig_gate()\n        if g is not None and g.ref_bank:\n            g.reset_live()\n        out = super().predict()\n        self._cqig_report_live(\"predict\")\n        return out\n\n    # ------------------------------------------------------------ path interpretation\n    @torch.no_grad()\n    def _channel_scores(self, graph, batch):\n        \"\"\"One forward; returns doc node ids and the four per-document score vectors\n        (fused, graph-alone raw, multi-view scorer, Qwen3 cosine), all in nodes.csv doc order.\"\"\"\n        with torch.amp.autocast(device_type=self.device.type, dtype=self.dtype, enabled=self.use_amp):\n            pred = self.model(graph, batch)\n        did = self.model._doc_ids\n        fused = pred[0, did].float()\n        graph_raw = self.model._raw_doc[0].float()\n        sem = self.model._s_op[0].float()\n        tag, row = self.model._sem_row[str(batch[\"id\"][0])]\n        tab = self.model._sem_tab[tag]\n        dense = tab[\"dense\"][row].float().to(fused.device)\n        return did, {\"fused\": fused, \"graph\": graph_raw, \"scorer\": sem, \"dense\": dense}, (tab, row)\n\n    def interpret(self, qids, out_path, probes_path=None, num_beam=10, path_topk=5,\n                  max_golds=2, top_views=3, do_paths=True, golds=None):\n        \"\"\"Path interpretations, NBFNet-style, for the graph channel of the fusion model.\n\n        For each requested query: the rank of every gold under each channel (fused, graph\n        alone, multi-view scorer, raw Qwen3 cosine), the top-k paths from the query's seed\n        frames to its best-ranked gold (beam search over the gradient of the graph score\n        w.r.t. each layer's edge weights, exactly the GFM-RAG / NBFNet recipe), and along\n        every path the CCMP responsibility of each hop's sender node at the layer it was\n        used, normalised by that layer's frontier mean (i.e. the gate the model applied).\n        Also the scorer's views that matched the gold best, with their probe text.\n        \"\"\"\n        import numpy as np\n        self.model.eval()\n        em = self.model.base.entity_model\n        em.num_beam, em.path_topk = int(num_beam), int(path_topk)\n        eta = float(getattr(em, \"resp_eta\", 0.5))\n        probes = {}\n        if probes_path and os.path.exists(probes_path):\n            for line in open(probes_path):\n                if line.strip():\n                    r = json.loads(line); probes[str(r[\"id\"])] = r.get(\"probes\", [])\n        want = [str(q) for q in qids]\n        results = []\n        for test_dataset in self.eval_graph_dataset_loader:\n            src = test_dataset.data\n            graph = src.graph.to(self.device)\n            data = src.test_data\n            id2node = src.id2node\n            id2rel = {v: k for k, v in src.rel2id.items()}\n            raw = {str(x[\"id\"]): x for x in src.raw_test_data}\n            pos = {}\n            for i in range(len(data)):\n                sid = str(data[i][\"id\"])\n                if sid in want: pos[sid] = i\n            missing = [q for q in want if q not in pos]\n            if missing:\n                print(f\"[interpret] {len(missing)} requested ids not in {test_dataset.name}: {missing[:5]}\")\n            for sid in want:\n                if sid not in pos: continue\n                item = data[pos[sid]]\n                batch = {\"question_embeddings\": item[\"question_embeddings\"].unsqueeze(0).to(self.device),\n                         \"start_nodes_mask\": item[\"start_nodes_mask\"].unsqueeze(0).to(self.device),\n                         \"target_nodes_mask\": item[\"target_nodes_mask\"].unsqueeze(0).to(self.device),\n                         \"id\": [sid]}\n                did, ch, (tab, row) = self._channel_scores(graph, batch)\n                n_doc = did.numel()\n                gold_pos = (batch[\"target_nodes_mask\"][0, did] > 0).nonzero(as_tuple=True)[0]\n                doc_name = [id2node[int(did[j])] for j in gold_pos.tolist()]\n                ranks = {}\n                for k, v in ch.items():\n                    ranks[k] = {doc_name[a]: int((v > v[j]).sum().item()) + 1 for a, j in enumerate(gold_pos.tolist())}\n                # `golds` pins the documents to interpret (e.g. the one gold the picker chose); otherwise\n                # the query's best-ranked golds under the fused score.\n                pinned = set((golds or {}).get(sid, []))\n                forced = [j for j in gold_pos.tolist() if id2node[int(did[j])] in pinned]\n                best = forced if forced else sorted(gold_pos.tolist(), key=lambda j: ranks[\"fused\"][id2node[int(did[j])]])[:max_golds]\n                # scorer views for each gold\n                Hq = np.asarray(tab[\"H\"][row])[:, tab[\"col\"]].astype(np.float32)   # [J, n_doc]\n                vmask = tab[\"mask\"][row].numpy() > 0\n                seeds = batch[\"start_nodes_mask\"][0].nonzero(as_tuple=True)[0].tolist()\n                rec = {\"id\": sid, \"question\": raw.get(sid, {}).get(\"question\", \"\"),\n                       \"stratum\": raw.get(sid, {}).get(\"stratum\"), \"golds\": doc_name, \"ranks\": ranks,\n                       \"n_doc\": int(n_doc), \"seeds\": [id2node[x] for x in seeds], \"targets\": []}\n                for j in best:\n                    gname = id2node[int(did[j])]\n                    tv = [(int(a), float(Hq[a, j])) for a in np.argsort(-Hq[:, j]) if vmask[a]][:top_views]\n                    views = [{\"view\": a, \"match\": m, \"text\": (probes.get(sid, [None] * (a + 1))[a] if a < len(probes.get(sid, [])) else None)}\n                             for a, m in tv]\n                    if not do_paths:        # scan mode: ranks and views only, no gradient beam search\n                        rec[\"targets\"].append({\"doc\": gname, \"rank\": {k: ranks[k][gname] for k in ranks},\n                                               \"dense_cos\": float(ch[\"dense\"][j]), \"views\": views,\n                                               \"frontier_mean_resp\": [], \"paths\": []})\n                        continue\n                    # ---- paths: gradient beam search on the GRAPH channel's score for this gold\n                    sample = dict(batch); tm = torch.zeros_like(batch[\"target_nodes_mask\"]); tm[0, int(did[j])] = 1.0\n                    sample[\"target_nodes_mask\"] = tm\n                    if getattr(em, \"resp_proj\", None) is not None:\n                        em._ccmp_seeds = sample[\"start_nodes_mask\"]\n                    em._keep_reach = True\n                    with torch.enable_grad(), torch.amp.autocast(device_type=self.device.type, dtype=self.dtype, enabled=self.use_amp):\n                        pr = self.model.base.visualize(graph, sample)\n                    em._keep_reach = False\n                    rp = [x[0].detach().float() for x in getattr(em, \"_resp_pred\", [])]\n                    reach = getattr(em, \"_reach_layers\", [])\n                    fm = []\n                    for l, y in enumerate(rp):\n                        r_ = reach[l] if l < len(reach) and reach[l] is not None else None\n                        fm.append(float(y[r_[0].bool()].mean()) if r_ is not None and r_.sum() > 0 else float(y.mean()))\n                    paths_out = []\n                    paths, weights = pr.get(int(did[j]), ([], []))\n                    for path, w in zip(paths, weights):\n                        hops = []\n                        for l, (h, t, r) in enumerate(path):\n                            hop = {\"layer\": l, \"head\": id2node[h], \"rel\": id2rel.get(r, str(r)), \"tail\": id2node[t]}\n                            if l < len(rp):\n                                y = float(rp[l][h]); hop[\"resp\"] = y; hop[\"frontier_mean\"] = fm[l]\n                                hop[\"gate\"] = (1 - eta) + eta * (y / max(fm[l], 1e-6))\n                                if l < len(reach) and reach[l] is not None:\n                                    hop[\"reached\"] = bool(reach[l][0, h] > 0)\n                            hops.append(hop)\n                        paths_out.append({\"weight\": float(w), \"hops\": hops})\n                    rec[\"targets\"].append({\"doc\": gname, \"rank\": {k: ranks[k][gname] for k in ranks},\n                                           \"dense_cos\": float(ch[\"dense\"][j]), \"views\": views,\n                                           \"frontier_mean_resp\": fm, \"paths\": paths_out})\n                    print(f\"[interpret] {sid} -> {gname}: ranks {rec['targets'][-1]['rank']} | {len(paths_out)} paths\")\n                if not do_paths and len(results) % 25 == 0:\n                    print(f\"[interpret] scanned {len(results) + 1}/{len(want)}\", flush=True)\n                results.append(rec)\n        os.makedirs(os.path.dirname(os.path.abspath(out_path)), exist_ok=True)\n        json.dump(results, open(out_path, \"w\"), indent=1)\n        print(f\"[interpret] wrote {len(results)} queries -> {out_path}\")\n        return results\n\n    def _cqig_maybe_calibrate(self, graph, batch, task_dataset):\n        \"\"\"Called from train_step. Keeps the TRAIN graph's statistics current; mu drifts as\n        the model trains, so it is recomputed every CQIG_RECAL steps.\"\"\"\n        g = self._cqig_gate()\n        if g is None:\n            return\n        self._cqig_step = getattr(self, \"_cqig_step\", 0) + 1\n        if not self._cqig_select_reference(batch, task_dataset):\n            g.mode = \"off\"                          # no bank yet: run ungated, not wrongly\n            return\n        key = self._cqig_key(task_dataset)\n        due = (not g.calibrated(key)) or (\n            self._cqig_recal > 0 and self._cqig_step % self._cqig_recal == 0)\n        if due:\n            self._cqig_train_key = self._cqig_calibrate(graph, task_dataset, \"train\")\n        else:\n            self.model.cqig_use_graph(key)          # eval may have left another graph active\n\n    def _cqig_for_inference(self, task_dataset):\n        \"\"\"Calibrate the graph about to be scored, from the SOURCE bank re-linked to it.\n\n        This is the zero-shot path: the target corpus supplies its vocabulary and its graph,\n        and nothing else. Its own queries are never inspected.\n        \"\"\"\n        g = self._cqig_gate()\n        if g is None:\n            return\n        if not g.ref_bank:\n            # The bank is frozen a few hundred training steps in. An evaluation before that\n            # must run UNGATED: the only calibrated graph is the train graph, and its mu has\n            # the wrong number of rows for this one.\n            g.mode = \"off\"\n            return\n        graph = task_dataset.graph.to(self.device)\n        key = self._cqig_key(task_dataset)\n        due = (not g.calibrated(key)) or (\n            self._cqig_recal > 0\n            and getattr(self, \"_cqig_infer_at\", -1) != getattr(self, \"_cqig_step\", 0))\n        if due:\n            self._cqig_calibrate(graph, task_dataset, \"inference (source refs re-linked)\")\n            self._cqig_infer_at = getattr(self, \"_cqig_step\", 0)\n        else:\n            self.model.cqig_use_graph(key)\n\n    def _create_task_dataset(self, dataset, is_train=True, **kw):\n        \"\"\"The one seam that train, evaluate AND predict all pass through.\n\n        Evaluation and prediction run on a different graph from training, and mu has one\n        row per node, so scoring the test graph against the train graph's statistics is not\n        merely wrong but shape-invalid. Calibrating here means every consumer gets the\n        right graph's statistics without each one needing its own hook.\n        \"\"\"\n        ds = super()._create_task_dataset(dataset, is_train=is_train, **kw)\n        g = self._cqig_gate()\n        if g is not None:\n            # The vocabulary rides along so CQIG can re-link its source problems by NAME.\n            src = getattr(dataset, \"data\", dataset)\n            ds.cqig_node2id = getattr(src, \"node2id\", None)\n            ds.cqig_id2node = getattr(src, \"id2node\", None)\n            if is_train:\n                self._cqig_train_key = self._cqig_key(ds)\n            else:\n                self._cqig_for_inference(ds)\n        return ds\n\n    def train_step(self, batch, task_dataset):\n        graph = task_dataset.graph.to(self.device)\n        batch = query_utils.cuda(batch, device=self.device)\n        self._cqig_maybe_calibrate(graph, batch, task_dataset)\n\n        pred = self.parallel_model(graph, batch)          # FUSED [B, N]\n        target = batch[\"target_nodes_mask\"]\n\n        total = torch.tensor(0.0, device=self.device, requires_grad=True)\n        step_metrics = {}\n\n        if self._objective == \"hardneg\":\n            did = self.model._doc_ids\n            # Whichever scorer the model's `semantic` setting selected: handcrafted\n            # operator or the learned sorted-MLP. Both contrastive terms below mine\n            # their negatives from it, so the graph is always trained to fix the misses\n            # of the scorer actually in use.\n            s_op = self.model._s_op                        # [B, n_doc] detached\n            tgt_doc = target[:, did]\n            raw = getattr(self.model, \"_raw_doc\", None)\n            # THE AUXILIARY LOSS MUST TRAIN WHAT THE FUSION RANKS ON. The fused score\n            # uses z(s_graph); z is scale-invariant, so a contrastive loss on the RAW\n            # graph score can be driven down simply by scaling every score up, a\n            # direction that leaves the fused ranking untouched. That is a free way for\n            # hn_graph to fall while hn_fused and validation do not move. Mining still\n            # uses the raw score, because top-k is scale-free either way.\n            raw_l = getattr(self.model, \"_raw_doc_z\", None)\n            if raw_l is None:\n                raw_l = raw                     # older reasoners that cache only the raw score\n            g_mine = raw.detach() if (raw is not None and self._hn_graph > 0) else None\n            # ONE LINEUP FOR BOTH TERMS. Same golds, same hubs, same randoms, so the only\n            # thing that differs between the fused and graph-alone losses is which score\n            # is being trained -- which is the whole point of having both.\n            lineups = self.build_lineups(tgt_doc, s_op, g_mine)\n            # (1) fused-score contrastive: trains gate/router + operator scalars + graph jointly\n            l_fused = self._contrastive_hardneg(\n                pred[:, did], tgt_doc, s_op, g_mine=g_mine, miss_w=self._miss_w_fused,\n                lineups=lineups\n            )\n            step_metrics[\"hn_fused\"] = l_fused.item()\n            total = total + l_fused\n            # (2) graph-alone contrastive: trains the GNN DIRECTLY (survives a near-zero gate/alpha)\n            # `_aux_ok` is False for the semantic-prior reasoner, which has no standalone\n            # graph ranking: its `_raw_doc` is the belief CORRECTION, kept for diagnostics.\n            # Training a contrastive on it would demand that the correction be a ranker in\n            # its own right, which is not what it is.\n            if self._aux_w > 0 and raw_l is not None and getattr(self.model, \"_aux_ok\", True):\n                if self._resid_prior:\n                    # REPLACES the standard graph-alone term rather than adding to it, so\n                    # the arm differs from its control in exactly one thing: which\n                    # documents count as negatives for the graph channel.\n                    prior_doc = self._prior_doc(graph, batch, did)\n                    l_graph, rstats = self._contrastive_resid(\n                        raw_l, tgt_doc, prior_doc, lineups)\n                    step_metrics.update(rstats)\n                else:\n                    l_graph = self._contrastive_hardneg(\n                        raw_l, tgt_doc, s_op, g_mine=g_mine, miss_w=self._miss_w_aux,\n                        lineups=lineups\n                    )\n                step_metrics[\"hn_graph\"] = l_graph.item()\n                total = total + self._aux_w * l_graph\n            # (3) CCMP: intermediate responsibility. ADDED to the endpoint terms, not\n            # substituted for them -- the graph must still be trained against the\n            # scorer's own mistakes, which is its job. This term only says WHICH\n            # intermediate nodes should be carrying the query while it does that.\n            if self._ccmp and self._ccmp_w > 0:\n                preds = getattr(self.model.base.entity_model, \"_resp_pred\", None)\n                if preds:\n                    tg = self._ccmp_targets(graph, batch, did, s_op, g_mine=g_mine)\n                    l_crp, cstats = self._ccmp_loss(preds, tg)\n                    if l_crp is not None:\n                        step_metrics[\"ccmp\"] = l_crp.item()\n                        step_metrics.update(cstats)\n                        _gs = getattr(self.model.base.entity_model,\n                                      \"_resp_gstat\", None)\n                        if _gs:\n                            step_metrics[\"ccmp_gmax\"] = max(x[1] for x in _gs)\n                            step_metrics[\"ccmp_gp95\"] = max(x[2] for x in _gs)\n                        total = total + self._ccmp_w * l_crp\n        else:\n            for sft_loss in self.loss_functions:\n                tids = graph.nodes_by_type[sft_loss.target_node_type]\n                loss = sft_loss.loss_fn(pred[:, tids], target[:, tids])\n                step_metrics[sft_loss.name] = loss.item()\n                total = total + sft_loss.weight * loss\n\n            # graph-alone auxiliary term (teach the GNN independently of the gate)\n            if self._aux_w > 0 and getattr(self.model, \"_raw_doc\", None) is not None:\n                did = self.model._doc_ids\n                aux = self._aux_loss_fn(self.model._raw_doc, target[:, did])\n                step_metrics[\"aux_graph\"] = aux.item()\n                total = total + self._aux_w * aux\n\n        # POPULARITY ANCHOR, only under semantic='mlp'. The learned scorer's popularity\n        # term is a network, and the fusion's ranking gradient reaches it exactly as the\n        # standalone run's did -- so it needs the same anchor, or p_hat stops meaning \"how\n        # generally matchable is this paper\" and starts meaning \"whatever lowers this loss\".\n        # Zero for the operator channel, which has no predictor.\n        if self._sem_pop_lambda > 0 and hasattr(self.model, \"semantic_aux_loss\"):\n            l_pop = self.model.semantic_aux_loss()\n            if torch.is_tensor(l_pop):\n                step_metrics[\"pop_anchor\"] = l_pop.item()\n                total = total + self._sem_pop_lambda * l_pop\n\n        step_metrics[\"loss\"] = total\n        return step_metrics\n", "/content/gfm-rag/gfmrag/trainers/base_trainer.py": "import logging\nimport os\nfrom abc import ABC, abstractmethod\nfrom dataclasses import dataclass\nfrom itertools import islice\nfrom typing import Any\n\nimport numpy as np\nimport torch\nimport torch.distributed as dist\nimport torch.nn as nn\nfrom torch.utils.data import DataLoader\nfrom tqdm import tqdm\n\nfrom gfmrag import utils\nfrom gfmrag.graph_index_datasets.graph_dataset_loader import (\n    GraphDataset,\n    GraphDatasetLoader,\n)\nfrom gfmrag.utils.wandb_utils import (\n    log_metrics,\n    log_model_checkpoint,\n)\n\nfrom .training_args import TrainingArguments\n\nlogger = logging.getLogger(__name__)\n# Disable on non-master processes\nif utils.get_rank() != 0:\n    logger.setLevel(logging.CRITICAL + 1)\n\n\n@dataclass\nclass TaskDataset:\n    \"\"\"Task-specific dataset for GFM-RAG models.\"\"\"\n\n    name: str\n    graph: Any\n    data_loader: DataLoader\n\n\nclass BaseTrainer(ABC):\n    \"\"\"\n    Base trainer class for GFM-RAG models, similar to HuggingFace Trainer.\n    \"\"\"\n\n    separator = \">\" * 30\n    line = \"-\" * 30\n\n    def __init__(\n        self,\n        output_dir: str,\n        args: TrainingArguments,\n        model: nn.Module,\n        optimizer: torch.optim.Optimizer,\n        train_graph_dataset_loader: GraphDatasetLoader | None = None,\n        eval_graph_dataset_loader: GraphDatasetLoader | None = None,\n        **kwargs: Any,\n    ) -> None:\n        self.output_dir = output_dir\n        self.model = model\n        self.optimizer = optimizer\n        self.args = args\n        self.train_graph_dataset_loader = train_graph_dataset_loader\n        self.eval_graph_dataset_loader = eval_graph_dataset_loader\n\n        # Evaluation strategy\n        self.eval_strategy = args.eval_strategy\n        self.eval_steps = args.eval_steps\n\n        # Set up distributed training\n        self.device = utils.get_device()\n        self.world_size = utils.get_world_size()\n        self.rank = utils.get_rank()\n\n        # Training state\n        self.state: dict[str, Any] = {\n            \"epoch\": 0,\n            \"global_step\": 0,\n            \"best_metric\": float(\"-inf\") if args.greater_is_better else float(\"inf\"),\n            \"best_epoch\": -1,\n        }\n\n        # Create output directory\n        os.makedirs(self.output_dir, exist_ok=True)\n\n        # Set up model for training\n        self._setup_model()\n\n    def _setup_model(self) -> None:\n        \"\"\"Set up the model for training.\"\"\"\n\n        self.model = self.model.to(self.device)\n        # Configure model precision based on config\n        self.model, self.dtype = utils.configure_model_precision(\n            self.model, self.device, self.args.dtype\n        )\n\n        self.use_amp = self.dtype != torch.float32\n        self.enable_grad_scaler = self.dtype not in [torch.float32, torch.bfloat16]\n        self.scaler = torch.amp.GradScaler(\n            self.device.type, enabled=self.enable_grad_scaler\n        )\n        # Resume AFTER the precision cast, not before. Loading optimizer state\n        # while the params are still fp32 makes load_state_dict cast exp_avg to\n        # fp32; after the bf16 cast the first Adam step then mixes fp32 state\n        # with bf16 grads and dies in _foreach_lerp_ (expected float, got\n        # BFloat16). Loading here matches the dtype layout a fresh run creates.\n        if self.args.resume_from_checkpoint:\n            self._load_checkpoint(self.args.resume_from_checkpoint)\n\n        if self.world_size > 1 and not self.args.split_graph_training:\n            self.parallel_model = nn.parallel.DistributedDataParallel(\n                self.model, device_ids=[self.device]\n            )\n        else:\n            self.parallel_model = self.model\n\n    def _load_checkpoint(self, checkpoint_path: str) -> None:\n        \"\"\"Load a checkpoint.\"\"\"\n        if os.path.exists(checkpoint_path):\n            logger.info(f\"Loading checkpoint from {checkpoint_path}\")\n            state = torch.load(\n                checkpoint_path, map_location=self.device, weights_only=False\n            )\n\n            # Load model state\n            if \"model\" in state:\n                self.model.load_state_dict(state[\"model\"], strict=False)\n\n            # Load optimizer state\n            if \"optimizer\" in state and hasattr(self, \"optimizer\"):\n                try:\n                    self.optimizer.load_state_dict(state[\"optimizer\"])\n                    logger.info(\"Loaded optimizer state from checkpoint\")\n                except Exception as e:\n                    logger.warning(f\"Could not load optimizer state: {e}\")\n\n            # Load training state\n            if \"epoch\" in state:\n                self.state[\"epoch\"] = state[\"epoch\"]\n            if \"global_step\" in state:\n                self.state[\"global_step\"] = state[\"global_step\"]\n            if \"best_metric\" in state:\n                self.state[\"best_metric\"] = state[\"best_metric\"]\n            if \"best_epoch\" in state:\n                self.state[\"best_epoch\"] = state[\"best_epoch\"]\n            # hasattr, because _load_checkpoint is reached from _setup_model(), which\n            # __init__ calls BEFORE it builds the AMP scaler. On the resume path\n            # self.scaler therefore does not exist yet. The optimizer load above\n            # already guards for the same reason; this line was missed, so every\n            # resume died with a bare AttributeError.\n            if \"scaler\" in state and hasattr(self, \"scaler\"):\n                self.scaler.load_state_dict(state[\"scaler\"])\n\n            logger.info(\n                f\"Resumed from epoch {self.state['epoch']}, global step {self.state['global_step']}\"\n            )\n        else:\n            logger.warning(f\"Checkpoint {checkpoint_path} does not exist\")\n\n    def _save_checkpoint(self, output_dir: str, is_best: bool = False) -> None:\n        \"\"\"Save a checkpoint.\"\"\"\n        if utils.get_rank() == 0:\n            state = {\n                \"model\": self.model.state_dict(),\n                \"optimizer\": self.optimizer.state_dict(),\n                \"scaler\": self.scaler.state_dict(),\n                \"epoch\": self.state[\"epoch\"],\n                \"global_step\": self.state[\"global_step\"],\n                \"best_metric\": self.state[\"best_metric\"],\n                \"best_epoch\": self.state[\"best_epoch\"],\n            }\n\n            # Save best checkpoint\n            if is_best:\n                best_path = os.path.join(output_dir, \"model_best.pth\")\n                torch.save(state, best_path)\n                logger.info(f\"Saved best model to {best_path}\")\n\n                # Log model checkpoint to wandb\n                log_model_checkpoint(\n                    best_path,\n                    f\"best-epoch-{self.state['epoch']}-step-{self.state['global_step']}\",\n                    metadata={\n                        \"epoch\": self.state[\"epoch\"],\n                        \"best_metric\": self.state[\"best_metric\"],\n                        \"best\": True,\n                    },\n                )\n            # Save regular checkpoint\n            elif not self.args.save_best_only:\n                checkpoint_path = os.path.join(\n                    output_dir,\n                    f\"checkpoint-epoch-{self.state['epoch']}-step-{self.state['global_step']}.pth\",\n                )\n                torch.save(state, checkpoint_path)\n                logger.info(f\"Saved checkpoint to {checkpoint_path}\")\n                log_model_checkpoint(\n                    checkpoint_path,\n                    f\"checkpoint-epoch-{self.state['epoch']}-step-{self.state['global_step']}\",\n                    metadata={\n                        \"epoch\": self.state[\"epoch\"],\n                        \"global_step\": self.state[\"global_step\"],\n                    },\n                )\n\n    def _log_metrics(\n        self, logs: dict[str, Any], step: int | None = None, prefix: str = \"\"\n    ) -> None:\n        \"\"\"Log metrics.\"\"\"\n        if utils.get_rank() == 0:\n            if step is None:\n                step = self.state[\"global_step\"]  # type: ignore\n\n            # Log to console\n            order = sorted(list(logs.keys()))\n            for key in order:\n                logger.info(f\"{key}: {logs[key]:.4f}\")\n\n            # Add Prefix to the logs\n            if prefix:\n                log_with_prefix = {f\"{prefix}/{k}\": v for k, v in logs.items()}\n            else:\n                log_with_prefix = logs\n\n            # Add step to logs\n            logs_with_step = {**log_with_prefix, \"step\": step}\n\n            # Log to wandb\n            log_metrics(logs_with_step)\n\n    @abstractmethod\n    def _create_task_dataset(\n        self, graph_dataset: GraphDataset, is_train: bool = True\n    ) -> TaskDataset:\n        \"\"\"\n        Create a task-specific dataset from the graph dataset\n\n        Args:\n            graph_dataset (GraphDataset): The graph dataset to use.\n            is_train (bool): Whether this is for training.\n\n        Returns:\n            TaskDataset\n        \"\"\"\n        pass\n\n    @abstractmethod\n    def train_step(\n        self, batch: Any, task_dataset: TaskDataset\n    ) -> dict[str, float | torch.Tensor]:\n        \"\"\"\n        Perform a single training step.\n\n        Args:\n            batch: Training batch from dataloader\n            task_dataset: Information about the current dataset\n\n        Returns:\n            Dictionary containing loss and other metrics\n        \"\"\"\n        pass\n\n    @abstractmethod\n    def evaluate(self) -> dict[str, float]:\n        \"\"\"\n        Perform the evaluation.\n\n        Returns:\n            Dictionary containing evaluation metrics\n        \"\"\"\n        pass\n\n    def train(self) -> None:\n        \"\"\"Main training loop.\"\"\"\n        if self.args.do_train:\n            logger.info(\"***** Running training *****\")\n            logger.info(f\"  Num epochs = {self.args.num_epoch}\")\n            logger.info(\n                f\"  Instantaneous batch size per device = {self.args.train_batch_size}\"\n            )\n            logger.info(\n                f\"  Total train batch size (w. parallel & distributed) = {self.args.train_batch_size * self.world_size}\"\n            )\n\n            start_epoch = self.state[\"epoch\"]\n\n            for epoch in range(start_epoch, self.args.num_epoch):  # type: ignore\n                self.state[\"epoch\"] = epoch + 1\n\n                if utils.get_rank() == 0:\n                    logger.info(f\"{'=' * 50}\")\n                    logger.info(f\"Epoch {self.state['epoch']} / {self.args.num_epoch}\")\n                    logger.info(f\"{'=' * 50}\")\n\n                # Training\n                self._train_epoch()\n                utils.synchronize()\n\n                # Evaluation by epoch\n                if (\n                    self.eval_strategy == \"epoch\"\n                    and self.eval_graph_dataset_loader is not None\n                ):\n                    eval_metrics = self.evaluate()\n                    self._log_metrics(eval_metrics, prefix=\"eval\")\n                    self._maybe_save_best_model(eval_metrics)\n\n                # Save checkpoint\n                if not self.args.save_best_only:\n                    self._save_checkpoint(self.output_dir)\n\n            utils.synchronize()\n            # Load best model at the end\n            if self.args.load_best_model_at_end:\n                best_model_path = os.path.join(self.output_dir, \"model_best.pth\")\n                if os.path.exists(best_model_path):\n                    logger.info(\"Loading best model for final evaluation\")\n                    self._load_checkpoint(best_model_path)\n\n            logger.info(\"Training completed!\")\n\n        # Final evaluation\n        if self.eval_graph_dataset_loader is not None and self.args.do_eval:\n            logger.info(\"***** Running final evaluation *****\")\n            final_metrics = self.evaluate()\n            self._log_metrics(final_metrics, prefix=\"final\")\n\n    def _train_epoch(self) -> None:\n        \"\"\"Train for one epoch.\"\"\"\n        if self.train_graph_dataset_loader is None:\n            logger.warning(\"No training dataset loader provided\")\n            return\n\n        self.parallel_model.train()\n\n        epoch_losses = []\n        epoch_metrics: dict[str, list[float]] = {}\n\n        # Set epoch for data loader\n        if hasattr(self.train_graph_dataset_loader, \"set_epoch\"):\n            self.train_graph_dataset_loader.set_epoch(self.state[\"epoch\"])  # type: ignore\n\n        for graph_dataset in self.train_graph_dataset_loader:\n            dataset_name = graph_dataset.name\n            task_dataset = self._create_task_dataset(graph_dataset, is_train=True)\n            data_loader = task_dataset.data_loader\n\n            # Set epoch for sampler\n            if hasattr(data_loader, \"sampler\") and hasattr(\n                data_loader.sampler, \"set_epoch\"\n            ):\n                data_loader.sampler.set_epoch(self.state[\"epoch\"])  # type: ignore\n\n            # Limit steps per epoch if specified\n            if self.args.max_steps_per_epoch:\n                data_iterator = islice(data_loader, self.args.max_steps_per_epoch)\n                total_steps = self.args.max_steps_per_epoch\n            else:\n                data_iterator = data_loader\n                total_steps = len(data_loader)\n\n            progress_bar = tqdm(\n                data_iterator,\n                desc=f\"Training {dataset_name} - Epoch {self.state['epoch']}\",\n                total=total_steps,\n                disable=not utils.is_main_process(),\n            )\n\n            for batch in progress_bar:\n                # Training step\n                with torch.amp.autocast(\n                    device_type=self.device.type, dtype=self.dtype, enabled=self.use_amp\n                ):\n                    step_metrics = self.train_step(batch, task_dataset)\n\n                    assert \"loss\" in step_metrics, (\n                        \"Training step must return 'loss' in metrics\"\n                    )\n\n                    # Backward pass\n                    loss = step_metrics[\"loss\"]\n                    self.scaler.scale(loss).backward()\n\n                    # Split-graph training: manual gradient sync (no DDP wrapper)\n                    if self.args.split_graph_training and self.world_size > 1:\n                        self.scaler.unscale_(self.optimizer)\n                        for param in self.model.parameters():\n                            if param.grad is not None:\n                                dist.all_reduce(param.grad, op=dist.ReduceOp.AVG)\n\n                    self.scaler.step(self.optimizer)\n                    self.scaler.update()\n                    self.optimizer.zero_grad()\n\n                epoch_losses.append(loss.item())  # type: ignore\n\n                # Convert step metrics to float for logging\n                step_metrics = {\n                    k: v.item() if isinstance(v, torch.Tensor) else v\n                    for k, v in step_metrics.items()\n                }\n\n                # Accumulate metrics\n                for key, value in step_metrics.items():\n                    if key not in epoch_metrics:\n                        epoch_metrics[key] = []\n                    epoch_metrics[key].append(value)\n\n                self.state[\"global_step\"] += 1\n\n                # Log step metrics\n                if self.state[\"global_step\"] % self.args.logging_steps == 0:\n                    self._log_metrics(step_metrics, prefix=\"train\")\n\n                # Evaluation by step\n                if (\n                    self.eval_strategy == \"step\"\n                    and self.eval_graph_dataset_loader is not None\n                    and self.eval_steps is not None\n                    and self.state[\"global_step\"] % self.eval_steps == 0\n                ):\n                    eval_metrics = self.evaluate()\n                    self._log_metrics(eval_metrics, prefix=\"eval\")\n                    self._maybe_save_best_model(eval_metrics)\n\n                    # Save checkpoint\n                    if not self.args.save_best_only:\n                        self._save_checkpoint(self.output_dir)\n\n                # Update progress bar. EVERY term, not just the total: the total is a\n                # weighted sum of three or four objectives and a change in it says nothing\n                # about which one moved. `ccmp_pos` in particular is the collapse detector\n                # -- if the gold-side share of supervised nodes goes to zero the target has\n                # degenerated and the run is already dead, which is worth seeing at step 50\n                # rather than after four epochs.\n                progress_bar.set_postfix(\n                    **{k: round(float(v), 4)\n                       for k, v in step_metrics.items()\n                       if k in (\"loss\", \"hn_fused\", \"hn_graph\", \"ccmp\",\n                                \"ccmp_conf\", \"ccmp_pos\", \"pop_anchor\")})\n\n        # Log epoch averages\n        if utils.get_rank() == 0:\n            epoch_avg_metrics = {\n                f\"epoch_{k}\": np.mean(v) for k, v in epoch_metrics.items()\n            }\n            epoch_avg_metrics[\"epoch\"] = self.state[\"epoch\"]\n            self._log_metrics(epoch_avg_metrics, prefix=\"train\")\n            # ONE LINE PER EPOCH, EVERY COMPONENT. _log_metrics goes to wandb, which the\n            # runs disable, so without this the per-term averages are computed and thrown\n            # away and only the total reaches stdout.\n            _ord = [\"loss\", \"hn_fused\", \"hn_graph\", \"ccmp\", \"aux_graph\", \"pop_anchor\",\n                    \"ccmp_conf\", \"ccmp_pos\", \"ccmp_nodes\", \"resid_cover\", \"resid_negs\"]\n            _have = [k for k in _ord if k in epoch_metrics] + \\\n                    [k for k in sorted(epoch_metrics) if k not in _ord]\n            print(\"[loss] epoch %d  \" % self.state[\"epoch\"]\n                  + \"  \".join(\"%s %.4f\" % (k, np.mean(epoch_metrics[k])) for k in _have),\n                  flush=True)\n\n            logger.info(\n                f\"Epoch {self.state['epoch']} completed - Average loss: {np.mean(epoch_losses):.4f}\"\n            )\n\n    def _maybe_save_best_model(self, eval_metrics: dict[str, float]) -> None:\n        \"\"\"Save model if it's the best so far.\"\"\"\n        if self.args.metric_for_best_model is None:\n            return\n\n        metric_value = eval_metrics.get(self.args.metric_for_best_model)\n        if metric_value is None:\n            logger.warning(\n                f\"Metric {self.args.metric_for_best_model} not found in eval metrics\"\n            )\n            return\n\n        is_best = (\n            self.args.greater_is_better and metric_value > self.state[\"best_metric\"]\n        ) or (\n            not self.args.greater_is_better and metric_value < self.state[\"best_metric\"]\n        )\n\n        if is_best:\n            self.state[\"best_metric\"] = metric_value\n            self.state[\"best_epoch\"] = self.state[\"epoch\"]\n            if utils.get_rank() == 0:\n                logger.info(\n                    f\"New best model! {self.args.metric_for_best_model}: {metric_value:.4f} at epoch {self.state['epoch']}\"\n                )\n            self._save_checkpoint(self.output_dir, is_best=True)\n\n        else:\n            if utils.get_rank() == 0:\n                logger.info(\n                    f\"Current best {self.args.metric_for_best_model}: {self.state['best_metric']:.4f} at epoch {self.state['best_epoch']}, not updated\"\n                )\n", "/content/gfm-rag/gfmrag/workflow/config/gfm_reasoner/sft_training_fusion.yaml": "# G-Reasoner SFT \u2014 CARGO fusion (v16sc graph + operator, learned jointly in ONE run).\n# = base sft_training.yaml, but: (1) model is FusionGraphReasoner (wraps GraphReasoner, fuses the\n# operator score into the document logits with a learned per-query gate); (2) losses are bce + pcr on\n# the FUSED score (NO mse distillation \u2014 the operator is a fused signal, not a teacher); (3) the\n# trainer adds a small graph-alone aux loss (weight AUX_W) so the from-scratch GNN learns alongside.\n# Operator scores come from env OPERATOR_SCORES (train) / OPERATOR_SCORES_TEST (test).\nhydra:\n  run:\n    dir: outputs/qa_finetune/${now:%Y-%m-%d}/${now:%H-%M-%S}\n  searchpath:\n    - pkg://gfmrag.workflow.config\n\ndefaults:\n  - _self_\n  - text_emb_model: qwen3\n  - wandb: default\n\nseed: 1024\ntimeout: 60\nsave_pretrained: no\nload_model_from_pretrained: null\n\ndatasets:\n  _target_: gfmrag.graph_index_datasets.GraphIndexDataset\n  cfgs:\n    root: ./data\n    force_reload: False\n    text_emb_model_cfgs: ${text_emb_model}\n  train_names:\n    - tomato_train_v16sc\n  valid_names:\n    - tomato_test_v16sc\n  init_datasets: True\n  feat_dim: 1024\n  max_datasets_in_memory: 10\n  data_loading_workers: 4\n\n# Fusion model: wraps the v16sc GraphReasoner; entity_model is instantiated and passed through.\n# The operator is recomputed live from cached ingredients (OPERATOR_COMPONENTS[_TEST]) so its\n# scalars w=[w0,w1,w2] and beta are LEARNED jointly (warm-started at fitted values, base LR).\nmodel:\n  _target_: gfmrag.models.fusion_reasoner.FusionGraphReasoner\n  gamma_init: 0.01         # gate near zero: on step 1 the model IS the operator (opt-in graph, protects same-domain)\n  gate_hidden: 8\n  op_lr_scale: 1.0         # operator scalars learn at the full base LR (4 params, warm-started, fast to fit)\n  # Which scorer supplies the semantic channel. \"operator\" is the handcrafted sum+max\n  # scorer with the leave-one-out anti-hub term; \"mlp\" is the learned sorted-input MLP\n  # with a predicted popularity discount, warm-started from SEMANTIC_CKPT/SEMANTIC_POPNET.\n  # Everything else about the fusion is identical, so the pair is a controlled comparison.\n  semantic: operator\n  semantic_train: true     # keep the learned scorer training under the fusion's ranking loss\n  # Cross-Query Informativeness Gating. cqig=false attaches no hooks at all, so the\n  # ungated arm is bit-identical to a build without it.\n  cqig: false\n  cqig_lam: 0.1            # initial max damping; NOT 0, which would be a dead gradient\n  # null on both means \"take it from the environment\" (CQIG_NORM / CQIG_LAYERS), which is\n  # how the notebook varies them per arm without pushing a string like \"3-6\" through\n  # Hydra's override quoting. Set them here to pin an arm from the config instead.\n  #   cqig_norm:   layer (default) | node | energy   -- see the header of cqig.py\n  #   cqig_layers: last (default) | all | \"6\" | \"3-6\" | \"3,5,6\"   -- 1-INDEXED\n  #   cqig_op:     gate (default) | centre | centre-fixed        -- CQIG_OP\n  #     Which OPERATOR the same statistic drives. gate scales the state by g; centre\n  #     subtracts (1-g)*mu, so an uninformative node sends its residual instead of a\n  #     quieter copy of itself; centre-fixed subtracts a constant lam*mu, which is the\n  #     query-independent rung of the ladder. lam=0 recovers the ungated model under all\n  #     three, so the ablation is still one scalar.\n  #   cqig_mu:     ref (default) | zero                          -- CQIG_MU\n  #     zero forces mu=0, which degenerates I to ||h||^2/s: an activation-magnitude gate\n  #     with no reference bank in it. THE ABLATION. If it reproduces the method's score,\n  #     the bank was never load-bearing. Illegal with a centring op, which would be a\n  #     literal no-op at mu=0.\n  cqig_norm: null\n  cqig_layers: null\n  cqig_op: null\n  cqig_mu: null\n  use_ent_emb: early-late-fusion\n  dtype: bfloat16\n  entity_model:\n    _target_: gfmrag.models.ultra.models.QueryNBFNet\n    input_dim: 1024\n    hidden_dims: [1024, 1024, 1024, 1024, 1024, 1024]\n    message_func: distmult\n    aggregate_func: sum\n    short_cut: yes\n    layer_norm: yes\n    return_hidden: True\n\n# Loss: gold supervision on the FUSED document score (no distillation term).\nlosses:\n  - name: bce_loss\n    loss:\n      _target_: gfmrag.losses.BCELoss\n      adversarial_temperature: 0.2\n    weight: 0.3\n    target_node_type: document\n  - name: pcr_loss\n    loss:\n      _target_: gfmrag.losses.ListCELoss\n    weight: 0.7\n    target_node_type: document\n\noptimizer:\n  _target_: torch.optim.AdamW\n  lr: 5.0e-4\n\ntrainer:\n  _target_: gfmrag.trainers.fusion_trainer.FusionSFTTrainer   # adds graph-alone aux loss (AUX_W)\n  args:\n    _target_: gfmrag.trainers.TrainingArguments\n    train_batch_size: 4\n    num_epoch: 10\n    logging_steps: 100\n    max_steps_per_epoch: null\n    resume_from_checkpoint: null\n    do_train: true\n    do_eval: true\n    save_best_only: yes\n    metric_for_best_model: document_mrr\n    dtype: ${model.dtype}\n    split_graph_inference: false\n    split_graph_training: false\n    split_graph_partition: contiguous\n  metrics: [mrr, hits@1, hits@2, hits@3, hits@5, hits@10, hits@20, recall@2, recall@3, recall@5, recall@10, recall@20]\n  target_types: [document]\n"}''')
    for p, c in FILES.items():
        os.makedirs(os.path.dirname(p), exist_ok=True)
        open(p, 'w').write(c)
        print('wrote', p, f'({len(c)} bytes)')
    import importlib, sys
    sys.path.insert(0, '/content/gfm-rag')
    for m in ['gfmrag.models.fusion_reasoner', 'gfmrag.trainers.fusion_trainer']:
        importlib.import_module(m); print('import OK:', m)
    print('fusion files ready')
else:
    print('skipped: every scan and path file is already on Drive, so the engine is not needed')


## 3b-bis. Current fusion sources (embedded overlay)

In [ ]:
# Only runs when cell 1 found a scan or a path search still to do; drawing the figures from
# cached interpretations needs neither the engine nor Qwen3.
if NEED_ENGINE:
    # 3b-bis. The CURRENT fusion sources, embedded (gfm_overlay.zip from the repo, built with the notebook).
    # The replayed cell above writes an older fusion_trainer.py whose interpret() lacks do_paths / golds /
    # min_hops: without this overlay the rank scan silently runs the slow beam search on every gold and
    # the hop figure has no structural floor. Nothing to upload; the zip travels inside this cell.
    import base64 as _b64, io as _io, zipfile as _zf
    _ov = _zf.ZipFile(_io.BytesIO(_b64.b64decode("...")))
    _ov.extractall("/content/gfm-rag"); print("overlay:", _ov.namelist())
    _ft = open("/content/gfm-rag/gfmrag/trainers/fusion_trainer.py").read()
    _need = ('def interpret(', 'do_paths', 'golds=None', 'min_hops')
    assert all(k in _ft for k in _need), "fusion_trainer.py lacks " + str([k for k in _need if k not in _ft])
    print("engine interpret() is current: do_paths, pinned golds, min_hops")
else:
    print('skipped: every scan and path file is already on Drive, so the engine is not needed')


## 3c. Config defaults

In [ ]:
# Only runs when cell 1 found a scan or a path search still to do; drawing the figures from
# cached interpretations needs neither the engine nor Qwen3.
if NEED_ENGINE:
    # Cell 3a is reused VERBATIM from the TOMATO notebook so the fusion model code
    # cannot drift. That file also carries a config whose DEFAULT train_names is
    # `tomato_train_v16sc`. run_model always overrides it on the command line, but a
    # default naming another corpus is exactly the shape of every contamination we
    # hit today, so it gets rewritten rather than trusted.
    import re
    CFGS = [p for p in ["/content/gfm-rag/gfmrag/workflow/config/gfm_reasoner/sft_training_fusion.yaml",
                        "/content/gfm-rag/gfmrag/workflow/config/gfm_reasoner/sft_training_belief.yaml"]
            if os.path.exists(p)]
    # BOTH configs, in one loop. The belief config is a copy of the fusion one with a
    # different model target, so a rewrite applied to only the first would leave the
    # second pointing at a TOMATO corpus -- the exact contamination this cell exists to
    # prevent, reintroduced by adding a second file.
    for cfg in CFGS:
        t = open(cfg).read()
        t = t.replace("tomato_train_v16sc", TRAIN).replace("tomato_test_v16sc", TEST)
        open(cfg, "w").write(t)
        # ASSERT ON WHAT MUST BE PRESENT, NOT ON WHAT MUST BE ABSENT. The old check counted
        # surviving `tomato_\w+` matches and required zero, which is correct only while the
        # target corpus is NOT tomato. On a TOMATO build TRAIN is itself `tomato_train_v16sc`,
        # so the replace is the identity, two names legitimately survive, and the assert fired
        # on a config that was in fact right. Checking that both intended names are present and
        # that no foreign corpus leaked is the same guarantee and holds for either target.
        assert TRAIN in t, f"{TRAIN} missing from {cfg}"
        assert TEST in t, f"{TEST} missing from {cfg}"
        assert "sir4" not in t, f"a sir4 dataset reference survived in {cfg}"
        print(f"{cfg.split('/')[-1]}: train_names/valid_names -> {TRAIN} / {TEST}")
    cfg = CFGS[0]
    print("  train_names:", TRAIN, "\n  valid_names:", TEST)

    # METRICS REPORTED PER EPOCH. The stock list is twelve columns of hits@k and
    # recall@k, none of which is what gets reported, and selection ran on
    # document_mrr. Narrow it to the four metrics that appear in the results table,
    # and select on nDCG@5.
    #
    # CompleteSet@5 is deliberately NOT here. utils.evaluate receives gold RANKS, not
    # ranked document ids, and knows nothing about sets.json, so the closest thing it
    # could compute is "all of this query's golds in the top 5" -- a much harsher
    # metric wearing the same name. It comes from score_sir4.py afterwards instead.
    for cfg in CFGS:
        t = open(cfg).read()
        t = re.sub(r"^  metrics: \[.*\]$", "  metrics: [mrr, ndcg@5, recall@3, recall@5]",
                   t, count=1, flags=re.M)
        t = t.replace("metric_for_best_model: document_mrr",
                      "metric_for_best_model: document_ndcg@5")
        open(cfg, "w").write(t)
        assert "metrics: [mrr, ndcg@5, recall@3, recall@5]" in t, f"metrics not rewritten in {cfg}"
        assert "document_ndcg@5" in t, f"selection metric not rewritten in {cfg}"
    cfg = CFGS[0]
    print("  metrics:  mrr, ndcg@5, recall@3, recall@5")
    print("  best on:  document_ndcg@5")

    # nDCG DOES NOT EXIST IN THE ENGINE. utils.evaluate handles mrr / recall@k /
    # hits@k / mape and raises ValueError on anything else, so selecting on ndcg@5
    # would die on the first evaluation. Patch the branch in. Binary relevance with
    # the same ideal-DCG treatment as score_sir4.py, verified identical on seven
    # cases including more golds than k and all golds outside the cutoff.
    QA = "/content/gfm-rag/gfmrag/utils/qa_utils.py"
    q = open(QA).read()
    if '_metric.startswith("ndcg@")' in q:
        print("  qa_utils: ndcg already patched")
    else:
        assert '        elif _metric == "mape":' in q, "mape anchor missing in qa_utils"
        _nd = [
            '        elif _metric.startswith("ndcg@"):',
            '            threshold = int(_metric[5:])',
            '            gain = torch.where(',
            '                answer_ranking <= threshold,',
            '                1.0 / torch.log2(answer_ranking.float() + 1.0),',
            '                torch.zeros_like(answer_ranking, dtype=torch.float),',
            '            )',
            '            dcg = variadic.variadic_sum(gain, num_hard)',
            '            disc = 1.0 / torch.log2(',
            '                torch.arange(1, threshold + 1, device=gain.device).float() + 1.0',
            '            )',
            '            idcg_table = torch.cat([torch.zeros(1, device=gain.device), disc.cumsum(0)])',
            '            idcg = idcg_table[num_hard.clamp(max=threshold)]',
            '            query_score = dcg / idcg.clamp(min=1e-9)',
            '        elif _metric == "mape":',
        ]
        q = q.replace('        elif _metric == "mape":', chr(10).join(_nd), 1)
        open(QA, "w").write(q)
        print("  qa_utils: ndcg@k branch added")
else:
    print('skipped: every scan and path file is already on Drive, so the engine is not needed')


## 3d. PyG version fix

In [ ]:
# Only runs when cell 1 found a scan or a path search still to do; drawing the figures from
# cached interpretations needs neither the engine nor Qwen3.
if NEED_ENGINE:
    # The ULTRA layers vendored inside the engine parse the PyG version as:
    #     pyg_version = [int(i) for i in torch_geometric.__version__.split(".")]
    # Colab now resolves torch-geometric to 2.6.1.post1, and int("post1") raises
    # ValueError deep inside the first message-passing call -- after the Qwen3 index
    # has already been built, so it costs a full setup to discover.
    #
    # Pinning PyG would also work, but it fights Colab's resolver and will drift
    # again at the next release. Parsing the version properly will not. Taking the
    # first three dotted components and keeping only the numeric ones handles
    # "2.6.1.post1" and "2.7.0+pt24cu121" alike.
    import os, torch_geometric
    OLD = 'pyg_version = [int(i) for i in torch_geometric.__version__.split(".")]'
    NEW = ('pyg_version = [int(i) for i in torch_geometric.__version__.split(".")[:3] '
           'if i.isdigit()]')
    hits = []
    for root, _, files in os.walk("/content/gfm-rag/gfmrag"):
        for f in files:
            if not f.endswith(".py"):
                continue
            fp = os.path.join(root, f)
            t = open(fp).read()
            if OLD in t:
                open(fp, "w").write(t.replace(OLD, NEW))
                hits.append(fp)
    if not hits:   # the embedded overlay (3b-bis) ships ultra/layers.py with its own isdigit() fix, so there may be nothing left to patch
        already = [os.path.join(r_, f_) for r_, _, fs in os.walk("/content/gfm-rag/gfmrag") for f_ in fs
                   if f_.endswith(".py") and "pyg_version" in open(os.path.join(r_, f_)).read() and "isdigit()" in open(os.path.join(r_, f_)).read()]
        assert already, "version-check pattern not found and no isdigit() fix present -- has the engine changed?"
        print("  already patched by the overlay:", already)
    parsed = [int(i) for i in torch_geometric.__version__.split(".")[:3] if i.isdigit()]
    for h in hits:
        print("  patched", h)
    print(f"torch_geometric {torch_geometric.__version__} now parses to {parsed}")
else:
    print('skipped: every scan and path file is already on Drive, so the engine is not needed')


## 3e. torchvision shim

In [ ]:
# Only runs when cell 1 found a scan or a path search still to do; drawing the figures from
# cached interpretations needs neither the engine nor Qwen3.
# (run verbatim: this cell writes indentation-sensitive source into a file)
import base64 as _b64
_src = _b64.b64decode("IyBUSEUgVFJBSU5JTkcgU1VCUFJPQ0VTUyBJUyBBIEZSRVNIIFBZVEhPTi4gYHB5dGhvbiAtbSBnZm1yYWcud29ya2Zsb3cuc2Z0X3RyYWluaW5nYCBkb2VzCiMgbm90IGluaGVyaXQgdGhpcyBrZXJuZWwncyBwYXRjaGVkIG1vZHVsZXMsIHNvIHRoZSBzaGltIGhhcyB0byBsaXZlIGluIGEgZmlsZSB0aGF0IHRoZQojIHN1YnByb2Nlc3MgaW1wb3J0cy4gc2Z0X3RyYWluaW5nLnB5IGlzIHRoYXQgZmlsZSwgYW5kIGlzIGFscmVhZHkgcGF0Y2hlZCB0d2ljZSBiZWxvdy4KaW1wb3J0IHRvcmNoLCB0b3JjaHZpc2lvbiwgdG9yY2h2aXNpb24uaW8sIGRhdGFzZXRzCnByaW50KGYidG9yY2gge3RvcmNoLl9fdmVyc2lvbl9ffSB8IHRvcmNodmlzaW9uIHt0b3JjaHZpc2lvbi5fX3ZlcnNpb25fX30gfCAiCiAgICAgIGYiZGF0YXNldHMge2RhdGFzZXRzLl9fdmVyc2lvbl9ffSB8IFZpZGVvUmVhZGVyICIKICAgICAgZiJ7aGFzYXR0cih0b3JjaHZpc2lvbi5pbywgJ1ZpZGVvUmVhZGVyJyl9IikKCl9TSElNID0gIiIiIyBQQVRDSCAobm90ZWJvb2spOiBkYXRhc2V0cycgdG9yY2ggZm9ybWF0dGVyIGltcG9ydHMgdG9yY2h2aXNpb24uaW8uVmlkZW9SZWFkZXIKIyB3aGVuZXZlciB0b3JjaHZpc2lvbiBpcyBpbiBzeXMubW9kdWxlcy4gUmVjZW50IHRvcmNodmlzaW9uIHJlbW92ZWQgdGhlIGxlZ2FjeSB2aWRlbyBBUEksIHNvCiMgdGhhdCBpbXBvcnQgcmFpc2VzIGluc2lkZSBldmVyeSBiYXRjaCBmZXRjaC4gVGhlIG5hbWUgaXMgb25seSBuZWVkZWQgZm9yIGFuIGlzaW5zdGFuY2UKIyBjaGVjayBhZ2FpbnN0IHRlbnNvciBkYXRhLCBzbyBhIHBsYWNlaG9sZGVyIGlzIHN1ZmZpY2llbnQgYW5kIGRvd25sb2FkcyBub3RoaW5nLgppbXBvcnQgdG9yY2h2aXNpb24uaW8gYXMgX3R2aW8KaWYgbm90IGhhc2F0dHIoX3R2aW8sICJWaWRlb1JlYWRlciIpOgogICAgY2xhc3MgX05vVmlkZW9SZWFkZXI6CiAgICAgICAgZGVmIF9faW5pdF9fKHNlbGYsICphLCAqKmspOgogICAgICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoInRvcmNodmlzaW9uIHZpZGVvIEFQSSByZW1vdmVkOyB0aGlzIHNoaW0gZXhpc3RzIG9ubHkgc28gIgogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgInRoZSBkYXRhc2V0cyB0b3JjaCBmb3JtYXR0ZXIgY2FuIGltcG9ydCB0aGUgbmFtZSIpCiAgICBfdHZpby5WaWRlb1JlYWRlciA9IF9Ob1ZpZGVvUmVhZGVyCiIiIgoKX1NURiA9ICIvY29udGVudC9nZm0tcmFnL2dmbXJhZy93b3JrZmxvdy9zZnRfdHJhaW5pbmcucHkiCl90ID0gb3BlbihfU1RGKS5yZWFkKCkKaWYgIl9Ob1ZpZGVvUmVhZGVyIiBpbiBfdDoKICAgIHByaW50KCJbc2tpcF0gc2Z0X3RyYWluaW5nLnB5IGFscmVhZHkgY2FycmllcyB0aGUgc2hpbSIpCmVsc2U6CiAgICAjIFBSRVBFTkRFRCwgbm90IGluamVjdGVkIGF0IGFuIGFuY2hvci4gVGhlIG90aGVyIHR3byBwYXRjaGVzIG9mIHRoaXMgZmlsZSBzZWFyY2ggZm9yIGEKICAgICMgbGluZSBpbnNpZGUgbWFpbigpOyB0aGlzIG9uZSBtdXN0IHJ1biBiZWZvcmUgYGRhdGFzZXRzYCBpcyBpbXBvcnRlZCBhbnl3aGVyZSwgc28gaXQgZ29lcwogICAgIyBhYm92ZSBldmVyeSBpbXBvcnQuIFByZXBlbmRpbmcgYWxzbyBsZWF2ZXMgdGhlaXIgYW5jaG9ycyB1bnRvdWNoZWQuCiAgICBvcGVuKF9TVEYsICJ3Iikud3JpdGUoX1NISU0gKyBfdCkKICAgIHByaW50KCJzaGltbWVkIiwgX1NURikKCiMgVGhpcyBrZXJuZWwgdG9vOiB0aGUgaW4ta2VybmVsIGdmbXJhZyBpbXBvcnQgYW5kIGFueSBpbi1ub3RlYm9vayBkYXRhc2V0IHVzZSBoaXQgdGhlIHNhbWUKIyBmb3JtYXR0ZXIuCmV4ZWMoX1NISU0pCnByaW50KCJWaWRlb1JlYWRlciBwcmVzZW50IG5vdzoiLCBoYXNhdHRyKHRvcmNodmlzaW9uLmlvLCAiVmlkZW9SZWFkZXIiKSk=").decode()
if NEED_ENGINE:
    _r = get_ipython().run_cell(_src, store_history=False)
    assert _r.success, 'setup cell failed; see the traceback above'
else:
    print('skipped: every scan and path file is already on Drive, so the engine is not needed')


## 3f. Diagnostics patch

In [ ]:
# Only runs when cell 1 found a scan or a path search still to do; drawing the figures from
# cached interpretations needs neither the engine nor Qwen3.
if NEED_ENGINE:
    STF = "/content/gfm-rag/gfmrag/workflow/sft_training.py"
    _src = open(STF).read()
    assert "_evaluate_stratified" in _src, "run the engine-install cell first"
    if "_evaluate_with_diag" in _src:
        print("component diagnostics already patched")
    else:
        _blk = "\n".join([
            "    # --- injected: component diagnostics, ZERO extra forward passes ---",
            "    import torch as _t",
            "    _diag = {'sem': [], 'gph': [], 'fus': [], 'gam': [], 'pri': []}",
            "    # THE STRUCTURAL PRIOR, scored as a channel in its own right. Personalised",
            "    # PageRank from the same seeds, no parameters, no training. It is the number",
            "    # the trained GNN has to beat for the learned component to be doing anything,",
            "    # and it was never in the table: a walk scores 0.245-0.273 nDCG@5 on these",
            "    # graphs while the 34M-parameter GNN scores 0.213-0.273. Three sparse mat-muls",
            "    # per eval batch, so it is free next to six dense GNN layers.",
            "    _PR = {'A': None, 'k': None}",
            "    # a=0 is NOT redundant with 'sem': it is the mixture's own zero point, so if the",
            "    # two disagree the arithmetic is wrong rather than the graph. Grid is log-spaced",
            "    # because the interesting region is a <= 0.05 and the learned 0.371 already lost.",
            "    _ASWEEP = [0.0, 0.005, 0.01, 0.02, 0.05, 0.1, 0.2, 0.35]",
            "    # GRAPH RECALL BUCKETED BY THE GOLD'S SEMANTIC RANK. The miss-weighted graph",
            "    # loss (MISS_W_AUX) claims to spend graph capacity on the golds the scorer",
            "    # buries. An aggregate cannot see that trade; this can. Buckets are on r_sem,",
            "    # the SAME quantity the loss weights by, so a working loss must lift the deep",
            "    # buckets while holding the shallow one. If the deep buckets stay flat and the",
            "    # shallow one falls, the graph cannot fit those golds and the reweighting is",
            "    # just moving gradient onto examples the architecture does not support.",
            "    _BUCKETS = [(1, 10), (11, 100), (101, 1000), (1001, 10**9)]",
            "    _BNAME = ['1-10', '11-100', '101-1k', '1k+']",
            "    _diag.update({'b%d' % _i: [] for _i in range(len(_BUCKETS))})",
            "    _diag.update({'a%g' % _a: [] for _a in _ASWEEP})",
            "    # THE SAME SWEEP FOR THE ADDITIVE FORM, and it runs in BOTH arms. The two forms",
            "    # are being compared as a 2x2 against the gate, so each one's free scalar has to",
            "    # be chosen by the same procedure or the comparison measures the tuning effort",
            "    # rather than the form. gamma is learned in the additive arm, but it froze at",
            "    # 0.0279 / 0.0288 / 0.0359 in three runs that scored 0.5207 / 0.5218 / 0.5340,",
            "    # so what gamma converges to is not obviously what gamma should be. Both channels",
            "    # are cached here, so this curve costs nothing and one run of EITHER arm now",
            "    # returns both curves. Grid brackets every gamma any arm has actually reached.",
            "    _GSWEEP = [0.0, 0.01, 0.02, 0.0359, 0.06, 0.1, 0.2, 0.4]",
            "    _diag.update({'g%g' % _g: [] for _g in _GSWEEP})",
            "    # SPLICE sweep. The mixture's a controls the DEPTH at which the graph's block",
            "    # lands; what nDCG@5 actually cares about is its SIZE. _ISWEEP controls size",
            "    # directly: keep semantic's top (5-m) untouched and hand ranks (5-m+1)..5 to the",
            "    # graph's own top m. Displacement is exactly m, known before the run, and the",
            "    # win condition is a hit-rate comparison rather than an emergent property of a",
            "    # scalar. m=0 is the semantic control. Uses only the graph's ORDER, so it is",
            "    # immune to the overconfident calibration that z()/tau_g are fighting.",
            "    _ISWEEP = [1, 2, 3]",
            "    _diag.update({'i%d' % _m: [] for _m in _ISWEEP})",
            "    _diag.update({'gh%d' % _j: [] for _j in range(3)})",
            "    _diag.update({'sh%d' % _j: [] for _j in range(5)})",
            "    _drec = {'on': False}",
            "    def _ndcg5(_sc, _tgt):",
            "        # binary relevance, same ideal-DCG convention as score_sir4.py",
            "        _k = min(5, _sc.shape[-1])",
            "        _top = _sc.topk(_k, dim=-1).indices",
            "        _hit = _tgt.gather(1, _top).float()",
            "        _disc = 1.0 / _t.log2(_t.arange(2, _k + 2, device=_hit.device).float())",
            "        _dcg = (_hit * _disc).sum(-1)",
            "        _n = _tgt.sum(-1).clamp(max=_k).long()",
            "        _tab = _t.cat([_t.zeros(1, device=_hit.device), _disc.cumsum(0)])",
            "        return _dcg / _tab[_n].clamp(min=1e-9)",
            "    def _dhook(_mod, _inp, _out):",
            "        if not _drec['on'] or len(_inp) < 2:",
            "            return",
            "        try:",
            "            _g, _b = _inp[0], _inp[1]",
            "            _did = _g.nodes_by_type['document']",
            "            _tgt = _b['target_nodes_mask'][:, _did]",
            "            _diag['fus'] += _ndcg5(_out[:, _did].float(), _tgt).tolist()",
            "            # prior: symmetrised, degree-normalised walk with restart from the",
            "            # query's seeds. Symmetrised because the KG is directed and a walk that",
            "            # only travels head->tail cannot reach a document from an entity that",
            "            # document mentions, which is the dominant path in this corpus.",
            "            if _b.get('start_nodes_mask') is not None:",
            "                _N = _g.num_nodes",
            "                if _PR['k'] != id(_g):",
            "                    _ei = _g.edge_index",
            "                    _rr = _t.cat([_ei[0], _ei[1]]); _cc = _t.cat([_ei[1], _ei[0]])",
            "                    _dg = _t.zeros(_N, device=_ei.device).index_add_(",
            "                        0, _rr, _t.ones(_rr.numel(), device=_ei.device))",
            "                    _PR['A'] = _t.sparse_coo_tensor(",
            "                        _t.stack([_cc, _rr]), 1.0 / _dg.clamp(min=1.0)[_rr],",
            "                        (_N, _N)).coalesce()",
            "                    _PR['k'] = id(_g)",
            "                _x0 = _b['start_nodes_mask'].float()",
            "                _x0 = _x0 / _x0.sum(-1, keepdim=True).clamp(min=1e-9)",
            "                _x = _x0",
            "                # autocast off: eval runs under bfloat16 AMP and sparse.mm has no",
            "                # bf16 kernel on every build. The prior is data, not a learned",
            "                # quantity, so nothing here should be cast or differentiated.",
            "                with _t.autocast(device_type=_x0.device.type, enabled=False):",
            "                    for _ in range(3):",
            "                        _x = 0.85 * _t.sparse.mm(_PR['A'], _x.t()).t() + 0.15 * _x0",
            "                _diag['pri'] += _ndcg5(_x[:, _did].float(), _tgt).tolist()",
            "            _sem = getattr(_mod, '_s_op', None)",
            "            _gph = getattr(_mod, '_raw_doc', None)",
            "            _gam = getattr(_mod, '_gamma', None)",
            "            if _sem is not None:",
            "                _diag['sem'] += _ndcg5(_sem.detach().float(), _tgt).tolist()",
            "            if _gph is not None:",
            "                _diag['gph'] += _ndcg5(_gph.detach().float(), _tgt).tolist()",
            "            if _gam is not None:",
            "                _diag['gam'] += _gam.detach().flatten().tolist()",
            "            if _sem is not None and _gph is not None:",
            "                _sd, _gd = _sem.detach().float(), _gph.detach().float()",
            "                _gtop = _gd.topk(min(10, _gd.shape[-1]), dim=-1).indices",
            "                for _bi in range(_tgt.shape[0]):",
            "                    _pos = _tgt[_bi].nonzero(as_tuple=True)[0]",
            "                    if _pos.numel() == 0:",
            "                        continue",
            "                    # true corpus rank of each gold under the semantic scorer,",
            "                    # matching how _contrastive_hardneg computes its weights",
            "                    _r = (_sd[_bi].unsqueeze(0) > _sd[_bi, _pos].unsqueeze(1)).sum(1) + 1",
            "                    _in = _t.isin(_pos, _gtop[_bi]).float()",
            "                    for _i, (_lo, _hi) in enumerate(_BUCKETS):",
            "                        _m = (_r >= _lo) & (_r <= _hi)",
            "                        if bool(_m.any()):",
            "                            _diag['b%d' % _i] += _in[_m].tolist()",
            "            # THE a-SWEEP. Costs a few tensor ops per batch and answers the only",
            "            # question the mixture arm exists to answer: is there ANY mixing weight",
            "            # at which the graph helps? The first run tested a_q=0.371 (learned) and",
            "            # lost 0.091 nDCG@5. One point is not a curve. Both channels are already",
            "            # cached here, so every a is free. tau_g is read off the model when it is",
            "            # there, so the sweep matches what the fusion actually computes.",
            "            if _sem is not None and _gph is not None and _ASWEEP:",
            "                _zz = lambda x: (x - x.mean(-1, keepdim=True)) / (x.std(-1, keepdim=True) + 1e-6)",
            "                _th = getattr(_mod, 'tau_g_hat', None)",
            "                _tg = 1.0 if _th is None else float(",
            "                    _mod.tau_min + (_mod.tau_max - _mod.tau_min) * _t.sigmoid(_th.float()))",
            "                _ls = _zz(_sem.detach().float())",
            "                _lg = _zz(_gph.detach().float()) / max(_tg, 1e-3)",
            "                _lps = _ls - _t.logsumexp(_ls, -1, keepdim=True)",
            "                _lpg = _lg - _t.logsumexp(_lg, -1, keepdim=True)",
            "                for _a in _ASWEEP:",
            "                    _f = _t.logaddexp(_t.log(_t.tensor(1.0 - _a)) + _lps,",
            "                                      _t.log(_t.tensor(_a)) + _lpg)",
            "                    _diag['a%g' % _a] += _ndcg5(_f, _tgt).tolist()",
            "                # the additive curve, on the SAME cached channels and the same queries,",
            "                # so a=x and gamma=y are paired per query and the two forms can be",
            "                # bootstrapped against each other rather than compared as two run-level",
            "                # point estimates 0.013 apart on n=331.",
            "                _rg = _t.relu(_zz(_gph.detach().float()))",
            "                for _g in _GSWEEP:",
            "                    _diag['g%g' % _g] += _ndcg5(_ls + _g * _rg, _tgt).tolist()",
            "                # splice: inject the graph's top m between semantic ranks (5-m) and",
            "                # (5-m+1). maximum() rather than scatter so a graph pick already in",
            "                # semantic's kept head is left where it is instead of demoted.",
            "                _gr = _zz(_gph.detach().float())",
            "                _sv = _ls.sort(-1, descending=True).values",
            "                for _m in _ISWEEP:",
            "                    _c = 5 - _m",
            "                    _mid = (_sv[:, _c - 1:_c] + _sv[:, _c:_c + 1]) / 2.0",
            "                    _off = _t.arange(_m, device=_ls.device).float() * 1e-4",
            "                    _inj = _t.full_like(_ls, -1e9)",
            "                    _inj.scatter_(1, _gr.topk(_m, -1).indices, _mid - _off)",
            "                    _diag['i%d' % _m] += _ndcg5(_t.maximum(_ls, _inj), _tgt).tolist()",
            "                # the decisive number: is the graph's own top-1 a gold more often",
            "                # than the semantic doc it would evict? if not, NO reordering of",
            "                # these two lists can raise nDCG@5 and the fusion form is not the",
            "                # problem. positions are per-rank hit rates, not nDCG.",
            "                _gh = _tgt.gather(1, _gr.topk(3, -1).indices).float()",
            "                _sh = _tgt.gather(1, _ls.topk(5, -1).indices).float()",
            "                for _j in range(3):",
            "                    _diag['gh%d' % _j] += _gh[:, _j].tolist()",
            "                for _j in range(5):",
            "                    _diag['sh%d' % _j] += _sh[:, _j].tolist()",
            "        except Exception as _e:",
            "            print('[diag] hook skipped:', _e)",
            "    trainer.model.register_forward_hook(_dhook)",
            "    _prev_eval = trainer.evaluate",
            "    def _evaluate_with_diag():",
            "        for _v in _diag.values():",
            "            _v.clear()",
            "        _drec['on'] = True",
            "        m = _prev_eval()",
            "        _drec['on'] = False",
            "        _mu = lambda x: (sum(x) / len(x)) if x else float('nan')",
            "        _lo = lambda x: min(x) if x else float('nan')",
            "        _hi = lambda x: max(x) if x else float('nan')",
            "        print(f\"[diag] nDCG@5  semantic {_mu(_diag['sem']):.4f} | \"",
            "              f\"graph {_mu(_diag['gph']):.4f} | fused {_mu(_diag['fus']):.4f}   \"",
            "              f\"gamma mean {_mu(_diag['gam']):.4f} min {_lo(_diag['gam']):.4f} \"",
            "              f\"max {_hi(_diag['gam']):.4f}\", flush=True)",
            "        _bhave = [_i for _i in range(len(_BUCKETS)) if _diag['b%d' % _i]]",
            "        if _bhave:",
            "            print('[bucket] graph recall@10 by gold semantic rank:  ' + '  '.join(",
            "                  '%s %.3f (n=%d)' % (_BNAME[_i], _mu(_diag['b%d' % _i]),",
            "                                      len(_diag['b%d' % _i])) for _i in _bhave), flush=True)",
            "            for _i in _bhave:",
            "                m['bucket/graph_r10_%s' % _BNAME[_i]] = _mu(_diag['b%d' % _i])",
            "        if _diag['pri']:",
            "            _pv, _gv = _mu(_diag['pri']), _mu(_diag['gph'])",
            "            print('[prior] parameter-free walk nDCG@5 %.4f   trained graph %.4f   '",
            "                  '(%+.4f)  -> %s' % (_pv, _gv, _gv - _pv,",
            "                  'the GNN adds something' if _gv - _pv > 0.02 else",
            "                  'THE GNN IS AT ITS OWN PRIOR: the learned component is ~free-lunch "
            "topology'), flush=True)",
            "            m['prior/ndcg@5_walk'] = _pv",
            "            m['prior/gnn_minus_walk'] = _gv - _pv",
            "        m['diag/ndcg@5_semantic'] = _mu(_diag['sem'])",
            "        m['diag/ndcg@5_graph'] = _mu(_diag['gph'])",
            "        m['diag/ndcg@5_fused'] = _mu(_diag['fus'])",
            "        m['diag/gamma_mean'] = _mu(_diag['gam'])",
            "        if _ASWEEP and _diag['a%g' % _ASWEEP[0]]:",
            "            _row = [(_a, _mu(_diag['a%g' % _a])) for _a in _ASWEEP]",
            "            _bi = max(range(len(_row)), key=lambda i: _row[i][1])",
            "            print('[sweep] fused nDCG@5 vs constant a:  '",
            "                  + '  '.join('a=%g %.4f' % r for r in _row), flush=True)",
            "            print('[sweep] best a=%g -> %.4f   vs semantic %.4f   (%+.4f)   '",
            "                  'a=0 check %.4f'",
            "                  % (_row[_bi][0], _row[_bi][1], _mu(_diag['sem']),",
            "                     _row[_bi][1] - _mu(_diag['sem']), _row[0][1]), flush=True)",
            "            for _a, _v in _row:",
            "                m['sweep/a%g' % _a] = _v",
            "            # the additive curve, printed next to it. These two lines ARE the 2x2's",
            "            # fusion axis at this gate setting: same queries, same checkpoint, same",
            "            # cached channels, so the only thing that differs between them is the form.",
            "            _grow = [(_g, _mu(_diag['g%g' % _g])) for _g in _GSWEEP]",
            "            _gi = max(range(len(_grow)), key=lambda i: _grow[i][1])",
            "            print('[gsweep] fused nDCG@5 vs constant gamma (additive):  '",
            "                  + '  '.join('g=%g %.4f' % r for r in _grow), flush=True)",
            "            print('[gsweep] best gamma=%g -> %.4f   vs semantic %.4f   (%+.4f)   '",
            "                  'gamma=0 check %.4f'",
            "                  % (_grow[_gi][0], _grow[_gi][1], _mu(_diag['sem']),",
            "                     _grow[_gi][1] - _mu(_diag['sem']), _grow[0][1]), flush=True)",
            "            print('[form] best mixture %+.4f  vs  best additive %+.4f   over semantic'",
            "                  % (_row[_bi][1] - _mu(_diag['sem']),",
            "                     _grow[_gi][1] - _mu(_diag['sem'])), flush=True)",
            "            for _g, _v in _grow:",
            "                m['sweep/g%g' % _g] = _v",
            "            # per-query vectors for the paired bootstrap. Aggregates cannot be",
            "            # bootstrapped after the fact and these are the only place the per-query",
            "            # numbers exist, so they get written next to the checkpoint.",
            "            try:",
            "                import json as _json, os as _os",
            "                _pq = {'semantic': _diag['sem'], 'graph': _diag['gph'],",
            "                       'fused': _diag['fus']}",
            "                _pq.update({'a%g' % _a: _diag['a%g' % _a] for _a in _ASWEEP})",
            "                _pq.update({'g%g' % _g: _diag['g%g' % _g] for _g in _GSWEEP})",
            "                with open(_os.path.join(_os.getcwd(), 'per_query_ndcg5.json'), 'w') as _fh:",
            "                    _json.dump(_pq, _fh)",
            "            except Exception as _e:",
            "                print('[diag] per-query dump skipped:', _e, flush=True)",
            "        if _diag['i%d' % _ISWEEP[0]]:",
            "            _ir = [(_m, _mu(_diag['i%d' % _m])) for _m in _ISWEEP]",
            "            _ib = max(range(len(_ir)), key=lambda i: _ir[i][1])",
            "            print('[splice] fused nDCG@5 vs graph block size m:  m=0 %.4f  ' % _mu(_diag['sem'])",
            "                  + '  '.join('m=%d %.4f' % r for r in _ir), flush=True)",
            "            print('[splice] best m=%d -> %.4f   vs semantic %.4f   (%+.4f)'",
            "                  % (_ir[_ib][0], _ir[_ib][1], _mu(_diag['sem']),",
            "                     _ir[_ib][1] - _mu(_diag['sem'])), flush=True)",
            "            for _m, _v in _ir:",
            "                m['splice/m%d' % _m] = _v",
            "        if _diag['gh0']:",
            "            _g1 = _mu(_diag['gh0'])",
            "            _s5 = _mu(_diag['sh4'])",
            "            print('[hit] gold rate by rank -- graph 1..3: '",
            "                  + ' '.join('%.3f' % _mu(_diag['gh%d' % _j]) for _j in range(3))",
            "                  + '   semantic 1..5: '",
            "                  + ' '.join('%.3f' % _mu(_diag['sh%d' % _j]) for _j in range(5)),",
            "                  flush=True)",
            "            print('[hit] graph@1 %.3f vs semantic@5 %.3f -> %s' % (_g1, _s5,",
            "                  'a swap at rank 5 gains in expectation' if _g1 > _s5 else",
            "                  'NO reordering of these two lists can raise nDCG@5'), flush=True)",
            "            m['hit/graph@1'] = _g1",
            "            m['hit/semantic@5'] = _s5",
            "        return m",
            "    trainer.evaluate = _evaluate_with_diag",
            "    trainer.train()",
        ])
        # Injected AFTER the stratified block, so _prev_eval wraps that one rather than
        # replacing it and silently dropping the per-slice metrics.
        assert _src.count("    trainer.train()") == 1, "unexpected number of trainer.train() calls"
        open(STF, "w").write(_src.replace("    trainer.train()", _blk, 1))
        print("patched sft_training.py -> per-epoch semantic/graph/fused nDCG@5 + gamma")
else:
    print('skipped: every scan and path file is already on Drive, so the engine is not needed')


## 3g. CCMP_LR patch

In [ ]:
# Only runs when cell 1 found a scan or a path search still to do; drawing the figures from
# cached interpretations needs neither the engine nor Qwen3.
if NEED_ENGINE:
    # Idempotent: re-running is a no-op. Patches the same file section 3d just rewrote,
    # so it must run AFTER it.
    STF = "/content/gfm-rag/gfmrag/workflow/sft_training.py"
    _src = open(STF).read()
    _ANCHOR = "    optimizer = instantiate(cfg.optimizer, model.parameters())"
    if "_ccmp_lr = os.environ.get" in _src:
        print("CCMP_LR parameter group already patched")
    else:
        assert _src.count(_ANCHOR) == 1, (
            f"expected exactly one optimizer construction, found {_src.count(_ANCHOR)}")
        _NEW = '\n'.join([
            '    _ccmp_lr = os.environ.get("CCMP_LR", "")',
            '    _head, _rest = [], []',
            '    if _ccmp_lr != "":',
            '        for _n, _p in model.named_parameters():',
            '            if not _p.requires_grad:',
            '                continue',
            '            (_head if any(k in _n for k in ("resp_proj", "resp_emb", "resp_head"))',
            '             else _rest).append(_p)',
            '        # Raise only when the head is genuinely missing on a CCMP arm. CCMP_LR',
            '        # reaches a control through dict(os.environ, ...) with no resp_* to attach',
            '        # to, and that is not an error -- it falls back to one group and says so.',
            '        if not _head and os.environ.get("CCMP", "0") == "1":',
            '            raise AssertionError(',
            '                "CCMP_LR set and CCMP=1 but no resp_* parameters exist. The head "',
            '                "was not built; check the [ccmp] construction line above.")',
            '    if _head:',
            '        optimizer = instantiate(cfg.optimizer,',
            '                                [{"params": _rest},',
            '                                 {"params": _head, "lr": float(_ccmp_lr)}])',
            '        logger.info(f"[ccmp] head lr {float(_ccmp_lr):g} on {len(_head)} tensors | "',
            '                    f"trunk lr {cfg.optimizer.lr:g} on {len(_rest)} tensors")',
            '    else:',
            '        optimizer = instantiate(cfg.optimizer, model.parameters())',
            '        if _ccmp_lr != "":',
            '            logger.info("[ccmp] CCMP_LR set but no responsibility head in this arm "',
            '                        "(control); one parameter group, unchanged.")',
        ])
        open(STF, "w").write(_src.replace(_ANCHOR, _NEW, 1))
        print("patched sft_training.py -> CCMP_LR parameter group")
        print("read the [ccmp] head lr line in the training log to confirm it attached")
else:
    print('skipped: every scan and path file is already on Drive, so the engine is not needed')


## 4. interpret_paths.py + analysis sections

In [ ]:
# 4. interpret_paths.py (the interpretation entry point) and the per-dataset analysis sections,
# stored as source (base64 JSON, they contain triple quotes) and run by the driver below per dataset.
# Only runs when cell 1 found a scan or a path search still to do; drawing the figures from
# cached interpretations needs neither the engine nor Qwen3.
# (run verbatim: this cell writes indentation-sensitive source into a file)
import base64 as _b64
_src = _b64.b64decode("IyB0aGUgaW50ZXJwcmV0YXRpb24gZW50cnkgcG9pbnQgKG5vdCBpbiB0aGUgdHJhaW5pbmcgbm90ZWJvb2tzJyBmdXNpb24gYmxvYikKb3BlbigiL2NvbnRlbnQvZ2ZtLXJhZy9nZm1yYWcvd29ya2Zsb3cvaW50ZXJwcmV0X3BhdGhzLnB5IiwgInciKS53cml0ZShyJycnIiIiCmludGVycHJldF9wYXRocy5weSAtLSBwYXRoIGludGVycHJldGF0aW9ucyBmb3IgYSB0cmFpbmVkIGZ1c2lvbiBjaGVja3BvaW50LgoKU2FtZSBjb25zdHJ1Y3Rpb24gYXMgc2Z0X3RyYWluaW5nIChjb25maWcsIGRhdGFzZXRzLCBtb2RlbCwgdHJhaW5lcikgd2l0aCBubyB0cmFpbmluZzogdGhlCmNoZWNrcG9pbnQgaXMgbG9hZGVkLCB0aGVuIEZ1c2lvblNGVFRyYWluZXIuaW50ZXJwcmV0KCkgcnVucyB0aGUgTkJGTmV0LXN0eWxlIGdyYWRpZW50IGJlYW0Kc2VhcmNoIGZyb20gZWFjaCByZXF1ZXN0ZWQgcXVlcnkncyBzZWVkIGZyYW1lcyB0byBpdHMgYmVzdC1yYW5rZWQgZ29sZCBhbmQgcmVjb3JkcyB0aGUgQ0NNUApyZXNwb25zaWJpbGl0eSBhbG9uZyBldmVyeSBwYXRoLiBPdXRwdXQ6IG9uZSBKU09OLgoKICAgIHB5dGhvbiAtbSBnZm1yYWcud29ya2Zsb3cuaW50ZXJwcmV0X3BhdGhzIC0tY29uZmlnLXBhdGggY29uZmlnL2dmbV9yZWFzb25lciBcXAogICAgICAgIC0tY29uZmlnLW5hbWUgc2Z0X3RyYWluaW5nX2Z1c2lvbiB0ZXh0X2VtYl9tb2RlbD1xd2VuM19zdCBcXAogICAgICAgIGRhdGFzZXRzLmNmZ3Mucm9vdD0uLi4gZGF0YXNldHMudHJhaW5fbmFtZXM9W0ddIGRhdGFzZXRzLnZhbGlkX25hbWVzPVtHXSBcXAogICAgICAgIG1vZGVsLnNlbWFudGljPW1scCBtb2RlbC5jcWlnPWZhbHNlIFxcCiAgICAgICAgK2ludGVycC5ja3B0PS9wYXRoL21vZGVsX2Jlc3QucHRoICtpbnRlcnAucWlkc19maWxlPS9wYXRoL3FpZHMuanNvbiBcXAogICAgICAgICtpbnRlcnAub3V0PS9wYXRoL3BhdGhzLmpzb24gK2ludGVycC5wcm9iZXM9L3BhdGgvcHJvYmVzX3Rlc3QuanNvbmwgXFwKICAgICAgICBoeWRyYS5ydW4uZGlyPS9wYXRoL3J1bgoiIiIKdHJ5OiAgIyBzYW1lIHRvcmNodmlzaW9uIHNoaW0gYXMgc2Z0X3RyYWluaW5nCiAgICBpbXBvcnQgdG9yY2h2aXNpb24uaW8gYXMgX3R2aW8KICAgIGlmIG5vdCBoYXNhdHRyKF90dmlvLCAiVmlkZW9SZWFkZXIiKToKICAgICAgICBjbGFzcyBfTm9WaWRlb1JlYWRlcjoKICAgICAgICAgICAgZGVmIF9faW5pdF9fKHNlbGYsICphLCAqKmspOgogICAgICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCJ0b3JjaHZpc2lvbiB2aWRlbyBBUEkgcmVtb3ZlZCIpCiAgICAgICAgX3R2aW8uVmlkZW9SZWFkZXIgPSBfTm9WaWRlb1JlYWRlcgpleGNlcHQgRXhjZXB0aW9uOgogICAgcGFzcwppbXBvcnQganNvbgppbXBvcnQgbG9nZ2luZwppbXBvcnQgb3MKCmltcG9ydCBoeWRyYQppbXBvcnQgdG9yY2gKZnJvbSBoeWRyYS5jb3JlLmh5ZHJhX2NvbmZpZyBpbXBvcnQgSHlkcmFDb25maWcKZnJvbSBoeWRyYS51dGlscyBpbXBvcnQgaW5zdGFudGlhdGUKZnJvbSBvbWVnYWNvbmYgaW1wb3J0IERpY3RDb25maWcsIE9tZWdhQ29uZgoKZnJvbSBnZm1yYWcgaW1wb3J0IHV0aWxzCmZyb20gZ2ZtcmFnLmdyYXBoX2luZGV4X2RhdGFzZXRzIGltcG9ydCBHcmFwaERhdGFzZXRMb2FkZXIKZnJvbSBnZm1yYWcudHJhaW5lcnMuc2Z0X3RyYWluZXIgaW1wb3J0IFNGVExvc3MKCmxvZ2dlciA9IGxvZ2dpbmcuZ2V0TG9nZ2VyKF9fbmFtZV9fKQoKCkBoeWRyYS5tYWluKGNvbmZpZ19wYXRoPSJjb25maWcvZ2ZtX3JhZyIsIGNvbmZpZ19uYW1lPSJzZnRfdHJhaW5pbmciLCB2ZXJzaW9uX2Jhc2U9Tm9uZSkKZGVmIG1haW4oY2ZnOiBEaWN0Q29uZmlnKSAtPiBOb25lOgogICAgdXRpbHMuaW5pdF9kaXN0cmlidXRlZF9tb2RlKGNmZy50aW1lb3V0KQogICAgdG9yY2gubWFudWFsX3NlZWQoY2ZnLnNlZWQpCiAgICBvdXRwdXRfZGlyID0gSHlkcmFDb25maWcuZ2V0KCkucnVudGltZS5vdXRwdXRfZGlyCiAgICBsb2dnZXIuaW5mbyhmIkNvbmZpZzpcbiB7T21lZ2FDb25mLnRvX3lhbWwoY2ZnKX0iKQogICAgZmVhdF9kaW0gPSBzZXQodXRpbHMuaW5pdF9tdWx0aV9kYXRhc2V0KGNmZywgMSwgMCkpCiAgICBhc3NlcnQgbGVuKGZlYXRfZGltKSA9PSAxCiAgICBtb2RlbCA9IGluc3RhbnRpYXRlKGNmZy5tb2RlbCwgZmVhdF9kaW09ZmVhdF9kaW0ucG9wKCkpCiAgICBja3B0ID0gY2ZnLmludGVycC5ja3B0CiAgICBzZCA9IHRvcmNoLmxvYWQoY2twdCwgbWFwX2xvY2F0aW9uPSJjcHUiLCB3ZWlnaHRzX29ubHk9RmFsc2UpWyJtb2RlbCJdCiAgICByZXMgPSBtb2RlbC5sb2FkX3N0YXRlX2RpY3Qoc2QsIHN0cmljdD1GYWxzZSkKICAgIHByaW50KGYiW2ludGVycHJldF0gbG9hZGVkIHtja3B0fToge2xlbihzZCl9IHRlbnNvcnMsIG1pc3Npbmcge2xlbihyZXMubWlzc2luZ19rZXlzKX0sICIKICAgICAgICAgIGYidW5leHBlY3RlZCB7bGVuKHJlcy51bmV4cGVjdGVkX2tleXMpfSIsIGZsdXNoPVRydWUpCiAgICBpZiByZXMudW5leHBlY3RlZF9rZXlzOgogICAgICAgIHByaW50KCJbaW50ZXJwcmV0XSB1bmV4cGVjdGVkIGtleXMgKGZpcnN0IDUpOiIsIHJlcy51bmV4cGVjdGVkX2tleXNbOjVdLCBmbHVzaD1UcnVlKQogICAgdmFsaWRfbG9hZGVyID0gR3JhcGhEYXRhc2V0TG9hZGVyKGNmZy5kYXRhc2V0cywgY2ZnLmRhdGFzZXRzLnZhbGlkX25hbWVzLCBzaHVmZmxlPUZhbHNlLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIG1heF9kYXRhc2V0c19pbl9tZW1vcnk9Y2ZnLmRhdGFzZXRzLm1heF9kYXRhc2V0c19pbl9tZW1vcnksCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgZGF0YV9sb2FkaW5nX3dvcmtlcnM9Y2ZnLmRhdGFzZXRzLmRhdGFfbG9hZGluZ193b3JrZXJzKQogICAgb3B0aW1pemVyID0gaW5zdGFudGlhdGUoY2ZnLm9wdGltaXplciwgbW9kZWwucGFyYW1ldGVycygpKQogICAgbG9zc19mdW5jdGlvbnMgPSBbU0ZUTG9zcyhuYW1lPWxjLm5hbWUsIGxvc3NfZm49aW5zdGFudGlhdGUobGMubG9zcyksIHdlaWdodD1sYy53ZWlnaHQsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIHRhcmdldF9ub2RlX3R5cGU9bGMudGFyZ2V0X25vZGVfdHlwZSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgaXNfZGlzdGlsbGF0aW9uX2xvc3M9bGMuZ2V0KCJpc19kaXN0aWxsYXRpb25fbG9zcyIsIEZhbHNlKSkKICAgICAgICAgICAgICAgICAgICAgIGZvciBsYyBpbiBjZmcubG9zc2VzXQogICAgdHJhaW5lciA9IGluc3RhbnRpYXRlKGNmZy50cmFpbmVyLCBvdXRwdXRfZGlyPW91dHB1dF9kaXIsIG1vZGVsPW1vZGVsLCBvcHRpbWl6ZXI9b3B0aW1pemVyLAogICAgICAgICAgICAgICAgICAgICAgICAgIGxvc3NfZnVuY3Rpb25zPWxvc3NfZnVuY3Rpb25zLCB0cmFpbl9ncmFwaF9kYXRhc2V0X2xvYWRlcj12YWxpZF9sb2FkZXIsCiAgICAgICAgICAgICAgICAgICAgICAgICAgZXZhbF9ncmFwaF9kYXRhc2V0X2xvYWRlcj12YWxpZF9sb2FkZXIpCiAgICBxaWRzID0ganNvbi5sb2FkKG9wZW4oY2ZnLmludGVycC5xaWRzX2ZpbGUpKQogICAgZ29sZHMgPSBqc29uLmxvYWQob3BlbihjZmcuaW50ZXJwLmdvbGRzX2ZpbGUpKSBpZiBjZmcuaW50ZXJwLmdldCgiZ29sZHNfZmlsZSIpIGVsc2UgTm9uZQogICAga3cgPSBkaWN0KHByb2Jlc19wYXRoPWNmZy5pbnRlcnAuZ2V0KCJwcm9iZXMiKSwKICAgICAgICAgICAgICBudW1fYmVhbT1pbnQoY2ZnLmludGVycC5nZXQoIm51bV9iZWFtIiwgMTApKSwgcGF0aF90b3BrPWludChjZmcuaW50ZXJwLmdldCgicGF0aF90b3BrIiwgNSkpLAogICAgICAgICAgICAgIG1heF9nb2xkcz1pbnQoY2ZnLmludGVycC5nZXQoIm1heF9nb2xkcyIsIDIpKSwgdG9wX3ZpZXdzPWludChjZmcuaW50ZXJwLmdldCgidG9wX3ZpZXdzIiwgMykpLAogICAgICAgICAgICAgIGRvX3BhdGhzPWJvb2woaW50KGNmZy5pbnRlcnAuZ2V0KCJwYXRocyIsIDEpKSksIGdvbGRzPWdvbGRzLAogICAgICAgICAgICAgIG5lY2Vzc2l0eT1ib29sKGludChjZmcuaW50ZXJwLmdldCgibmVjZXNzaXR5IiwgMCkpKSwgZGlzdHJhY3Rvcj1ib29sKGludChjZmcuaW50ZXJwLmdldCgiZGlzdHJhY3RvciIsIDApKSksCiAgICAgICAgICAgICAgZHVtcF9rPWludChjZmcuaW50ZXJwLmdldCgiZHVtcF9zY29yZXMiLCAwKSkpCiAgICBpbXBvcnQgaW5zcGVjdCAgICMgb2xkZXIgZnVzaW9uIGJsb2JzICh0aGUgcXVhbGl0YXRpdmUgbm90ZWJvb2tzKSBoYXZlIGludGVycHJldCgpIHdpdGhvdXQgdGhlIGxhdGVyIGtleXdvcmRzCiAgICBhY2NlcHRlZCA9IGluc3BlY3Quc2lnbmF0dXJlKHRyYWluZXIuaW50ZXJwcmV0KS5wYXJhbWV0ZXJzCiAgICBkcm9wcGVkID0gW2sgZm9yIGsgaW4ga3cgaWYgayBub3QgaW4gYWNjZXB0ZWRdCiAgICBpZiBkcm9wcGVkOgogICAgICAgIHByaW50KGYiW2ludGVycHJldF0gdGhpcyB0cmFpbmVyJ3MgaW50ZXJwcmV0KCkgbGFja3Mge2Ryb3BwZWR9OyBub3QgcGFzc2VkIiwgZmx1c2g9VHJ1ZSkKICAgIHRyYWluZXIuaW50ZXJwcmV0KHFpZHMsIGNmZy5pbnRlcnAub3V0LCAqKntrOiB2IGZvciBrLCB2IGluIGt3Lml0ZW1zKCkgaWYgayBpbiBhY2NlcHRlZH0pCiAgICB2YWxpZF9sb2FkZXIuc2h1dGRvd24oKQogICAgdXRpbHMuc3luY2hyb25pemUoKQogICAgdXRpbHMuY2xlYW51cCgpCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIG1haW4oKQonJycpCnByaW50KCJ3cm90ZSBpbnRlcnByZXRfcGF0aHMucHkiKQo=").decode()
if NEED_ENGINE:
    _r = get_ipython().run_cell(_src, store_history=False)
    assert _r.success, 'setup cell failed; see the traceback above'
else:
    print('skipped: every scan and path file is already on Drive, so the engine is not needed')
import base64 as _b64
SECTIONS = json.loads(_b64.b64decode("...").decode())
print('sections:', list(SECTIONS))


## 5. Run every dataset: pick golds, decompose the gate, draw the figure

In [ ]:
# 5. Per dataset: graphs + checkpoint, component tables (engine runs only), pick the golds, run the gate
# decomposition (trainer.gate_decomposition via interpret_paths.py mode=gate_decomp), draw the figure.
import base64 as _b64
# the CURRENT interpret_paths.py (the setup blob writes an older one) and the figure script, from the repo
if os.path.isdir("/content/gfm-rag"):
    open("/content/gfm-rag/gfmrag/workflow/interpret_paths.py", "wb").write(_b64.b64decode("IiIiCmludGVycHJldF9wYXRocy5weSAtLSBwYXRoIGludGVycHJldGF0aW9ucyBmb3IgYSB0cmFpbmVkIGZ1c2lvbiBjaGVja3BvaW50LgoKU2FtZSBjb25zdHJ1Y3Rpb24gYXMgc2Z0X3RyYWluaW5nIChjb25maWcsIGRhdGFzZXRzLCBtb2RlbCwgdHJhaW5lcikgd2l0aCBubyB0cmFpbmluZzogdGhlCmNoZWNrcG9pbnQgaXMgbG9hZGVkLCB0aGVuIEZ1c2lvblNGVFRyYWluZXIuaW50ZXJwcmV0KCkgcnVucyB0aGUgTkJGTmV0LXN0eWxlIGdyYWRpZW50IGJlYW0Kc2VhcmNoIGZyb20gZWFjaCByZXF1ZXN0ZWQgcXVlcnkncyBzZWVkIGZyYW1lcyB0byBpdHMgYmVzdC1yYW5rZWQgZ29sZCBhbmQgcmVjb3JkcyB0aGUgQ0NNUApyZXNwb25zaWJpbGl0eSBhbG9uZyBldmVyeSBwYXRoLiBPdXRwdXQ6IG9uZSBKU09OLgoKICAgIHB5dGhvbiAtbSBnZm1yYWcud29ya2Zsb3cuaW50ZXJwcmV0X3BhdGhzIC0tY29uZmlnLXBhdGggY29uZmlnL2dmbV9yZWFzb25lciBcXAogICAgICAgIC0tY29uZmlnLW5hbWUgc2Z0X3RyYWluaW5nX2Z1c2lvbiB0ZXh0X2VtYl9tb2RlbD1xd2VuM19zdCBcXAogICAgICAgIGRhdGFzZXRzLmNmZ3Mucm9vdD0uLi4gZGF0YXNldHMudHJhaW5fbmFtZXM9W0ddIGRhdGFzZXRzLnZhbGlkX25hbWVzPVtHXSBcXAogICAgICAgIG1vZGVsLnNlbWFudGljPW1scCBtb2RlbC5jcWlnPWZhbHNlIFxcCiAgICAgICAgK2ludGVycC5ja3B0PS9wYXRoL21vZGVsX2Jlc3QucHRoICtpbnRlcnAucWlkc19maWxlPS9wYXRoL3FpZHMuanNvbiBcXAogICAgICAgICtpbnRlcnAub3V0PS9wYXRoL3BhdGhzLmpzb24gK2ludGVycC5wcm9iZXM9L3BhdGgvcHJvYmVzX3Rlc3QuanNvbmwgXFwKICAgICAgICBoeWRyYS5ydW4uZGlyPS9wYXRoL3J1bgoiIiIKdHJ5OiAgIyBzYW1lIHRvcmNodmlzaW9uIHNoaW0gYXMgc2Z0X3RyYWluaW5nCiAgICBpbXBvcnQgdG9yY2h2aXNpb24uaW8gYXMgX3R2aW8KICAgIGlmIG5vdCBoYXNhdHRyKF90dmlvLCAiVmlkZW9SZWFkZXIiKToKICAgICAgICBjbGFzcyBfTm9WaWRlb1JlYWRlcjoKICAgICAgICAgICAgZGVmIF9faW5pdF9fKHNlbGYsICphLCAqKmspOgogICAgICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCJ0b3JjaHZpc2lvbiB2aWRlbyBBUEkgcmVtb3ZlZCIpCiAgICAgICAgX3R2aW8uVmlkZW9SZWFkZXIgPSBfTm9WaWRlb1JlYWRlcgpleGNlcHQgRXhjZXB0aW9uOgogICAgcGFzcwppbXBvcnQganNvbgppbXBvcnQgbG9nZ2luZwppbXBvcnQgb3MKCmltcG9ydCBoeWRyYQppbXBvcnQgdG9yY2gKZnJvbSBoeWRyYS5jb3JlLmh5ZHJhX2NvbmZpZyBpbXBvcnQgSHlkcmFDb25maWcKZnJvbSBoeWRyYS51dGlscyBpbXBvcnQgaW5zdGFudGlhdGUKZnJvbSBvbWVnYWNvbmYgaW1wb3J0IERpY3RDb25maWcsIE9tZWdhQ29uZgoKZnJvbSBnZm1yYWcgaW1wb3J0IHV0aWxzCmZyb20gZ2ZtcmFnLmdyYXBoX2luZGV4X2RhdGFzZXRzIGltcG9ydCBHcmFwaERhdGFzZXRMb2FkZXIKZnJvbSBnZm1yYWcudHJhaW5lcnMuc2Z0X3RyYWluZXIgaW1wb3J0IFNGVExvc3MKCmxvZ2dlciA9IGxvZ2dpbmcuZ2V0TG9nZ2VyKF9fbmFtZV9fKQoKCkBoeWRyYS5tYWluKGNvbmZpZ19wYXRoPSJjb25maWcvZ2ZtX3JhZyIsIGNvbmZpZ19uYW1lPSJzZnRfdHJhaW5pbmciLCB2ZXJzaW9uX2Jhc2U9Tm9uZSkKZGVmIG1haW4oY2ZnOiBEaWN0Q29uZmlnKSAtPiBOb25lOgogICAgdXRpbHMuaW5pdF9kaXN0cmlidXRlZF9tb2RlKGNmZy50aW1lb3V0KQogICAgdG9yY2gubWFudWFsX3NlZWQoY2ZnLnNlZWQpCiAgICBvdXRwdXRfZGlyID0gSHlkcmFDb25maWcuZ2V0KCkucnVudGltZS5vdXRwdXRfZGlyCiAgICBsb2dnZXIuaW5mbyhmIkNvbmZpZzpcbiB7T21lZ2FDb25mLnRvX3lhbWwoY2ZnKX0iKQogICAgZmVhdF9kaW0gPSBzZXQodXRpbHMuaW5pdF9tdWx0aV9kYXRhc2V0KGNmZywgMSwgMCkpCiAgICBhc3NlcnQgbGVuKGZlYXRfZGltKSA9PSAxCiAgICBtb2RlbCA9IGluc3RhbnRpYXRlKGNmZy5tb2RlbCwgZmVhdF9kaW09ZmVhdF9kaW0ucG9wKCkpCiAgICBja3B0ID0gY2ZnLmludGVycC5ja3B0CiAgICBzZCA9IHRvcmNoLmxvYWQoY2twdCwgbWFwX2xvY2F0aW9uPSJjcHUiLCB3ZWlnaHRzX29ubHk9RmFsc2UpWyJtb2RlbCJdCiAgICAjIGFuIG9sZGVyIGNoZWNrcG9pbnQgKHRoZSBKdWx5IHYxIGZ1c2lvbikgY2FuIGNhcnJ5IGEgdGVuc29yIG9mIGFub3RoZXIgc2hhcGUgKGUuZy4gdGhlIDEtZmVhdHVyZQogICAgIyBnYXRlKTsgc3RyaWN0PUZhbHNlIGRvZXMgbm90IHRvbGVyYXRlIHRoYXQsIHNvIHN1Y2ggdGVuc29ycyBhcmUgZHJvcHBlZCBhbmQgcmVwb3J0ZWQKICAgIG1zZCA9IG1vZGVsLnN0YXRlX2RpY3QoKQogICAgYmFkID0gW2sgZm9yIGssIHYgaW4gc2QuaXRlbXMoKSBpZiBrIGluIG1zZCBhbmQgdHVwbGUobXNkW2tdLnNoYXBlKSAhPSB0dXBsZSh2LnNoYXBlKV0KICAgIGZvciBrIGluIGJhZDoKICAgICAgICBzZC5wb3AoaykKICAgIGlmIGJhZDoKICAgICAgICBwcmludChmIltpbnRlcnByZXRdIGRyb3BwZWQge2xlbihiYWQpfSB0ZW5zb3JzIHdob3NlIHNoYXBlIGRpZmZlcnMgZnJvbSB0aGlzIG1vZGVsOiB7YmFkWzo1XX0iLCBmbHVzaD1UcnVlKQogICAgcmVzID0gbW9kZWwubG9hZF9zdGF0ZV9kaWN0KHNkLCBzdHJpY3Q9RmFsc2UpCiAgICBwcmludChmIltpbnRlcnByZXRdIGxvYWRlZCB7Y2twdH06IHtsZW4oc2QpfSB0ZW5zb3JzLCBtaXNzaW5nIHtsZW4ocmVzLm1pc3Npbmdfa2V5cyl9LCAiCiAgICAgICAgICBmInVuZXhwZWN0ZWQge2xlbihyZXMudW5leHBlY3RlZF9rZXlzKX0iLCBmbHVzaD1UcnVlKQogICAgaWYgcmVzLnVuZXhwZWN0ZWRfa2V5czoKICAgICAgICBwcmludCgiW2ludGVycHJldF0gdW5leHBlY3RlZCBrZXlzIChmaXJzdCA1KToiLCByZXMudW5leHBlY3RlZF9rZXlzWzo1XSwgZmx1c2g9VHJ1ZSkKICAgIHZhbGlkX2xvYWRlciA9IEdyYXBoRGF0YXNldExvYWRlcihjZmcuZGF0YXNldHMsIGNmZy5kYXRhc2V0cy52YWxpZF9uYW1lcywgc2h1ZmZsZT1GYWxzZSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBtYXhfZGF0YXNldHNfaW5fbWVtb3J5PWNmZy5kYXRhc2V0cy5tYXhfZGF0YXNldHNfaW5fbWVtb3J5LAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIGRhdGFfbG9hZGluZ193b3JrZXJzPWNmZy5kYXRhc2V0cy5kYXRhX2xvYWRpbmdfd29ya2VycykKICAgIG9wdGltaXplciA9IGluc3RhbnRpYXRlKGNmZy5vcHRpbWl6ZXIsIG1vZGVsLnBhcmFtZXRlcnMoKSkKICAgIGxvc3NfZnVuY3Rpb25zID0gW1NGVExvc3MobmFtZT1sYy5uYW1lLCBsb3NzX2ZuPWluc3RhbnRpYXRlKGxjLmxvc3MpLCB3ZWlnaHQ9bGMud2VpZ2h0LAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICB0YXJnZXRfbm9kZV90eXBlPWxjLnRhcmdldF9ub2RlX3R5cGUsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIGlzX2Rpc3RpbGxhdGlvbl9sb3NzPWxjLmdldCgiaXNfZGlzdGlsbGF0aW9uX2xvc3MiLCBGYWxzZSkpCiAgICAgICAgICAgICAgICAgICAgICBmb3IgbGMgaW4gY2ZnLmxvc3Nlc10KICAgIHRyYWluZXIgPSBpbnN0YW50aWF0ZShjZmcudHJhaW5lciwgb3V0cHV0X2Rpcj1vdXRwdXRfZGlyLCBtb2RlbD1tb2RlbCwgb3B0aW1pemVyPW9wdGltaXplciwKICAgICAgICAgICAgICAgICAgICAgICAgICBsb3NzX2Z1bmN0aW9ucz1sb3NzX2Z1bmN0aW9ucywgdHJhaW5fZ3JhcGhfZGF0YXNldF9sb2FkZXI9dmFsaWRfbG9hZGVyLAogICAgICAgICAgICAgICAgICAgICAgICAgIGV2YWxfZ3JhcGhfZGF0YXNldF9sb2FkZXI9dmFsaWRfbG9hZGVyKQogICAgcWlkcyA9IGpzb24ubG9hZChvcGVuKGNmZy5pbnRlcnAucWlkc19maWxlKSkKICAgIGdvbGRzID0ganNvbi5sb2FkKG9wZW4oY2ZnLmludGVycC5nb2xkc19maWxlKSkgaWYgY2ZnLmludGVycC5nZXQoImdvbGRzX2ZpbGUiKSBlbHNlIE5vbmUKICAgIGlmIHN0cihjZmcuaW50ZXJwLmdldCgibW9kZSIsICJpbnRlcnByZXQiKSkgPT0gImdhdGVfZGVjb21wIjoKICAgICAgICAjIHdoaWNoIGdhdGVzIG1vdmUgYSByb3V0ZSdzIHdlaWdodDogc2FtZSB3ZWlnaHRzLCB0aGUgZ2F0ZSByZXN0cmljdGVkIHBlciBsYXllciAvIHBlciBzZW5kZXIgc2V0CiAgICAgICAgdHJhaW5lci5nYXRlX2RlY29tcG9zaXRpb24ocWlkcywgY2ZnLmludGVycC5vdXQsIGdvbGRzPWdvbGRzLCBudW1fYmVhbT1pbnQoY2ZnLmludGVycC5nZXQoIm51bV9iZWFtIiwgMTApKSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBwYXRoX3RvcGs9aW50KGNmZy5pbnRlcnAuZ2V0KCJwYXRoX3RvcGsiLCA1KSksIG1heF9nb2xkcz1pbnQoY2ZnLmludGVycC5nZXQoIm1heF9nb2xkcyIsIDIpKSkKICAgIGVsc2U6CiAgICAgICAgdHJhaW5lci5pbnRlcnByZXQocWlkcywgY2ZnLmludGVycC5vdXQsIHByb2Jlc19wYXRoPWNmZy5pbnRlcnAuZ2V0KCJwcm9iZXMiKSwKICAgICAgICAgICAgICAgICAgICAgICAgICBudW1fYmVhbT1pbnQoY2ZnLmludGVycC5nZXQoIm51bV9iZWFtIiwgMTApKSwgcGF0aF90b3BrPWludChjZmcuaW50ZXJwLmdldCgicGF0aF90b3BrIiwgNSkpLAogICAgICAgICAgICAgICAgICAgICAgICAgIG1heF9nb2xkcz1pbnQoY2ZnLmludGVycC5nZXQoIm1heF9nb2xkcyIsIDIpKSwgdG9wX3ZpZXdzPWludChjZmcuaW50ZXJwLmdldCgidG9wX3ZpZXdzIiwgMykpLAogICAgICAgICAgICAgICAgICAgICAgICAgIGRvX3BhdGhzPWJvb2woaW50KGNmZy5pbnRlcnAuZ2V0KCJwYXRocyIsIDEpKSksIGdvbGRzPWdvbGRzLAogICAgICAgICAgICAgICAgICAgICAgICAgIG5lY2Vzc2l0eT1ib29sKGludChjZmcuaW50ZXJwLmdldCgibmVjZXNzaXR5IiwgMCkpKSwgZGlzdHJhY3Rvcj1ib29sKGludChjZmcuaW50ZXJwLmdldCgiZGlzdHJhY3RvciIsIDApKSksCiAgICAgICAgICAgICAgICAgICAgICAgICAgZHVtcF9rPWludChjZmcuaW50ZXJwLmdldCgiZHVtcF9zY29yZXMiLCAwKSkpCiAgICB2YWxpZF9sb2FkZXIuc2h1dGRvd24oKQogICAgdXRpbHMuc3luY2hyb25pemUoKQogICAgdXRpbHMuY2xlYW51cCgpCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIG1haW4oKQo="))
    print("interpret_paths.py written from the repo copy (gate_decomp mode)")
os.makedirs(f"{S4}/eval", exist_ok=True)
open(f"{S4}/eval/gate_decomp_fig.py", "wb").write(_b64.b64decode("IyEvdXNyL2Jpbi9lbnYgcHl0aG9uMwoiIiIKZ2F0ZV9kZWNvbXBfZmlnLnB5IC0tIHdoZXJlIGRvZXMgdGhlIENDTVAgZ2F0ZSBhY3Q/IFJlYWRzIGdhdGVfZGVjb21wXzxkYXRhc2V0Pi5qc29uIHdyaXR0ZW4gYnkKdHJhaW5lci5nYXRlX2RlY29tcG9zaXRpb24oKSAoY29sYWJfY2NtcF9nYXRlX2RlY29tcC5pcHluYikgYW5kIGRyYXdzIHRoZSBkZWNvbXBvc2l0aW9uLgoKRm9yIGV2ZXJ5IChxdWVyeSwgZ29sZCkgdGhlIGZpbGUgaG9sZHMgdGhlIHRvcCByb3V0ZXMgdW5kZXIgdGhlIGZ1bGwgZ2F0ZSBhbmQsIGZvciBzaXggaW5mZXJlbmNlLXRpbWUKZ2F0aW5nIGNvbmRpdGlvbnMgb24gdGhlIFNBTUUgdHJhaW5lZCB3ZWlnaHRzLCB0aGUgd2VpZ2h0IG9mIHRob3NlIHNhbWUgcm91dGVzIGV2YWx1YXRlZCBkaXJlY3RseSBhbG9uZwp0aGVpciBlZGdlcywgdGhlIGdvbGQncyBncmFwaCBzY29yZSBhbmQgaXRzIGdyYXBoIC8gZnVzZWQgcmFuazoKCiAgICBnYXRlX29uICAgICAgICAgICAgICAgICAgICAgICB0aGUgbW9kZWwgYXMgcnVuCiAgICBnYXRlX29mZiAgICAgICAgICAgICAgICAgICAgICBldmVyeSBnYXRlIDEgKHRoZSAibm8gQ0NNUCIgYXJtKQogICAgZ2F0ZV9sYXllcnNfYXR0cmlidXRlZCAgICAgICAgZ2F0ZSBvbmx5IGF0IHRoZSBsYXllcnMgdGhlIHJvdXRlIGlzIGF0dHJpYnV0ZWQgdG8gKDAgLi4gTC0xKQogICAgZ2F0ZV9sYXllcnNfbGF0ZXIgICAgICAgICAgICAgZ2F0ZSBvbmx5IGF0IHRoZSBsYXRlciBsYXllcnMgKEwgLi4gNSkKICAgIGdhdGVfcm91dGVfc2VuZGVyc19vbmx5ICAgICAgIGdhdGUgb25seSBvbiB0aGUgcm91dGUncyBvd24gc2VuZGVyIG5vZGVzLCAxIGV2ZXJ5d2hlcmUgZWxzZQogICAgZ2F0ZV9hbGxfYnV0X3JvdXRlX3NlbmRlcnMgICAgZ2F0ZSBvbiBldmVyeSBub2RlIGV4Y2VwdCB0aGUgcm91dGUncyBzZW5kZXJzCgpPdXRwdXRzIChwcmVmaXggLS1vdXQpOgogICAgPG91dD4ucGRmLy5wbmcgICBwYW5lbCBBOiBvbmUgZXhhbXBsZSAoLS1leGFtcGxlIGdvbGQgaWQsIGRlZmF1bHQgdGhlIGZpcnN0IHRhcmdldCk6IHJvdXRlLTEKICAgICAgICAgICAgICAgICAgICAgd2VpZ2h0IGFuZCBncmFwaCByYW5rIHVuZGVyIGVhY2ggY29uZGl0aW9uLgogICAgICAgICAgICAgICAgICAgICBwYW5lbCBCOiBvdmVyIGFsbCBnb2xkcywgbWVkaWFuIHJhdGlvIG9mIHJvdXRlLTEgd2VpZ2h0IHRvIHRoZSBnYXRlLW9mZiB3ZWlnaHQKICAgICAgICAgICAgICAgICAgICAgcGVyIGNvbmRpdGlvbiwgd2l0aCB0aGUgaW50ZXJxdWFydGlsZSByYW5nZSwgYW5kIHRoZSBzaGFyZSBvZiBnb2xkcyB3aG9zZSBncmFwaAogICAgICAgICAgICAgICAgICAgICByYW5rIGltcHJvdmVzIG92ZXIgZ2F0ZS1vZmYuCiAgICA8b3V0Pi5tZCAgICAgICAgIHRoZSBzYW1lIG51bWJlcnMgYXMgYSB0YWJsZSwgcGx1cyB0aGUgZXhhbXBsZSdzIHJvdXRlcyB3aXRoIHRoZWlyIGdhdGVzLgoKdXNhZ2U6IGdhdGVfZGVjb21wX2ZpZy5weSAtLWRlY29tcCBvdXRwdXRzL3NjYW4vc2lyNF9iaW9sb2d5L2dhdGVfZGVjb21wX3NpcjRfYmlvbG9neS5qc29uIFxcCiAgICAgICAgICAgLS1kb2NzIGtnLWNvbnN0cnVjdGlvbi9kYXRhL3NpcjRfYmlvbG9neV90ZXN0L3Jhdy9kb2N1bWVudHMuanNvbiBcXAogICAgICAgICAgIFstLWV4YW1wbGUgMTAuMTAwNy9zMTEyNjMtMDIzLTAxODMxLTldIC0tb3V0IHJlc3VsdHMvcXVhbGl0YXRpdmUvZmlnX2dhdGVfZGVjb21wX2Jpb2xvZ3kKIiIiCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBhcmdwYXJzZQppbXBvcnQganNvbgppbXBvcnQgbWF0aAppbXBvcnQgc3RhdGlzdGljcyBhcyBzdAoKQ09ORFMgPSBbKCJnYXRlX29mZiIsICJnYXRlIG9mZlxuKG5vIENDTVApIiksICgiZ2F0ZV9sYXllcnNfYXR0cmlidXRlZCIsICJnYXRlIG9ubHkgYXRcbnJvdXRlIGxheWVyc1xuMC4uTC0xIiksCiAgICAgICAgICgiZ2F0ZV9sYXllcnNfbGF0ZXIiLCAiZ2F0ZSBvbmx5IGF0XG5sYXRlciBsYXllcnNcbkwuLjUiKSwgKCJnYXRlX3JvdXRlX3NlbmRlcnNfb25seSIsICJnYXRlIG9ubHkgb25cbnJvdXRlXG5zZW5kZXJzIiksCiAgICAgICAgICgiZ2F0ZV9hbGxfYnV0X3JvdXRlX3NlbmRlcnMiLCAiZ2F0ZSBvbmx5IG9uXG5vdGhlclxubm9kZXMiKSwgKCJnYXRlX29uIiwgImdhdGUgb25cbihmdWxsIENDTVApIildCgoKZGVmIHRpdGxlX29mKGRvY19pZCwgZG9jcywgbj03MCk6CiAgICB0ID0gZG9jcy5nZXQoZG9jX2lkLCBkb2NfaWQpLnNwbGl0KCIuICIpWzBdCiAgICByZXR1cm4gdCBpZiBsZW4odCkgPD0gbiBlbHNlIHRbOiBuIC0gMV0gKyAi4oCmIgoKCmRlZiBtYWluKCk6CiAgICBhcCA9IGFyZ3BhcnNlLkFyZ3VtZW50UGFyc2VyKCkKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1kZWNvbXAiLCByZXF1aXJlZD1UcnVlKTsgYXAuYWRkX2FyZ3VtZW50KCItLWRvY3MiLCByZXF1aXJlZD1UcnVlKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLWV4YW1wbGUiLCBkZWZhdWx0PU5vbmUsIGhlbHA9ImdvbGQgaWQgb2YgdGhlIGV4YW1wbGUgZm9yIHBhbmVsIEEgKGRlZmF1bHQ6IGZpcnN0IHRhcmdldCkiKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLXJvdXRlIiwgdHlwZT1pbnQsIGRlZmF1bHQ9MCwgaGVscD0id2hpY2ggYmFzZWxpbmUgcm91dGUgdG8gZm9sbG93ICgwID0gdG9wKSIpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tb3V0IiwgcmVxdWlyZWQ9VHJ1ZSkKICAgIGEgPSBhcC5wYXJzZV9hcmdzKCkKICAgIGRvY3MgPSBqc29uLmxvYWQob3BlbihhLmRvY3MpKQogICAgZGF0YSA9IGpzb24ubG9hZChvcGVuKGEuZGVjb21wKSkKICAgIHRhcmdldHMgPSBbKHIsIHQpIGZvciByIGluIGRhdGEgZm9yIHQgaW4gclsidGFyZ2V0cyJdXQogICAgYXNzZXJ0IHRhcmdldHMsICJubyB0YXJnZXRzIGluIHRoZSBkZWNvbXBvc2l0aW9uIGZpbGUiCgogICAgIyAtLS0tIGFnZ3JlZ2F0ZSBvdmVyIGdvbGRzOiByYXRpbyBvZiB0aGUgZm9sbG93ZWQgcm91dGUncyB3ZWlnaHQgdG8gaXRzIGdhdGUtb2ZmIHdlaWdodCwgcGVyIGNvbmRpdGlvbgogICAgYWdnID0ge2M6IFtdIGZvciBjLCBfIGluIENPTkRTfTsgcmFua19iZXR0ZXIgPSB7YzogMCBmb3IgYywgXyBpbiBDT05EU307IHJhbmtfbiA9IDAKICAgIGZvciByLCB0IGluIHRhcmdldHM6CiAgICAgICAgQyA9IHRbImNvbmRpdGlvbnMiXQogICAgICAgIHdfb2ZmID0gQ1siZ2F0ZV9vZmYiXVsicm91dGVfd2VpZ2h0c19kaXJlY3QiXQogICAgICAgIGlmIGxlbih3X29mZikgPD0gYS5yb3V0ZSBvciB3X29mZlthLnJvdXRlXSBpcyBOb25lIG9yIHdfb2ZmW2Eucm91dGVdIDw9IDA6CiAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgcmFua19uICs9IDEKICAgICAgICBmb3IgYywgXyBpbiBDT05EUzoKICAgICAgICAgICAgdyA9IENbY11bInJvdXRlX3dlaWdodHNfZGlyZWN0Il1bYS5yb3V0ZV0KICAgICAgICAgICAgaWYgdyBpcyBub3QgTm9uZSBhbmQgdyA+IDA6CiAgICAgICAgICAgICAgICBhZ2dbY10uYXBwZW5kKHcgLyB3X29mZlthLnJvdXRlXSkKICAgICAgICAgICAgaWYgQ1tjXVsicmFuayJdWyJncmFwaCJdIDwgQ1siZ2F0ZV9vZmYiXVsicmFuayJdWyJncmFwaCJdOgogICAgICAgICAgICAgICAgcmFua19iZXR0ZXJbY10gKz0gMQoKICAgICMgLS0tLSB0aGUgZXhhbXBsZQogICAgZXggPSBuZXh0KCgociwgdCkgZm9yIHIsIHQgaW4gdGFyZ2V0cyBpZiB0WyJkb2MiXSA9PSBhLmV4YW1wbGUpLCB0YXJnZXRzWzBdKSBpZiBhLmV4YW1wbGUgZWxzZSB0YXJnZXRzWzBdCiAgICByX2V4LCB0X2V4ID0gZXg7IEMgPSB0X2V4WyJjb25kaXRpb25zIl0KICAgIHJvdXRlID0gdF9leFsicm91dGVzIl1bYS5yb3V0ZV0gaWYgbGVuKHRfZXhbInJvdXRlcyJdKSA+IGEucm91dGUgZWxzZSBOb25lCgogICAgIyAtLS0tIG1hcmtkb3duCiAgICBtZCA9IFtmIiMgQ0NNUCBnYXRlIGRlY29tcG9zaXRpb246IHthLmRlY29tcH0iLCAiIiwKICAgICAgICAgIGYie2xlbih0YXJnZXRzKX0gZ29sZHM7IHtyYW5rX259IHdpdGggYSBwb3NpdGl2ZSBnYXRlLW9mZiB3ZWlnaHQgb24gcm91dGUge2Eucm91dGUgKyAxfS4gU2FtZSB0cmFpbmVkIHdlaWdodHMgaW4gZXZlcnkgcm93OyBvbmx5IHRoZSBpbmZlcmVuY2UtdGltZSBnYXRlIG1hc2sgY2hhbmdlcy4iLCAiIiwKICAgICAgICAgICJ8IGNvbmRpdGlvbiB8IG1lZGlhbiB3IC8gd19vZmYgfCBJUVIgfCBzaGFyZSBvZiBnb2xkcyB3aXRoIGEgYmV0dGVyIGdyYXBoIHJhbmsgdGhhbiBnYXRlIG9mZiB8IiwgInwtLS18LS06fC0tLXwtLTp8Il0KICAgIGZvciBjLCBsYWIgaW4gQ09ORFM6CiAgICAgICAgdiA9IGFnZ1tjXQogICAgICAgIGlmIHY6CiAgICAgICAgICAgIHExLCBxMyA9IHN0LnF1YW50aWxlcyh2LCBuPTQpWzBdLCBzdC5xdWFudGlsZXModiwgbj00KVsyXQogICAgICAgICAgICBtZC5hcHBlbmQoZiJ8IHtsYWIucmVwbGFjZShjaHIoMTApLCAnICcpfSB8IHtzdC5tZWRpYW4odik6LjNmfSB8IHtxMTouMmZ9IHRvIHtxMzouMmZ9IHwge3JhbmtfYmV0dGVyW2NdIC8gbWF4KDEsIHJhbmtfbik6LjJmfSB8IikKICAgIG1kICs9IFsiIiwgZiIjIyBFeGFtcGxlOiB7cl9leFsnaWQnXX0gLT4ge3RpdGxlX29mKHRfZXhbJ2RvYyddLCBkb2NzKX0iLCAiIiwKICAgICAgICAgICBmImF0dHJpYnV0ZWQgbGF5ZXJzIDAuLnt0X2V4WydhdHRyaWJ1dGVkX2xheWVycyddIC0gMX07IHt0X2V4WyduX3JvdXRlX3NlbmRlcnMnXX0gcm91dGUgc2VuZGVyczsgZnJvbnRpZXItbWVhbiByZXNwb25zaWJpbGl0eSBwZXIgbGF5ZXIgIgogICAgICAgICAgICsgIiwgIi5qb2luKGYie3g6LjNmfSIgZm9yIHggaW4gdF9leFsiZnJvbnRpZXJfbWVhbl9yZXNwIl0pLCAiIl0KICAgIGlmIHJvdXRlOgogICAgICAgIG1kLmFwcGVuZCgicm91dGUgIiArIHN0cihhLnJvdXRlICsgMSkgKyAiOiAiICsgIiAtPiAiLmpvaW4oZiJ7aFsnaGVhZCddfSBbe2hbJ3JlbCddfV0gKGcge2guZ2V0KCdnYXRlJywgMSk6LjRmfSwgcmVzcCB7aC5nZXQoJ3Jlc3AnLCBmbG9hdCgnbmFuJykpOi4zZn0pIiBmb3IgaCBpbiByb3V0ZVsiaG9wcyJdKSArIGYiIC0+IGdvbGQ7IGJlYW0gd2VpZ2h0IHtyb3V0ZVsnd2VpZ2h0X2JlYW0nXTouMmZ9IikKICAgIG1kICs9IFsiIiwgInwgY29uZGl0aW9uIHwgcm91dGUgd2VpZ2h0IHwgZ3JhcGggcmFuayB8IGZ1c2VkIHJhbmsgfCBnb2xkIGdyYXBoIHNjb3JlIHwgZ2FwIHRvIHRoZSB0b3AgZ3JhcGggc2NvcmUgfCIsICJ8LS0tfC0tOnwtLTp8LS06fC0tOnwtLTp8Il0KICAgIGZvciBjLCBsYWIgaW4gQ09ORFM6CiAgICAgICAgdyA9IENbY11bInJvdXRlX3dlaWdodHNfZGlyZWN0Il1bYS5yb3V0ZV0gaWYgbGVuKENbY11bInJvdXRlX3dlaWdodHNfZGlyZWN0Il0pID4gYS5yb3V0ZSBlbHNlIE5vbmUKICAgICAgICBtZC5hcHBlbmQoZiJ8IHtsYWIucmVwbGFjZShjaHIoMTApLCAnICcpfSB8IHsnbi9hJyBpZiB3IGlzIE5vbmUgZWxzZSBmJ3t3Oi4yZn0nfSB8IHtDW2NdWydyYW5rJ11bJ2dyYXBoJ119IHwge0NbY11bJ3JhbmsnXVsnZnVzZWQnXX0gfCB7Q1tjXVsnZ3JhcGhfc2NvcmUnXTouM2Z9IHwge0NbY11bJ2dyYXBoX3Njb3JlX2dhcF90b190b3AnXTouM2Z9IHwiKQogICAgb3BlbihhLm91dCArICIubWQiLCAidyIpLndyaXRlKCJcbiIuam9pbihtZCkgKyAiXG4iKQoKICAgICMgLS0tLSBmaWd1cmUKICAgIGltcG9ydCBtYXRwbG90bGliCiAgICBtYXRwbG90bGliLnVzZSgiQWdnIikKICAgIGltcG9ydCBtYXRwbG90bGliLnB5cGxvdCBhcyBwbHQKICAgIGZpZywgYXhlcyA9IHBsdC5zdWJwbG90cygxLCAyLCBmaWdzaXplPSgxMi41LCA0LjApLCBncmlkc3BlY19rdz17IndpZHRoX3JhdGlvcyI6IFsxLjA1LCAxXX0pCiAgICBsYWJzID0gW2xhYiBmb3IgXywgbGFiIGluIENPTkRTXTsgeHMgPSBsaXN0KHJhbmdlKGxlbihDT05EUykpKQogICAgIyBBOiBleGFtcGxlCiAgICBheCA9IGF4ZXNbMF0KICAgIHcgPSBbQ1tjXVsicm91dGVfd2VpZ2h0c19kaXJlY3QiXVthLnJvdXRlXSBpZiBsZW4oQ1tjXVsicm91dGVfd2VpZ2h0c19kaXJlY3QiXSkgPiBhLnJvdXRlIGVsc2UgTm9uZSBmb3IgYywgXyBpbiBDT05EU10KICAgIHJrID0gW0NbY11bInJhbmsiXVsiZ3JhcGgiXSBmb3IgYywgXyBpbiBDT05EU10KICAgIGNvbHMgPSBbIiM3ZjdmN2YiLCAiIzllY2FlMSIsICIjMzE4MmJkIiwgIiNhMWQ5OWIiLCAiIzMxYTM1NCIsICIjZTY1NTBkIl0KICAgIGF4LmJhcih4cywgWzAgaWYgdiBpcyBOb25lIGVsc2UgdiBmb3IgdiBpbiB3XSwgY29sb3I9Y29scywgZWRnZWNvbG9yPSJibGFjayIsIGxpbmV3aWR0aD0wLjUpCiAgICBmb3IgeCwgdiwgayBpbiB6aXAoeHMsIHcsIHJrKToKICAgICAgICBheC50ZXh0KHgsICgwIGlmIHYgaXMgTm9uZSBlbHNlIHYpICogMS4wMSwgZiJyYW5rIHtrfSIsIGhhPSJjZW50ZXIiLCB2YT0iYm90dG9tIiwgZm9udHNpemU9OCkKICAgIGF4LnNldF94dGlja3MoeHMpOyBheC5zZXRfeHRpY2tsYWJlbHMobGFicywgZm9udHNpemU9Nyk7IGF4LnNldF95bGFiZWwoZiJ3ZWlnaHQgb2Ygcm91dGUge2Eucm91dGUgKyAxfSAoc2FtZSByb3V0ZSwgc2FtZSB3ZWlnaHRzKSIsIGZvbnRzaXplPTgpCiAgICBheC5zZXRfdGl0bGUoZiJBLiB7dGl0bGVfb2YodF9leFsnZG9jJ10sIGRvY3MsIDYwKX0iLCBmb250c2l6ZT05LCBsb2M9ImxlZnQiKQogICAgYXguZ3JpZChheGlzPSJ5IiwgbHM9Ii0tIiwgYWxwaGE9MC40KQogICAgIyBCOiBhZ2dyZWdhdGUKICAgIGF4ID0gYXhlc1sxXQogICAgbWVkID0gW3N0Lm1lZGlhbihhZ2dbY10pIGlmIGFnZ1tjXSBlbHNlIGZsb2F0KCJuYW4iKSBmb3IgYywgXyBpbiBDT05EU10KICAgIGxvID0gW3N0LnF1YW50aWxlcyhhZ2dbY10sIG49NClbMF0gaWYgbGVuKGFnZ1tjXSkgPiAzIGVsc2UgZmxvYXQoIm5hbiIpIGZvciBjLCBfIGluIENPTkRTXQogICAgaGkgPSBbc3QucXVhbnRpbGVzKGFnZ1tjXSwgbj00KVsyXSBpZiBsZW4oYWdnW2NdKSA+IDMgZWxzZSBmbG9hdCgibmFuIikgZm9yIGMsIF8gaW4gQ09ORFNdCiAgICBheC5iYXIoeHMsIG1lZCwgY29sb3I9Y29scywgZWRnZWNvbG9yPSJibGFjayIsIGxpbmV3aWR0aD0wLjUsCiAgICAgICAgICAgeWVycj1bW20gLSBsIGZvciBtLCBsIGluIHppcChtZWQsIGxvKV0sIFtoIC0gbSBmb3IgbSwgaCBpbiB6aXAobWVkLCBoaSldXSwgY2Fwc2l6ZT0zKQogICAgYXguYXhobGluZSgxLjAsIGNvbG9yPSJibGFjayIsIGx3PTAuOCwgbHM9IjoiKQogICAgZm9yIHgsIGMgaW4gemlwKHhzLCBbYyBmb3IgYywgXyBpbiBDT05EU10pOgogICAgICAgIGF4LnRleHQoeCwgMC4wMywgZiJ7cmFua19iZXR0ZXJbY10gLyBtYXgoMSwgcmFua19uKTouMCV9XG5iZXR0ZXJcbnJhbmsiLCBoYT0iY2VudGVyIiwgdmE9ImJvdHRvbSIsIGZvbnRzaXplPTcsIGNvbG9yPSJ3aGl0ZSIgaWYgeCAhPSAwIGVsc2UgImJsYWNrIikKICAgIGF4LnNldF94dGlja3MoeHMpOyBheC5zZXRfeHRpY2tsYWJlbHMobGFicywgZm9udHNpemU9Nyk7IGF4LnNldF95bGFiZWwoInJvdXRlLTEgd2VpZ2h0IC8gZ2F0ZS1vZmYgd2VpZ2h0IChtZWRpYW4sIElRUikiLCBmb250c2l6ZT04KQogICAgYXguc2V0X3RpdGxlKGYiQi4gYWxsIHtyYW5rX259IGdvbGRzIiwgZm9udHNpemU9OSwgbG9jPSJsZWZ0Iik7IGF4LmdyaWQoYXhpcz0ieSIsIGxzPSItLSIsIGFscGhhPTAuNCkKICAgIGZpZy50aWdodF9sYXlvdXQoKQogICAgZm9yIGV4dCBpbiAoInBkZiIsICJwbmciKToKICAgICAgICBmaWcuc2F2ZWZpZyhmInthLm91dH0ue2V4dH0iLCBkcGk9MjAwLCBiYm94X2luY2hlcz0idGlnaHQiKQogICAgIyAtLS0tIHBhbmVsIEMgKHNlcGFyYXRlIGZpbGUpOiBnYXRlIGF0dHJpYnV0aW9uIGFsb25nIHRoZSBleGFtcGxlJ3Mgcm91dGUsIHBlciBzZW5kZXIgYW5kIGxheWVyCiAgICBnYSA9IHRfZXguZ2V0KCJnYXRlX2F0dHJpYnV0aW9uIikgb3Ige30KICAgIGlmIHJvdXRlIGFuZCAicm91dGVfc2VuZGVycyIgaW4gZ2E6CiAgICAgICAgc2VuZGVycyA9IFtoWyJoZWFkIl0gZm9yIGggaW4gcm91dGVbImhvcHMiXV0KICAgICAgICBmaWcyLCBheCA9IHBsdC5zdWJwbG90cyhmaWdzaXplPSg2LjUsIDMuMikpCiAgICAgICAgbkwgPSBsZW4obmV4dChpdGVyKGdhWyJyb3V0ZV9zZW5kZXJzIl0udmFsdWVzKCkpKVsiY29udHJpYiJdKQogICAgICAgIHdpZHRoID0gMC44IC8gbWF4KDEsIGxlbihzZW5kZXJzKSkKICAgICAgICBmb3Igc2ksIHNuYW1lIGluIGVudW1lcmF0ZShzZW5kZXJzKToKICAgICAgICAgICAgcmVjID0gZ2FbInJvdXRlX3NlbmRlcnMiXS5nZXQoc25hbWUpCiAgICAgICAgICAgIGlmIG5vdCByZWM6IGNvbnRpbnVlCiAgICAgICAgICAgIHhzMiA9IFtsICsgKHNpIC0gKGxlbihzZW5kZXJzKSAtIDEpIC8gMikgKiB3aWR0aCBmb3IgbCBpbiByYW5nZShuTCldCiAgICAgICAgICAgIGF4LmJhcih4czIsIHJlY1siY29udHJpYiJdLCB3aWR0aD13aWR0aCAqIDAuOTUsIGxhYmVsPWYie3NuYW1lWzozOF19IChnIHBlciBsYXllcjogIiArICIsICIuam9pbihmIntnOi4yZn0iIGZvciBnIGluIHJlY1siZ2F0ZSJdKSArICIpIikKICAgICAgICBheC5heGhsaW5lKDAsIGNvbG9yPSJibGFjayIsIGx3PTAuOCkKICAgICAgICBheC5zZXRfeHRpY2tzKHJhbmdlKG5MKSk7IGF4LnNldF94dGlja2xhYmVscyhbZiJsYXllciB7bH0iIGZvciBsIGluIHJhbmdlKG5MKV0pCiAgICAgICAgYXguc2V0X3lsYWJlbCgiQ0NNUCBjb250cmlidXRpb24gdG8gdGhlIGdvbGQncyBzY29yZVxu4oiCcy/iiIJnIMOXIChnIOKIkiAxKSIsIGZvbnRzaXplPTgpCiAgICAgICAgYXguc2V0X3RpdGxlKGYiQy4gd2hlcmUgQ0NNUCBhY3RzIG9uIHJvdXRlIHthLnJvdXRlICsgMX06IHN1bSBvdmVyIHJvdXRlIHNlbmRlcnMge2dhLmdldCgnc3VtX2NvbnRyaWJfcm91dGVfc2VuZGVycycsIGZsb2F0KCduYW4nKSk6LjNmfSwgIgogICAgICAgICAgICAgICAgICAgICBmImFsbCBub2RlcyB7Z2EuZ2V0KCdzdW1fY29udHJpYl9hbGxfbm9kZXMnLCBmbG9hdCgnbmFuJykpOi4zZn0sIGFjdHVhbCDOlHNjb3JlIG9u4oiSb2ZmIHtnYS5nZXQoJ2RlbHRhX3Njb3JlX29uX21pbnVzX29mZicsIGZsb2F0KCduYW4nKSk6LjNmfSIsIGZvbnRzaXplPTgsIGxvYz0ibGVmdCIpCiAgICAgICAgYXgubGVnZW5kKGZvbnRzaXplPTYuNSwgbG9jPSJiZXN0Iik7IGF4LmdyaWQoYXhpcz0ieSIsIGxzPSItLSIsIGFscGhhPTAuNCkKICAgICAgICBmaWcyLnRpZ2h0X2xheW91dCgpCiAgICAgICAgZm9yIGV4dCBpbiAoInBkZiIsICJwbmciKToKICAgICAgICAgICAgZmlnMi5zYXZlZmlnKGYie2Eub3V0fV9hdHRyLntleHR9IiwgZHBpPTIwMCwgYmJveF9pbmNoZXM9InRpZ2h0IikKICAgICAgICBtZC5hcHBlbmQoIlxuIyMgR2F0ZSBhdHRyaWJ1dGlvbiAocm91dGUgIiArIHN0cihhLnJvdXRlICsgMSkgKyAiKVxuIikKICAgICAgICBtZC5hcHBlbmQoInwgc2VuZGVyIHwgIiArICIgfCAiLmpvaW4oZiJsYXllciB7bH0iIGZvciBsIGluIHJhbmdlKG5MKSkgKyAiIHwgdG90YWwgfCIpCiAgICAgICAgbWQuYXBwZW5kKCJ8LS0tfCIgKyAiLS06fCIgKiAobkwgKyAxKSkKICAgICAgICBmb3Igc25hbWUgaW4gc2VuZGVyczoKICAgICAgICAgICAgcmVjID0gZ2FbInJvdXRlX3NlbmRlcnMiXS5nZXQoc25hbWUpCiAgICAgICAgICAgIGlmIHJlYzogbWQuYXBwZW5kKGYifCB7c25hbWVbOjUwXX0gfCAiICsgIiB8ICIuam9pbihmIntjOisuM2Z9IChnIHtnOi4yZn0pIiBmb3IgYywgZyBpbiB6aXAocmVjWyJjb250cmliIl0sIHJlY1siZ2F0ZSJdKSkgKyBmIiB8IHtyZWNbJ2NvbnRyaWJfdG90YWwnXTorLjNmfSB8IikKICAgICAgICBtZC5hcHBlbmQoZiJcbnN1bSBvdmVyIGFsbCBub2RlcyB7Z2EuZ2V0KCdzdW1fY29udHJpYl9hbGxfbm9kZXMnLCAwKTorLjNmfSB2cyBhY3R1YWwgzpRzY29yZSAoZ2F0ZSBvbiDiiJIgb2ZmKSB7Z2EuZ2V0KCdkZWx0YV9zY29yZV9vbl9taW51c19vZmYnLCAwKTorLjNmfTsgcGVyLWxheWVyIHN1bSBvdmVyIGFsbCBub2RlczogIiArICIsICIuam9pbihmInt2OisuM2Z9IiBmb3IgdiBpbiBnYS5nZXQoInBlcl9sYXllcl9zdW1fYWxsIiwgW10pKSkKICAgICAgICBtZC5hcHBlbmQoIlxudG9wIG5vZGVzIGJ5IHxjb250cmlidXRpb258IChvbiByb3V0ZSBtYXJrZWQgKik6ICIgKyAiOyAiLmpvaW4oZiJ7JyonIGlmIG5bJ29uX3JvdXRlJ10gZWxzZSAnJ317blsnbm9kZSddWzo0MF19IHtuWydjb250cmliX3RvdGFsJ106Ky4zZn0iIGZvciBuIGluIGdhLmdldCgidG9wX25vZGVzIiwgW10pWzoxMF0pKQogICAgICAgIG9wZW4oYS5vdXQgKyAiLm1kIiwgInciKS53cml0ZSgiXG4iLmpvaW4obWQpICsgIlxuIikKICAgIGlmIEZhbHNlOgogICAgICAgIGZpZy5zYXZlZmlnKGYie2Eub3V0fS57ZXh0fSIsIGRwaT0yMDAsIGJib3hfaW5jaGVzPSJ0aWdodCIpCiAgICBwcmludCgiXG4iLmpvaW4obWRbOjEyXSkpOyBwcmludChmIlxud3JvdGUge2Eub3V0fS5tZC8ucGRmLy5wbmciKQoKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICBtYWluKCkK"))
print("eval/gate_decomp_fig.py written; notebook built 2026-09-09 12:43")

def set_dataset(d):
    global DATASET, S, SCAN_DATASET, CACHE, QUERIES, env
    DATASET = SCAN_DATASET = d; S = SPEC[d]
    CACHE = f"{DRIVE}/outputs/{d}/cache"; os.makedirs(CACHE, exist_ok=True)
    QUERIES = f"{DATA_ROOT}/{d}_test/raw/test.json"
    os.environ["CARGO_DATASET"] = d
    env = dict(os.environ, CARGO_ROOT=CARGO_ROOT, CARGO_DATASET=d, PYTHONUNBUFFERED="1")
    cp.set_dataset(d); print(cp.banner())
    for k in list(os.environ):
        if k.startswith(("CCMP", "ROUTE", "STRAT_", "CQIG", "RESID_", "MISS_W")): os.environ.pop(k)

GD = {}
GD["pick"] = r"""
# --- the golds to decompose: every (query, gold) the showcase run interpreted with a valid route on the CCMP arm,
# cross-field first, deepest under Qwen3 cosine first, capped at GD_MAX_GOLDS; the listed examples always included
_hops = json.load(open(f"{SCAN_OUT}/hops_{GD_ARM}.json"))
_cand = []
for r in _hops:
    seeds = set(r["seeds"])
    for t in r["targets"]:
        if any(p["hops"] and p["hops"][0]["head"] in seeds for p in t["paths"]):
            _cand.append((0 if (r.get("stratum") == "cross") else 1, -t["rank"]["dense"], r["id"], t["doc"]))
_cand.sort()
_ex = set(GD_EXAMPLES.get(DATASET, []))
_keep = [c for c in _cand if c[3] in _ex] + [c for c in _cand if c[3] not in _ex][:GD_MAX_GOLDS]
GD_PIN = {}
for _, _, q, g in _keep: GD_PIN.setdefault(q, []).append(g)
GD_QIDS = f"{SCAN_OUT}/qids_gate_decomp.json"; json.dump(sorted(GD_PIN), open(GD_QIDS, "w"))
GD_GOLDS = f"{SCAN_OUT}/golds_gate_decomp.json"; json.dump(GD_PIN, open(GD_GOLDS, "w"))
print(f"{DATASET}: {len(_keep)} golds on {len(GD_PIN)} queries (of {len(_cand)} interpreted with a route); examples present:",
      [g for g in _ex if any(c[3] == g for c in _keep)])
"""
GD["decomp"] = r"""
# --- run the decomposition (six gate masks per gold, same weights) --------------------------------------
GD_OUT = f"{SCAN_OUT}/gate_decomp_{DATASET}.json"
if os.path.exists(GD_OUT) and FORCE not in ("decomp", "all"): print("[cached]", os.path.relpath(GD_OUT, DRIVE))
else:
    name, ckpt, graph, skey, gate = ARMS[0]
    env_, info = model_env(ckpt, graph, skey, gate=True)
    print(f"[gate-decomp] {name}: {os.path.relpath(ckpt, DRIVE)} {info}")
    rl = f"{RUNS}/gate_decomp_{DATASET}"; os.makedirs(rl, exist_ok=True)
    rc = sh("python -u -m gfmrag.workflow.interpret_paths " + hydra_common(graph) +
            f"+interp.ckpt={ckpt} +interp.qids_file={GD_QIDS} +interp.out={GD_OUT} +interp.golds_file={GD_GOLDS} "
            f"+interp.mode=gate_decomp +interp.num_beam=10 +interp.path_topk=5 +interp.max_golds=4 hydra.run.dir={rl}",
            "/content/gfm-rag", extra=env_, log=f"{rl}/console.log", check=False)
    assert rc == 0 and os.path.exists(GD_OUT), f"gate decomposition failed (exit {rc}); read {rl}/console.log"
"""
GD["fig"] = r"""
# --- figure + table -------------------------------------------------------------------------------
GD_FIG = f"{SCAN_OUT}/fig_gate_decomp_{DATASET}"
cmd = [sys.executable, "-u", "eval/gate_decomp_fig.py", "--decomp", f"{SCAN_OUT}/gate_decomp_{DATASET}.json",
       "--docs", f"{DATA_ROOT}/{DATASET}_test/raw/documents.json", "--out", GD_FIG]
if GD_EXAMPLES.get(DATASET): cmd += ["--example", GD_EXAMPLES[DATASET][0]]
sh(cmd, S4)
from IPython.display import Image, display
display(Image(GD_FIG + ".png"))
print(open(GD_FIG + ".md").read()[:6000])
"""

def sections_for(d):
    P = PLAN[d]
    todo = ["graphs", "model_env"] if P["decomp"] else ["graphs", "components", "model_env"]
    todo += ["pick", "decomp", "fig"]
    return todo
ALL = dict(SECTIONS); ALL.update(GD)

DONE, FAILED = [], {}
for _d in DATASETS:
    print(f"\n\n#################### {_d} ####################")
    if not (PLAN[_d]["ckpt"] and PLAN[_d]["hops"]):
        FAILED[_d] = "checkpoint or hops_frame_ccmp.json missing on Drive"; print(f"[{_d}] SKIPPED: {FAILED[_d]}"); continue
    if not PLAN[_d]["decomp"] and not os.path.isdir("/content/gfm-rag"):
        FAILED[_d] = "needs the engine but it is not installed"; print(f"[{_d}] SKIPPED: {FAILED[_d]}; re-run cell 1 then cells 2b-3g"); continue
    set_dataset(_d)
    _todo = sections_for(_d); print("sections:", ", ".join(_todo))
    try:
        for _name in _todo:
            print(f"\n======== {_d}: {_name} ========")
            _r = get_ipython().run_cell(ALL[_name], store_history=False)
            assert _r.success, f"section '{_name}' failed; see the traceback above"
        DONE.append(_d)
    except Exception as _e:
        FAILED[_d] = str(_e)[:300]; print(f"[{_d}] FAILED: {_e}")
print("\nfinished:", DONE, "| failed:", FAILED)
